# FraudLab · baseline R18/R20 e código atual R23

**Esta versão foi salva sem execução no Kaggle.** As células não contêm resultados de uma execução nesta plataforma. Os treinamentos locais e as submissões realmente processadas estão documentados no [repositório público e suas evidências](https://github.com/aikolopes-DS/fraudlab). O ranking é uma observação datada e provisória.

As seções 1–6 preservam o **baseline histórico R18/R20 executável**: offset próprio com BPE, frequência, pooling e oito interações, além do LightGBM próprio. Executam CV estratificada 5/42 e, depois, dois refits em todos os rótulos conforme a R20. Seus pesos são **56,25% offset / 18,75% LGB / 25% GPT-2 de Masaya Kawamata**. **Esse baseline não é a receita selecionada atual.** O apêndice R23 disponibiliza separadamente o código próprio atualizado e seus comandos, sem executar treinamento ou inferência.

O GPT-2 foi treinado pelo autor externo. Refits não fabricam OOF nem AUC nova; a reserva antiga não é independente destes protocolos. Código próprio MIT; previsões externas v2 CC BY 4.0. Tokenizador e vocabulários oficiais GPT-2: OpenAI. Dados sob os termos da competição.

## 1. Fonte fixada e execução offline

Revisão [8a1c7fd141b43e41b0d43e3a07b0c5ad0bf41ae5](https://github.com/aikolopes-DS/fraudlab/tree/8a1c7fd141b43e41b0d43e3a07b0c5ad0bf41ae5). Cada módulo abaixo é exportado com `git show` dessa revisão; os hashes conferem a integridade. Não há `pip install`, clone remoto ou execução de código de treinamento de terceiros. O notebook usa as bibliotecas já instaladas na imagem Kaggle; o preflight interrompe se forem incompatíveis. Os 22 módulos vêm do core 8a1c7fd. Os dois vocabulários oficiais estão incorporados, comprimidos e fixados por SHA-256; o cache é preparado antes da tokenização, sem rede. tiktoken 0.14.0 e regex 2026.9.29 são obrigatórios. As previsões externas vêm de um input v2 com hashes fixados. Não é necessária GPU.

In [ ]:
from pathlib import Path
Path('fraudlab').mkdir(exist_ok=True)


In [ ]:
%%writefile fraudlab/__init__.py
"""Small, auditable experiments for binary tabular classification."""

__version__ = "0.1.0"


In [ ]:
%%writefile fraudlab/__main__.py
"""Run with python -m fraudlab; defaults never download or submit data."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

from .data import inspect_data, validate_submission, write_json
from .experiment import blend, train


def parser() -> argparse.ArgumentParser:
    root = argparse.ArgumentParser(description="Auditable Kaggle fraud experiments.")
    commands = root.add_subparsers(dest="command", required=True)
    inspection = commands.add_parser(
        "inspect", help="Write aggregate data checks and SHA-256 fingerprints."
    )
    inspection.add_argument("--data", default="data/raw")
    inspection.add_argument("--output", default="artifacts/profile.json")
    inspection.add_argument("--id-column", default="id")
    inspection.add_argument("--target", default="is_fraud")

    training = commands.add_parser(
        "train",
        help="Fit cross-validation folds and write OOF, models and test predictions.",
    )
    training.add_argument("--data", default="data/raw")
    training.add_argument(
        "--original-data",
        help="Pinned original CSV, appended only to outer-training rows.",
    )
    training.add_argument(
        "--original-sha256", help="Required SHA-256 pin when --original-data is used."
    )
    training.add_argument("--original-id-column", default="transaction_id")
    training.add_argument("--output", default="artifacts/runs")
    training.add_argument("--name", required=True)
    training.add_argument(
        "--phase",
        choices=("development", "exploratory_full_data_cv"),
        default="development",
        help="Label the validation protocol; exploratory full-data CV has no independent holdout.",
    )
    training.add_argument(
        "--model",
        choices=(
            "logistic",
            "glm",
            "catboost",
            "lightgbm",
            "mlp",
            "offset",
            "tabm",
            "neural_offset",
        ),
        required=True,
    )
    training.add_argument(
        "--feature-set",
        choices=(
            "raw",
            "behavior",
            "digits",
            "value_digits",
            "paired_digits",
            "bpe_digits",
            "token_digits",
            "joint_bpe",
        ),
        default="raw",
    )
    training.add_argument("--id-column", default="id")
    training.add_argument("--target", default="is_fraud")
    training.add_argument(
        "--split", choices=("stratified", "group", "time"), default="stratified"
    )
    training.add_argument(
        "--validation-note",
        required=True,
        help="Explain why this split matches the data and intended generalization.",
    )
    training.add_argument("--group-column")
    training.add_argument("--time-column")
    training.add_argument("--folds", type=int, default=3)
    training.add_argument("--seed", type=int, default=42)
    training.add_argument(
        "--holdout-fraction",
        type=float,
        default=0.0,
        help="Reserve this fraction before CV; never evaluate it during model selection.",
    )
    training.add_argument("--holdout-seed", type=int, default=2026)
    training.add_argument("--threads", type=int, default=8)
    training.add_argument(
        "--fixed-training",
        action="store_true",
        help="Use the configured tree/epoch count without early stopping; useful for preselected full-data CV.",
    )
    training.add_argument(
        "--device",
        choices=("auto", "cpu", "cuda"),
        default="auto",
        help="Auto uses CPU for CatBoost and CUDA when available for neural models.",
    )
    training.add_argument("--logistic-c", type=float, default=1.0)
    training.add_argument(
        "--glm-inner-folds",
        type=int,
        choices=(5, 10),
        default=5,
        help="Inner target-encoding partitions for GLM/offset; outer CV is unchanged.",
    )
    training.add_argument(
        "--glm-value-one-hot",
        action="store_true",
        help="One-hot encode every categorical GLM input with fold-local rare pooling and sparse L-BFGS.",
    )
    training.add_argument(
        "--glm-frequency",
        action="store_true",
        help="Add training-fold log category counts to GLM/offset inputs.",
    )
    training.add_argument(
        "--glm-transductive-frequency",
        action="store_true",
        help="Use label-free official train+test category counts; exploratory CV only.",
    )
    training.add_argument(
        "--glm-pooled-tokens",
        action="store_true",
        help="Add 16 inner-cross-fitted hierarchical decimal/token logits to GLM/offset.",
    )
    training.add_argument(
        "--glm-spline-knots",
        type=int,
        default=0,
        help="Add cubic quantile splines of continuous features to GLM/offset; 0 disables them.",
    )
    training.add_argument("--iterations", type=int, default=2000)
    training.add_argument("--depth", type=int, default=6)
    training.add_argument("--learning-rate", type=float, default=0.05)
    training.add_argument("--l2-leaf-reg", type=float, default=3.0)
    training.add_argument("--num-leaves", type=int, default=4)
    training.add_argument("--min-child-samples", type=int, default=100)
    training.add_argument("--target-smoothing", type=float, default=20.0)
    training.add_argument("--epochs", type=int, default=30)
    training.add_argument(
        "--hidden-dims",
        type=int,
        nargs="+",
        help="Defaults: MLP 128 64; TabM/neural_offset require the fixed architecture 256 256.",
    )
    training.add_argument("--dropout", type=float, default=0.1)
    training.add_argument("--neural-learning-rate", type=float, default=0.001)
    training.add_argument("--weight-decay", type=float, default=0.0001)
    training.add_argument(
        "--neural-numeric-transform",
        choices=("standard", "quantile"),
        default="standard",
    )
    training.add_argument("--batch-size", type=int, default=4096)
    training.add_argument(
        "--neural-categorical-transform",
        choices=("embedding", "target"),
        help="Defaults: MLP embedding; TabM/neural_offset require target.",
    )
    training.add_argument(
        "--patience",
        type=int,
        help="Defaults: CatBoost 100 iterations; MLP 5 epochs; TabM/neural_offset 6 inner epochs.",
    )

    blending = commands.add_parser(
        "blend", help="Average compatible runs with explicit, normalized weights."
    )
    blending.add_argument("--runs", type=Path, nargs="+", required=True)
    blending.add_argument("--weights", type=float, nargs="+")
    blending.add_argument(
        "--space",
        choices=("probability", "logit", "rank", "rank_logit"),
        default="probability",
        help="Combine probabilities, logits or whole-vector normalized ranks.",
    )
    blending.add_argument("--name", required=True)
    blending.add_argument("--output", type=Path, default=Path("artifacts/runs"))
    blending.add_argument("--id-column", default="id")
    blending.add_argument("--target", default="is_fraud")

    validation = commands.add_parser(
        "validate-submission",
        help="Require exact schema, ID order and finite probabilities.",
    )
    validation.add_argument("--file", type=Path, required=True)
    validation.add_argument(
        "--sample", type=Path, default=Path("data/raw/sample_submission.csv")
    )
    validation.add_argument("--id-column", default="id")
    validation.add_argument("--target", default="is_fraud")

    holdout = commands.add_parser(
        "evaluate-holdout",
        help="Evaluate frozen development choices once on the reserved independent holdout.",
    )
    holdout.add_argument("--runs", type=Path, nargs="+", required=True)
    holdout.add_argument("--weights", type=float, nargs="+")
    holdout.add_argument(
        "--output", type=Path, default=Path("artifacts/holdout/evaluation.json")
    )
    holdout.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")
    holdout.add_argument(
        "--development-run",
        type=Path,
        help="The matching pre-holdout development blend, for an explicit provenance chain.",
    )
    final = commands.add_parser(
        "refit",
        help="Refit one frozen selected model on all labeled data, without early stopping.",
    )
    final.add_argument("--run", type=Path, required=True)
    final.add_argument(
        "--holdout-report", type=Path, default=Path("artifacts/holdout/evaluation.json")
    )
    final.add_argument("--output", type=Path, default=Path("artifacts/runs"))
    final.add_argument("--name", required=True)
    public = commands.add_parser(
        "import-public-gpt2",
        help="Validate the approved licensed public GPT-2 arrays and convert logits to the OOF contract.",
    )
    public.add_argument("--source", type=Path, required=True)
    public.add_argument("--data", type=Path, default=Path("data/raw"))
    public.add_argument("--output", type=Path, default=Path("artifacts/runs"))
    public.add_argument("--name", default="explore5-public-gpt2")
    return root


def validate_training_config(config: dict) -> None:
    if (
        config.get("original_data")
        or config.get("original_sha256")
        or config.get("original_id_column", "transaction_id") != "transaction_id"
    ):
        from .original import validate_config

        validate_config(config)
    if config.get("hidden_dims") is None:
        config["hidden_dims"] = (
            [256, 256] if config["model"] in ("tabm", "neural_offset") else [128, 64]
        )
    if config.get("neural_categorical_transform") is None:
        config["neural_categorical_transform"] = (
            "target" if config["model"] in ("tabm", "neural_offset") else "embedding"
        )
    if config["model"] == "tabm":
        if config.get("fixed_training") or config.get("refit"):
            raise ValueError(
                "TabM requires inner epoch selection; --fixed-training/refit is unsupported."
            )
        if config["hidden_dims"] != [256, 256]:
            raise ValueError("TabM requires --hidden-dims 256 256.")
        if config["neural_categorical_transform"] != "target":
            raise ValueError("TabM requires --neural-categorical-transform target.")
    if config["model"] == "neural_offset":
        try:
            from .neural_offset import NeuralOffsetConfig
        except ImportError as error:
            raise RuntimeError(
                "Neural offset requires the optional fraudlab[neural] dependencies."
            ) from error

        if config.get("patience") is None:
            config["patience"] = 6
        NeuralOffsetConfig.from_training_config(config)
    glm_models = ("glm", "offset", "neural_offset")
    inner_folds = config.get("glm_inner_folds", 5)
    if inner_folds not in (5, 10) or (
        inner_folds != 5 and config["model"] not in glm_models
    ):
        raise ValueError(
            "glm_inner_folds must be 5 or 10; 10 requires GLM/offset/neural_offset."
        )
    if config.get("glm_value_one_hot") and config["model"] != "glm":
        raise ValueError("glm_value_one_hot is supported only for GLM.")
    if config.get("glm_frequency") and config["model"] not in glm_models:
        raise ValueError(
            "glm_frequency is supported only for GLM/offset/neural_offset."
        )
    if config.get("glm_transductive_frequency"):
        from .transductive import validate_config

        validate_config(config)
    if config.get("glm_pooled_tokens"):
        if config["model"] not in glm_models or inner_folds != 5:
            raise ValueError(
                "glm_pooled_tokens requires GLM/offset/neural_offset with glm_inner_folds=5."
            )
        from .pooled import manifest

        config["pooled_token_encoding"] = manifest(random_state=config["seed"])
    knots = config.get("glm_spline_knots", 0)
    if knots < 0 or knots == 1 or (knots and config["model"] not in glm_models):
        raise ValueError(
            "glm_spline_knots must be 0 or at least 2, for GLM/offset/neural_offset only."
        )
    if config.get("phase") == "exploratory_full_data_cv" and (
        config["holdout_fraction"]
        or config["split"] != "stratified"
        or config["folds"] != 5
        or config["seed"] != 42
    ):
        raise ValueError(
            "Exploratory full-data comparison requires stratified 5-fold/seed42 CV with no independent holdout."
        )
    if config["id_column"] == config["target"]:
        raise ValueError("ID and target must be different columns.")
    if not config["validation_note"].strip():
        raise ValueError("A non-empty validation note is required.")
    if not 0 <= config["holdout_fraction"] < 0.5:
        raise ValueError("holdout_fraction must be in [0, 0.5).")
    if config["holdout_fraction"] and config["split"] != "stratified":
        raise ValueError(
            "The independent random holdout is supported only for stratified IID experiments."
        )
    for name in (
        "threads",
        "iterations",
        "epochs",
        "batch_size",
        "patience",
        "depth",
        "num_leaves",
        "min_child_samples",
    ):
        if config[name] < 1:
            raise ValueError(f"{name} must be positive.")
    if config["num_leaves"] < 2:
        raise ValueError("num_leaves must be at least two.")
    import math

    for name in ("learning_rate", "neural_learning_rate", "logistic_c"):
        if not math.isfinite(config[name]) or config[name] <= 0:
            raise ValueError(f"{name} must be positive.")
    if not 0 <= config["dropout"] < 1 or any(
        width < 1 for width in config["hidden_dims"]
    ):
        raise ValueError(
            "Dropout must be in [0, 1) and hidden layer widths must be positive."
        )
    if (
        not math.isfinite(config["l2_leaf_reg"])
        or not math.isfinite(config["weight_decay"])
        or config["l2_leaf_reg"] < 0
        or config["weight_decay"] < 0
        or not math.isfinite(config["target_smoothing"])
        or config["target_smoothing"] < 0
    ):
        raise ValueError("Regularization values must be nonnegative.")
    if config["split"] != "group" and config["group_column"]:
        raise ValueError("--group-column requires --split group.")
    if config["split"] != "time" and config["time_column"]:
        raise ValueError("--time-column requires --split time.")
    if (
        config["group_column"] in (config["target"], config["id_column"])
        or config["time_column"] == config["target"]
    ):
        raise ValueError(
            "The target cannot define a split; entity grouping must use an actual entity column."
        )


def main() -> None:
    arguments = parser().parse_args()
    try:
        if arguments.command == "inspect":
            profile = inspect_data(
                Path(arguments.data), arguments.id_column, arguments.target
            )
            write_json(Path(arguments.output), profile)
            print(
                json.dumps(
                    {
                        "profile": arguments.output,
                        "rows": profile["rows"],
                        "target": profile["target"],
                        "duplicate_feature_rows": profile["duplicate_feature_rows"],
                        "potential_split_columns": profile["potential_split_columns"],
                    },
                    indent=2,
                )
            )
        elif arguments.command == "train":
            config = vars(arguments).copy()
            config.pop("command")
            if config["patience"] is None:
                config["patience"] = {"mlp": 5, "tabm": 6, "neural_offset": 6}.get(
                    config["model"], 100
                )
            validate_training_config(config)
            metrics = train(config)
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in (
                            "model",
                            "roc_auc",
                            "fold_scores",
                            "fold_std",
                            "elapsed_seconds",
                        )
                    },
                    indent=2,
                )
            )
        elif arguments.command == "blend":
            metrics = blend(
                arguments.runs,
                arguments.weights,
                arguments.output,
                arguments.name,
                arguments.id_column,
                arguments.target,
                arguments.space,
            )
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in ("model", "roc_auc", "fold_scores", "fold_std")
                    },
                    indent=2,
                )
            )
        elif arguments.command == "evaluate-holdout":
            from .holdout import evaluate

            print(
                json.dumps(
                    evaluate(
                        arguments.runs,
                        arguments.weights,
                        arguments.output,
                        arguments.device,
                        arguments.development_run,
                    ),
                    indent=2,
                )
            )
        elif arguments.command == "refit":
            from .refit import refit

            metrics = refit(
                arguments.run,
                arguments.holdout_report,
                arguments.output,
                arguments.name,
            )
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in ("model", "phase", "rows", "elapsed_seconds")
                    },
                    indent=2,
                )
            )
        elif arguments.command == "import-public-gpt2":
            from .external import import_public_gpt2

            metrics = import_public_gpt2(
                arguments.source, arguments.data, arguments.output, arguments.name
            )
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in ("model", "phase", "roc_auc", "fold_scores")
                    },
                    indent=2,
                )
            )
        else:
            frame, sample = pd.read_csv(arguments.file), pd.read_csv(arguments.sample)
            validate_submission(frame, sample, arguments.id_column, arguments.target)
            print(
                json.dumps(
                    {"valid": True, "rows": len(frame), "file": str(arguments.file)}
                )
            )
    except (ValueError, RuntimeError, OSError) as error:
        raise SystemExit(f"Error: {error}") from error


if __name__ == "__main__":
    main()


In [ ]:
%%writefile fraudlab/bpe.py
"""Deterministic GPT-2 numeric keys, with no labels or learned statistics.

Hypothesis: azizdzhon's numeric BPE notebook and Shallow-Tree Stack v31.
Implementation: our own feature extraction using OpenAI's tokenizer API.
No notebook code, merge tables, predictions or fitted statistics are reused.
"""

from __future__ import annotations

from functools import lru_cache
from importlib.metadata import version

import numpy as np
import pandas as pd

from .strings import categorical_strings

COLUMNS = ("amount_usd", "account_balance_usd")
SUFFIXES = ("__gpt2_first_key", "__gpt2_second_key", "__gpt2_second_cents_key")
TOKENIZER_VERSION = "0.14.0"
ENCODING = "gpt2"
ENCODING_FILES = (
    {
        "url": "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/vocab.bpe",
        "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
    },
    {
        "url": "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/encoder.json",
        "sha256": "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783",
    },
)


@lru_cache(maxsize=1)
def _encoding():
    import tiktoken

    if version("tiktoken") != TOKENIZER_VERSION:
        raise RuntimeError(f"This hypothesis requires tiktoken=={TOKENIZER_VERSION}.")
    # The pinned implementation verifies both official asset hashes above.
    return tiktoken.get_encoding(ENCODING)


def _cents(series: pd.Series) -> np.ndarray:
    if not pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series):
        raise ValueError(f"{series.name} must contain numeric USD amounts.")
    values = series.to_numpy(dtype=np.float64)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f"{series.name} requires finite, nonnegative USD amounts.")
    if (values > (2**53 - 1) / 100).any():
        raise ValueError(f"{series.name} overflows the supported USD cent range.")
    return np.rint(values * 100).astype(np.int64)


def bpe_keys(
    frame: pd.DataFrame,
    excluded: tuple[str, ...] = (),
) -> pd.DataFrame:
    """Return six categorical keys; excluded columns produce no keys.

    USD is quantized with np.rint(float64_values * 100), matching value_digits.
    The integer text is one ASCII space followed by the unsigned base-ten integer.
    Token IDs are categorical labels; an absent second token is -1.
    Integers with more than two tokens intentionally retain only the first two.
    """
    result = pd.DataFrame(index=frame.index)
    for name in COLUMNS:
        if name in excluded:
            continue
        if name not in frame:
            raise ValueError(f"Missing BPE input column: {name}")
        cents = _cents(frame[name])
        integers, inverse = np.unique(cents // 100, return_inverse=True)
        tokens = [_encoding().encode_ordinary(f" {int(value)}") for value in integers]
        first = np.asarray([row[0] for row in tokens], dtype=np.int32)[inverse]
        second = np.asarray(
            [row[1] if len(row) > 1 else -1 for row in tokens], dtype=np.int32
        )[inverse]
        result[name + SUFFIXES[0]] = categorical_strings(first)
        result[name + SUFFIXES[1]] = categorical_strings(second)
        result[name + SUFFIXES[2]] = categorical_strings(
            np.char.add(np.char.add(second.astype(str), ":"), (cents % 100).astype(str))
        )
    return result


def manifest() -> dict:
    """Pin the tokenizer and official vocabulary assets in experiment evidence."""
    encoding = _encoding()
    return {
        "tiktoken": TOKENIZER_VERSION,
        "encoding": encoding.name,
        "vocabulary_size": encoding.max_token_value + 1,
        "encoding_files": list(ENCODING_FILES),
        "input_columns": list(COLUMNS),
        "output_suffixes": list(SUFFIXES),
        "representation": "cents=np.rint(float64_USD*100); ' ' + str(cents // 100)",
        "rounding": "NumPy rint; nearest representable cent, halfway ties to even",
        "second_token_missing": "-1",
        "third_key": "second_token_id + ':' + str(cents % 100)",
        "more_than_two_tokens": "retain first two; discard later integer tokens",
        "label_usage": "none in extraction; fold-local cross-fitted TE downstream",
    }


In [ ]:
%%writefile fraudlab/data.py
"""Input contracts and aggregate inspection. Raw data stays local."""

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd


def fingerprint(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def validate_ids(frame: pd.DataFrame, id_column: str) -> None:
    if (
        frame.empty
        or id_column not in frame
        or frame[id_column].isna().any()
        or frame[id_column].duplicated().any()
    ):
        raise ValueError(
            f"{id_column}: rows and IDs must exist; IDs must be non-null and unique."
        )


def validate_submission(
    frame: pd.DataFrame,
    expected: pd.DataFrame,
    id_column: str = "id",
    target: str = "is_fraud",
) -> None:
    if list(frame.columns) != [id_column, target]:
        raise ValueError(
            f"Submission columns must be exactly {[id_column, target]} in that order."
        )
    validate_ids(frame, id_column)
    validate_ids(expected, id_column)
    if frame[id_column].tolist() != expected[id_column].tolist():
        raise ValueError(
            "Submission IDs must match the expected IDs and their order exactly."
        )
    try:
        probabilities = frame[target].to_numpy(dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError("Submission probabilities must be numeric.") from error
    if (
        not np.isfinite(probabilities).all()
        or ((probabilities < 0) | (probabilities > 1)).any()
    ):
        raise ValueError(
            "Submission probabilities must be finite values between 0 and 1."
        )


def load_data(
    directory: Path, id_column: str = "id", target: str = "is_fraud"
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(directory / "train.csv")
    test = pd.read_csv(directory / "test.csv")
    sample = pd.read_csv(directory / "sample_submission.csv")
    if train.empty or test.empty:
        raise ValueError("Training and test data must not be empty.")
    validate_ids(train, id_column)
    validate_ids(test, id_column)
    if set(train[id_column]) & set(test[id_column]):
        raise ValueError("Training and test row IDs must be disjoint.")
    if (
        target not in train
        or train[target].isna().any()
        or set(train[target].unique()) != {0, 1}
    ):
        raise ValueError(
            f"{target}: expected a non-null binary target containing both 0 and 1."
        )
    features = [column for column in train if column not in (id_column, target)]
    if not features or set(test.columns) != {id_column, *features}:
        raise ValueError(
            "Training and test feature schemas must match, excluding target and ID."
        )
    if list(sample.columns) != [id_column, target]:
        raise ValueError("sample_submission.csv has an unexpected schema.")
    validate_ids(sample, id_column)
    if set(sample[id_column]) != set(test[id_column]):
        raise ValueError("Sample submission IDs must cover the test IDs exactly.")
    return train, test[[id_column, *features]], sample


def submission(
    test_ids: pd.Series, predictions: np.ndarray, sample: pd.DataFrame, target: str
) -> pd.DataFrame:
    values = pd.Series(np.asarray(predictions), index=test_ids)
    result = sample.copy()
    result[target] = values.reindex(sample[test_ids.name]).to_numpy()
    validate_submission(result, sample, test_ids.name, target)
    return result


def inspect_data(
    directory: Path, id_column: str = "id", target: str = "is_fraud"
) -> dict:
    train, test, _ = load_data(directory, id_column, target)
    features = [column for column in test if column != id_column]
    columns = {}
    for name in features:
        columns[name] = {
            "dtype": str(train[name].dtype),
            "train_missing": int(train[name].isna().sum()),
            "test_missing": int(test[name].isna().sum()),
            "train_unique": int(train[name].nunique(dropna=False)),
            "test_unique": int(test[name].nunique(dropna=False)),
        }
    duplicate_mask = train.duplicated(features, keep=False)
    return {
        "rows": {"train": len(train), "test": len(test)},
        "target": {
            "name": target,
            "positive_rate": float(train[target].mean()),
            "counts": {str(k): int(v) for k, v in train[target].value_counts().items()},
        },
        "features": columns,
        "duplicate_feature_rows": int(duplicate_mask.sum()),
        "train_test_feature_overlap": int(
            pd.util.hash_pandas_object(train[features], index=False)
            .isin(pd.util.hash_pandas_object(test[features], index=False))
            .sum()
        ),
        "potential_split_columns": [
            name
            for name in features
            if any(
                word in name.lower()
                for word in (
                    "date",
                    "time",
                    "customer",
                    "account",
                    "user",
                    "card_id",
                    "merchant_id",
                )
            )
        ],
        "validation_warning": "Inspect time, entities and duplicate rows before choosing a split. Candidate names alone do not establish the required split.",
        "data_hashes": {
            name: fingerprint(directory / name)
            for name in ("train.csv", "test.csv", "sample_submission.csv")
        },
    }


In [ ]:
%%writefile fraudlab/experiment.py
"""Cross-validation, immutable run artifacts and explicit-weight blending."""

from __future__ import annotations

import importlib.metadata
import json
import platform
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from .data import (
    fingerprint,
    load_data,
    submission,
    validate_ids,
    validate_submission,
    write_json,
)
from .features import transform_features
from .models import feature_types, fit_predict
from .splits import make_splits


def versions() -> dict:
    result = {"python": platform.python_version()}
    for package in (
        "fraudlab",
        "numpy",
        "pandas",
        "scikit-learn",
        "catboost",
        "lightgbm",
        "torch",
        "tabm",
        "rtdl_num_embeddings",
        "tiktoken",
        "regex",
    ):
        try:
            result[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            continue
    return result


def git_revision() -> str | None:
    result = subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=False
    )
    return result.stdout.strip() if result.returncode == 0 else None


def source_provenance(feature_set: str | None = None) -> dict:
    status = subprocess.run(
        ["git", "status", "--porcelain"], capture_output=True, text=True, check=False
    )
    result = {
        "source_hashes": {
            file.name: fingerprint(file)
            for file in sorted(Path(__file__).parent.glob("*.py"))
        },
        "git_revision": git_revision(),
        "git_dirty": bool(status.stdout.strip()) if status.returncode == 0 else None,
        "source_hashes_timing": "captured before reading data or fitting models",
    }
    if feature_set == "bpe_digits":
        from .bpe import manifest

        result["feature_encoding"] = manifest()
    elif feature_set == "token_digits":
        from .tokens import manifest

        result["feature_encoding"] = manifest()
    elif feature_set == "joint_bpe":
        from .joints import manifest

        result["feature_encoding"] = manifest()
    return result


def create_run(output: Path, name: str) -> Path:
    if not name or Path(name).name != name or name in (".", ".."):
        raise ValueError("Run name must be a single directory name.")
    directory = output / name
    try:
        directory.mkdir(parents=True, exist_ok=False)
    except FileExistsError as error:
        raise FileExistsError(
            f"Run already exists: {directory}. Choose a new run name."
        ) from error
    return directory


def train(config: dict) -> dict:
    started = time.perf_counter()
    if "_frequency_reference" in config:
        raise ValueError(
            "Frequency reference maps are runtime-only, not persisted configuration."
        )
    if config.get("glm_transductive_frequency"):
        from .transductive import (
            build_reference_counts,
            validate_config as validate_transductive,
        )

        validate_transductive(config)
    original_features = original_labels = original_metadata = None
    if (
        config.get("original_data")
        or config.get("original_sha256")
        or config.get("original_id_column", "transaction_id") != "transaction_id"
    ):
        from .original import load_original, validate_config, verify_unchanged

        validate_config(config)
    provenance = source_provenance(config.get("feature_set"))
    data_dir = Path(config["data"])
    frame, test, sample = load_data(data_dir, config["id_column"], config["target"])
    target, id_column = config["target"], config["id_column"]
    total_rows = len(frame)
    if config.get("original_data"):
        original_path = Path(config["original_data"])
        original_features, original_labels, original_metadata = load_original(
            original_path,
            config["original_sha256"],
            frame,
            test,
            original_id_column=config.get("original_id_column", "transaction_id"),
            id_column=id_column,
            target=target,
        )
        original_features = transform_features(
            original_features, config.get("feature_set", "raw"), (id_column, target)
        )
    holdout_fraction = config.get("holdout_fraction", 0.0)
    if holdout_fraction:
        development, _ = train_test_split(
            np.arange(len(frame)),
            test_size=holdout_fraction,
            stratify=frame[target],
            random_state=config["holdout_seed"],
        )
        frame = frame.iloc[np.sort(development)].reset_index(drop=True)
    frame = transform_features(
        frame, config.get("feature_set", "raw"), (id_column, target)
    )
    test = transform_features(
        test, config.get("feature_set", "raw"), (id_column, target)
    )
    excluded = [id_column, target]
    if config["split"] == "group":
        excluded.append(config["group_column"])
    if config["split"] == "time":
        excluded.append(config["time_column"])
    features = [name for name in frame if name not in excluded]
    if not features:
        raise ValueError(
            "No model features remain after excluding ID, target and split columns."
        )
    if original_features is not None and original_features.columns.tolist() != features:
        raise ValueError(
            "Engineered original feature columns must match competition order."
        )
    frequency_metadata = None
    fit_config = config
    if config.get("glm_transductive_frequency"):
        corpus_hashes = {
            name: fingerprint(data_dir / name)
            for name in ("train.csv", "test.csv", "sample_submission.csv")
        }
        references, frequency_metadata = build_reference_counts(
            frame, test, features, corpus_hashes, excluded=(id_column, target)
        )
        config = {**config, "transductive_frequency": frequency_metadata}
        fit_config = {**config, "_frequency_reference": references}
    splits = make_splits(
        frame,
        target,
        config["split"],
        config["folds"],
        config["seed"],
        config["group_column"],
        config["time_column"],
    )
    run = create_run(Path(config["output"]), config["name"])
    write_json(run / "config.json", config)
    config_hash = fingerprint(run / "config.json")
    write_json(
        run / "status.json",
        {"status": "running", "started_at": datetime.now(timezone.utc).isoformat()},
    )
    hashes = {
        name: fingerprint(data_dir / name)
        for name in ("train.csv", "test.csv", "sample_submission.csv")
    }
    oof = np.full(len(frame), np.nan)
    fold_ids = np.full(len(frame), -1, dtype=int)
    test_predictions = np.zeros(len(test))
    scores, details = [], []
    try:
        if (
            frequency_metadata is not None
            and hashes != frequency_metadata["corpus_hashes"]
        ):
            raise ValueError(
                "Official corpus bytes changed while constructing reference counts."
            )
        for fold, (training_rows, validation_rows) in enumerate(splits):
            training = frame.iloc[training_rows][features]
            labels = frame.iloc[training_rows][target].to_numpy()
            if original_features is not None:
                verify_unchanged(original_path, config["original_sha256"])
                training = pd.concat([training, original_features], ignore_index=True)
                labels = np.concatenate([labels, original_labels])
            valid_prediction, test_prediction, model_details = fit_predict(
                config["model"],
                training,
                labels,
                frame.iloc[validation_rows][features],
                frame.iloc[validation_rows][target].to_numpy(),
                test[features],
                fit_config,
                config["seed"] + fold,
                run / f"fold-{fold}",
            )
            if original_features is not None:
                verify_unchanged(original_path, config["original_sha256"])
                model_details.update(
                    training_rows=len(training),
                    competition_training_rows=len(training_rows),
                    additional_training_rows=len(original_labels),
                )
            if (
                not np.isfinite(valid_prediction).all()
                or not np.isfinite(test_prediction).all()
            ):
                raise ValueError("Model produced non-finite predictions.")
            oof[validation_rows] = valid_prediction
            fold_ids[validation_rows] = fold
            test_predictions += test_prediction / len(splits)
            score = float(
                roc_auc_score(frame.iloc[validation_rows][target], valid_prediction)
            )
            scores.append(score)
            details.append(model_details)
            print(
                f"fold={fold} roc_auc={score:.8f} train={len(training)} validation={len(validation_rows)}",
                flush=True,
            )
        covered = np.isfinite(oof)
        oof_frame = pd.DataFrame(
            {
                id_column: frame.loc[covered, id_column],
                "fold": fold_ids[covered],
                "target": frame.loc[covered, target],
                "prediction": oof[covered],
            }
        )
        result = submission(test[id_column], test_predictions, sample, target)
        if original_features is not None:
            verify_unchanged(original_path, config["original_sha256"])
        oof_frame.to_csv(run / "oof.csv", index=False)
        result.to_csv(run / "predictions.csv", index=False)
        result.to_csv(run / "submission.csv", index=False)
        numeric, categorical = feature_types(frame[features])
        metrics = {
            "status": "complete",
            "phase": config.get("phase", "development"),
            "config_sha256": config_hash,
            "model": config["model"],
            "roc_auc": float(roc_auc_score(frame.loc[covered, target], oof[covered])),
            "fold_scores": scores,
            "fold_std": float(np.std(scores)),
            "rows": {
                "train": total_rows,
                "test": len(test),
                "oof": int(covered.sum()),
                **(
                    {"development": len(frame), "holdout": total_rows - len(frame)}
                    if holdout_fraction
                    else {}
                ),
            },
            "features": {
                "count": len(features),
                "numeric": len(numeric),
                "categorical": len(categorical),
                "names": features,
                "excluded": list(dict.fromkeys(excluded)),
            },
            "target": {"name": target, "positive_rate": float(frame[target].mean())},
            "seed": config["seed"],
            "validation": {
                "method": config["split"],
                "n_splits": len(splits),
                "note": config["validation_note"],
                "group_column": config["group_column"],
                "time_column": config["time_column"],
                "holdout_fraction": holdout_fraction,
                "holdout_seed": (
                    config.get("holdout_seed") if holdout_fraction else None
                ),
                "independent_holdout": bool(holdout_fraction),
                "early_stopping_uses_validation": config["model"]
                in ("catboost", "lightgbm", "mlp")
                and not config.get("fixed_training"),
                **(
                    {
                        "inner_epoch_selection": {
                            "fraction": 0.15,
                            "seed": 42,
                            "metric": "roc_auc",
                            "fresh_outer_refit": True,
                        }
                    }
                    if config["model"] in ("tabm", "neural_offset")
                    else {}
                ),
            },
            "elapsed_seconds": round(time.perf_counter() - started, 3),
            "versions": versions(),
            "data_hashes": hashes,
            **provenance,
            "model_details": {
                **details[0],
                "feature_set": config.get("feature_set", "raw"),
                "folds": details,
            },
            "created_at": datetime.now(timezone.utc).isoformat(),
            "submission_sha256": fingerprint(run / "submission.csv"),
            "limitations": [
                "OOF scores used for model selection are not an independent final assessment.",
                "Public leaderboard and final private leaderboard scores must be obtained from Kaggle.",
                "Temporal OOF, when selected, excludes the initial training-only window.",
                *(
                    [
                        "Exploratory full-data CV uses all labels, including the historical holdout; no independent holdout remains for this stage."
                    ]
                    if config.get("phase") == "exploratory_full_data_cv"
                    else []
                ),
            ],
        }
        if original_metadata is not None:
            metrics["additional_training_data"] = {
                **original_metadata,
                "component_run": str(run),
                "component_model": config["model"],
                "component_scope": "this fitted component only; other blend sources are not retrained",
                "fold_training_rows": [fold["training_rows"] for fold in details],
            }
        if frequency_metadata is not None:
            metrics["transductive_frequency"] = frequency_metadata
            metrics["validation"]["transductive_feature_counts"] = {
                "rows": frequency_metadata["rows"]["total"],
                "label_usage": "none",
                "counts_sha256": frequency_metadata["counts_sha256"],
                "scalers_and_targets": "outer-training only",
            }
        write_json(run / "metrics.json", metrics)
        write_json(run / "status.json", {"status": "complete"})
        return metrics
    except Exception as error:
        write_json(run / "status.json", {"status": "failed", "error": str(error)})
        raise


def blend(
    runs: list[Path],
    weights: list[float] | None,
    output: Path,
    name: str,
    id_column: str = "id",
    target: str = "is_fraud",
    space: str = "probability",
) -> dict:
    from .scores import LOGIT_CLIP, SPACES, blend_scores

    started = time.perf_counter()
    provenance = source_provenance()
    if space not in SPACES:
        raise ValueError(f"Unknown score space: {space}")
    if len(runs) < 2:
        raise ValueError("Blending requires at least two runs.")
    coefficients = np.asarray(
        weights if weights is not None else [1.0] * len(runs), dtype=float
    ).copy()
    if (
        coefficients.shape != (len(runs),)
        or not np.isfinite(coefficients).all()
        or (coefficients < 0).any()
        or not np.isfinite(coefficients.sum())
        or coefficients.sum() <= 0
    ):
        raise ValueError(
            "Provide one finite nonnegative weight per run, with a positive total."
        )
    coefficients /= coefficients.sum()
    metrics = [
        json.loads((run / "metrics.json").read_text(encoding="utf-8")) for run in runs
    ]
    phases = [metric.get("phase") == "full_data_refit" for metric in metrics]
    if any(phases):
        if not all(phases):
            raise ValueError("Cannot mix development OOF runs with full-data refits.")
        if space != "probability":
            raise ValueError("Full-data refits support probability blending only.")
        from .refit import blend_refits

        return blend_refits(runs, weights, output, name, id_column, target)
    for coefficient, metric in zip(coefficients, metrics):
        components = metric.get("model_details", {}).get("external_components", [])
        if not components or coefficient <= 0:
            continue
        source_space = metric.get("model_details", {}).get(
            "mixture_space", "probability"
        )
        if (
            source_space != "probability"
            or any(
                component.get("mixture_space", "probability") != "probability"
                for component in components
            )
            or (space != "probability" and metric.get("model") == "blend")
        ):
            raise ValueError(
                "Cannot flatten nested external mixtures across nonlinear score spaces."
            )
        if space != "probability" and (
            len(components) != 1 or components[0].get("weight", 1.0) != 1.0
        ):
            raise ValueError(
                "Nonlinear external blending requires an atomic unit-weight source."
            )
    oofs = [pd.read_csv(run / "oof.csv") for run in runs]
    predictions = [pd.read_csv(run / "predictions.csv") for run in runs]
    reference, expected = oofs[0], predictions[0]
    validate_ids(reference, id_column)
    for index, (score, oof, prediction) in enumerate(zip(metrics, oofs, predictions)):
        if (
            score.get("status") != "complete"
            or score["data_hashes"] != metrics[0]["data_hashes"]
        ):
            raise ValueError(
                "Blend sources must be complete runs with identical data fingerprints."
            )
        if not reference[[id_column, "fold", "target"]].equals(
            oof[[id_column, "fold", "target"]]
        ):
            raise ValueError(
                "Blend sources must use exactly the same OOF rows, order, folds and labels."
            )
        if score["validation"] != metrics[0]["validation"]:
            # Models can differ in early stopping; the actual split contract cannot.
            fields = (
                "method",
                "n_splits",
                "group_column",
                "time_column",
                "holdout_fraction",
            )
            if any(
                score["validation"].get(field) != metrics[0]["validation"].get(field)
                for field in fields
            ):
                raise ValueError("Blend sources must use the same validation strategy.")
            if metrics[0]["validation"].get("holdout_fraction", 0) and score[
                "validation"
            ].get("holdout_seed") != metrics[0]["validation"].get("holdout_seed"):
                raise ValueError("Blend sources must reserve exactly the same holdout.")
        validate_submission(prediction, expected, id_column, target)
        if (
            not np.isfinite(oof["prediction"]).all()
            or not oof["prediction"].between(0, 1).all()
        ):
            raise ValueError(f"Source {index} has invalid OOF probabilities.")
    oof_result, prediction_result = reference.copy(), expected.copy()
    if space == "probability":
        # Keep the original arithmetic and its summation order exactly.
        oof_result["prediction"] = sum(
            weight * frame["prediction"].to_numpy()
            for weight, frame in zip(coefficients, oofs)
        )
        prediction_result[target] = sum(
            weight * frame[target].to_numpy()
            for weight, frame in zip(coefficients, predictions)
        )
    else:
        oof_result["prediction"] = blend_scores(
            [frame["prediction"].to_numpy() for frame in oofs], weights, space=space
        )
        prediction_result[target] = blend_scores(
            [frame[target].to_numpy() for frame in predictions], weights, space=space
        )
    fold_scores = [
        float(roc_auc_score(group["target"], group["prediction"]))
        for _, group in oof_result.groupby("fold")
    ]
    run = create_run(output, name)
    oof_result.to_csv(run / "oof.csv", index=False)
    prediction_result.to_csv(run / "predictions.csv", index=False)
    prediction_result.to_csv(run / "submission.csv", index=False)
    external_components = {}
    for weight, metric, source in zip(coefficients, metrics, runs):
        if weight <= 0:
            continue
        for component in metric.get("model_details", {}).get("external_components", []):
            entry = {**component, "mixture_space": space}
            if space == "probability":
                entry["weight"] = float(weight) * component.get("weight", 1.0)
                key = entry["predictions_source_url"]
            else:
                entry.pop("weight", None)
                entry.update(
                    coefficient=float(weight),
                    coefficient_space=space,
                    coefficient_interpretation="direct source coefficient, not a probability share",
                    source_run=str(source),
                    source_prediction_space="probability",
                )
                key = (entry["predictions_source_url"], str(source))
            if key in external_components:
                field = "weight" if space == "probability" else "coefficient"
                external_components[key][field] += entry[field]
            else:
                external_components[key] = entry
    result = {
        **metrics[0],
        "model": "blend",
        "external_components": list(external_components.values()),
        "roc_auc": float(roc_auc_score(oof_result["target"], oof_result["prediction"])),
        "fold_scores": fold_scores,
        "fold_std": float(np.std(fold_scores)),
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "created_at": datetime.now(timezone.utc).isoformat(),
        **provenance,
        "model_details": {
            "method": {
                "probability": "weighted_probability_average",
                "logit": "sigmoid_weighted_logit_average",
                "rank": "weighted_normalized_rank_average",
                "rank_logit": "sigmoid_weighted_rank_logit_average",
            }[space],
            "sources": [str(source) for source in runs],
            "weights": coefficients.tolist(),
            "external_components": list(external_components.values()),
            "mixture_space": space,
        },
        "submission_sha256": fingerprint(run / "submission.csv"),
    }
    blend_config = {
        "sources": [str(source) for source in runs],
        "weights": coefficients.tolist(),
    }
    if space != "probability":
        blend_config["space"] = space
        result["model_details"]["score_transform"] = {
            "input_space": "probability",
            "coefficient_space": space,
            "logit_clip": LOGIT_CLIP if space in ("logit", "rank_logit") else None,
            "rank_method": "average" if space in ("rank", "rank_logit") else None,
            "rank_normalization": (
                "rank/(n+1)" if space in ("rank", "rank_logit") else None
            ),
            "rank_scope": (
                "full OOF vector and full test vector separately; no labels"
                if space in ("rank", "rank_logit")
                else None
            ),
        }
    write_json(
        run / "config.json",
        blend_config,
    )
    result["config_sha256"] = fingerprint(run / "config.json")
    write_json(run / "metrics.json", result)
    write_json(run / "status.json", {"status": "complete"})
    return result


In [ ]:
%%writefile fraudlab/external.py
"""Import one explicitly approved public prediction release; never execute its code."""

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from .data import fingerprint, load_data, submission, write_json
from .experiment import create_run, source_provenance, versions
from .models import feature_types
from .splits import make_splits

MANIFEST_SHA256 = "0083d23b46f71311ceeed66874b093ec927d25d98b37f6d2e78b1de17f0f22d0"
DATASET = "https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof"
CODE = "https://github.com/Matapanino/s6e11-gpt2-target-first"
CODE_REVISION = "8446fb73c11583df4f7006aa08fa34472249b318"


def import_public_gpt2(source: Path, data: Path, output: Path, name: str) -> dict:
    provenance = source_provenance()
    if fingerprint(source / "manifest.json") != MANIFEST_SHA256:
        raise ValueError(
            "Public manifest does not match the reviewed, approved release."
        )
    manifest = json.loads((source / "manifest.json").read_text(encoding="utf-8"))
    hashes = {
        filename: fingerprint(data / filename)
        for filename in ("train.csv", "test.csv", "sample_submission.csv")
    }
    if (
        hashes["train.csv"] != manifest["inputs"]["train_csv"]["sha256"]
        or hashes["test.csv"] != manifest["inputs"]["test_csv"]["sha256"]
    ):
        raise ValueError(
            "Public arrays are not aligned with these exact train/test CSV bytes."
        )
    expected = {
        **manifest["sha256"],
        "fold_ids.npy": manifest["inputs"]["fold_ids"]["sha256"],
    }
    arrays = {}
    for filename in ("oof.npy", "test.npy", "fold_ids.npy"):
        if fingerprint(source / filename) != expected[filename]:
            raise ValueError(f"Public array hash mismatch: {filename}")
        arrays[filename] = np.load(source / filename, allow_pickle=False)
    train, test, sample = load_data(data)
    for filename, rows in (("oof.npy", len(train)), ("test.npy", len(test))):
        array = arrays[filename]
        if (
            array.shape != (rows,)
            or array.dtype != np.float64
            or not np.isfinite(array).all()
        ):
            raise ValueError(
                f"Expected finite float64 logits in original CSV row order: {filename}"
            )
    fold_ids = arrays["fold_ids.npy"]
    if fold_ids.shape != (len(train),) or fold_ids.dtype != np.int8:
        raise ValueError("Expected int8 fold IDs, one per original training row.")
    expected_folds = np.full(len(train), -1, dtype=np.int8)
    for fold, (_, validation) in enumerate(
        make_splits(train, "is_fraud", "stratified", 5, 42)
    ):
        expected_folds[validation] = fold
    if not np.array_equal(fold_ids, expected_folds):
        raise ValueError(
            "Public folds differ from exact StratifiedKFold(5, shuffle=True, seed42)."
        )
    # Stable sigmoid: the approved arrays contain log-odds, not probabilities.
    oof_probability = np.exp(-np.logaddexp(0.0, -arrays["oof.npy"]))
    test_probability = np.exp(-np.logaddexp(0.0, -arrays["test.npy"]))
    score = float(roc_auc_score(train["is_fraud"], oof_probability))
    if abs(score - manifest["auc"]["pooled"]) > 1e-10:
        raise ValueError(
            "Reproduced public OOF AUC differs from the approved manifest."
        )
    fold_scores = [
        float(
            roc_auc_score(
                train.loc[fold_ids == fold, "is_fraud"],
                oof_probability[fold_ids == fold],
            )
        )
        for fold in range(5)
    ]
    result = submission(test["id"], test_probability, sample, "is_fraud")
    run = create_run(output, name)
    pd.DataFrame(
        {
            "id": train["id"],
            "fold": fold_ids,
            "target": train["is_fraud"],
            "prediction": oof_probability,
        }
    ).to_csv(run / "oof.csv", index=False)
    result.to_csv(run / "predictions.csv", index=False)
    result.to_csv(run / "submission.csv", index=False)
    raw_features = train.drop(columns=["id", "is_fraud"])
    numeric, categorical = feature_types(raw_features)
    component = {
        "name": "GPT-2 target-first",
        "kind": "gpt2_target_first",
        "source_type": "external_predictions",
        "trained_by": "Masaya Kawamata",
        "model_architecture": "GPT-2 target-first classifier",
        "training_code_url": CODE,
        "training_code_commit": CODE_REVISION,
        "code_license": "MIT",
        "predictions_source_url": DATASET,
        "predictions_source_version": 2,
        "predictions_download_ref": "masayakawamata/s6e11-gpt2-target-first-oof/2",
        "predictions_license": "CC BY 4.0",
        "data_license": "CC BY 4.0 (prediction dataset)",
        "method_credits": [
            "Chris Deotte: target-first GPT-2 method",
            "Borisov et al.: GReaT / Language Models are Realistic Tabular Data Generators",
            "uditjain13: original Credit Card Fraud Detection 2026 dataset",
        ],
        "training_executed_locally": False,
        "trained_by_us": False,
        "oof_auc": score,
        "validation": {
            "method": "StratifiedKFold",
            "n_splits": 5,
            "shuffle": True,
            "seed": 42,
        },
        "inference": {
            "method": "target-first label likelihood ratio",
            "target_first": True,
            "likelihood_ratio": True,
            "prior": manifest["settings"]["prior"],
            "orders": manifest["settings"]["orders"],
            "order_aggregation": manifest["settings"]["combine"],
            "test_aggregation": "mean of fold log-odds",
            "test_ensemble_folds": 5,
            "source_test_aggregation": manifest["settings"]["test"],
            "import_transform": "sigmoid(log_odds), float64",
        },
    }
    metrics = {
        "status": "complete",
        "model": "external_gpt2",
        "external_components": [component],
        "phase": "exploratory_full_data_cv",
        "roc_auc": score,
        "fold_scores": fold_scores,
        "fold_std": float(np.std(fold_scores)),
        "rows": {"train": len(train), "test": len(test), "oof": len(train)},
        "features": {
            "count": raw_features.shape[1],
            "numeric": len(numeric),
            "categorical": len(categorical),
            "names": raw_features.columns.tolist(),
            "excluded": ["id", "is_fraud"],
        },
        "target": {
            "name": "is_fraud",
            "positive_rate": float(train["is_fraud"].mean()),
        },
        "seed": 42,
        "validation": {
            "method": "stratified",
            "n_splits": 5,
            "note": "Exploratory full-data CV with externally trained predictions; all labels, including the historical holdout, were used in fold training. The old holdout is not independent for this stage.",
            "group_column": None,
            "time_column": None,
            "holdout_fraction": 0.0,
            "holdout_seed": None,
            "independent_holdout": False,
            "early_stopping_uses_validation": False,
        },
        "model_details": {
            "external_components": [component],
            "training_executed_locally": False,
        },
        "external_provenance": {
            "manifest_sha256": MANIFEST_SHA256,
            "array_sha256": expected,
            "fold_ids_reproduced": True,
            "row_alignment": "Exact raw train/test SHA256 and original row order; no sorting by ID",
        },
        "data_hashes": hashes,
        "versions": versions(),
        **provenance,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "submission_sha256": fingerprint(run / "submission.csv"),
        "limitations": [
            "This neural component was trained by an external author, not by FraudLab. Preserve CC BY 4.0 attribution.",
            "No independent holdout is available for this exploratory full-data stage; the earlier holdout report applies only to the original protocol.",
            "Public test predictions average fold log-odds before sigmoid; OOF rows use only their excluded-fold model.",
        ],
    }
    write_json(
        run / "config.json",
        {
            "model": "external_gpt2",
            "phase": "exploratory_full_data_cv",
            "source": DATASET,
            "manifest_sha256": MANIFEST_SHA256,
            "logits_to_probabilities": "sigmoid",
        },
    )
    write_json(run / "metrics.json", metrics)
    write_json(run / "status.json", {"status": "complete"})
    return metrics


In [ ]:
%%writefile fraudlab/features.py
"""Small deterministic hypotheses; every feature set is compared by ablation."""

from __future__ import annotations

import numpy as np
import pandas as pd

from .strings import categorical_strings

CONTINUOUS = (
    "amount_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "account_balance_usd",
    "velocity_score",
    "merchant_risk_score",
)


def transform_features(
    frame: pd.DataFrame,
    feature_set: str = "raw",
    excluded: tuple[str, ...] = ("id", "is_fraud"),
) -> pd.DataFrame:
    if feature_set == "joint_bpe":
        from .joints import joint_keys

        keys = joint_keys(frame, excluded)
        base = transform_features(frame, "bpe_digits", excluded)
        return pd.concat([base, keys], axis=1)
    if feature_set == "token_digits":
        from .tokens import token_keys

        keys = token_keys(frame, excluded)
        base = transform_features(frame, "value_digits", excluded)
        return pd.concat([base, keys], axis=1)
    if feature_set == "bpe_digits":
        from .bpe import bpe_keys

        base = transform_features(frame, "value_digits", excluded)
        return pd.concat([base, bpe_keys(frame, excluded)], axis=1)
    if feature_set == "raw":
        return frame.copy()
    if feature_set not in ("behavior", "digits", "value_digits", "paired_digits"):
        raise ValueError(f"Unknown feature set: {feature_set}")
    result = frame.copy()
    for name in CONTINUOUS:
        if name in result and name not in excluded:
            values = result[name].to_numpy(dtype=float)
            result[name + "__log1p"] = np.sign(values) * np.log1p(np.abs(values))
    for name, period in (("time_of_day_hour", 24), ("day_of_week", 7)):
        if name in result and name not in excluded:
            angle = 2 * np.pi * result[name] / period
            result[name + "__sin"] = np.sin(angle)
            result[name + "__cos"] = np.cos(angle)
    if {"amount_usd", "account_balance_usd"}.issubset(result) and not {
        "amount_usd",
        "account_balance_usd",
    }.intersection(excluded):
        result["amount_balance_ratio"] = result["amount_usd"] / (
            np.abs(result["account_balance_usd"]) + 1
        )
    if feature_set in ("digits", "value_digits", "paired_digits"):
        # These views test synthetic rounding patterns; they are not banking rules.
        for name in CONTINUOUS:
            if name not in result or name in excluded:
                continue
            values = np.abs(result[name].to_numpy(dtype=float))
            if not np.isfinite(values).all():
                raise ValueError("Digit views require finite numeric inputs.")
            cents = np.rint(values * 100).astype(np.int64)
            magnitude = np.floor(np.log10(np.maximum(values, 1e-12))).astype(int)
            leading = np.floor(values / np.power(10.0, magnitude)).astype(int)
            result[name + "__value_key"] = categorical_strings(cents)
            result[name + "__cents_key"] = categorical_strings(cents % 100)
            result[name + "__leading_key"] = categorical_strings(leading)
            if feature_set in ("value_digits", "paired_digits"):
                leading_two = np.floor(values / np.power(10.0, magnitude - 1)).astype(
                    int
                )
                result[name + "__integer_key"] = categorical_strings(cents // 100)
                result[name + "__tenths_key"] = categorical_strings(cents // 10)
                result[name + "__leading_two_key"] = categorical_strings(leading_two)
        if feature_set in ("value_digits", "paired_digits"):
            for name in frame.select_dtypes(include="number"):
                if name not in CONTINUOUS and name not in excluded:
                    result[name + "__value_key"] = frame[name].astype(str)
    if feature_set == "paired_digits":
        # Four domain/audit hypotheses fixed before CV; no label-based pair search.
        pairs = (
            ("cvv_retry_count", None, "auth_method", None),
            ("velocity_score", 5, "distance_from_home_km", 5),
            ("velocity_score", 5, "txn_count_last_24h", 3),
            ("velocity_score", 5, "hours_since_last_txn", 2),
        )
        for left, left_width, right, right_width in pairs:
            if not {left, right}.issubset(frame) or {left, right}.intersection(
                excluded
            ):
                continue
            left_key = frame[left] // left_width if left_width else frame[left]
            right_key = frame[right] // right_width if right_width else frame[right]
            # Tuple text escapes embedded separators and keeps the pair unambiguous.
            result[f"{left}__{right}__pair"] = pd.Series(
                list(zip(left_key, right_key)), index=frame.index
            ).astype(str)
    return result


In [ ]:
%%writefile fraudlab/frequency.py
"""Label-free category counts: fold-local by default, optionally fixed."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class FrequencyEncoder(TransformerMixin, BaseEstimator):
    """Encode log1p support from training or explicit reference; unknowns are zero."""

    # Older persisted encoders have no instance attribute for this new option.
    reference_counts = None

    def __init__(self, reference_counts=None):
        self.reference_counts = reference_counts

    def fit(self, frame: pd.DataFrame, y=None):
        if (
            not isinstance(frame, pd.DataFrame)
            or not frame.columns.is_unique
            or not all(isinstance(name, str) for name in frame.columns)
        ):
            raise ValueError(
                "FrequencyEncoder requires unique string DataFrame columns."
            )
        self.feature_names_in_ = np.asarray(frame.columns, dtype=object)
        self.n_features_in_ = len(self.feature_names_in_)
        if self.reference_counts is None:
            self.counts_ = {
                name: frame[name].value_counts(dropna=False, sort=False)
                for name in self.feature_names_in_
            }
        else:
            if not isinstance(self.reference_counts, dict) or set(
                self.reference_counts
            ) != set(frame):
                raise ValueError(
                    "Reference counts must match frequency columns exactly."
                )
            self.counts_ = {}
            for name in self.feature_names_in_:
                counts = self.reference_counts[name]
                if (
                    not isinstance(counts, pd.Series)
                    or not len(counts)
                    or not all(isinstance(key, str) for key in counts.index)
                    or not counts.index.is_unique
                    or counts.index.hasnans
                    or not pd.api.types.is_numeric_dtype(counts)
                    or pd.api.types.is_bool_dtype(counts)
                    or pd.api.types.is_complex_dtype(counts)
                ):
                    raise ValueError(
                        "Reference counts need unique string keys and numeric counts."
                    )
                values = counts.to_numpy(dtype=np.float64)
                if (
                    not np.isfinite(values).all()
                    or (values <= 0).any()
                    or (values != np.floor(values)).any()
                    or (values >= 2**63).any()
                ):
                    raise ValueError(
                        "Reference counts must be finite positive integers."
                    )
                observed = frame[name].value_counts(dropna=False, sort=False)
                available = counts.reindex(observed.index)
                if (
                    available.isna().any()
                    or (available.to_numpy() < observed.to_numpy()).any()
                ):
                    raise ValueError("Reference counts must cover the fitting rows.")
                self.counts_[name] = counts.astype("int64").copy()
        return self

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        check_is_fitted(self, "counts_")
        if (
            not isinstance(frame, pd.DataFrame)
            or frame.columns.tolist() != self.feature_names_in_.tolist()
        ):
            raise ValueError("Frequency columns must match training columns and order.")
        encoded = np.empty((len(frame), self.n_features_in_), dtype=np.float64)
        for index, name in enumerate(self.feature_names_in_):
            counts = frame[name].map(self.counts_[name]).astype(np.float64).fillna(0)
            encoded[:, index] = np.log1p(counts.to_numpy(dtype=np.float64))
        return encoded

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        check_is_fitted(self, "counts_")
        if input_features is not None and not np.array_equal(
            input_features, self.feature_names_in_
        ):
            raise ValueError("Frequency feature names must match training columns.")
        return np.asarray(
            [f"{name}__log_count" for name in self.feature_names_in_], dtype=object
        )


In [ ]:
%%writefile fraudlab/holdout.py
"""One final evaluation after development choices have been frozen."""

from __future__ import annotations

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from .data import fingerprint, load_data, write_json
from .features import transform_features
from .models import predict_saved


def evaluate(
    runs: list[Path],
    weights: list[float] | None,
    output: Path,
    device: str = "auto",
    development_run: Path | None = None,
) -> dict:
    if output.exists():
        raise FileExistsError(
            "Holdout evaluation already exists; do not reuse it for model selection."
        )
    if not runs:
        raise ValueError("At least one frozen source run is required.")
    coefficients = np.asarray(
        weights if weights is not None else [1.0] * len(runs), dtype=float
    )
    if (
        len(coefficients) != len(runs)
        or not np.isfinite(coefficients).all()
        or not np.isfinite(coefficients.sum())
        or (coefficients < 0).any()
        or coefficients.sum() <= 0
    ):
        raise ValueError(
            "Provide finite, nonnegative weights with a finite positive total."
        )
    coefficients /= coefficients.sum()
    configs = [
        json.loads((run / "config.json").read_text(encoding="utf-8")) for run in runs
    ]
    metrics = [
        json.loads((run / "metrics.json").read_text(encoding="utf-8")) for run in runs
    ]
    reference = configs[0]
    if not reference.get("holdout_fraction"):
        raise ValueError("Sources must have reserved a holdout before training.")
    fields = ("data", "target", "id_column", "holdout_fraction", "holdout_seed")
    for run, config, metric in zip(runs, configs, metrics):
        if (
            metric.get("config_sha256")
            and fingerprint(run / "config.json") != metric["config_sha256"]
        ):
            raise ValueError("Source configuration changed after training.")
        if config["model"] != metric["model"] or config.get(
            "feature_set", "raw"
        ) != metric["model_details"].get("feature_set", "raw"):
            raise ValueError(
                "Source model or feature set no longer matches its training metadata."
            )
        if any(
            config.get(field) != metric["validation"].get(field)
            for field in ("holdout_fraction", "holdout_seed")
        ):
            raise ValueError(
                "Source config no longer matches the holdout recorded at training time."
            )
        if (
            any(config.get(field) != reference.get(field) for field in fields)
            or metric.get("status") != "complete"
            or metric["data_hashes"] != metrics[0]["data_hashes"]
        ):
            raise ValueError(
                "Sources must be complete runs sharing the same untouched holdout and input data."
            )
    data_dir = Path(reference["data"])
    for filename, expected in metrics[0]["data_hashes"].items():
        if fingerprint(data_dir / filename) != expected:
            raise ValueError("Input files changed after training.")
    frame, _, _ = load_data(data_dir, reference["id_column"], reference["target"])
    _, held_out = train_test_split(
        np.arange(len(frame)),
        test_size=reference["holdout_fraction"],
        stratify=frame[reference["target"]],
        random_state=reference["holdout_seed"],
    )
    frame = frame.iloc[np.sort(held_out)].reset_index(drop=True)
    held_out_ids = set(frame[reference["id_column"]])
    for run in runs:
        if held_out_ids.intersection(
            pd.read_csv(run / "oof.csv", usecols=[reference["id_column"]])[
                reference["id_column"]
            ]
        ):
            raise ValueError("Holdout rows were found in development OOF predictions.")
    output.parent.mkdir(parents=True, exist_ok=True)
    # Persist the chosen configuration before the first holdout prediction is evaluated.
    selection = {
        "sources": [str(run) for run in runs],
        "weights": coefficients.tolist(),
        "metrics_hashes": [fingerprint(run / "metrics.json") for run in runs],
        "config_hashes": [fingerprint(run / "config.json") for run in runs],
        "frozen_at": datetime.now(timezone.utc).isoformat(),
    }
    if development_run is not None:
        dev_config = json.loads(
            (development_run / "config.json").read_text(encoding="utf-8")
        )
        dev_metric = json.loads(
            (development_run / "metrics.json").read_text(encoding="utf-8")
        )
        if (
            [Path(source).resolve() for source in dev_config["sources"]]
            != [source.resolve() for source in runs]
            or not np.allclose(dev_config["weights"], coefficients, rtol=0, atol=1e-12)
            or dev_metric.get("status") != "complete"
            or dev_metric["data_hashes"] != metrics[0]["data_hashes"]
        ):
            raise ValueError(
                "Development blend does not match the frozen sources, weights and input data."
            )
        selection.update(
            source_dev_run=str(development_run),
            development_roc_auc=dev_metric["roc_auc"],
            development_metrics_sha256=fingerprint(development_run / "metrics.json"),
        )
    partition_key = hashlib.sha256(
        json.dumps(
            {
                "train_sha256": metrics[0]["data_hashes"]["train.csv"],
                "target": reference["target"],
                "seed": reference["holdout_seed"],
                "fraction": reference["holdout_fraction"],
            },
            sort_keys=True,
        ).encode()
    ).hexdigest()
    ledger_dir = data_dir / ".holdout-evaluations"
    ledger_dir.mkdir(exist_ok=True)
    ledger = ledger_dir / f"{partition_key}.json"
    selection_key = hashlib.sha256(
        json.dumps(
            {key: value for key, value in selection.items() if key != "frozen_at"},
            sort_keys=True,
        ).encode()
    ).hexdigest()
    if ledger.exists():
        prior = json.loads(ledger.read_text(encoding="utf-8"))
        if prior["status"] == "complete" or prior["selection_key"] != selection_key:
            raise ValueError(
                "This independent holdout partition is already reserved or evaluated; do not retune from it."
            )
    else:
        # The dataset-local ledger follows this partition even if --output changes.
        # A failed evaluation may resume only with the identical frozen sources.
        with ledger.open("x", encoding="utf-8") as stream:
            json.dump(
                {
                    "status": "started",
                    "selection_key": selection_key,
                    "selection": selection,
                },
                stream,
                indent=2,
            )
    write_json(output.with_name(output.stem + "-frozen-selection.json"), selection)
    predictions = np.zeros(len(frame))
    individual = []
    for weight, run, config, metric in zip(coefficients, runs, configs, metrics):
        model_predictions = np.zeros(len(frame))
        inputs = transform_features(
            frame,
            config.get("feature_set", "raw"),
            (config["id_column"], config["target"]),
        )[metric["features"]["names"]]
        for fold in range(metric["validation"]["n_splits"]):
            model_predictions += (
                predict_saved(config["model"], run / f"fold-{fold}", inputs, device)
                / metric["validation"]["n_splits"]
            )
        predictions += weight * model_predictions
        individual.append(
            float(roc_auc_score(frame[reference["target"]], model_predictions))
        )
    if (
        not np.isfinite(predictions).all()
        or ((predictions < 0) | (predictions > 1)).any()
    ):
        raise ValueError("Invalid holdout predictions.")
    result = {
        "status": "complete",
        "kind": "independent_holdout",
        "evaluation_source_hashes": {
            file.name: fingerprint(file)
            for file in sorted(Path(__file__).parent.glob("*.py"))
        },
        "roc_auc": float(roc_auc_score(frame[reference["target"]], predictions)),
        "individual_roc_auc": individual,
        "rows": len(frame),
        "positive_rows": int(frame[reference["target"]].sum()),
        "holdout_seed": reference["holdout_seed"],
        "holdout_fraction": reference["holdout_fraction"],
        "selection": selection,
        "data_hashes": metrics[0]["data_hashes"],
        "evaluated_at": datetime.now(timezone.utc).isoformat(),
        "limitation": "The selection was frozen before this evaluation. Using this result to retune would make the holdout part of model selection.",
    }
    pd.DataFrame(
        {
            reference["id_column"]: frame[reference["id_column"]],
            "target": frame[reference["target"]],
            "prediction": predictions,
        }
    ).to_csv(output.with_suffix(".csv"), index=False)
    write_json(output, result)
    write_json(
        ledger,
        {
            "status": "complete",
            "selection_key": selection_key,
            "report_sha256": fingerprint(output),
            "evaluated_at": result["evaluated_at"],
        },
    )
    return result


In [ ]:
%%writefile fraudlab/joints.py
"""Eight fixed interaction hypotheses, with no learned selection or statistics.

Public-source inspiration: megayak's joint-key ideas as credited in goodpjw2008's
Part 12, version 16. This extraction is independently written. No G-statistic
screen, quantile fitting, third-party code or prediction arrays are used.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

JOINTS = (
    ("foreign_ip", "is_foreign_transaction", "exact", "ip_country_mismatch", "exact"),
    ("category_risk10", "merchant_category", "exact", "merchant_risk_score", "floor10"),
    ("category_amount_log2", "merchant_category", "exact", "amount_usd", "log2"),
    (
        "scam_velocity5",
        "is_ai_generated_scam_attempt",
        "exact",
        "velocity_score",
        "floor5",
    ),
    ("newmerchant_weekday", "is_new_merchant", "exact", "day_of_week", "exact"),
    ("hour_weekday", "time_of_day_hour", "exact", "day_of_week", "exact"),
    ("age_cardyears", "customer_age", "floor10", "card_age_months", "floor12"),
    ("auth_channel", "auth_method", "exact", "channel", "exact"),
)
OUTPUT_NAMES = tuple(name + "__joint_key" for name, *_ in JOINTS)
MAX_VALUE = 2**53 - 1
# Exact float64 boundaries avoid log2(1+x) rounding up immediately below a tie.
LOG2_BOUNDARIES = np.asarray([2**power - 1 for power in range(54)], dtype=np.float64)
FORMULAS = {
    "exact": "unchanged string; numeric value normalized to float64",
    "floor5": "floor(float64(x)/5)",
    "floor10": "floor(float64(x)/10)",
    "floor12": "floor(float64(x)/12)",
    "log2": "floor(log2(1+x)); interval k starts at exact boundary 2**k-1",
}


def _numeric(series: pd.Series, *, allow_bool: bool = False) -> np.ndarray:
    if (
        not pd.api.types.is_numeric_dtype(series)
        or pd.api.types.is_complex_dtype(series)
        or (pd.api.types.is_bool_dtype(series) and not allow_bool)
    ):
        raise ValueError(f"{series.name} requires real numeric joint inputs.")
    values = series.to_numpy(dtype=np.float64, na_value=np.nan)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f"{series.name} requires finite nonnegative joint inputs.")
    if (values > MAX_VALUE).any():
        raise ValueError(f"{series.name} exceeds the supported joint integer range.")
    return np.where(values == 0, 0.0, values)


def _view(series: pd.Series, mode: str) -> list:
    if mode == "exact":
        if pd.api.types.is_numeric_dtype(series):
            return _numeric(series, allow_bool=True).tolist()
        values = series.tolist()
        if not all(isinstance(value, str) for value in values):
            raise ValueError(f"{series.name} requires nonmissing string categories.")
        return values
    values = _numeric(series)
    if mode == "log2":
        bins = np.searchsorted(LOG2_BOUNDARIES, values, side="right") - 1
    else:
        bins = np.floor(values / int(mode.removeprefix("floor")))
    return bins.astype(np.int64).tolist()


def joint_keys(frame: pd.DataFrame, excluded: tuple[str, ...] = ()) -> pd.DataFrame:
    """Return fixed categorical tuple keys; skip a pair if either field is excluded.

    Only the declared fields are read. IDs/targets/other columns have no effect.
    There is no fitted state, category vocabulary or dependence on other rows.
    """
    if not isinstance(frame, pd.DataFrame) or not frame.columns.is_unique:
        raise ValueError("Joint keys require unique DataFrame columns.")
    result = pd.DataFrame(index=frame.index)
    views = {}
    for name, left, left_mode, right, right_mode in JOINTS:
        if {left, right}.intersection(excluded):
            continue
        for column, mode in ((left, left_mode), (right, right_mode)):
            if column not in frame:
                raise ValueError(f"Missing joint input column: {column}")
            if (column, mode) not in views:
                views[column, mode] = _view(frame[column], mode)
        pairs = zip(views[left, left_mode], views[right, right_mode])
        result[name + "__joint_key"] = pd.Series(
            [repr(pair) for pair in pairs], index=frame.index, dtype=object
        )
    return result


def manifest() -> dict:
    from .bpe import manifest as bpe_manifest

    return {
        "algorithm": "bpe_digits_plus_eight_fixed_joint_categories",
        "base_feature_set": "bpe_digits",
        "bpe": bpe_manifest(),
        "joint_encoding": {
            "output_count": 8,
            "pairs": [
                {
                    "output": name + "__joint_key",
                    "left": left,
                    "left_view": FORMULAS[left_mode],
                    "right": right,
                    "right_view": FORMULAS[right_mode],
                }
                for name, left, left_mode, right, right_mode in JOINTS
            ],
            "category_representation": "repr((left, right)); strings escaped, numbers canonicalized as above; categorical labels, not ordinal magnitudes",
            "numeric_domain": "finite nonnegative float64 at most 2**53-1; negative zero normalized; bins use no cent rounding",
            "log2_boundaries": "2**k-1 for integer k=0..53, left-inclusive via searchsorted(side='right')-1",
            "excluded_fields": "Exclude the entire pair if either input is excluded",
            "selection": "Eight hypotheses fixed before fitting; public-source-informed exploratory selection; no G scan, label ranking or quantile fitting",
            "label_usage": "none in extraction; downstream TE/frequency/scaling remain fitted inside outer training only",
            "concept_source": {
                "url": "https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298",
                "version": 16,
                "source_cell_zero_based": 74,
                "source_body_sha256": "835772526d7efcc70f771aa446690cb1956676a457f40b08d48760bbf3336baf",
                "credit": "megayak's joint-key ideas; independent extraction, no source code copied or executed",
            },
        },
    }


In [ ]:
%%writefile fraudlab/models.py
"""Small tabular models, each fitted inside its training fold."""

from __future__ import annotations

import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    QuantileTransformer,
    SplineTransformer,
    StandardScaler,
    TargetEncoder,
)
from threadpoolctl import threadpool_limits

from .features import CONTINUOUS


def feature_types(frame: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric = frame.select_dtypes(include="number").columns.tolist()
    return numeric, [name for name in frame if name not in numeric]


def clean_features(
    frame: pd.DataFrame,
    numeric: list[str] | None = None,
    categorical: list[str] | None = None,
) -> pd.DataFrame:
    result = frame.copy()
    if numeric is None or categorical is None:
        numeric, categorical = feature_types(result)
    for name in numeric:
        result[name] = pd.to_numeric(result[name], errors="raise")
    result[numeric] = result[numeric].replace([np.inf, -np.inf], np.nan)
    for name in categorical:
        result[name] = result[name].astype("string").fillna("__MISSING__").astype(str)
    return result


def log_odds(values: np.ndarray) -> np.ndarray:
    probabilities = np.clip(values, 1e-6, 1 - 1e-6)
    return np.log(probabilities / (1 - probabilities))


def fit_logistic(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    numeric, categorical = feature_types(training)
    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                Pipeline(
                    [
                        (
                            "impute",
                            SimpleImputer(strategy="median", keep_empty_features=True),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                numeric,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore", min_frequency=5, dtype=np.float32
                ),
                categorical,
            ),
        ],
        sparse_threshold=1.0,
    )
    model = Pipeline(
        [
            ("preprocessing", preprocessing),
            (
                "classifier",
                LogisticRegression(
                    C=config["logistic_c"], max_iter=1000, random_state=seed
                ),
            ),
        ]
    )
    with threadpool_limits(limits=config["threads"]):
        model.fit(training, y)
    joblib.dump(model, path.with_suffix(".joblib"))
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        {
            "algorithm": "logistic_regression",
            "C": config["logistic_c"],
            "solver": "lbfgs",
            "max_iter": 1000,
            "iterations_used": int(model.named_steps["classifier"].n_iter_[0]),
            "preprocessing": "Fold-local median imputation, standard scaling, one-hot encoding; unknown categories ignored; category min_frequency=5.",
        },
    )


def fit_catboost(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    from catboost import CatBoostClassifier

    _, categorical = feature_types(training)
    device = "GPU" if config["device"] == "cuda" else "CPU"
    parameters = {
        "iterations": config["iterations"],
        "depth": config["depth"],
        "learning_rate": config["learning_rate"],
        "l2_leaf_reg": config["l2_leaf_reg"],
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": seed,
        "thread_count": config["threads"],
        "task_type": device,
        "allow_writing_files": False,
        "verbose": False,
    }
    model = CatBoostClassifier(**parameters)
    if config.get("refit") or config.get("fixed_training"):
        model.fit(training, y, cat_features=categorical)
    else:
        model.fit(
            training,
            y,
            cat_features=categorical,
            eval_set=(validation, validation_y),
            early_stopping_rounds=config["patience"],
            use_best_model=True,
        )
    model.save_model(str(path.with_suffix(".cbm")))
    best_iteration = model.get_best_iteration()
    details = {
        **parameters,
        "best_iteration": int(best_iteration) if best_iteration is not None else None,
        "trees_used": int(model.tree_count_),
        "resolved_parameters": model.get_all_params(),
        "early_stopping_patience": (
            None
            if config.get("refit") or config.get("fixed_training")
            else config["patience"]
        ),
        "preprocessing": "Numeric missing values handled by CatBoost; categorical missing values mapped to __MISSING__; native ordered categorical statistics.",
    }
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


class NeuralPreprocessor:
    """Training-fold-only numeric statistics and categorical vocabularies."""

    def __init__(
        self,
        numeric_transform: str = "standard",
        seed: int = 42,
        categorical_transform: str = "embedding",
        smoothing: float = 20.0,
    ):
        self.numeric_transform = numeric_transform
        self.seed = seed
        self.categorical_transform = categorical_transform
        self.smoothing = smoothing

    def fit(self, frame: pd.DataFrame) -> "NeuralPreprocessor":
        self.numeric, self.categorical = feature_types(frame)
        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)
        self.scaler = (
            QuantileTransformer(
                n_quantiles=min(1000, len(frame)),
                output_distribution="normal",
                subsample=min(100000, len(frame)),
                random_state=self.seed,
            )
            if self.numeric_transform == "quantile"
            else StandardScaler()
        )
        if self.numeric:
            self.scaler.fit(self.imputer.fit_transform(frame[self.numeric]))
        self.vocabularies = {
            name: {
                value: index + 1
                for index, value in enumerate(sorted(frame[name].unique()))
            }
            for name in self.categorical
        }
        self.cardinalities = [
            len(self.vocabularies[name]) + 1 for name in self.categorical
        ]
        return self

    def fit_transform(
        self, frame: pd.DataFrame, y: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray]:
        self.fit(frame)
        if self.categorical_transform == "embedding":
            return self.transform(frame)
        numbers = (
            self.scaler.transform(self.imputer.transform(frame[self.numeric])).astype(
                np.float32
            )
            if self.numeric
            else np.empty((len(frame), 0), dtype=np.float32)
        )
        self.cardinalities = []
        self.target_encoder = TargetEncoder(
            target_type="binary",
            smooth=self.smoothing,
            cv=5,
            shuffle=True,
            random_state=self.seed,
        )
        self.target_scaler = StandardScaler()
        if self.categorical:
            encoded = self.target_encoder.fit_transform(frame[self.categorical], y)
            encoded = self.target_scaler.fit_transform(log_odds(encoded)).astype(
                np.float32
            )
            numbers = np.column_stack([numbers, encoded])
        return numbers, np.empty((len(frame), 0), dtype=np.int64)

    def transform(self, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        numbers = (
            self.scaler.transform(self.imputer.transform(frame[self.numeric])).astype(
                np.float32
            )
            if self.numeric
            else np.empty((len(frame), 0), dtype=np.float32)
        )
        if getattr(self, "categorical_transform", "embedding") == "target":
            if self.categorical:
                encoded = self.target_encoder.transform(frame[self.categorical])
                encoded = self.target_scaler.transform(log_odds(encoded)).astype(
                    np.float32
                )
                numbers = np.column_stack([numbers, encoded])
            return numbers, np.empty((len(frame), 0), dtype=np.int64)
        categories = (
            np.column_stack(
                [
                    frame[name]
                    .map(self.vocabularies[name])
                    .fillna(0)
                    .to_numpy(dtype=np.int64)
                    for name in self.categorical
                ]
            )
            if self.categorical
            else np.empty((len(frame), 0), dtype=np.int64)
        )
        return numbers, categories


def build_network(
    torch,
    numeric_count: int,
    cardinalities: list[int],
    hidden_dims: list[int],
    dropout: float,
):
    nn = torch.nn
    embedding_dims = [
        min(32, max(2, int(np.sqrt(count)) + 1)) for count in cardinalities
    ]

    class TabularMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.embeddings = nn.ModuleList(
                [
                    nn.Embedding(count, width, padding_idx=0)
                    for count, width in zip(cardinalities, embedding_dims)
                ]
            )
            width = numeric_count + sum(embedding_dims)
            layers = []
            for next_width in hidden_dims:
                layers.extend(
                    [nn.Linear(width, next_width), nn.ReLU(), nn.Dropout(dropout)]
                )
                width = next_width
            layers.append(nn.Linear(width, 1))
            self.layers = nn.Sequential(*layers)

        def forward(self, numbers, categories):
            pieces = [
                numbers,
                *[
                    embedding(categories[:, i])
                    for i, embedding in enumerate(self.embeddings)
                ],
            ]
            return self.layers(torch.cat(pieces, dim=1)).squeeze(1)

    return TabularMLP(), embedding_dims


def fit_neural(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    try:
        import torch
    except ImportError as error:
        raise RuntimeError(
            "Neural training requires PyTorch: install fraudlab[neural]."
        ) from error

    torch.set_num_threads(config["threads"])
    torch.manual_seed(seed)
    if config["device"] == "cuda" and not torch.cuda.is_available():
        raise ValueError(
            "CUDA was requested but is not available in this PyTorch installation."
        )
    device = torch.device(
        "cuda" if config["device"] != "cpu" and torch.cuda.is_available() else "cpu"
    )
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    preprocessing = NeuralPreprocessor(
        config.get("neural_numeric_transform", "standard"),
        seed,
        config.get("neural_categorical_transform", "embedding"),
        config.get("target_smoothing", 20.0),
    )
    arrays = [
        preprocessing.fit_transform(training, y),
        preprocessing.transform(validation),
        preprocessing.transform(test),
    ]
    tensors = [
        tuple(torch.as_tensor(value, device=device) for value in pair)
        for pair in arrays
    ]
    labels = torch.as_tensor(y, dtype=torch.float32, device=device)
    model, embedding_dims = build_network(
        torch,
        arrays[0][0].shape[1],
        preprocessing.cardinalities,
        config["hidden_dims"],
        config["dropout"],
    )
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["neural_learning_rate"],
        weight_decay=config["weight_decay"],
    )
    criterion = torch.nn.BCEWithLogitsLoss()

    def predict(pair) -> np.ndarray:
        model.eval()
        with torch.no_grad():
            return np.concatenate(
                [
                    torch.sigmoid(
                        model(
                            pair[0][start : start + config["batch_size"]],
                            pair[1][start : start + config["batch_size"]],
                        )
                    )
                    .cpu()
                    .numpy()
                    for start in range(0, len(pair[0]), config["batch_size"])
                ]
            )

    best_auc, stale, best_epoch, best_state = -np.inf, 0, 0, None
    epochs_completed = 0
    for epoch in range(config["epochs"]):
        model.train()
        order = torch.randperm(len(y), device=device)
        for start in range(0, len(y), config["batch_size"]):
            selected = order[start : start + config["batch_size"]]
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(
                model(tensors[0][0][selected], tensors[0][1][selected]),
                labels[selected],
            )
            if not torch.isfinite(loss):
                raise ValueError(
                    "Neural loss became non-finite; check features or lower the learning rate."
                )
            loss.backward()
            optimizer.step()
        epochs_completed = epoch + 1
        if config.get("refit") or config.get("fixed_training"):
            best_epoch = epochs_completed
            if epochs_completed == config["epochs"]:
                best_state = {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in model.state_dict().items()
                }
            continue
        score = roc_auc_score(validation_y, predict(tensors[1]))
        if score > best_auc + 1e-6:
            best_auc, best_epoch, stale = score, epoch + 1, 0
            best_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
        else:
            stale += 1
        if stale >= config["patience"]:
            break
    if best_state is None:
        raise ValueError("Neural training did not produce a usable checkpoint.")
    model.load_state_dict(best_state)
    details = {
        "architecture": {
            "numeric_features": int(arrays[0][0].shape[1]),
            "raw_numeric_features": len(preprocessing.numeric),
            "categorical_transform": config.get(
                "neural_categorical_transform", "embedding"
            ),
            "categorical_cardinalities": preprocessing.cardinalities,
            "embedding_dims": embedding_dims,
            "hidden_dims": config["hidden_dims"],
            "dropout": config["dropout"],
            "hidden_activation": "ReLU",
            "normalization": "none",
            "output_activation": "sigmoid",
        },
        "optimizer": "AdamW",
        "loss": "BCEWithLogitsLoss",
        "learning_rate": config["neural_learning_rate"],
        "weight_decay": config["weight_decay"],
        "batch_size": config["batch_size"],
        "epochs_completed": epochs_completed,
        "best_epoch": best_epoch,
        "early_stopping_patience": (
            None
            if config.get("refit") or config.get("fixed_training")
            else config["patience"]
        ),
        "device": str(device),
        "numeric_transform": config.get("neural_numeric_transform", "standard"),
        "target_encoder": (
            {
                "internal_folds": 5,
                "smooth": config.get("target_smoothing", 20.0),
                "unknown": "training prior",
                "log_odds": True,
            }
            if config.get("neural_categorical_transform") == "target"
            else None
        ),
        "preprocessing": "Fold-local numeric median imputation and "
        + (
            "quantile-to-normal transformation"
            if config.get("neural_numeric_transform") == "quantile"
            else "standard scaling"
        )
        + (
            "; categorical target-encoding logits cross-fitted into five inner partitions, then scaled using training-only statistics."
            if config.get("neural_categorical_transform") == "target"
            else "; categorical training vocabularies with index 0 reserved for unknown categories."
        ),
    }
    torch.save(
        {"state_dict": best_state, "architecture": details["architecture"]},
        path.with_suffix(".pt"),
    )
    joblib.dump(preprocessing, path.with_suffix(".preprocessing.joblib"))
    return predict(tensors[1]), predict(tensors[2]), details


def fit_predict(
    name: str,
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    numeric, categorical = feature_types(training)
    training, validation, test = [
        clean_features(frame, numeric, categorical)
        for frame in (training, validation, test)
    ]
    if name == "logistic":
        return fit_logistic(training, y, validation, test, config, seed, path)
    if name == "glm":
        return fit_glm(training, y, validation, test, config, seed, path)
    if name == "offset":
        from .offset import fit_offset

        result = fit_offset(training, y, validation, test, config, seed, path)
        if config.get("glm_pooled_tokens"):
            from .pooled import manifest

            result[2]["pooled_tokens"] = manifest(random_state=seed)
        if config.get("glm_transductive_frequency"):
            result[2]["transductive_frequency"] = config.get("transductive_frequency")
            result[2]["preprocessing"] = (
                "Numeric scaling and target statistics use outer training only; category counts "
                "use official train+test features without labels. The same cross-fitted matrix "
                "fits GLM and residual trees; inference sums their margins."
            )
        return result
    if name == "catboost":
        return fit_catboost(
            training, y, validation, validation_y, test, config, seed, path
        )
    if name == "mlp":
        return fit_neural(
            training, y, validation, validation_y, test, config, seed, path
        )
    if name == "tabm":
        try:
            from .tabm import fit_tabm
        except ImportError as error:
            raise RuntimeError(
                "TabM requires the optional fraudlab[tabm] dependencies."
            ) from error
        # Deliberately do not forward outer validation labels to TabM.
        return fit_tabm(training, y, validation, test, config, seed, path)
    if name == "neural_offset":
        try:
            from .neural_offset import fit_neural_offset
        except ImportError as error:
            raise RuntimeError(
                "Neural offset requires the optional fraudlab[neural] dependencies."
            ) from error
        # Outer validation labels cannot enter epoch selection or either GLM fit.
        result = fit_neural_offset(training, y, validation, test, config, seed, path)
        if config.get("glm_pooled_tokens"):
            from .pooled import manifest

            result[2]["pooled_tokens"] = manifest(random_state=seed)
        return result
    if name == "lightgbm":
        return fit_lightgbm(
            training, y, validation, validation_y, test, config, seed, path
        )
    raise ValueError(f"Unknown model: {name}")


def predict_saved(
    name: str, path: Path, frame: pd.DataFrame, device: str = "auto"
) -> np.ndarray:
    """Load only locally trained checkpoints; never download executable artifacts."""
    if name in ("logistic", "glm", "lightgbm", "offset"):
        return joblib.load(path.with_suffix(".joblib")).predict_proba(
            clean_features(frame)
        )[:, 1]
    if name == "catboost":
        from catboost import CatBoostClassifier

        model = CatBoostClassifier()
        model.load_model(str(path.with_suffix(".cbm")))
        return model.predict_proba(clean_features(frame))[:, 1]
    if name == "tabm":
        from .tabm import predict_saved_tabm

        return predict_saved_tabm(path, frame, device)
    if name == "neural_offset":
        from .neural_offset import predict_saved_neural_offset

        return predict_saved_neural_offset(path, frame, device)
    if name != "mlp":
        raise ValueError(f"Unknown saved model: {name}")
    import torch

    if device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is not available.")
    selected_device = torch.device(
        "cuda" if device != "cpu" and torch.cuda.is_available() else "cpu"
    )
    preprocessing = joblib.load(path.with_suffix(".preprocessing.joblib"))
    checkpoint = torch.load(
        path.with_suffix(".pt"), map_location="cpu", weights_only=True
    )
    architecture = checkpoint["architecture"]
    model, _ = build_network(
        torch,
        architecture["numeric_features"],
        architecture["categorical_cardinalities"],
        architecture["hidden_dims"],
        architecture["dropout"],
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.to(selected_device).eval()
    prepared = preprocessing.transform(
        clean_features(frame, preprocessing.numeric, preprocessing.categorical)
    )
    predictions = []
    with torch.no_grad():
        for start in range(0, len(frame), 8192):
            tensors = [
                torch.as_tensor(values[start : start + 8192], device=selected_device)
                for values in prepared
            ]
            predictions.append(torch.sigmoid(model(*tensors)).cpu().numpy())
    return np.concatenate(predictions)


def fit_lightgbm(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    from lightgbm import LGBMClassifier, early_stopping

    numeric, categorical = feature_types(training)
    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                SimpleImputer(strategy="median", keep_empty_features=True),
                numeric,
            ),
            (
                "categorical",
                TargetEncoder(
                    target_type="binary",
                    smooth=config.get("target_smoothing", "auto"),
                    cv=5,
                    shuffle=True,
                    random_state=seed,
                ),
                categorical,
            ),
        ]
    )
    # fit_transform cross-fits encodings inside this outer training fold. Validation
    # and test use mappings learned only from that training fold.
    encoded_training = preprocessing.fit_transform(training, y)
    encoded_validation = preprocessing.transform(validation)
    parameters = {
        "n_estimators": config["iterations"],
        "num_leaves": config["num_leaves"],
        "learning_rate": config["learning_rate"],
        "min_child_samples": config["min_child_samples"],
        "reg_lambda": config["l2_leaf_reg"],
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "random_state": seed,
        "n_jobs": config["threads"],
        "deterministic": True,
        "force_col_wise": True,
    }
    classifier = LGBMClassifier(**parameters)
    if config.get("refit") or config.get("fixed_training"):
        classifier.fit(encoded_training, y)
    else:
        classifier.fit(
            encoded_training,
            y,
            eval_X=encoded_validation,
            eval_y=validation_y,
            eval_metric="auc",
            callbacks=[
                early_stopping(
                    config["patience"], first_metric_only=True, verbose=False
                )
            ],
        )
    model = Pipeline([("preprocessing", preprocessing), ("classifier", classifier)])
    joblib.dump(model, path.with_suffix(".joblib"))
    details = {
        **parameters,
        "best_iteration": int(classifier.best_iteration_ or classifier.n_estimators_),
        "target_encoder": {
            "internal_folds": 5,
            "smooth": config.get("target_smoothing", "auto"),
            "unknown": "training prior",
        },
        "preprocessing": "Fold-local median imputation and supervised categorical target encoding, internally cross-fitted into five partitions.",
    }
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


def make_glm_pipeline(
    training: pd.DataFrame,
    config: dict,
    seed: int,
) -> Pipeline:
    """Construct fold-local preprocessing and a regularized logistic model."""
    numeric, categorical = feature_types(training)
    nominal = [name for name in categorical if "__" not in name]
    value_one_hot = config.get("glm_value_one_hot", False)
    one_hot_columns = categorical if value_one_hot else nominal
    encoding = Pipeline(
        [
            (
                "target",
                TargetEncoder(
                    target_type="binary",
                    smooth=config["target_smoothing"],
                    cv=config.get("glm_inner_folds", 5),
                    shuffle=True,
                    random_state=seed,
                ),
            ),
            ("log_odds", FunctionTransformer(log_odds)),
            ("scale", StandardScaler()),
        ]
    )
    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                Pipeline(
                    [
                        (
                            "impute",
                            SimpleImputer(strategy="median", keep_empty_features=True),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                numeric,
            ),
            ("target_logits", encoding, categorical),
            (
                "one_hot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=value_one_hot,
                    min_frequency=5 if value_one_hot else None,
                    dtype=np.float32,
                ),
                one_hot_columns,
            ),
        ],
        sparse_threshold=1.0 if value_one_hot else 0.3,
    )
    knots = config.get("glm_spline_knots", 0)
    if config.get("glm_frequency"):
        from .frequency import FrequencyEncoder

        if (
            config.get("glm_transductive_frequency")
            and config.get("_frequency_reference") is None
        ):
            raise ValueError("Transductive frequency needs runtime reference counts.")
        if config.get("_frequency_reference") is not None and not config.get(
            "glm_transductive_frequency"
        ):
            raise ValueError(
                "Reference counts require the transductive frequency flag."
            )
        preprocessing.transformers.append(
            (
                "category_frequency",
                Pipeline(
                    [
                        (
                            "frequency",
                            FrequencyEncoder(
                                reference_counts=config.get("_frequency_reference")
                            ),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                categorical,
            )
        )
    if config.get("glm_pooled_tokens"):
        from .pooled import COLUMNS, PooledTokenEncoder

        if config.get("glm_inner_folds", 5) != 5:
            raise ValueError("Pooled tokens require five inner folds.")
        preprocessing.transformers.append(
            (
                "pooled_token_logits",
                Pipeline(
                    [
                        ("pooled", PooledTokenEncoder(random_state=seed)),
                        ("log_odds", FunctionTransformer(log_odds)),
                        ("scale", StandardScaler()),
                    ]
                ),
                list(COLUMNS),
            )
        )
    continuous = [name for name in CONTINUOUS if name in numeric]
    if knots:
        if not continuous:
            raise ValueError("Numeric splines require a supported continuous feature.")
        preprocessing.transformers.append(
            (
                "numeric_splines",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="median")),
                        (
                            "spline",
                            SplineTransformer(
                                n_knots=knots,
                                degree=3,
                                knots="quantile",
                                include_bias=False,
                                extrapolation="constant",
                            ),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                continuous,
            )
        )
    classifier = LogisticRegression(
        C=config["logistic_c"],
        solver="lbfgs" if value_one_hot else "newton-cholesky",
        max_iter=2000 if value_one_hot else 100,
        tol=1e-7,
        random_state=seed,
    )
    return Pipeline([("preprocessing", preprocessing), ("classifier", classifier)])


def fit_glm(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    model = make_glm_pipeline(training, config, seed)
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", ConvergenceWarning)
        with threadpool_limits(limits=config["threads"]):
            model.fit(training, y)
    for warning in captured:
        warnings.warn(warning.message, warning.category, stacklevel=2)
    classifier = model.named_steps["classifier"]
    iterations_used = int(classifier.n_iter_[0])
    converged = (
        not any(
            issubclass(warning.category, ConvergenceWarning) for warning in captured
        )
        and iterations_used < classifier.max_iter
    )
    if config.get("glm_value_one_hot") and not converged:
        raise RuntimeError(
            "GLM value one-hot did not converge; "
            f"iterations_used={iterations_used}, max_iter={classifier.max_iter}. "
            "The candidate is invalid under the fixed protocol."
        )
    joblib.dump(model, path.with_suffix(".joblib"))
    details = {
        "algorithm": "logistic_regression_with_cross_fitted_target_logits",
        "C": config["logistic_c"],
        "solver": classifier.solver,
        "max_iter": classifier.max_iter,
        "tol": classifier.tol,
        "iterations_used": iterations_used,
        "converged": converged,
        "value_one_hot": (
            {
                "features": feature_types(training)[1],
                "min_frequency": 5,
                "handle_unknown": "ignore",
                "sparse_output": True,
            }
            if config.get("glm_value_one_hot")
            else None
        ),
        "category_frequency": (
            (
                "log1p official train+test feature counts, no labels; outer-training scale only"
                if config.get("glm_transductive_frequency")
                else "log1p counts fitted on outer training rows only"
            )
            if config.get("glm_frequency")
            else None
        ),
        "numeric_splines": (
            {
                "n_knots": config.get("glm_spline_knots", 0),
                "degree": 3,
                "knots": "quantile fitted only on outer training rows",
                "features": [name for name in CONTINUOUS if name in training],
                "extrapolation": "constant",
            }
            if config.get("glm_spline_knots", 0)
            else None
        ),
        "target_encoder": {
            "internal_folds": config.get("glm_inner_folds", 5),
            "smooth": config["target_smoothing"],
            "unknown": "training prior",
            "log_odds": True,
        },
        "preprocessing": "Fold-local numeric imputation and scaling, cross-fitted target-encoding logits with scaling, plus "
        + (
            "sparse one-hot encoding of all categorical features; counts below 5 pooled per training fold; unknown categories ignored."
            if config.get("glm_value_one_hot")
            else "one-hot encoding of original categorical features."
        ),
    }
    if config.get("glm_pooled_tokens"):
        from .pooled import manifest

        details["pooled_tokens"] = manifest(random_state=seed)
    if config.get("glm_transductive_frequency"):
        details["transductive_frequency"] = config.get("transductive_frequency")
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


In [ ]:
%%writefile fraudlab/neural_offset.py
"""A fresh neural correction to a frozen GLM logit, selected inside outer training.

The outer validation labels are deliberately absent from this module's API.
Both models learn from the identical, internally cross-fitted training matrix.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits

from .models import clean_features, feature_types


@dataclass(frozen=True)
class NeuralOffsetConfig:
    hidden_width: int = 256
    dropout: float = 0.1
    learning_rate: float = 0.001
    weight_decay: float = 0.0001
    batch_size: int = 4096
    epochs: int = 40
    patience: int = 6
    gradient_clip: float = 1.0
    inner_fraction: float = 0.15
    inner_seed: int = 42

    @classmethod
    def from_training_config(cls, config: dict):
        if config.get("fixed_training") or config.get("refit"):
            raise ValueError("Neural offset requires inner epoch selection.")
        hidden_dims = config.get("hidden_dims")
        if hidden_dims is not None and hidden_dims != [256, 256]:
            raise ValueError("Neural offset requires --hidden-dims 256 256.")
        if config.get("glm_value_one_hot"):
            raise ValueError("Neural offset requires the dense GLM representation.")
        if config.get("neural_numeric_transform", "standard") != "standard":
            raise ValueError("Neural offset uses the GLM's standard numeric scaling.")
        if config.get("neural_categorical_transform") not in (None, "target"):
            raise ValueError("Neural offset uses the GLM's cross-fitted target logits.")
        settings = cls(
            dropout=config.get("dropout", 0.1),
            learning_rate=config.get("neural_learning_rate", 0.001),
            weight_decay=config.get("weight_decay", 0.0001),
            batch_size=config.get("batch_size", 4096),
            epochs=config.get("epochs", 40),
            patience=6 if config.get("patience") is None else config["patience"],
        )
        if not 1 <= settings.epochs <= 40 or settings.patience < 1:
            raise ValueError("Neural offset needs 1..40 epochs and positive patience.")
        if settings.batch_size < 1 or settings.learning_rate <= 0:
            raise ValueError("Batch size and learning rate must be positive.")
        if not 0 <= settings.dropout < 1 or settings.weight_decay < 0:
            raise ValueError("Invalid dropout or weight decay.")
        return settings


def build_residual(input_features: int, settings: NeuralOffsetConfig):
    width = settings.hidden_width
    model = torch.nn.Sequential(
        torch.nn.Linear(input_features, width),
        torch.nn.SiLU(),
        torch.nn.Dropout(settings.dropout),
        torch.nn.Linear(width, width),
        torch.nn.SiLU(),
        torch.nn.Dropout(settings.dropout),
        torch.nn.Linear(width, 1),
    )
    # The initial combined predictor is the GLM with its margin cast to float32.
    torch.nn.init.zeros_(model[-1].weight)
    torch.nn.init.zeros_(model[-1].bias)
    return model


def prepare_training(frame, y, config, seed):
    # Local import permits later lazy dispatch without a circular import.
    from .models import make_glm_pipeline

    glm = make_glm_pipeline(frame, config, seed)
    preprocessing = glm.named_steps["preprocessing"]
    classifier = glm.named_steps["classifier"]
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", ConvergenceWarning)
        with threadpool_limits(limits=config["threads"]):
            encoded = preprocessing.fit_transform(frame, y)
            classifier.fit(encoded, y)
            margin = classifier.decision_function(encoded)
    for warning in captured:
        warnings.warn(warning.message, warning.category, stacklevel=2)
    if (
        any(issubclass(w.category, ConvergenceWarning) for w in captured)
        or int(classifier.n_iter_[0]) >= classifier.max_iter
    ):
        raise RuntimeError(
            "Neural offset GLM did not converge; candidate invalid under fixed protocol."
        )
    # NEVER transform training again with the full target maps.
    return glm, _float_matrix(encoded), np.asarray(margin, dtype=np.float32)


def _float_matrix(encoded):
    if hasattr(encoded, "toarray"):
        raise ValueError("Neural offset requires a dense GLM training matrix.")
    matrix = np.ascontiguousarray(encoded, dtype=np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError("Neural offset received nonfinite encoded features.")
    return matrix


def transform_frame(glm, frame):
    encoded = glm.named_steps["preprocessing"].transform(frame)
    margin = glm.named_steps["classifier"].decision_function(encoded)
    return _float_matrix(encoded), np.asarray(margin, dtype=np.float32)


@torch.no_grad()
def predict_array(model, numbers, margin, batch_size):
    model.eval()
    return np.concatenate(
        [
            (offset + model(batch).squeeze(-1)).sigmoid().cpu().numpy()
            for batch, offset in zip(
                numbers.split(batch_size), margin.split(batch_size)
            )
        ]
    )


def _train(numbers, y, margin, settings, seed, device, *, validation=None, epochs=None):
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    model = build_residual(numbers.shape[1], settings).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=settings.learning_rate,
        weight_decay=settings.weight_decay,
    )
    x = torch.as_tensor(numbers, device=device)
    labels = torch.as_tensor(y, dtype=torch.float32, device=device)
    offsets = torch.as_tensor(margin, dtype=torch.float32, device=device).detach()
    best_auc, best_epoch, stale, history = -np.inf, 0, 0, []
    for epoch in range(1, (epochs or settings.epochs) + 1):
        model.train()
        for selected in torch.randperm(len(y), device=device).split(
            settings.batch_size
        ):
            optimizer.zero_grad(set_to_none=True)
            logits = offsets[selected] + model(x[selected]).squeeze(-1)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                logits, labels[selected]
            )
            if not torch.isfinite(loss):
                raise ValueError("Neural offset loss became nonfinite.")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), settings.gradient_clip)
            optimizer.step()
        if validation is None:
            best_epoch = epoch
            continue
        vx, vy, vm = validation
        score = roc_auc_score(vy, predict_array(model, vx, vm, settings.batch_size))
        history.append({"epoch": epoch, "inner_auc": float(score)})
        print(
            f"neural_offset seed={seed} inner_epoch={epoch} roc_auc={score:.8f}",
            flush=True,
        )
        if score > best_auc + 1e-6:
            best_auc, best_epoch, stale = score, epoch, 0
        else:
            stale += 1
        if stale >= settings.patience:
            break
    return model, best_epoch, history, epoch


def fit_neural_offset(training, y, validation, test, config, seed, path: Path):
    """Choose E internally, discard that stage, and refit both models from scratch."""
    settings = NeuralOffsetConfig.from_training_config(config)
    config = {
        "feature_set": "bpe_digits",
        "glm_frequency": True,
        "glm_pooled_tokens": True,
        "glm_inner_folds": 5,
        "glm_spline_knots": 0,
        "target_smoothing": 50.0,
        "logistic_c": 0.3,
        "threads": 4,
        **config,
    }
    torch.set_num_threads(config["threads"])
    requested_device = config.get("device", "cpu")
    if requested_device not in ("cpu", "cuda", "auto"):
        raise ValueError("Unknown neural offset device.")
    if requested_device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is not available.")
    device = torch.device(
        "cuda" if requested_device != "cpu" and torch.cuda.is_available() else "cpu"
    )
    numeric, categorical = feature_types(training)
    training = clean_features(training, numeric, categorical)
    y = np.asarray(y)
    inner_train, inner_valid = train_test_split(
        np.arange(len(y)),
        test_size=settings.inner_fraction,
        random_state=settings.inner_seed,
        stratify=y,
    )
    inner_glm, inner_x, inner_margin = prepare_training(
        training.iloc[inner_train], y[inner_train], config, seed
    )
    inner_classifier = inner_glm.named_steps["classifier"]
    inner_glm_iterations = int(inner_classifier.n_iter_[0])
    inner_glm_max_iter = inner_classifier.max_iter
    inner_vx, inner_vm = transform_frame(inner_glm, training.iloc[inner_valid])
    selection_model, selected_epoch, history, selection_epochs = _train(
        inner_x,
        y[inner_train],
        inner_margin,
        settings,
        seed,
        device,
        validation=(
            torch.as_tensor(inner_vx, device=device),
            y[inner_valid],
            torch.as_tensor(inner_vm, device=device),
        ),
    )
    # No preprocessing, GLM, network weights or optimizer crosses this boundary.
    del (
        selection_model,
        inner_glm,
        inner_classifier,
        inner_x,
        inner_margin,
        inner_vx,
        inner_vm,
    )
    print(
        f"neural_offset seed={seed} fresh_outer_refit_epochs={selected_epoch}",
        flush=True,
    )
    glm, numbers, margin = prepare_training(training, y, config, seed)
    model, _, _, refit_epochs = _train(
        numbers, y, margin, settings, seed, device, epochs=selected_epoch
    )
    predictions = []
    for frame in (validation, test):
        vx, vm = transform_frame(glm, clean_features(frame, numeric, categorical))
        predictions.append(
            predict_array(
                model,
                torch.as_tensor(vx, device=device),
                torch.as_tensor(vm, device=device),
                settings.batch_size,
            )
        )
    epoch_metadata = {
        "selected_epoch": selected_epoch,
        "selection_epochs_completed": selection_epochs,
        "refit_epochs_completed": refit_epochs,
        "inner_history": history,
        "inner_training_rows": len(inner_train),
        "inner_validation_rows": len(inner_valid),
        "refit_rows": len(training),
        "inner_glm_iterations_used": inner_glm_iterations,
        "inner_glm_converged": inner_glm_iterations < inner_glm_max_iter,
    }
    architecture = {
        "name": "GLM_logit_plus_neural_residual",
        "input_features": numbers.shape[1],
        "hidden_dims": [256, 256],
        "activation": "SiLU",
        "dropout": settings.dropout,
        "zero_initialized_head": True,
        "glm_margin_dtype": "float32",
        "parameters": sum(p.numel() for p in model.parameters()),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {"glm": glm, "numeric": numeric, "categorical": categorical},
        path.with_suffix(".glm.joblib"),
    )
    torch.save(
        {
            "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "config": asdict(settings),
            "architecture": architecture,
            "epoch_metadata": epoch_metadata,
        },
        path.with_suffix(".pt"),
    )
    return *predictions, {
        "architecture": architecture,
        "config": asdict(settings),
        **epoch_metadata,
        "glm": {
            key: config[key]
            for key in (
                "feature_set",
                "logistic_c",
                "target_smoothing",
                "glm_inner_folds",
                "glm_frequency",
                "glm_pooled_tokens",
                "glm_spline_knots",
            )
        },
        "glm_iterations_used": int(glm.named_steps["classifier"].n_iter_[0]),
        "glm_converged": int(glm.named_steps["classifier"].n_iter_[0])
        < glm.named_steps["classifier"].max_iter,
        "optimizer": "AdamW",
        "loss": "unweighted mean BCEWithLogits",
        "prediction": "sigmoid(frozen GLM decision_function + neural residual)",
        "training_encoding": "same cross-fitted matrix for GLM and residual; no full-map training retransform",
        "epoch_selection": "inner 15% AUC; fresh GLM, preprocessing, network and AdamW for full outer refit",
        "outer_validation_usage": "inference only; no labels accepted",
        "seed": seed,
        "threads": torch.get_num_threads(),
        "device": str(device),
    }


def predict_saved_neural_offset(path: Path, frame: pd.DataFrame, device="cpu"):
    if device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is not available.")
    device = "cuda" if device != "cpu" and torch.cuda.is_available() else "cpu"
    payload = torch.load(path.with_suffix(".pt"), map_location="cpu", weights_only=True)
    settings = NeuralOffsetConfig(**payload["config"])
    model = build_residual(payload["architecture"]["input_features"], settings).to(
        device
    )
    model.load_state_dict(payload["state_dict"])
    saved = joblib.load(path.with_suffix(".glm.joblib"))
    numbers, margin = transform_frame(
        saved["glm"], clean_features(frame, saved["numeric"], saved["categorical"])
    )
    return predict_array(
        model,
        torch.as_tensor(numbers, device=device),
        torch.as_tensor(margin, device=device),
        settings.batch_size,
    )


In [ ]:
%%writefile fraudlab/offset.py
"""Fixed small trees correct a GLM margin using one cross-fitted training matrix."""

from __future__ import annotations

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits


class OffsetClassifier:
    """Persist both terms: LightGBM does not add its training init_score at inference."""

    def __init__(self, glm, residual):
        self.glm = glm
        self.residual = residual

    def predict_proba(self, frame: pd.DataFrame) -> np.ndarray:
        encoded = self.glm.named_steps["preprocessing"].transform(frame)
        margin = self.glm.named_steps["classifier"].decision_function(encoded)
        margin = margin + self.residual.predict(encoded, raw_score=True)
        probability = np.exp(-np.logaddexp(0.0, -margin))
        return np.column_stack((1 - probability, probability))


def fit_offset(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    from lightgbm import LGBMClassifier

    # Local import keeps model dispatch free to import this module lazily.
    from .models import make_glm_pipeline

    glm = make_glm_pipeline(training, config, seed)
    preprocessing = glm.named_steps["preprocessing"]
    classifier = glm.named_steps["classifier"]
    with threadpool_limits(limits=config["threads"]):
        encoded_training = preprocessing.fit_transform(training, y)
        classifier.fit(encoded_training, y)
        initial_score = classifier.decision_function(encoded_training)
    parameters = {
        "n_estimators": config["iterations"],
        "num_leaves": config["num_leaves"],
        "learning_rate": config["learning_rate"],
        "reg_lambda": config["l2_leaf_reg"],
        "min_child_samples": config["min_child_samples"],
        "objective": "binary",
        "verbosity": -1,
        "random_state": seed,
        "n_jobs": config["threads"],
        "deterministic": True,
        "force_col_wise": True,
    }
    residual = LGBMClassifier(**parameters)
    residual.fit(encoded_training, y, init_score=initial_score)
    model = OffsetClassifier(glm, residual)
    joblib.dump(model, path.with_suffix(".joblib"))
    details = {
        "algorithm": "glm_logit_offset_with_lightgbm_residual",
        "prediction": "sigmoid(GLM decision_function + LightGBM raw_score)",
        "fixed_training": True,
        "early_stopping_patience": None,
        "best_iteration": int(residual.n_estimators_),
        "encoded_features": int(encoded_training.shape[1]),
        "glm": {
            "C": config["logistic_c"],
            "solver": classifier.solver,
            "iterations_used": int(classifier.n_iter_[0]),
            "spline_knots": config.get("glm_spline_knots", 0),
        },
        "target_encoder": {
            "internal_folds": config.get("glm_inner_folds", 5),
            "smooth": config["target_smoothing"],
            "unknown": "training prior",
            "log_odds": True,
        },
        "residual_parameters": parameters,
        "preprocessing": (
            "Fold-local GLM preprocessing, including internally cross-fitted target "
            "logits, is fitted once. The identical encoded training matrix fits the "
            "GLM, supplies its init_score and fits residual trees. Training rows are "
            "never re-encoded with the full target map. Validation and test use "
            "training-only mappings. Both fitted models are saved together."
        ),
    }
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


In [ ]:
%%writefile fraudlab/original.py
"""Validate explicitly pinned original data for outer-training-only augmentation."""

from __future__ import annotations

import hashlib
from io import BytesIO
from pathlib import Path
import re

import numpy as np
import pandas as pd

from .data import fingerprint, validate_ids

NUMERIC = (
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
)
NOMINAL = ("merchant_category", "card_type", "auth_method", "channel", "device_type")
FLAGS = (
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
)
COLUMNS = NUMERIC + NOMINAL + FLAGS
AUDITED_SHA256 = "4e32b1ad4880cae452d1f2e5aa53224d38eb913c185455c00a3759df5eb3da61"
SOURCE_URL = "https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026/versions/5"


def validate_config(config: dict) -> None:
    path, digest = config.get("original_data"), config.get("original_sha256")
    if not path:
        if (
            digest
            or config.get("original_id_column", "transaction_id") != "transaction_id"
        ):
            raise ValueError("Original-data options require --original-data.")
        return
    if not isinstance(digest, str) or not re.fullmatch(r"[a-fA-F0-9]{64}", digest):
        raise ValueError("--original-data requires a valid --original-sha256.")
    if (
        config.get("phase") != "exploratory_full_data_cv"
        or config.get("split") != "stratified"
        or config.get("holdout_fraction", 0)
        or config.get("folds") != 5
        or config.get("seed") != 42
        or config.get("refit")
    ):
        raise ValueError(
            "Original data requires exploratory stratified CV5/42, no holdout/refit."
        )
    if config.get("model") in ("catboost", "lightgbm", "mlp") and not config.get(
        "fixed_training"
    ):
        raise ValueError(
            "Original augmentation cannot use outer-validation early stopping."
        )


def verify_unchanged(path: Path, expected_sha256: str) -> None:
    if fingerprint(path) != expected_sha256.lower():
        raise ValueError(
            "Original CSV bytes changed; SHA-256 no longer matches the pin."
        )


def load_original(
    path: Path,
    expected_sha256: str,
    competition: pd.DataFrame,
    test: pd.DataFrame,
    *,
    original_id_column="transaction_id",
    id_column="id",
    target="is_fraud",
) -> tuple[pd.DataFrame, np.ndarray, dict]:
    """Return only the 24 ordered features and labels; never carry source IDs."""
    if not isinstance(expected_sha256, str) or not re.fullmatch(
        r"[a-fA-F0-9]{64}", expected_sha256
    ):
        raise ValueError("A valid original SHA-256 pin is required.")
    if original_id_column == target or {original_id_column, id_column, target} & set(
        COLUMNS
    ):
        raise ValueError("Original IDs and target must not be feature columns.")
    raw = path.read_bytes()
    digest = hashlib.sha256(raw).hexdigest()
    if digest != expected_sha256.lower():
        raise ValueError("Original CSV SHA-256 does not match the expected pin.")
    original = pd.read_csv(BytesIO(raw))
    if set(original) != {*COLUMNS, original_id_column, target}:
        raise ValueError(
            "Original schema must contain exactly 24 common features, ID and target."
        )
    order = [name for name in competition if name not in (id_column, target)]
    if (
        len(order) != 24
        or set(order) != set(COLUMNS)
        or set(test) != {*COLUMNS, id_column}
    ):
        raise ValueError(
            "Competition schema must contain the same 24 original features."
        )
    validate_ids(original, original_id_column)
    if original.isna().any().any():
        raise ValueError("Original CSV must not contain missing values.")
    labels = original[target]
    if not pd.api.types.is_numeric_dtype(labels) or set(labels.unique()) != {0, 1}:
        raise ValueError("Original target must contain binary numeric labels 0 and 1.")
    for name in order:
        values = original[name]
        if values.dtype != competition[name].dtype or values.dtype != test[name].dtype:
            raise ValueError(f"Original dtype differs from competition: {name}")
        if name in NUMERIC:
            if (
                not pd.api.types.is_numeric_dtype(values)
                or pd.api.types.is_bool_dtype(values)
                or pd.api.types.is_complex_dtype(values)
                or not np.isfinite(values.to_numpy()).all()
            ):
                raise ValueError(f"Original numeric feature is invalid: {name}")
        elif name in FLAGS:
            if not pd.api.types.is_bool_dtype(values):
                raise ValueError(f"Original flag must be boolean: {name}")
        elif not values.map(lambda value: isinstance(value, str)).all():
            raise ValueError(f"Original nominal feature must contain strings: {name}")
    features = original[order].copy()
    if features.duplicated().any():
        raise ValueError("Original CSV contains duplicate feature rows.")
    hashes = pd.util.hash_pandas_object(features, index=False)
    for name, reference in (("train", competition), ("test", test)):
        reference_features = reference[order]
        candidates = reference_features.loc[
            pd.util.hash_pandas_object(reference_features, index=False).isin(hashes)
        ]
        if (
            not candidates.empty
            and not features.merge(candidates.drop_duplicates(), on=order).empty
        ):
            raise ValueError(f"Original features overlap competition {name} rows.")
    audited = digest == AUDITED_SHA256
    metadata = {
        "source": str(path),
        "sha256": digest,
        "source_url": SOURCE_URL if audited else None,
        "version": 5 if audited else None,
        "license": "CC0-1.0" if audited else None,
        "provenance": (
            "audited public original"
            if audited
            else "explicit local CSV; public provenance not asserted"
        ),
        "rows": len(original),
        "positives": int(labels.sum()),
        "usage": "outer-training only",
        "excluded_columns": [original_id_column],
        "competition_validation_rows_added": 0,
        "competition_oof_rows_added": 0,
        "feature_overlap": "zero exact feature rows against competition train and test",
    }
    verify_unchanged(path, digest)
    return features, labels.to_numpy(dtype=np.int64), metadata


In [ ]:
%%writefile fraudlab/pooled.py
"""Hierarchical decimal/token target rates with honest inner cross-fitting.

Concept: megayak's token pooling, credited in goodpjw2008's Part 12 notebook
(version 16, gboost.py cell). This implementation is independently written;
no third-party code is copied or executed. The hypothesis concerns observed
numeric patterns, not a verified data-generating mechanism.
"""

from __future__ import annotations

from decimal import Decimal

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.validation import check_is_fitted

POOL2 = (
    "amount_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "account_balance_usd",
)
POOL1 = ("velocity_score", "merchant_risk_score")
COLUMNS = POOL2 + POOL1
GROUPS = ((0, 1, 2, 3), (4, 5, 6, 7), (8, 9), (10, 11))
OUTPUT_NAMES = (
    tuple(f"{name}__pooled_decimal_rate" for name in POOL2)
    + tuple(f"{name}__pooled_first_decimal_rate" for name in POOL2)
    + tuple(f"{name}__pooled_decimal_rate" for name in POOL1)
    + tuple(f"{name}__pooled_last_integer_token_rate" for name in POOL1)
    + tuple(f"{name}__pooled_markov_rate" for name in POOL2)
)


def canonical_decimal(value: float) -> str:
    """Represent a finite nonnegative float64 without exponent or new rounding."""
    if np.iscomplexobj(value):
        raise ValueError("Decimal token inputs must be real numbers.")
    number = float(value)
    if not np.isfinite(number) or number < 0:
        raise ValueError("Decimal token inputs must be finite and nonnegative.")
    text = format(Decimal(str(number if number else 0.0)), "f")
    return text if "." in text else text + ".0"


def _keys(frame: pd.DataFrame) -> list[np.ndarray]:
    from .bpe import _encoding

    decimals, first, last, pairs = {}, {}, {}, {}
    tokenizer = _encoding()
    integer_tokens = {}
    for name in COLUMNS:
        series = frame[name]
        if (
            not pd.api.types.is_numeric_dtype(series)
            or pd.api.types.is_bool_dtype(series)
            or pd.api.types.is_complex_dtype(series)
        ):
            raise ValueError(f"{name} requires numeric decimal token inputs.")
        values = series.to_numpy(dtype=np.float64)
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError(f"{name} requires finite nonnegative values.")
        unique, inverse = np.unique(values, return_inverse=True)
        parts = [canonical_decimal(value).split(".", 1) for value in unique]
        integer = [part[0] for part in parts]
        for text in set(integer) - integer_tokens.keys():
            integer_tokens[text] = tokenizer.encode_ordinary(" " + text)[-1]
        dec = [part[1] for part in parts]
        token = [integer_tokens[text] for text in integer]
        decimals[name] = np.asarray(dec, dtype=object)[inverse]
        first[name] = np.asarray([text[0] for text in dec], dtype=object)[inverse]
        last[name] = np.asarray(token, dtype=np.int32)[inverse]
        if name in POOL2:
            pairs[name] = np.asarray(
                [f"{key}:{text}" for key, text in zip(token, dec)], dtype=object
            )[inverse]
    return (
        [decimals[name] for name in POOL2]
        + [first[name] for name in POOL2]
        + [decimals[name] for name in POOL1]
        + [last[name] for name in POOL1]
        + [pairs[name] for name in POOL2]
    )


def manifest(random_state: int = 42) -> dict:
    from .bpe import ENCODING, ENCODING_FILES, TOKENIZER_VERSION

    return {
        "algorithm": "hierarchical_pooled_decimal_token_rates",
        "input_columns": list(COLUMNS),
        "output_features": list(OUTPUT_NAMES),
        "output_count": 16,
        "groups": [
            {"fields": list(POOL2), "key": "complete decimal text"},
            {"fields": list(POOL2), "key": "first decimal digit"},
            {"fields": list(POOL1), "key": "complete decimal text"},
            {"fields": list(POOL1), "key": "last GPT-2 integer token ID"},
        ],
        "markov_fields": list(POOL2),
        "markov_key": "last GPT-2 integer token ID + complete decimal text",
        "smoothing": {
            "pooled_to_prior": 50.0,
            "field_to_pool": 2000.0,
            "markov_to_field_decimal": 200.0,
        },
        "crossfit": {
            "n_splits": 5,
            "shuffle": True,
            "random_state": random_state,
            "seed_scope": "model seed; CV runner passes run seed + outer fold index",
            "prior": "inner-training mean only",
            "inference": "outer-training mappings only",
        },
        "canonical_decimal": "Decimal(str(float64)), fixed notation, at least .0; normalize negative zero; no cent rounding or decimal padding",
        "integer_token": "last token of one ASCII space + canonical integer part",
        "tokenizer": {
            "tiktoken": TOKENIZER_VERSION,
            "encoding": ENCODING,
            "encoding_files": list(ENCODING_FILES),
        },
        "pipeline": "rates -> clipped log_odds -> outer-training StandardScaler",
        "concept_sources": [
            {
                "author": "megayak",
                "url": "https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack",
                "scope": "token pooling concept credited as v7 in the Part 12 source",
            },
            {
                "author": "goodpjw2008",
                "url": "https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298",
                "version": 16,
                "cell": 74,
                "source_body_sha256": "835772526d7efcc70f771aa446690cb1956676a457f40b08d48760bbf3336baf",
            },
        ],
        "implementation": "independently written; concept credit only, no source code copied or executed",
        "interpretation": "numeric-pattern hypothesis; generator not verified",
    }


class PooledTokenEncoder(TransformerMixin, BaseEstimator):
    """Pool target rates across fields, then shrink field and conditional rates.

    fit_transform returns inner-held-out rates and retains full-training maps
    only for later transform calls. IDs and any extra columns are ignored.
    """

    def __init__(
        self,
        *,
        n_splits=5,
        random_state=42,
        pool_smoothing=50.0,
        field_smoothing=2000.0,
        markov_smoothing=200.0,
    ):
        self.n_splits = n_splits
        self.random_state = random_state
        self.pool_smoothing = pool_smoothing
        self.field_smoothing = field_smoothing
        self.markov_smoothing = markov_smoothing

    def _validate(self, frame, y=None):
        if not isinstance(frame, pd.DataFrame) or not frame.columns.is_unique:
            raise ValueError("PooledTokenEncoder requires unique DataFrame columns.")
        missing = set(COLUMNS) - set(frame)
        if missing:
            raise ValueError(f"Missing pooled-token columns: {sorted(missing)}")
        if y is None:
            return None
        labels = np.asarray(y, dtype=np.float64)
        if (
            labels.shape != (len(frame),)
            or not len(labels)
            or not np.isin(labels, [0, 1]).all()
        ):
            raise ValueError("PooledTokenEncoder requires one binary label per row.")
        if (
            not isinstance(self.n_splits, int)
            or isinstance(self.n_splits, bool)
            or self.n_splits < 2
        ):
            raise ValueError("n_splits must be an integer of at least two.")
        for value in (self.pool_smoothing, self.field_smoothing, self.markov_smoothing):
            if not np.isfinite(value) or value <= 0:
                raise ValueError("Pooled-token smoothing values must be positive.")
        return labels

    def _learn(self, keys, labels, rows):
        prior = float(labels[rows].mean())
        fields, pools, markov = [], [], []
        for group in GROUPS:
            tables = [
                pd.DataFrame({"key": keys[column][rows], "y": labels[rows]})
                .groupby("key", sort=False)["y"]
                .agg(["sum", "count"])
                for column in group
            ]
            pooled = pd.concat(tables).groupby(level=0, sort=False).sum()
            rate = (pooled["sum"] + self.pool_smoothing * prior) / (
                pooled["count"] + self.pool_smoothing
            )
            pools.append(rate)
            for table in tables:
                fields.append(
                    (table["sum"] + self.field_smoothing * rate.reindex(table.index))
                    / (table["count"] + self.field_smoothing)
                )
        for column in range(12, 16):
            markov.append(
                pd.DataFrame({"key": keys[column][rows], "y": labels[rows]})
                .groupby("key", sort=False)["y"]
                .agg(["sum", "count"])
            )
        return {"prior": prior, "pools": pools, "fields": fields, "markov": markov}

    def _encode(self, keys, maps, rows):
        result = np.empty((len(rows), 16), dtype=np.float64)
        for index, group in enumerate(GROUPS):
            for column in group:
                query = pd.Series(keys[column][rows], copy=False)
                fallback = query.map(maps["pools"][index]).fillna(maps["prior"])
                result[:, column] = query.map(maps["fields"][column]).fillna(fallback)
        for index in range(4):
            counts = maps["markov"][index].reindex(keys[index + 12][rows], fill_value=0)
            rate = (
                counts["sum"].to_numpy() + self.markov_smoothing * result[:, index]
            ) / (counts["count"].to_numpy() + self.markov_smoothing)
            result[:, index + 12] = np.where(
                counts["count"].to_numpy() == 0, result[:, index], rate
            )
        return result

    def _retain(self, maps):
        self.maps_ = maps
        self.feature_names_in_ = np.asarray(COLUMNS, dtype=object)
        self.n_features_in_ = len(COLUMNS)

    def fit(self, frame, y):
        labels = self._validate(frame, y)
        keys = _keys(frame)
        self._retain(self._learn(keys, labels, np.arange(len(frame))))
        return self

    def fit_transform(self, frame, y, **fit_params):
        labels = self._validate(frame, y)
        if (
            min(np.count_nonzero(labels == 0), np.count_nonzero(labels == 1))
            < self.n_splits
        ):
            raise ValueError("Cross-fitting requires at least n_splits rows per class.")
        keys = _keys(frame)
        result = np.empty((len(frame), 16), dtype=np.float64)
        splitter = StratifiedKFold(
            n_splits=self.n_splits, shuffle=True, random_state=self.random_state
        )
        for training, validation in splitter.split(np.zeros(len(frame)), labels):
            maps = self._learn(keys, labels, training)
            result[validation] = self._encode(keys, maps, validation)
        self._retain(self._learn(keys, labels, np.arange(len(frame))))
        return result

    def transform(self, frame):
        check_is_fitted(self, "maps_")
        self._validate(frame)
        return self._encode(_keys(frame), self.maps_, np.arange(len(frame)))

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "maps_")
        if input_features is not None and list(input_features) != list(COLUMNS):
            raise ValueError(
                "Pooled-token feature names must match the six input columns."
            )
        return np.asarray(OUTPUT_NAMES, dtype=object)


In [ ]:
%%writefile fraudlab/refit.py
"""Refit a frozen choice once on all labeled rows, without another selection loop."""

from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from .data import fingerprint, load_data, submission, validate_submission, write_json
from .experiment import create_run, source_provenance, versions
from .features import transform_features
from .models import feature_types, fit_predict


def refit(source: Path, holdout_report: Path, output: Path, name: str) -> dict:
    started = time.perf_counter()
    config = json.loads((source / "config.json").read_text(encoding="utf-8"))
    if config.get("original_data"):
        raise ValueError("Original-data augmentation has no full-data refit contract.")
    if config.get("glm_transductive_frequency"):
        raise ValueError("Transductive frequency has no full-data refit contract.")
    if config["model"] == "neural_offset":
        raise ValueError(
            "Neural offset uses the CV checkpoint ensemble; a full-data refit protocol is not defined."
        )
    if config["model"] == "tabm":
        raise ValueError(
            "TabM uses the CV checkpoint ensemble; a full-data refit protocol is not defined."
        )
    provenance = source_provenance(config.get("feature_set"))
    selected = json.loads((source / "metrics.json").read_text(encoding="utf-8"))
    holdout = json.loads(holdout_report.read_text(encoding="utf-8"))
    if (
        selected.get("status") != "complete"
        or holdout.get("kind") != "independent_holdout"
        or holdout.get("status") != "complete"
    ):
        raise ValueError(
            "A completed development run and frozen independent holdout evaluation are required."
        )
    sources = [Path(path).resolve() for path in holdout["selection"]["sources"]]
    if source.resolve() not in sources:
        raise ValueError("This model was not part of the frozen holdout selection.")
    index = sources.index(source.resolve())
    if holdout["selection"]["weights"][index] <= 0:
        raise ValueError("A model with zero frozen weight does not need a final refit.")
    if (
        fingerprint(source / "metrics.json")
        != holdout["selection"]["metrics_hashes"][index]
        or fingerprint(source / "config.json")
        != holdout["selection"]["config_hashes"][index]
    ):
        raise ValueError(
            "Development metrics or configuration changed after the selection was frozen."
        )
    data_dir = Path(config["data"])
    if any(
        fingerprint(data_dir / filename) != expected
        for filename, expected in selected["data_hashes"].items()
    ):
        raise ValueError("Input data changed after development training.")
    folds = selected["model_details"]["folds"]
    if config["model"] == "mlp":
        config["epochs"] = int(
            round(float(np.median([fold["best_epoch"] for fold in folds])))
        )
    elif config["model"] in ("catboost", "lightgbm"):
        config["iterations"] = int(
            round(
                float(
                    np.median(
                        [
                            (
                                fold["trees_used"]
                                if config["model"] == "catboost"
                                else fold["best_iteration"]
                            )
                            for fold in folds
                        ]
                    )
                )
            )
        )
    config.update(
        refit=True,
        holdout_fraction=0.0,
        name=name,
        output=str(output),
        selection_source=str(source),
        holdout_report=str(holdout_report),
    )
    frame, test, sample = load_data(data_dir, config["id_column"], config["target"])
    target, id_column = config["target"], config["id_column"]
    frame = transform_features(
        frame, config.get("feature_set", "raw"), (id_column, target)
    )
    test = transform_features(
        test, config.get("feature_set", "raw"), (id_column, target)
    )
    features = selected["features"]["names"]
    run = create_run(output, name)
    write_json(run / "config.json", config)
    write_json(run / "status.json", {"status": "running"})
    try:
        _, predictions, details = fit_predict(
            config["model"],
            frame[features],
            frame[target].to_numpy(),
            test[features],
            None,
            test[features],
            config,
            config["seed"],
            run / "model",
        )
        result = submission(test[id_column], predictions, sample, target)
        result.to_csv(run / "predictions.csv", index=False)
        result.to_csv(run / "submission.csv", index=False)
        numeric, categorical = feature_types(frame[features])
        metrics = {
            "status": "complete",
            "model": config["model"],
            "phase": "full_data_refit",
            "roc_auc": None,
            "fold_scores": [],
            "fold_std": None,
            "rows": {"train": len(frame), "test": len(test), "oof": 0},
            "features": {
                "count": len(features),
                "numeric": len(numeric),
                "categorical": len(categorical),
                "names": features,
                "excluded": selected["features"]["excluded"],
            },
            "target": {"name": target, "positive_rate": float(frame[target].mean())},
            "seed": config["seed"],
            "validation": {
                "method": "no_new_selection_full_data_refit",
                "n_splits": 0,
                "early_stopping_uses_validation": False,
            },
            "selection": {
                "source_dev_run": str(source),
                "development_roc_auc": selected["roc_auc"],
                "holdout_report_sha256": fingerprint(holdout_report),
                "frozen_iterations_rule": "median best epoch/tree count across development folds; model hyperparameters unchanged",
            },
            "elapsed_seconds": round(time.perf_counter() - started, 3),
            "versions": versions(),
            "data_hashes": selected["data_hashes"],
            **provenance,
            "model_details": {
                **details,
                "feature_set": config.get("feature_set", "raw"),
            },
            "created_at": datetime.now(timezone.utc).isoformat(),
            "submission_sha256": fingerprint(run / "submission.csv"),
            "limitations": [
                "All labels were used to refit a previously frozen choice. No new independent score was measured for this final model.",
                "Wall-clock training times from concurrent runs are not controlled performance benchmarks.",
            ],
        }
        write_json(run / "metrics.json", metrics)
        write_json(run / "status.json", {"status": "complete"})
        return metrics
    except Exception as error:
        write_json(run / "status.json", {"status": "failed", "error": str(error)})
        raise


def blend_refits(
    runs: list[Path],
    weights: list[float] | None,
    output: Path,
    name: str,
    id_column: str,
    target: str,
) -> dict:
    """Combine final predictions with exactly the pre-holdout frozen weights."""
    started = time.perf_counter()
    provenance = source_provenance()
    configs = [
        json.loads((run / "config.json").read_text(encoding="utf-8")) for run in runs
    ]
    metrics = [
        json.loads((run / "metrics.json").read_text(encoding="utf-8")) for run in runs
    ]
    report_path = Path(configs[0]["holdout_report"])
    report = json.loads(report_path.read_text(encoding="utf-8"))
    report_hash = fingerprint(report_path)
    frozen = {
        Path(source).resolve(): weight
        for source, weight in zip(
            report["selection"]["sources"], report["selection"]["weights"]
        )
        if weight > 0
    }
    source_paths = [Path(config["selection_source"]).resolve() for config in configs]
    if set(source_paths) != set(frozen) or len(set(source_paths)) != len(source_paths):
        raise ValueError(
            "Final blend must contain every nonzero frozen source exactly once."
        )
    coefficients = np.array([frozen[source] for source in source_paths], dtype=float)
    if weights is not None:
        requested = np.array(weights, dtype=float)
        if len(requested) != len(coefficients) or not np.allclose(
            requested / requested.sum(), coefficients, rtol=0, atol=1e-12
        ):
            raise ValueError(
                "Final blend weights cannot change after independent holdout evaluation."
            )
    for config, metric in zip(configs, metrics):
        if (
            metric.get("status") != "complete"
            or metric.get("phase") != "full_data_refit"
            or metric["data_hashes"] != metrics[0]["data_hashes"]
            or metric["selection"]["holdout_report_sha256"] != report_hash
            or fingerprint(Path(config["holdout_report"])) != report_hash
        ):
            raise ValueError(
                "Final sources must be completed refits of the same frozen selection and input data."
            )
    predictions = [pd.read_csv(run / "predictions.csv") for run in runs]
    result = predictions[0].copy()
    for frame in predictions:
        validate_submission(frame, result, id_column, target)
    result[target] = sum(
        weight * frame[target].to_numpy()
        for weight, frame in zip(coefficients, predictions)
    )
    validate_submission(result, predictions[0], id_column, target)
    run = create_run(output, name)
    result.to_csv(run / "predictions.csv", index=False)
    result.to_csv(run / "submission.csv", index=False)
    details = {
        "method": "weighted_probability_average",
        "sources": [str(source) for source in runs],
        "weights": coefficients.tolist(),
    }
    metadata = {
        **metrics[0],
        **provenance,
        "model": "blend",
        "phase": "full_data_refit",
        "roc_auc": None,
        "fold_scores": [],
        "fold_std": None,
        "model_details": details,
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "submission_sha256": fingerprint(run / "submission.csv"),
        "selection": {
            "source_dev_run": report["selection"].get("source_dev_run"),
            "development_roc_auc": report["selection"].get("development_roc_auc"),
            "development_metrics_sha256": report["selection"].get(
                "development_metrics_sha256"
            ),
            "holdout_report_sha256": report_hash,
            "weights_frozen_before_holdout": True,
        },
    }
    write_json(run / "config.json", details)
    write_json(run / "metrics.json", metadata)
    write_json(run / "status.json", {"status": "complete"})
    return metadata


In [ ]:
%%writefile fraudlab/scores.py
"""Pure weighted score combinations; ranking never reads target labels."""

from __future__ import annotations

import numpy as np
from scipy.stats import rankdata

SPACES = ("probability", "logit", "rank", "rank_logit")
LOGIT_CLIP = 1e-6


def blend_scores(columns, weights=None, *, space="probability") -> np.ndarray:
    """Blend whole aligned vectors; rank OOF and test in separate calls."""
    if space not in SPACES:
        raise ValueError(f"Unknown score space: {space}")
    values = [np.asarray(column, dtype=np.float64) for column in columns]
    if not values or any(
        column.ndim != 1
        or not len(column)
        or column.shape != values[0].shape
        or not np.isfinite(column).all()
        or ((column < 0) | (column > 1)).any()
        for column in values
    ):
        raise ValueError("Scores must be aligned nonempty finite vectors in [0, 1].")
    coefficients = np.asarray(
        weights if weights is not None else [1.0] * len(values), dtype=np.float64
    ).copy()
    if (
        coefficients.shape != (len(values),)
        or not np.isfinite(coefficients).all()
        or (coefficients < 0).any()
        or not np.isfinite(coefficients.sum())
        or coefficients.sum() <= 0
    ):
        raise ValueError("Provide finite nonnegative weights with a positive total.")
    coefficients /= coefficients.sum()
    if space in ("rank", "rank_logit"):
        values = [
            rankdata(column, method="average") / (len(column) + 1) for column in values
        ]
    if space in ("logit", "rank_logit"):
        values = [np.clip(column, LOGIT_CLIP, 1 - LOGIT_CLIP) for column in values]
        values = [np.log(column / (1 - column)) for column in values]
    result = sum(weight * column for weight, column in zip(coefficients, values))
    return (
        np.exp(-np.logaddexp(0.0, -result))
        if space in ("logit", "rank_logit")
        else result
    )


In [ ]:
%%writefile fraudlab/splits.py
"""Explicit split strategies shared by every model."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold,
    TimeSeriesSplit,
)


def make_splits(
    frame: pd.DataFrame,
    target: str,
    method: str,
    folds: int,
    seed: int,
    group_column: str | None = None,
    time_column: str | None = None,
) -> list[tuple[np.ndarray, np.ndarray]]:
    if folds < 2:
        raise ValueError("At least two folds are required.")
    y = frame[target]
    if method == "stratified":
        if y.value_counts().min() < folds:
            raise ValueError("Every class needs at least one row per fold.")
        splits = list(
            StratifiedKFold(folds, shuffle=True, random_state=seed).split(frame, y)
        )
    elif method == "group":
        if not group_column or group_column not in frame:
            raise ValueError("Group validation requires an existing --group-column.")
        groups = frame[group_column]
        if groups.isna().any() or groups.nunique() < folds:
            raise ValueError(
                "Group column must be non-null and contain at least one group per fold."
            )
        splits = list(
            StratifiedGroupKFold(folds, shuffle=True, random_state=seed).split(
                frame, y, groups
            )
        )
    elif method == "time":
        if not time_column or time_column not in frame:
            raise ValueError("Time validation requires an existing --time-column.")
        values = frame[time_column]
        if not pd.api.types.is_numeric_dtype(values):
            values = pd.to_datetime(values, errors="raise", utc=True)
        if values.isna().any():
            raise ValueError("Time column must not contain missing values.")
        # Split unique timestamps, so simultaneous events never cross a boundary.
        times = np.sort(values.unique())
        if len(times) <= folds:
            raise ValueError("Time validation needs more unique timestamps than folds.")
        splits = [
            (
                np.flatnonzero(values.isin(times[a])),
                np.flatnonzero(values.isin(times[b])),
            )
            for a, b in TimeSeriesSplit(folds).split(times)
        ]
    else:
        raise ValueError(f"Unknown validation method: {method}")
    seen: set[int] = set()
    for training, validation in splits:
        if set(training) & set(validation) or seen & set(validation):
            raise ValueError(
                "Training and validation rows must be disjoint; each OOF row is predicted once."
            )
        seen.update(validation)
        if y.iloc[training].nunique() != 2 or y.iloc[validation].nunique() != 2:
            raise ValueError(
                "Every training and validation fold needs both classes for ROC-AUC."
            )
    return splits


In [ ]:
%%writefile fraudlab/strings.py
"""Object string arrays whose repeated values reference the same Python string."""

import numpy as np


def categorical_strings(values: np.ndarray) -> np.ndarray:
    """Preserve integer/Unicode text while converting only distinct values.

    This is storage sharing, not a learned vocabulary or pandas categorical dtype.
    Float/object inputs are excluded because their formatting is not interchangeable
    with pandas' existing raw-value conversion.
    """
    values = np.asarray(values)
    if values.ndim != 1 or values.dtype.kind not in "iuU":
        raise ValueError(
            "Shared categorical strings require integer or Unicode arrays."
        )
    unique, inverse = np.unique(values, return_inverse=True)
    return unique.astype(str).astype(object)[inverse]


In [ ]:
%%writefile fraudlab/tabm.py
"""TabM with inner epoch selection and fresh outer-training refits.

The outer validation labels are deliberately absent from this module's API.
Architecture: https://github.com/yandex-research/tabm (Apache-2.0).
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from .models import NeuralPreprocessor, clean_features
from rtdl_num_embeddings import PiecewiseLinearEmbeddings, compute_bins
from tabm import TabM


@dataclass(frozen=True)
class TabMConfig:
    n_blocks: int = 2
    d_block: int = 256
    k: int = 16
    dropout: float = 0.1
    d_embedding: int = 12
    n_bins: int = 48
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 1024
    epochs: int = 40
    patience: int = 6
    gradient_clip: float = 1.0
    inner_fraction: float = 0.15
    inner_seed: int = 42
    numeric_transform: str = "standard"
    target_smoothing: float = 20.0

    @classmethod
    def from_training_config(cls, config: dict):
        if config.get("fixed_training") or config.get("refit"):
            raise ValueError(
                "TabM requires inner epoch selection; fixed-training/full-data refit is not defined."
            )
        if config.get("neural_categorical_transform", "target") != "target":
            raise ValueError("TabM requires --neural-categorical-transform target.")
        if config.get("hidden_dims", [256, 256]) != [256, 256]:
            raise ValueError("TabM architecture is fixed: --hidden-dims 256 256.")
        return cls(
            dropout=config["dropout"],
            learning_rate=config["neural_learning_rate"],
            weight_decay=config["weight_decay"],
            batch_size=config["batch_size"],
            epochs=config["epochs"],
            patience=config["patience"],
            numeric_transform=config.get("neural_numeric_transform", "standard"),
            target_smoothing=config["target_smoothing"],
        )


def build_tabm(bins: list[torch.Tensor], config: TabMConfig) -> TabM:
    return TabM.make(
        n_num_features=len(bins),
        d_out=1,
        num_embeddings=PiecewiseLinearEmbeddings(
            bins, config.d_embedding, activation=False, version="B"
        ),
        arch_type="tabm",
        n_blocks=config.n_blocks,
        d_block=config.d_block,
        k=config.k,
        dropout=config.dropout,
    )


def member_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """One BCE per row AND member. Never BCE on the average prediction."""
    if logits.ndim != 3 or logits.shape[-1] != 1:
        raise ValueError("Expected [batch, k, 1] logits.")
    return torch.nn.functional.binary_cross_entropy_with_logits(
        logits[..., 0], y[:, None].expand_as(logits[..., 0])
    )


def mean_probability(logits: torch.Tensor) -> torch.Tensor:
    return logits[..., 0].sigmoid().mean(dim=1)


def prepare_training(frame: pd.DataFrame, y: np.ndarray, seed: int, config: TabMConfig):
    preprocessing = NeuralPreprocessor(
        numeric_transform=config.numeric_transform,
        seed=seed,
        categorical_transform="target",
        smoothing=config.target_smoothing,
    )
    numbers, categories = preprocessing.fit_transform(frame, y)
    assert categories.shape == (len(frame), 0)
    # The official binning API rejects constants. Select using training only;
    # the same fixed mask is applied to every later partition and checkpoint.
    active = np.any(numbers != numbers[0], axis=0)
    if not active.any():
        raise ValueError("TabM needs at least one nonconstant training feature.")
    numbers = np.ascontiguousarray(numbers[:, active])
    bins = compute_bins(torch.from_numpy(numbers), n_bins=config.n_bins)
    return preprocessing, active, numbers, bins


@torch.no_grad()
def predict_array(model, numbers: torch.Tensor, batch_size: int) -> np.ndarray:
    model.eval()
    return np.concatenate(
        [
            mean_probability(model(batch)).cpu().numpy()
            for batch in numbers.split(batch_size)
        ]
    )


def _train(numbers, y, bins, config, seed, device, *, validation=None, epochs=None):
    """Select E on inner validation, or train exactly E from a new seed state."""
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    model = build_tabm(bins, config).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    x = torch.as_tensor(numbers, device=device)
    labels = torch.as_tensor(y, dtype=torch.float32, device=device)
    best_auc, best_epoch, stale, history = -np.inf, 0, 0, []
    for epoch in range(1, (epochs or config.epochs) + 1):
        model.train()
        for selected in torch.randperm(len(y), device=device).split(config.batch_size):
            optimizer.zero_grad(set_to_none=True)
            loss = member_loss(model(x[selected]), labels[selected])
            if not torch.isfinite(loss):
                raise ValueError("TabM loss became nonfinite.")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
            optimizer.step()
        if validation is None:
            best_epoch = epoch
            continue
        vx, vy = validation
        score = roc_auc_score(vy, predict_array(model, vx, config.batch_size))
        history.append({"epoch": epoch, "inner_auc": float(score)})
        print(f"tabm seed={seed} inner_epoch={epoch} roc_auc={score:.8f}", flush=True)
        if score > best_auc + 1e-6:
            best_auc, best_epoch, stale = score, epoch, 0
        else:
            stale += 1
        if stale >= config.patience:
            break
    return model, best_epoch, history, epoch


def fit_tabm(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
):
    """Outer validation is inference-only: its labels cannot enter this API."""
    settings = TabMConfig.from_training_config(config)
    torch.set_num_threads(config["threads"])
    if config["device"] == "cuda" and not torch.cuda.is_available():
        raise ValueError(
            "CUDA was requested but is not available in this PyTorch installation."
        )
    device = torch.device(
        "cuda" if config["device"] != "cpu" and torch.cuda.is_available() else "cpu"
    )
    inner_train, inner_valid = train_test_split(
        np.arange(len(y)),
        test_size=settings.inner_fraction,
        random_state=settings.inner_seed,
        stratify=y,
    )
    inner_pre, inner_active, inner_x, inner_bins = prepare_training(
        training.iloc[inner_train], y[inner_train], seed, settings
    )
    inner_vx, _ = inner_pre.transform(training.iloc[inner_valid])
    selection_model, selected_epoch, history, selection_epochs_completed = _train(
        inner_x,
        y[inner_train],
        inner_bins,
        settings,
        seed,
        device,
        validation=(
            torch.as_tensor(inner_vx[:, inner_active], device=device),
            y[inner_valid],
        ),
    )
    # Discard ALL inner preprocessing, bins, weights and optimizer state.
    del selection_model, inner_pre, inner_active, inner_x, inner_bins, inner_vx
    print(f"tabm seed={seed} fresh_outer_refit_epochs={selected_epoch}", flush=True)
    preprocessing, active, outer_x, bins = prepare_training(training, y, seed, settings)
    model, _, _, refit_epochs_completed = _train(
        outer_x, y, bins, settings, seed, device, epochs=selected_epoch
    )
    predictions = []
    for frame in (validation, test):
        numbers, _ = preprocessing.transform(frame)
        predictions.append(
            predict_array(
                model,
                torch.as_tensor(numbers[:, active], device=device),
                settings.batch_size,
            )
        )
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {"preprocessing": preprocessing, "active": active},
        path.with_suffix(".preprocessing.joblib"),
    )
    torch.save(
        {
            "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "bins": [b.cpu() for b in bins],
            "config": asdict(settings),
            "selected_epoch": selected_epoch,
        },
        path.with_suffix(".pt"),
    )
    return *predictions, {
        "architecture": {
            "name": "TabM",
            "n_blocks": settings.n_blocks,
            "d_block": settings.d_block,
            "k": settings.k,
            "dropout": settings.dropout,
            "d_out": 1,
            "num_embeddings": {
                "name": "PiecewiseLinearEmbeddings",
                "version": "B",
                "activation": False,
                "d_embedding": settings.d_embedding,
                "n_bins": settings.n_bins,
            },
            "parameters": sum(parameter.numel() for parameter in model.parameters()),
        },
        "config": asdict(settings),
        "selected_epoch": selected_epoch,
        "selection_epochs_completed": selection_epochs_completed,
        "refit_epochs_completed": refit_epochs_completed,
        "inner_history": history,
        "inner_training_rows": len(inner_train),
        "inner_validation_rows": len(inner_valid),
        "refit_rows": len(training),
        "input_features": len(active),
        "active_features": int(active.sum()),
        "active_columns": np.flatnonzero(active).tolist(),
        "numeric_features": preprocessing.numeric,
        "categorical_features": preprocessing.categorical,
        "bin_edges_per_active_feature": [len(b) for b in bins],
        "optimizer": "AdamW",
        "loss": "mean BCEWithLogits over rows and members",
        "inference": "mean sigmoid probability over members",
        "learning_rate": settings.learning_rate,
        "weight_decay": settings.weight_decay,
        "batch_size": settings.batch_size,
        "device": str(device),
        "seed": seed,
        "threads": torch.get_num_threads(),
        "target_encoder": {
            "internal_folds": 5,
            "smooth": settings.target_smoothing,
            "log_odds": True,
            "training_values": "crossfit",
        },
        "numeric_transform": settings.numeric_transform,
        "bins_fit": "training only per stage; quantiles, no labels",
        "epoch_selection": "inner 15% AUC; fresh full outer-training refit",
        "outer_validation_usage": "inference only; no labels accepted",
    }


def predict_saved_tabm(path: Path, frame: pd.DataFrame, device: str = "cpu"):
    if device == "cuda" and not torch.cuda.is_available():
        raise ValueError(
            "CUDA was requested but is not available in this PyTorch installation."
        )
    device = "cuda" if device != "cpu" and torch.cuda.is_available() else "cpu"
    payload = torch.load(path.with_suffix(".pt"), map_location="cpu", weights_only=True)
    config = TabMConfig(**payload["config"])
    model = build_tabm(payload["bins"], config).to(device)
    model.load_state_dict(payload["state_dict"])
    saved = joblib.load(path.with_suffix(".preprocessing.joblib"))
    preprocessing = saved["preprocessing"]
    numbers, _ = preprocessing.transform(
        clean_features(frame, preprocessing.numeric, preprocessing.categorical)
    )
    return predict_array(
        model,
        torch.as_tensor(numbers[:, saved["active"]], device=device),
        config.batch_size,
    )


In [ ]:
%%writefile fraudlab/tokens.py
"""Complete-number GPT-2 keys and integer-prefix views, without labels.

Inspired by megayak's numeric-token views and goodpjw2008's Part 12 notebook
(version 16). Independently implemented; no notebook code or predictions are
copied. Canonical float64 text intentionally differs from original CSV lexemes.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from .bpe import ENCODING, ENCODING_FILES, TOKENIZER_VERSION, _encoding
from .pooled import COLUMNS, canonical_decimal

SUFFIXES = (
    "__fulltoken_first_key",
    "__fulltoken_second_key",
    "__fulltoken_last_key",
    "__fulltoken_length_key",
    "__integer_tens_key",
    "__integer_length_first_key",
    "__integer_length_two_key",
)


def token_keys(frame: pd.DataFrame, excluded: tuple[str, ...] = ()) -> pd.DataFrame:
    """Return seven categorical keys per allowed continuous column.

    Inputs must also fit the exact-cent range inherited by value_digits. The
    complete decimal text retains fractional cents; there is no cent rounding.
    A missing second token is -1. Prefixes come from floor(x), with zero length1.
    """
    result = pd.DataFrame(index=frame.index)
    encoding = _encoding()
    for name in COLUMNS:
        if name in excluded:
            continue
        if name not in frame:
            raise ValueError(f"Missing full-token input column: {name}")
        series = frame[name]
        if (
            not pd.api.types.is_numeric_dtype(series)
            or pd.api.types.is_bool_dtype(series)
            or pd.api.types.is_complex_dtype(series)
        ):
            raise ValueError(f"{name} requires numeric full-token inputs.")
        values = series.to_numpy(dtype=np.float64)
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError(f"{name} requires finite nonnegative values.")
        if (values > (2**53 - 1) / 100).any():
            raise ValueError(
                f"{name} exceeds the exact-cent range inherited from value_digits."
            )
        unique, inverse = np.unique(values, return_inverse=True)
        keys = []
        for value in unique:
            text = canonical_decimal(value)
            pieces = encoding.encode_ordinary(" " + text)
            integer = text.split(".", 1)[0]
            keys.append(
                (
                    str(pieces[0]),
                    str(pieces[1] if len(pieces) > 1 else -1),
                    str(pieces[-1]),
                    str(len(pieces)),
                    str(int(integer) // 10),
                    f"{len(integer)}:{integer[:1]}",
                    f"{len(integer)}:{integer[:2]}",
                )
            )
        columns = np.asarray(keys, dtype=object).reshape(-1, len(SUFFIXES))[inverse]
        for index, suffix in enumerate(SUFFIXES):
            result[name + suffix] = columns[:, index]
    return result


def manifest() -> dict:
    encoding = _encoding()
    return {
        "algorithm": "complete_numeric_token_and_integer_prefix_keys",
        "tiktoken": TOKENIZER_VERSION,
        "encoding": ENCODING,
        "vocabulary_size": encoding.max_token_value + 1,
        "encoding_files": list(ENCODING_FILES),
        "input_columns": list(COLUMNS),
        "output_suffixes": list(SUFFIXES),
        "output_count": len(COLUMNS) * len(SUFFIXES),
        "base_feature_set": "value_digits",
        "canonical_text": "float64 -> Decimal(str(float(value))) -> fixed decimal notation, at least .0; one leading ASCII space",
        "zero": "-0.0 normalized to 0.0",
        "fractional_cents": "retained; no cent rounding or decimal padding",
        "csv_lexemes": "not preserved; trailing zeros and exponent spelling may differ",
        "second_token_missing": "-1",
        "integer_keys": "floor(x/10), integer-digit-length:first-digit, integer-digit-length:first-two-digits; floor(0) has length1",
        "numeric_range": "finite, nonnegative, at most (2**53-1)/100 to support inherited exact-cent views",
        "category_semantics": "all seven keys are categorical labels, not ordered token magnitudes",
        "label_usage": "none; only the six declared input columns are read",
        "inspiration": [
            "https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack",
            "https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298",
        ],
    }


In [ ]:
%%writefile fraudlab/transductive.py
"""Label-free categorical counts over official training and test features."""

from __future__ import annotations

import hashlib
import json

import pandas as pd

from .models import clean_features, feature_types


def validate_config(config: dict) -> None:
    if not config.get("glm_transductive_frequency"):
        return
    if (
        not config.get("glm_frequency")
        or config.get("model") not in ("glm", "offset")
        or config.get("phase") != "exploratory_full_data_cv"
        or config.get("split") != "stratified"
        or config.get("folds") != 5
        or config.get("seed") != 42
        or config.get("holdout_fraction", 0)
        or config.get("original_data")
        or config.get("refit")
    ):
        raise ValueError(
            "Transductive frequency requires GLM/offset, glm_frequency and exploratory CV5/42 without original data, holdout or refit."
        )


def counts_sha256(counts: dict[str, pd.Series]) -> str:
    """Hash sorted UTF-8 category/count records with unambiguous JSON framing."""
    digest = hashlib.sha256()
    for name in sorted(counts):
        column = counts[name]
        digest.update((json.dumps([name], ensure_ascii=False) + "\n").encode("utf-8"))
        for key, value in column.sort_index().items():
            digest.update(
                (json.dumps([key, int(value)], ensure_ascii=False) + "\n").encode(
                    "utf-8"
                )
            )
    return digest.hexdigest()


def build_reference_counts(
    training: pd.DataFrame,
    test: pd.DataFrame,
    feature_columns: list[str],
    corpus_hashes: dict,
    excluded: tuple[str, ...] = ("id", "is_fraud"),
) -> tuple[dict[str, pd.Series], dict]:
    """Count one column at a time; never concatenate a million-row wide frame."""
    columns = [name for name in feature_columns if name not in excluded]
    if (
        training.empty
        or test.empty
        or not training.columns.is_unique
        or not test.columns.is_unique
        or len(columns) != len(set(columns))
        or not all(isinstance(name, str) for name in columns)
        or not set(columns).issubset(training)
        or not set(columns).issubset(test)
    ):
        raise ValueError(
            "Transductive counts require nonempty frames and matching feature columns."
        )
    # Zero-row selection preserves exactly feature_types' dtype semantics.
    _, categorical = feature_types(training.iloc[:0][columns])
    if not categorical:
        raise ValueError("Transductive frequency requires categorical features.")
    reference = {}
    for name in categorical:
        left = clean_features(training[[name]], [], [name])[name].value_counts(
            sort=False
        )
        right = clean_features(test[[name]], [], [name])[name].value_counts(sort=False)
        reference[name] = left.add(right, fill_value=0).astype("int64")
    metadata = {
        "scope": "transductive official train + test feature counts",
        "label_usage": "none",
        "rows": {
            "train": len(training),
            "test": len(test),
            "total": len(training) + len(test),
        },
        "corpus_hashes": dict(corpus_hashes),
        "columns": categorical,
        "counts_sha256": counts_sha256(reference),
        "normalization": "clean_features categorical string conversion and __MISSING__",
        "supervised_statistics": "target encoding, pooled target rates and all scalers fit on outer training only",
    }
    return reference, metadata


In [ ]:
import hashlib
import importlib.metadata as package_metadata
import json
import re
import shutil
import subprocess
import sys

SOURCE_SHA256 = {'fraudlab/__init__.py': '14479eff95a0233e8bf2d9f359843c36f53f36b92cc95c3e452f75bfb2982dcc',
 'fraudlab/__main__.py': 'b8fc5d9e4cf77f7319f9e2f7956f490bb71da6ce3c6842e279f46194672a7fae',
 'fraudlab/bpe.py': 'fb71799ca075c657a8344f85d6a7e91653d1f319e24b5e287c17565cea456d4e',
 'fraudlab/data.py': 'f22355cc51fa6e6162b24e639923f40a11a23af00afcb65553090176271c8326',
 'fraudlab/experiment.py': '85a0995157f4b4453206d88a41ee1196765f7bb5cd6cf9dd55d5dd89ba87bb7a',
 'fraudlab/external.py': '05ec4ec64e4a2c5cf1137c949646bed14fe83fb04506b8f2f973c2c7633c5e08',
 'fraudlab/features.py': '184fe327468194f34563b25aee2268348ca854d59e4916f8e3eacdd1b6420424',
 'fraudlab/frequency.py': 'de06b4f581f6864066d94b140d67e4ed16f7cef054a30ef20bc408a972279ce6',
 'fraudlab/holdout.py': 'e3702891766326710c3b56dd7a7fe69f2f778acea3316665f130aa6955e2f4e3',
 'fraudlab/joints.py': '620851854c56aceceba02d5855fe22525e748973e80ffe0f83040f5e75a271e8',
 'fraudlab/models.py': 'e00d858bac42b79362ef408cbb7f359bca44a4828931b0e4943d660a3eb095b5',
 'fraudlab/neural_offset.py': '042eec5ea088477bb37b045ace2f537c6e6e39ff49b3e810dc5e501b4ea8e487',
 'fraudlab/offset.py': 'a4ea551e4ddf89aedc87f313eb07425aebc60f7d967986524a873a73565fdc4f',
 'fraudlab/original.py': '4db9d8d54929ba9d9a36fd55b8f8655611a800d22fade59af079e0497ce7b934',
 'fraudlab/pooled.py': '6f15acb31c8a878efc262410a0ddf3f6b733b1a636a7bfa917758d568164601a',
 'fraudlab/refit.py': '981074be27e2ed24aa45dbaabeda32677f83ae59aa74f12dd6a56de152fde4e9',
 'fraudlab/scores.py': '8bae07482b56eb488a388c54ecd7ec0cf87a8644d9e98e53d7c1fcc75d2e58a1',
 'fraudlab/splits.py': '41215280332984010f5b616bbf6a675c623b7af3e3f00604eb3238e79c709af9',
 'fraudlab/strings.py': 'f3248a9c0e7bbda655e29ecac02886dfa4cd4f34f35007ad90cf99a17efccbb0',
 'fraudlab/tabm.py': 'df22ff1a120c8a6fde59052ea7a1b509ff1b7e7acc83d774f670a1daf4a4e667',
 'fraudlab/tokens.py': '8e3abfcbe2e437eee859d120f2a89b0a45bd84dfe389a06c11b622ee5ac9cbb4',
 'fraudlab/transductive.py': 'b2848df445f94d1135df27b99e7586048c279860b244dcf9544c30687263ca3b'}
for filename, expected in SOURCE_SHA256.items():
    assert hashlib.sha256(Path(filename).read_bytes()).hexdigest() == expected, filename

if shutil.which('git') is None:
    raise RuntimeError('A imagem precisa do executável git para registrar a proveniência; nenhum acesso remoto será feito.')

requirements = {'numpy': ((1, 24), 3), 'pandas': ((2, 0), 3),
                'scikit-learn': ((1, 3), 2), 'lightgbm': ((4, 7), 5)}
versions = {}
for package, (minimum, upper_major) in requirements.items():
    installed = package_metadata.version(package)
    version = tuple(map(int, re.findall(r'\d+', installed)[:2]))
    if version < minimum or version[0] >= upper_major:
        raise RuntimeError(f'{package} {installed} incompatível com esta revisão offline; use uma imagem Kaggle compatível.')
    versions[package] = installed
print(json.dumps(versions, indent=2))


In [ ]:
TOKENIZER_ASSETS = [{'cache_key': '6d1cbeee0f20b3d9449abfede4726ed8212e3aee',
  'gzip_base64': '...',
  'sha256': '1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5',
  'url': 'https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/vocab.bpe'},
 {'cache_key': '6c7ea1a7e38e3a7f062df639a5b80947f075ffe6',
  'gzip_base64': '...',
  'sha256': '196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783',
  'url': 'https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/encoder.json'}]

import base64
import gzip
import os

for package, expected in {'tiktoken': '0.14.0', 'regex': '2026.9.29'}.items():
    installed = package_metadata.version(package)
    if installed != expected:
        raise RuntimeError(f'{package}=={expected} is required offline; found {installed}.')
tokenizer_cache = Path('/kaggle/temp/fraudlab-tokenizer-cache')
tokenizer_cache.mkdir(parents=True, exist_ok=True)
for asset in TOKENIZER_ASSETS:
    body = gzip.decompress(base64.b64decode(asset['gzip_base64']))
    assert hashlib.sha256(body).hexdigest() == asset['sha256']
    assert hashlib.sha1(asset['url'].encode()).hexdigest() == asset['cache_key']
    (tokenizer_cache / asset['cache_key']).write_bytes(body)
os.environ['TIKTOKEN_CACHE_DIR'] = str(tokenizer_cache.resolve())
from fraudlab.bpe import _encoding
encoding = _encoding()
assert encoding.name == 'gpt2' and encoding.max_token_value + 1 == 50257
print({'offline_tokenizer': encoding.name, 'verified_assets': len(TOKENIZER_ASSETS)})


## 2. Inputs e créditos

Adicione a competição S6E11 e o dataset GPT-2 **versão 2** aos inputs. Os dados e arrays são verificados por SHA-256; não há download durante a execução. [Masaya Kawamata](https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof) treinou o GPT-2 externo; previsões CC BY 4.0 e [código MIT](https://github.com/Matapanino/s6e11-gpt2-target-first) no commit `8446fb73c11583df4f7006aa08fa34472249b318`. Método target-first: Chris Deotte; GReaT: Borisov et al.; dados originais: uditjain13. Os arrays são logits: no teste o autor fez a média dos cinco logits antes da sigmoid. O importador aprovado conserva conversão, autoria e hashes.

BPE e pooling usam o vocabulário OpenAI, sem treinar a rede GPT-2. Pooling: crédito conceitual Megayak/Goodpjw2008; implementação própria e gerador não verificado.

In [ ]:
RECIPE = {'protocol_id': 'r18-cv-and-r20-full-refit-public25',
 'phase': 'exploratory_full_data_cv',
 'output_phase': 'full_data_refit',
 'source_revision': '8a1c7fd141b43e41b0d43e3a07b0c5ad0bf41ae5',
 'source_sha256': {'fraudlab/__init__.py': '14479eff95a0233e8bf2d9f359843c36f53f36b92cc95c3e452f75bfb2982dcc',
                   'fraudlab/__main__.py': 'b8fc5d9e4cf77f7319f9e2f7956f490bb71da6ce3c6842e279f46194672a7fae',
                   'fraudlab/bpe.py': 'fb71799ca075c657a8344f85d6a7e91653d1f319e24b5e287c17565cea456d4e',
                   'fraudlab/data.py': 'f22355cc51fa6e6162b24e639923f40a11a23af00afcb65553090176271c8326',
                   'fraudlab/experiment.py': '85a0995157f4b4453206d88a41ee1196765f7bb5cd6cf9dd55d5dd89ba87bb7a',
                   'fraudlab/external.py': '05ec4ec64e4a2c5cf1137c949646bed14fe83fb04506b8f2f973c2c7633c5e08',
                   'fraudlab/features.py': '184fe327468194f34563b25aee2268348ca854d59e4916f8e3eacdd1b6420424',
                   'fraudlab/frequency.py': 'de06b4f581f6864066d94b140d67e4ed16f7cef054a30ef20bc408a972279ce6',
                   'fraudlab/holdout.py': 'e3702891766326710c3b56dd7a7fe69f2f778acea3316665f130aa6955e2f4e3',
                   'fraudlab/joints.py': '620851854c56aceceba02d5855fe22525e748973e80ffe0f83040f5e75a271e8',
                   'fraudlab/models.py': 'e00d858bac42b79362ef408cbb7f359bca44a4828931b0e4943d660a3eb095b5',
                   'fraudlab/neural_offset.py': '042eec5ea088477bb37b045ace2f537c6e6e39ff49b3e810dc5e501b4ea8e487',
                   'fraudlab/offset.py': 'a4ea551e4ddf89aedc87f313eb07425aebc60f7d967986524a873a73565fdc4f',
                   'fraudlab/original.py': '4db9d8d54929ba9d9a36fd55b8f8655611a800d22fade59af079e0497ce7b934',
                   'fraudlab/pooled.py': '6f15acb31c8a878efc262410a0ddf3f6b733b1a636a7bfa917758d568164601a',
                   'fraudlab/refit.py': '981074be27e2ed24aa45dbaabeda32677f83ae59aa74f12dd6a56de152fde4e9',
                   'fraudlab/scores.py': '8bae07482b56eb488a388c54ecd7ec0cf87a8644d9e98e53d7c1fcc75d2e58a1',
                   'fraudlab/splits.py': '41215280332984010f5b616bbf6a675c623b7af3e3f00604eb3238e79c709af9',
                   'fraudlab/strings.py': 'f3248a9c0e7bbda655e29ecac02886dfa4cd4f34f35007ad90cf99a17efccbb0',
                   'fraudlab/tabm.py': 'df22ff1a120c8a6fde59052ea7a1b509ff1b7e7acc83d774f670a1daf4a4e667',
                   'fraudlab/tokens.py': '8e3abfcbe2e437eee859d120f2a89b0a45bd84dfe389a06c11b622ee5ac9cbb4',
                   'fraudlab/transductive.py': 'b2848df445f94d1135df27b99e7586048c279860b244dcf9544c30687263ca3b'},
 'competition': 'playground-series-s6e11',
 'rows': {'train': 699960, 'test': 299984},
 'data_hashes': {'train.csv': '87eaa10b8d17a7e4f657b8da1c66242431338bdd8b6ad45861521ac0c0e9fbd5',
                 'test.csv': '10f516959020357b565abaaa4fd25baae97fc2f536d0805695bb7cc15c1660f0',
                 'sample_submission.csv': '13fc6ed8b73de5329884ec490f2a0b750cf29a7cda772b78cfa0cf486f6f35fa'},
 'common': {'phase': 'exploratory_full_data_cv',
            'id_column': 'id',
            'target': 'is_fraud',
            'split': 'stratified',
            'folds': 5,
            'seed': 42,
            'holdout_fraction': 0.0,
            'fixed_training': True,
            'validation_note': 'Exploratory full-data StratifiedKFold5/42, same699960 OOF '
                               'IDs/labels/folds as R10. Inner-crossfit5 target and pooled '
                               'estimates, outer-training-only scaling/frequency. No original '
                               'augmentation, transductive counts, global G scan, independent '
                               'holdout or outer-validation early stopping.'},
 'models': {'offset': {'name': 'round18-joint-bpe-pooled-offset',
                       'model': 'offset',
                       'feature_set': 'joint_bpe',
                       'glm_pooled_tokens': True,
                       'glm_frequency': True,
                       'glm_inner_folds': 5,
                       'target_smoothing': 50,
                       'logistic_c': 0.3,
                       'glm_spline_knots': 0,
                       'iterations': 1000,
                       'num_leaves': 4,
                       'learning_rate': 0.03,
                       'l2_leaf_reg': 5,
                       'min_child_samples': 100,
                       'device': 'cpu',
                       'threads': 4},
            'lightgbm': {'name': 'explore5-lgbm',
                         'model': 'lightgbm',
                         'threads': 6,
                         'device': 'auto',
                         'iterations': 1438,
                         'learning_rate': 0.04,
                         'l2_leaf_reg': 5.0,
                         'num_leaves': 4,
                         'min_child_samples': 100,
                         'target_smoothing': 20.0,
                         'patience': 150,
                         'feature_set': 'value_digits'}},
 'cv_hybrid_name': 'round18-pooled-share100-public25',
 'own_weights': {'offset': 0.75, 'lightgbm': 0.25},
 'hybrid_weights': {'own': 0.75, 'external_gpt2': 0.25},
 'effective_weights': {'offset': 0.5625, 'lightgbm': 0.1875, 'external_gpt2': 0.25},
 'tokenizer': {'tiktoken': '0.14.0',
               'regex': '2026.9.29',
               'encoding': 'gpt2',
               'offline_assets_embedded': True,
               'encoding_files': [{'url': 'https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/vocab.bpe',
                                   'sha256': '1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5'},
                                  {'url': 'https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/encoder.json',
                                   'sha256': '196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783'}]},
 'public_component': {'name': 'GPT-2 target-first',
                      'kind': 'gpt2_target_first',
                      'source_type': 'external_predictions',
                      'trained_by': 'Masaya Kawamata',
                      'model_architecture': 'GPT-2 target-first classifier',
                      'training_code_url': 'https://github.com/Matapanino/s6e11-gpt2-target-first',
                      'training_code_commit': '8446fb73c11583df4f7006aa08fa34472249b318',
                      'code_license': 'MIT',
                      'predictions_source_url': 'https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof',
                      'predictions_source_version': 2,
                      'predictions_download_ref': 'masayakawamata/s6e11-gpt2-target-first-oof/2',
                      'predictions_license': 'CC BY 4.0',
                      'data_license': 'CC BY 4.0 (prediction dataset)',
                      'method_credits': ['Chris Deotte: target-first GPT-2 method',
                                         'Borisov et al.: GReaT / Language Models are Realistic '
                                         'Tabular Data Generators',
                                         'uditjain13: original Credit Card Fraud Detection 2026 '
                                         'dataset'],
                      'training_executed_locally': False,
                      'trained_by_us': False,
                      'oof_auc': 0.9500272037932734,
                      'validation': {'method': 'StratifiedKFold',
                                     'n_splits': 5,
                                     'shuffle': True,
                                     'seed': 42},
                      'inference': {'method': 'target-first label likelihood ratio',
                                    'target_first': True,
                                    'likelihood_ratio': True,
                                    'prior': "training rate of the fold's fit rows",
                                    'orders': 10,
                                    'order_aggregation': 'log-sum-exp over orders per label, then '
                                                         'the difference, plus the prior',
                                    'test_aggregation': 'mean of fold log-odds',
                                    'test_ensemble_folds': 5,
                                    'source_test_aggregation': 'mean over fold models of the '
                                                               'per-fold log-odds',
                                    'import_transform': 'sigmoid(log_odds), float64'},
                      'weight': 0.25,
                      'mixture_space': 'probability',
                      'manifest_sha256': '0083d23b46f71311ceeed66874b093ec927d25d98b37f6d2e78b1de17f0f22d0',
                      'array_sha256': {'oof.npy': '3468aaaeb7851739ec0de1e06b780da6d2777fcaeef638a6852c56e68e67a206',
                                       'test.npy': '6c27aff7be744f7a0cc17809e0cd04fcdea0664234af88e3040a25235fb67906',
                                       'fold_ids.npy': '1d459900f6c3a001ff971c09d9c607b383bdc732105c54701587189e52eeab45'},
                      'run': 'explore5-public-gpt2-v2'},
 'refit': {'operation': 'portable_selected_cv_refit',
           'seed': 42,
           'names': {'offset': 'round20-refit-joint-offset', 'lightgbm': 'round20-refit-lgbm'},
           'hybrid_name': 'round20-refit-joint-public25',
           'roc_auc': None,
           'fold_scores': [],
           'oof_rows': 0,
           'provenance': 'New hashes of this notebook execution; not byte-identity to local '
                         'audited script outputs',
           'validation_labels': None},
 'observed_local_results': {'own_offset_oof_auc': 0.9511314157843096,
                            'hybrid_oof_auc': 0.9516524825255244,
                            'refit_oof_auc': None,
                            'refit_public_submission': '58483010',
                            'refit_public_auc': 0.95162,
                            'ranking': 657,
                            'total_teams': 2126,
                            'observed_at': '2026-11-17T21:59:16.262839+00:00',
                            'scope': 'Recorded local R18 CV and separately processed R20 refit '
                                     'submission; no cloud execution or new refit OOF claimed'},
 'historical_protocol': {'source_revision': '1fe34b540f3ea28276da652c3dd1de02726897f4',
                         'independent_holdout_applies_to_hybrid': False,
                         'regenerate_argument': '--protocol artifacts/holdout/evaluation.json',
                         'note': 'The earlier 3-fold/15-percent holdout protocol is historical '
                                 'only; no independent holdout remains for the hybrid.'},
 'output': 'submission.csv',
 'cv_output': 'submission-cv.csv',
 'execution_manifest': 'notebook-execution-manifest.json'}
RECIPE_SHA256 = 'acbbd8519739186582bbd21a5127b364a2c3947b87ab94f0f372e3e6a6bb90fe'
EXPORT = {'checkpoint_byte_identity_to_local_runs_claimed': False,
 'core_revision': '8a1c7fd141b43e41b0d43e3a07b0c5ad0bf41ae5',
 'generator_sha256': '628c063173ce5caeb029504dac9a11fe19e2efb81b2cecb515e7b0a66963a3c8',
 'origin': 'portable R18/R20 notebook cells; not the local artifact-audit script'}

candidates = [Path('/kaggle/input/competitions/playground-series-s6e11'),
              Path('/kaggle/input/playground-series-s6e11')]
data = next((p for p in candidates if (p / 'train.csv').is_file()), None)
if data is None:
    raise FileNotFoundError('Adicione playground-series-s6e11 aos inputs.')
manifest_hash = RECIPE['public_component']['manifest_sha256']
public_source = next((p.parent for p in Path('/kaggle/input').rglob('manifest.json')
                      if hashlib.sha256(p.read_bytes()).hexdigest() == manifest_hash), None)
if public_source is None:
    raise FileNotFoundError('Adicione o dataset GPT-2 versão 2 aprovado.')

def digest(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            result.update(block)
    return result.hexdigest()

def check_inputs():
    current = hashlib.sha256(json.dumps(
        RECIPE, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert current == RECIPE_SHA256, 'Recipe changed'
    for filename, expected in SOURCE_SHA256.items():
        assert digest(filename) == expected, filename
    for filename, expected in RECIPE['data_hashes'].items():
        assert digest(data / filename) == expected, filename
    assert digest(public_source / 'manifest.json') == manifest_hash
    for filename, expected in RECIPE['public_component']['array_sha256'].items():
        assert digest(public_source / filename) == expected, filename
    for asset in TOKENIZER_ASSETS:
        assert digest(tokenizer_cache / asset['cache_key']) == asset['sha256']

check_inputs()
import fraudlab
assert Path(fraudlab.__file__).resolve() == Path('fraudlab/__init__.py').resolve()
runs = Path('artifacts/runs')
def run(*arguments):
    subprocess.run([sys.executable, '-m', 'fraudlab', *map(str, arguments)], check=True)

run('inspect', '--data', data, '--output', 'artifacts/profile.json')
public_run = runs / RECIPE['public_component']['run']
run('import-public-gpt2', '--source', public_source, '--data', data,
    '--output', runs, '--name', public_run.name)


## 3. R18: configurações fixas e validação dos argumentos

Offset: `joint_bpe` + frequência do treino + pooling, 92 colunas e 223 dimensões codificadas. GLM Newton-Cholesky C=0,3, TE interno 5/50 e pooling 50/2.000/200. Residual: 1.000 árvores, quatro folhas, taxa 0,03, L2=5 e mínimo 100 por folha. LightGBM separado: `value_digits`, 1.438 árvores, quatro folhas, taxa 0,04 e TE20. CV estratificada 5/42 em todos os rótulos, sem holdout independente, dados originais adicionais, frequência transdutiva ou early stopping externo. O preflight abaixo valida a CLI antes dos treinos.

In [ ]:

from fraudlab.__main__ import parser, validate_training_config

def training_arguments(role):
    configuration = {'data': str(data), 'output': str(runs),
                     **RECIPE['common'], **RECIPE['models'][role]}
    arguments = ['train']
    for key, value in configuration.items():
        if value is None or value is False:
            continue
        arguments.append('--' + key.replace('_', '-'))
        if value is not True:
            arguments.extend(map(str, value if isinstance(value, list) else [value]))
    return arguments

for role in ('offset', 'lightgbm'):
    config = vars(parser().parse_args(training_arguments(role)))
    config.pop('command')
    if config['patience'] is None:
        config['patience'] = 100
    validate_training_config(config)
    assert config['fixed_training'] and config['folds'] == 5
    assert config['seed'] == 42 and config['holdout_fraction'] == 0
    assert not config.get('original_data') and not config.get('glm_transductive_frequency')
assert [RECIPE['effective_weights'][r] for r in ('offset', 'lightgbm', 'external_gpt2')] == [0.5625, 0.1875, 0.25]
print({'validated_models': list(RECIPE['models']), 'fit_performed_by_this_cell': False})


## 4. Executar a CV R18 e conservar seu CSV

Os dez ajustes são sequenciais. Cada OOF vem do treino que excluiu seu fold; o teste usa a média de cinco probabilidades por família. A composição é direta: **56,25% offset / 18,75% LGB / 25% Masaya**, em probabilidades. A OOF local histórica selecionada foi 0,9516524825255244; ela não é output deste notebook salvo. Esta execução produzirá seus próprios artefatos e hashes.

In [ ]:

check_inputs()
run(*training_arguments('offset'))
run(*training_arguments('lightgbm'))
offset_run = runs / RECIPE['models']['offset']['name']
lgbm_run = runs / RECIPE['models']['lightgbm']['name']
hybrid_run = runs / RECIPE['cv_hybrid_name']
weights = [RECIPE['effective_weights'][role]
           for role in ('offset', 'lightgbm', 'external_gpt2')]
run('blend', '--runs', offset_run, lgbm_run, public_run,
    '--weights', *weights, '--output', runs, '--name', hybrid_run.name)
run('validate-submission', '--file', hybrid_run / 'submission.csv',
    '--sample', data / 'sample_submission.csv')
shutil.copyfile(hybrid_run / 'submission.csv', 'submission-cv.csv')
cv_metric = json.loads((hybrid_run / 'metrics.json').read_text())
assert cv_metric['phase'] == 'exploratory_full_data_cv'
assert cv_metric['validation']['independent_holdout'] is False
assert cv_metric['rows']['oof'] == RECIPE['rows']['train']
component = cv_metric['model_details']['external_components']
assert len(component) == 1 and component[0]['trained_by_us'] is False
assert component[0]['weight'] == 0.25
assert component[0]['predictions_source_version'] == 2
check_inputs()
CV_ARTIFACT_SHA256 = {
    str(p / name): digest(p / name)
    for p in (offset_run, lgbm_run, public_run, hybrid_run)
    for name in ('config.json', 'metrics.json', 'oof.csv', 'predictions.csv')}
print({'cv_file': 'submission-cv.csv', 'sha256': digest('submission-cv.csv'),
       'this_execution_oof_auc': cv_metric['roc_auc']})


## 5. R20 portátil: refits em 100%, sem nova OOF

Dois novos ajustes usam todos os 699.960 rótulos, seed42 e os mesmos parâmetros. O meta não existe neste protocolo: R21 é outra pesquisa e não é executada aqui. A validação passada ao fit contém uma linha de teste e `validation_y=None`; não seleciona árvores. Encodings supervisionados permanecem cruzados no treino.

Esta implementação portátil usa `fit_predict` do core8a1 e guarda os hashes **desta execução**, sem exigir ou alegar identidade com os checkpoints locais. O script local auditado usa outro contrato de proveniência. Cada checkpoint novo é recarregado em 1.024 testes; falha de convergência interrompe, sem retry. Metadados: `full_data_refit`, AUC nula, folds vazios, zero OOF. A AUC da CV acima aparece somente como referência de desenvolvimento.

In [ ]:

import gc
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from fraudlab.data import load_data, submission, validate_submission, write_json
from fraudlab.experiment import create_run, source_provenance, versions
from fraudlab.features import transform_features
from fraudlab.models import fit_predict, predict_saved

raw, raw_test, sample = load_data(data, 'id', 'is_fraud')
assert len(raw) == RECIPE['rows']['train'] and len(raw_test) == RECIPE['rows']['test']
labels = raw['is_fraud'].to_numpy()
reference = json.loads((hybrid_run / 'metrics.json').read_text())
assert reference['status'] == 'complete' and reference['data_hashes'] == RECIPE['data_hashes']
assert reference['model_details']['weights'] == weights
full_common = {
    'phase': 'full_data_refit', 'roc_auc': None, 'fold_scores': [], 'fold_std': None,
    'rows': {'train': len(raw), 'test': len(raw_test), 'oof': 0},
    'data_hashes': RECIPE['data_hashes'], 'seed': 42, 'versions': versions(),
    'development_roc_auc': reference['roc_auc'],
    'development_roc_auc_scope': 'This execution R18 exploratory CV; not an AUC of refits',
    'reference_cv_run': hybrid_run.name,
    'reference_cv_metrics_sha256': digest(hybrid_run / 'metrics.json'),
    'notebook_export': {**EXPORT, 'recipe_sha256': RECIPE_SHA256},
}

def full_refit(role, source):
    check_inputs()
    for filename in ('config.json', 'metrics.json'):
        assert digest(source / filename) == CV_ARTIFACT_SHA256[str(source / filename)]
    original = json.loads((source / 'config.json').read_text())
    source_metric = json.loads((source / 'metrics.json').read_text())
    assert original['model'] == role and original['fixed_training']
    assert original['phase'] == 'exploratory_full_data_cv'
    assert not original.get('original_data') and not original.get('glm_transductive_frequency')
    name = RECIPE['refit']['names'][role]
    config = {**original, 'name': name, 'phase': 'full_data_refit', 'refit': True,
              'validation_note': 'Portable fixed all-label refit; no new OOF or holdout'}
    execution_keys = {'name', 'phase', 'refit', 'validation_note'}
    assert {k: v for k, v in config.items() if k not in execution_keys} == {
        k: v for k, v in original.items() if k not in execution_keys}
    directory = create_run(runs, name)
    write_json(directory / 'status.json', {'status': 'running'})
    write_json(directory / 'config.json', config)
    try:
        training = transform_features(raw.drop(columns=['id', 'is_fraud']),
                                      config['feature_set'])
        testing = transform_features(raw_test.drop(columns='id'), config['feature_set'])
        assert training.columns.tolist() == testing.columns.tolist() == source_metric['features']['names']
        stem = directory / 'model'
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter('always', ConvergenceWarning)
            _, prediction, detail = fit_predict(
                role, training, labels, testing.iloc[:1], None, testing, config, 42, stem)
        for warning in captured:
            warnings.warn(warning.message, warning.category)
        if any(issubclass(w.category, ConvergenceWarning) for w in captured):
            raise RuntimeError('Refit did not converge; no automatic retry')
        assert detail['best_iteration'] == config['iterations']
        checkpoint_hash = digest(stem.with_suffix('.joblib'))
        if role == 'offset':
            classifier = joblib.load(stem.with_suffix('.joblib')).glm.named_steps['classifier']
            assert int(classifier.n_iter_[0]) < classifier.max_iter
            assert detail['encoded_features'] == 223
        replay = predict_saved(role, stem, testing.iloc[:1024], device='cpu')
        error = float(np.max(np.abs(replay - prediction[:1024])))
        assert np.allclose(replay, prediction[:1024], rtol=0, atol=1e-12)
        assert digest(stem.with_suffix('.joblib')) == checkpoint_hash
        final = submission(raw_test['id'], prediction, sample, 'is_fraud')
        for filename in ('predictions.csv', 'submission.csv'):
            final.to_csv(directory / filename, index=False)
        validate_submission(pd.read_csv(directory / 'submission.csv'), sample, 'id', 'is_fraud')
        check_inputs()
        for filename in ('config.json', 'metrics.json'):
            assert digest(source / filename) == CV_ARTIFACT_SHA256[str(source / filename)]
        metric = {**full_common, **source_provenance(config['feature_set']),
                  'status': 'complete', 'model': role,
                  'features': source_metric['features'], 'model_details': detail,
                  'source_cv_run': source.name,
                  'source_cv_config_sha256': digest(source / 'config.json'),
                  'config_sha256': digest(directory / 'config.json'),
                  'checkpoint_sha256': checkpoint_hash, 'reload_rows': len(replay),
                  'reload_max_abs_error': error,
                  'external_components': [],
                  'submission_sha256': digest(directory / 'submission.csv')}
        assert metric['roc_auc'] is None and metric['fold_scores'] == []
        assert metric['rows']['oof'] == 0 and not (directory / 'oof.csv').exists()
        write_json(directory / 'metrics.json', metric)
        write_json(directory / 'status.json', {'status': 'complete'})
        return directory
    except Exception as error:
        write_json(directory / 'status.json', {'status': 'failed', 'error': str(error)})
        raise


In [ ]:

refit_offset = full_refit('offset', offset_run)
gc.collect()
refit_lgbm = full_refit('lightgbm', lgbm_run)
gc.collect()


## 6. R20: composição final e manifesto desta execução

O blend geral continua recusando fases mistas. Esta composição especializada combina os dois novos refits completos com as previsões externas aprovadas, valida os IDs e probabilidades e mantém os pesos 56,25/18,75/25. Gera `submission.csv` e `notebook-execution-manifest.json`, sem envio automático e sem nova AUC de refit. O arquivo CV permanece separado.

In [ ]:

paths = [refit_offset, refit_lgbm, public_run]
inputs = {str(p / name): digest(p / name) for p in paths
          for name in ('config.json', 'metrics.json', 'predictions.csv')}
inputs.update(CV_ARTIFACT_SHA256)
inputs.update({str(p / 'model.joblib'): digest(p / 'model.joblib') for p in paths[:2]})
for path, expected in inputs.items():
    assert digest(path) == expected, path
metrics = [json.loads((p / 'metrics.json').read_text()) for p in paths]
assert all(m['status'] == 'complete' and m['data_hashes'] == RECIPE['data_hashes']
           for m in metrics)
assert all(m['phase'] == 'full_data_refit' and m['roc_auc'] is None
           and m['fold_scores'] == [] and m['rows']['oof'] == 0 for m in metrics[:2])
assert metrics[2]['phase'] == 'exploratory_full_data_cv'
for path, metric in zip(paths[:2], metrics[:2]):
    assert digest(path / 'model.joblib') == metric['checkpoint_sha256']
external = [{**c, 'weight': 0.25}
            for c in metrics[2]['model_details']['external_components']]
assert len(external) == 1 and external[0]['trained_by_us'] is False
assert external[0]['predictions_source_version'] == 2
assert external[0]['predictions_license'] == 'CC BY 4.0'
frames = [pd.read_csv(p / 'predictions.csv') for p in paths]
for frame in frames:
    validate_submission(frame, sample, 'id', 'is_fraud')
probability = sum(w * frame['is_fraud'].to_numpy() for w, frame in zip(weights, frames))
final = submission(raw_test['id'], probability, sample, 'is_fraud')
directory = create_run(runs, RECIPE['refit']['hybrid_name'])
config = {'operation': 'portable_selected_cv_refit_composition',
          'sources': list(map(str, paths)), 'weights': weights,
          'inputs_sha256': inputs, 'notebook_export': full_common['notebook_export']}
write_json(directory / 'config.json', config)
for filename in ('predictions.csv', 'submission.csv'):
    final.to_csv(directory / filename, index=False)
written = pd.read_csv(directory / 'submission.csv')
validate_submission(written, sample, 'id', 'is_fraud')
assert np.allclose(written['is_fraud'], probability, rtol=0, atol=1e-15)
check_inputs()
for path, expected in inputs.items():
    assert digest(path) == expected, path
final_metric = {
    **full_common, **source_provenance(), 'status': 'complete', 'model': 'blend',
    'features': metrics[0]['features'],
    'config_sha256': digest(directory / 'config.json'),
    'model_details': {'method': 'weighted_probability_average',
                      'sources': list(map(str, paths)), 'weights': weights,
                      'external_components': external},
    'external_components': external,
    'submission_sha256': digest(directory / 'submission.csv'),
}
assert final_metric['roc_auc'] is None and final_metric['rows']['oof'] == 0
assert not (directory / 'oof.csv').exists()
write_json(directory / 'metrics.json', final_metric)
write_json(directory / 'status.json', {'status': 'complete'})
shutil.copyfile(directory / 'submission.csv', 'submission.csv')
outputs = {str(p): digest(p) for folder in [refit_offset, refit_lgbm, directory]
           for p in folder.iterdir() if p.is_file()}
outputs['submission.csv'] = digest('submission.csv')
assert outputs['submission.csv'] == final_metric['submission_sha256']
execution_manifest = {
    'origin': EXPORT, 'source_sha256': SOURCE_SHA256, 'recipe_sha256': RECIPE_SHA256,
    'data_sha256': RECIPE['data_hashes'], 'input_artifacts_sha256': inputs,
    'output_artifacts_sha256': outputs,
    'phase': 'full_data_refit', 'roc_auc': None, 'fold_scores': [], 'oof_rows': 0,
    'development_roc_auc': reference['roc_auc'],
    'development_roc_auc_scope': full_common['development_roc_auc_scope'],
    'external_training_executed_here': False, 'external_probability_weight': 0.25,
    'kaggle_submission_sent': False,
}
write_json(Path('notebook-execution-manifest.json'), execution_manifest)
print({'file': 'submission.csv', 'sha256': outputs['submission.csv'],
       'phase': 'full_data_refit', 'roc_auc': None, 'oof_rows': 0})


## Resultados locais registrados e limites

R18 CV híbrida local: **0.9516524825**, sem holdout independente. R20 local: submissão **58483010**, score público **0.95162**, posição **657/2126** em **2026-11-17T21:59:16.262839+00:00**. A referência CV pertence à R18; não é AUC do refit. Esses fatos vêm das evidências do repositório e não de execução deste notebook salvo. Os módulos são os mesmos blobs 8a1, mas bibliotecas, caminhos e ambiente podem mudar os bytes dos novos modelos. Não se alega identidade com os checkpoints locais.

A reserva histórica de 15% não é independente desta seleção e não é reavaliada. Gerar CSV, salvar notebook e obter score são estados distintos. Nenhuma submissão é automática; ranking é provisório, primeiras posições não foram alcançadas e a classificação privada final depende do encerramento.

## Apêndice R23 · solução atual, código aberto e reprodução local

O baseline acima produz arquivos da R18/R20. A **R23 atual** combina probabilidades de
**42,1875% offset próprio + 14,0625% RealMLP própria + 18,75% LightGBM próprio + 25% GPT-2 externo**.
A RealMLP foi treinada localmente em todos os 699.960 rótulos: 16 membros, camadas
512→256→128, três épocas fixadas pela mediana [3,2,3,3,3] da CV R22. Reutilizamos os
dois modelos completos da R20. A referência OOF R22 é 0,9517074903582017; **R23 não
gera uma nova OOF nem holdout independente**. Score público, posição e data abaixo
vêm de registros locais da plataforma; não são outputs de uma execução deste notebook.

Os 23 módulos próprios do core [`11b139f`](https://github.com/aikolopes-DS/fraudlab/tree/11b139f77206101d3f5beeb5648fb459f4cd304a)
e os dois scripts de [`c95ff46`](https://github.com/aikolopes-DS/fraudlab/tree/c95ff468619d299c94764b45e5e15af5ed9f4674)
são exportados abaixo em **`r23-source/`**, com os planos e hashes. Essas células
apenas gravam código para consulta: não substituem `fraudlab/`, não importam esse
código, não treinam, não preveem e não enviam submissões. Não instalam dependências.
O código próprio é MIT. PyTabKit e os demais pacotes conservam suas próprias licenças;
as previsões GPT-2 externas continuam atribuídas a Masaya Kawamata, CC BY 4.0.

Para treinar fora deste notebook, siga o [guia de reprodução](https://github.com/aikolopes-DS/fraudlab/blob/main/docs/reproduce.md).
A CLI RealMLP usa os argumentos de `evidence/round22-plan.json`. O script R23 é um
**replay auditado condicionado aos artefatos exatos** dos planos, inclusive checkpoints
e recibos locais; não é um comando autossuficiente para um clone vazio. Esses
artefatos não estão incorporados aqui. Uma nova reprodução exige seus próprios
ajustes, auditorias, hashes e processamento no Kaggle e não herda o score registrado.

Exemplos para uma sessão local preparada (texto de referência; não executado aqui):

```powershell
$fraudR22Plan = Get-Content evidence/round22-plan.json -Raw | ConvertFrom-Json
$fraudR22Args = @($fraudR22Plan.cli_argv)
python -m fraudlab @fraudR22Args
python scripts/refit_neural_cv.py --plan evidence/round23-plan.json --workspace . --core-worktree .local/worktrees/realmlp-cv
```

O segundo comando requer a fonte limpa pinada, versões exatas e todos os inputs/
recibos já produzidos e auditados; o primeiro inicia treinamento real caso seja
copiado e executado localmente. Nenhum deles é chamado pelo apêndice.

In [ ]:
from pathlib import Path
for folder in ('src/fraudlab', 'scripts', 'evidence'):
    (Path('r23-source') / folder).mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile r23-source/src/fraudlab/__init__.py
"""Small, auditable experiments for binary tabular classification."""

__version__ = "0.1.0"


In [ ]:
%%writefile r23-source/src/fraudlab/__main__.py
"""Run with python -m fraudlab; defaults never download or submit data."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

from .data import inspect_data, validate_submission, write_json
from .experiment import blend, train


def parser() -> argparse.ArgumentParser:
    root = argparse.ArgumentParser(description="Auditable Kaggle fraud experiments.")
    commands = root.add_subparsers(dest="command", required=True)
    inspection = commands.add_parser(
        "inspect", help="Write aggregate data checks and SHA-256 fingerprints."
    )
    inspection.add_argument("--data", default="data/raw")
    inspection.add_argument("--output", default="artifacts/profile.json")
    inspection.add_argument("--id-column", default="id")
    inspection.add_argument("--target", default="is_fraud")

    training = commands.add_parser(
        "train",
        help="Fit cross-validation folds and write OOF, models and test predictions.",
    )
    training.add_argument("--data", default="data/raw")
    training.add_argument(
        "--original-data",
        help="Pinned original CSV, appended only to outer-training rows.",
    )
    training.add_argument(
        "--original-sha256", help="Required SHA-256 pin when --original-data is used."
    )
    training.add_argument("--original-id-column", default="transaction_id")
    training.add_argument("--output", default="artifacts/runs")
    training.add_argument("--name", required=True)
    training.add_argument(
        "--phase",
        choices=("development", "exploratory_full_data_cv"),
        default="development",
        help="Label the validation protocol; exploratory full-data CV has no independent holdout.",
    )
    training.add_argument(
        "--model",
        choices=(
            "logistic",
            "glm",
            "catboost",
            "lightgbm",
            "mlp",
            "offset",
            "tabm",
            "neural_offset",
            "realmlp",
        ),
        required=True,
    )
    training.add_argument(
        "--feature-set",
        choices=(
            "raw",
            "behavior",
            "digits",
            "value_digits",
            "paired_digits",
            "bpe_digits",
            "token_digits",
            "joint_bpe",
        ),
        default="raw",
    )
    training.add_argument("--id-column", default="id")
    training.add_argument("--target", default="is_fraud")
    training.add_argument(
        "--split", choices=("stratified", "group", "time"), default="stratified"
    )
    training.add_argument(
        "--validation-note",
        required=True,
        help="Explain why this split matches the data and intended generalization.",
    )
    training.add_argument("--group-column")
    training.add_argument("--time-column")
    training.add_argument("--folds", type=int, default=3)
    training.add_argument("--seed", type=int, default=42)
    training.add_argument(
        "--holdout-fraction",
        type=float,
        default=0.0,
        help="Reserve this fraction before CV; never evaluate it during model selection.",
    )
    training.add_argument("--holdout-seed", type=int, default=2026)
    training.add_argument("--threads", type=int, default=8)
    training.add_argument(
        "--fixed-training",
        action="store_true",
        help="Use the configured tree/epoch count without early stopping; useful for preselected full-data CV.",
    )
    training.add_argument(
        "--device",
        choices=("auto", "cpu", "cuda"),
        default="auto",
        help="Auto uses CPU for CatBoost and CUDA when available for neural models.",
    )
    training.add_argument("--logistic-c", type=float, default=1.0)
    training.add_argument(
        "--glm-inner-folds",
        type=int,
        choices=(5, 10),
        default=5,
        help="Inner target-encoding partitions for GLM/offset; outer CV is unchanged.",
    )
    training.add_argument(
        "--glm-value-one-hot",
        action="store_true",
        help="One-hot encode every categorical GLM input with fold-local rare pooling and sparse L-BFGS.",
    )
    training.add_argument(
        "--glm-frequency",
        action="store_true",
        help="Add training-fold log category counts to GLM/offset inputs.",
    )
    training.add_argument(
        "--glm-transductive-frequency",
        action="store_true",
        help="Use label-free official train+test category counts; exploratory CV only.",
    )
    training.add_argument(
        "--glm-pooled-tokens",
        action="store_true",
        help="Add 16 inner-cross-fitted hierarchical decimal/token logits to GLM/offset.",
    )
    training.add_argument(
        "--glm-spline-knots",
        type=int,
        default=0,
        help="Add cubic quantile splines of continuous features to GLM/offset; 0 disables them.",
    )
    training.add_argument("--iterations", type=int, default=2000)
    training.add_argument("--depth", type=int, default=6)
    training.add_argument("--learning-rate", type=float, default=0.05)
    training.add_argument("--l2-leaf-reg", type=float, default=3.0)
    training.add_argument("--num-leaves", type=int, default=4)
    training.add_argument("--min-child-samples", type=int, default=100)
    training.add_argument(
        "--target-smoothing",
        type=lambda value: "auto" if value == "auto" else float(value),
        default=20.0,
    )
    training.add_argument("--epochs", type=int, default=30)
    training.add_argument(
        "--hidden-dims",
        type=int,
        nargs="+",
        help="Defaults: MLP 128 64; TabM/neural_offset 256 256; RealMLP 512 256 128.",
    )
    training.add_argument("--dropout", type=float, default=0.1)
    training.add_argument("--neural-learning-rate", type=float, default=0.001)
    training.add_argument("--weight-decay", type=float, default=0.0001)
    training.add_argument(
        "--neural-numeric-transform",
        choices=("standard", "quantile", "realmlp"),
    )
    training.add_argument("--batch-size", type=int, default=4096)
    training.add_argument(
        "--neural-categorical-transform",
        choices=("embedding", "target"),
        help="Defaults: MLP embedding; TabM/neural_offset require target.",
    )
    training.add_argument(
        "--patience",
        type=int,
        help="Defaults: CatBoost 100 iterations; MLP 5 epochs; TabM/neural_offset 6; RealMLP cap3.",
    )

    blending = commands.add_parser(
        "blend", help="Average compatible runs with explicit, normalized weights."
    )
    blending.add_argument("--runs", type=Path, nargs="+", required=True)
    blending.add_argument("--weights", type=float, nargs="+")
    blending.add_argument(
        "--space",
        choices=("probability", "logit", "rank", "rank_logit"),
        default="probability",
        help="Combine probabilities, logits or whole-vector normalized ranks.",
    )
    blending.add_argument("--name", required=True)
    blending.add_argument("--output", type=Path, default=Path("artifacts/runs"))
    blending.add_argument("--id-column", default="id")
    blending.add_argument("--target", default="is_fraud")

    validation = commands.add_parser(
        "validate-submission",
        help="Require exact schema, ID order and finite probabilities.",
    )
    validation.add_argument("--file", type=Path, required=True)
    validation.add_argument(
        "--sample", type=Path, default=Path("data/raw/sample_submission.csv")
    )
    validation.add_argument("--id-column", default="id")
    validation.add_argument("--target", default="is_fraud")

    holdout = commands.add_parser(
        "evaluate-holdout",
        help="Evaluate frozen development choices once on the reserved independent holdout.",
    )
    holdout.add_argument("--runs", type=Path, nargs="+", required=True)
    holdout.add_argument("--weights", type=float, nargs="+")
    holdout.add_argument(
        "--output", type=Path, default=Path("artifacts/holdout/evaluation.json")
    )
    holdout.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")
    holdout.add_argument(
        "--development-run",
        type=Path,
        help="The matching pre-holdout development blend, for an explicit provenance chain.",
    )
    final = commands.add_parser(
        "refit",
        help="Refit one frozen selected model on all labeled data, without early stopping.",
    )
    final.add_argument("--run", type=Path, required=True)
    final.add_argument(
        "--holdout-report", type=Path, default=Path("artifacts/holdout/evaluation.json")
    )
    final.add_argument("--output", type=Path, default=Path("artifacts/runs"))
    final.add_argument("--name", required=True)
    public = commands.add_parser(
        "import-public-gpt2",
        help="Validate the approved licensed public GPT-2 arrays and convert logits to the OOF contract.",
    )
    public.add_argument("--source", type=Path, required=True)
    public.add_argument("--data", type=Path, default=Path("data/raw"))
    public.add_argument("--output", type=Path, default=Path("artifacts/runs"))
    public.add_argument("--name", default="explore5-public-gpt2")
    return root


def validate_training_config(config: dict) -> None:
    if (
        config.get("original_data")
        or config.get("original_sha256")
        or config.get("original_id_column", "transaction_id") != "transaction_id"
    ):
        from .original import validate_config

        validate_config(config)
    if config.get("hidden_dims") is None:
        config["hidden_dims"] = (
            [512, 256, 128]
            if config["model"] == "realmlp"
            else (
                [256, 256]
                if config["model"] in ("tabm", "neural_offset")
                else [128, 64]
            )
        )
    if config.get("neural_numeric_transform") is None:
        config["neural_numeric_transform"] = (
            "realmlp" if config["model"] == "realmlp" else "standard"
        )
    if config.get("neural_categorical_transform") is None:
        config["neural_categorical_transform"] = (
            "target" if config["model"] in ("tabm", "neural_offset") else "embedding"
        )
    if config["model"] == "tabm":
        if config.get("fixed_training") or config.get("refit"):
            raise ValueError(
                "TabM requires inner epoch selection; --fixed-training/refit is unsupported."
            )
        if config["hidden_dims"] != [256, 256]:
            raise ValueError("TabM requires --hidden-dims 256 256.")
        if config["neural_categorical_transform"] != "target":
            raise ValueError("TabM requires --neural-categorical-transform target.")
    if config["model"] == "neural_offset":
        try:
            from .neural_offset import NeuralOffsetConfig
        except ImportError as error:
            raise RuntimeError(
                "Neural offset requires the optional fraudlab[neural] dependencies."
            ) from error

        if config.get("patience") is None:
            config["patience"] = 6
        NeuralOffsetConfig.from_training_config(config)
    if config["model"] == "realmlp":
        try:
            from .realmlp import RealMLPConfig
        except ImportError as error:
            raise RuntimeError(
                "RealMLP requires the optional fraudlab[realmlp] dependencies."
            ) from error
        if config.get("patience") is None:
            config["patience"] = 3
        RealMLPConfig.from_training_config(config)
    elif (
        config["neural_numeric_transform"] == "realmlp"
        or config["target_smoothing"] == "auto"
    ):
        raise ValueError(
            "RealMLP preprocessing/automatic target smoothing requires --model realmlp."
        )
    glm_models = ("glm", "offset", "neural_offset")
    inner_folds = config.get("glm_inner_folds", 5)
    if inner_folds not in (5, 10) or (
        inner_folds != 5 and config["model"] not in glm_models
    ):
        raise ValueError(
            "glm_inner_folds must be 5 or 10; 10 requires GLM/offset/neural_offset."
        )
    if config.get("glm_value_one_hot") and config["model"] != "glm":
        raise ValueError("glm_value_one_hot is supported only for GLM.")
    if config.get("glm_frequency") and config["model"] not in glm_models:
        raise ValueError(
            "glm_frequency is supported only for GLM/offset/neural_offset."
        )
    if config.get("glm_transductive_frequency"):
        from .transductive import validate_config

        validate_config(config)
    if config.get("glm_pooled_tokens"):
        if config["model"] not in glm_models or inner_folds != 5:
            raise ValueError(
                "glm_pooled_tokens requires GLM/offset/neural_offset with glm_inner_folds=5."
            )
        from .pooled import manifest

        config["pooled_token_encoding"] = manifest(random_state=config["seed"])
    knots = config.get("glm_spline_knots", 0)
    if knots < 0 or knots == 1 or (knots and config["model"] not in glm_models):
        raise ValueError(
            "glm_spline_knots must be 0 or at least 2, for GLM/offset/neural_offset only."
        )
    if config.get("phase") == "exploratory_full_data_cv" and (
        config["holdout_fraction"]
        or config["split"] != "stratified"
        or config["folds"] != 5
        or config["seed"] != 42
    ):
        raise ValueError(
            "Exploratory full-data comparison requires stratified 5-fold/seed42 CV with no independent holdout."
        )
    if config["id_column"] == config["target"]:
        raise ValueError("ID and target must be different columns.")
    if not config["validation_note"].strip():
        raise ValueError("A non-empty validation note is required.")
    if not 0 <= config["holdout_fraction"] < 0.5:
        raise ValueError("holdout_fraction must be in [0, 0.5).")
    if config["holdout_fraction"] and config["split"] != "stratified":
        raise ValueError(
            "The independent random holdout is supported only for stratified IID experiments."
        )
    for name in (
        "threads",
        "iterations",
        "epochs",
        "batch_size",
        "patience",
        "depth",
        "num_leaves",
        "min_child_samples",
    ):
        if config[name] < 1:
            raise ValueError(f"{name} must be positive.")
    if config["num_leaves"] < 2:
        raise ValueError("num_leaves must be at least two.")
    import math

    for name in ("learning_rate", "neural_learning_rate", "logistic_c"):
        if not math.isfinite(config[name]) or config[name] <= 0:
            raise ValueError(f"{name} must be positive.")
    if not 0 <= config["dropout"] < 1 or any(
        width < 1 for width in config["hidden_dims"]
    ):
        raise ValueError(
            "Dropout must be in [0, 1) and hidden layer widths must be positive."
        )
    if (
        not math.isfinite(config["l2_leaf_reg"])
        or not math.isfinite(config["weight_decay"])
        or config["l2_leaf_reg"] < 0
        or config["weight_decay"] < 0
        or (
            config["target_smoothing"] != "auto"
            and (
                not math.isfinite(config["target_smoothing"])
                or config["target_smoothing"] < 0
            )
        )
    ):
        raise ValueError("Regularization values must be nonnegative.")
    if config["split"] != "group" and config["group_column"]:
        raise ValueError("--group-column requires --split group.")
    if config["split"] != "time" and config["time_column"]:
        raise ValueError("--time-column requires --split time.")
    if (
        config["group_column"] in (config["target"], config["id_column"])
        or config["time_column"] == config["target"]
    ):
        raise ValueError(
            "The target cannot define a split; entity grouping must use an actual entity column."
        )


def main() -> None:
    arguments = parser().parse_args()
    try:
        if arguments.command == "inspect":
            profile = inspect_data(
                Path(arguments.data), arguments.id_column, arguments.target
            )
            write_json(Path(arguments.output), profile)
            print(
                json.dumps(
                    {
                        "profile": arguments.output,
                        "rows": profile["rows"],
                        "target": profile["target"],
                        "duplicate_feature_rows": profile["duplicate_feature_rows"],
                        "potential_split_columns": profile["potential_split_columns"],
                    },
                    indent=2,
                )
            )
        elif arguments.command == "train":
            config = vars(arguments).copy()
            config.pop("command")
            if config["patience"] is None:
                config["patience"] = {
                    "mlp": 5,
                    "tabm": 6,
                    "neural_offset": 6,
                    "realmlp": 3,
                }.get(config["model"], 100)
            validate_training_config(config)
            metrics = train(config)
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in (
                            "model",
                            "roc_auc",
                            "fold_scores",
                            "fold_std",
                            "elapsed_seconds",
                        )
                    },
                    indent=2,
                )
            )
        elif arguments.command == "blend":
            metrics = blend(
                arguments.runs,
                arguments.weights,
                arguments.output,
                arguments.name,
                arguments.id_column,
                arguments.target,
                arguments.space,
            )
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in ("model", "roc_auc", "fold_scores", "fold_std")
                    },
                    indent=2,
                )
            )
        elif arguments.command == "evaluate-holdout":
            from .holdout import evaluate

            print(
                json.dumps(
                    evaluate(
                        arguments.runs,
                        arguments.weights,
                        arguments.output,
                        arguments.device,
                        arguments.development_run,
                    ),
                    indent=2,
                )
            )
        elif arguments.command == "refit":
            from .refit import refit

            metrics = refit(
                arguments.run,
                arguments.holdout_report,
                arguments.output,
                arguments.name,
            )
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in ("model", "phase", "rows", "elapsed_seconds")
                    },
                    indent=2,
                )
            )
        elif arguments.command == "import-public-gpt2":
            from .external import import_public_gpt2

            metrics = import_public_gpt2(
                arguments.source, arguments.data, arguments.output, arguments.name
            )
            print(
                json.dumps(
                    {
                        key: metrics[key]
                        for key in ("model", "phase", "roc_auc", "fold_scores")
                    },
                    indent=2,
                )
            )
        else:
            frame, sample = pd.read_csv(arguments.file), pd.read_csv(arguments.sample)
            validate_submission(frame, sample, arguments.id_column, arguments.target)
            print(
                json.dumps(
                    {"valid": True, "rows": len(frame), "file": str(arguments.file)}
                )
            )
    except (ValueError, RuntimeError, OSError) as error:
        raise SystemExit(f"Error: {error}") from error


if __name__ == "__main__":
    main()


In [ ]:
%%writefile r23-source/src/fraudlab/bpe.py
"""Deterministic GPT-2 numeric keys, with no labels or learned statistics.

Hypothesis: azizdzhon's numeric BPE notebook and Shallow-Tree Stack v31.
Implementation: our own feature extraction using OpenAI's tokenizer API.
No notebook code, merge tables, predictions or fitted statistics are reused.
"""

from __future__ import annotations

from functools import lru_cache
from importlib.metadata import version

import numpy as np
import pandas as pd

from .strings import categorical_strings

COLUMNS = ("amount_usd", "account_balance_usd")
SUFFIXES = ("__gpt2_first_key", "__gpt2_second_key", "__gpt2_second_cents_key")
TOKENIZER_VERSION = "0.14.0"
ENCODING = "gpt2"
ENCODING_FILES = (
    {
        "url": "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/vocab.bpe",
        "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
    },
    {
        "url": "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/encoder.json",
        "sha256": "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783",
    },
)


@lru_cache(maxsize=1)
def _encoding():
    import tiktoken

    if version("tiktoken") != TOKENIZER_VERSION:
        raise RuntimeError(f"This hypothesis requires tiktoken=={TOKENIZER_VERSION}.")
    # The pinned implementation verifies both official asset hashes above.
    return tiktoken.get_encoding(ENCODING)


def _cents(series: pd.Series) -> np.ndarray:
    if not pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series):
        raise ValueError(f"{series.name} must contain numeric USD amounts.")
    values = series.to_numpy(dtype=np.float64)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f"{series.name} requires finite, nonnegative USD amounts.")
    if (values > (2**53 - 1) / 100).any():
        raise ValueError(f"{series.name} overflows the supported USD cent range.")
    return np.rint(values * 100).astype(np.int64)


def bpe_keys(
    frame: pd.DataFrame,
    excluded: tuple[str, ...] = (),
) -> pd.DataFrame:
    """Return six categorical keys; excluded columns produce no keys.

    USD is quantized with np.rint(float64_values * 100), matching value_digits.
    The integer text is one ASCII space followed by the unsigned base-ten integer.
    Token IDs are categorical labels; an absent second token is -1.
    Integers with more than two tokens intentionally retain only the first two.
    """
    result = pd.DataFrame(index=frame.index)
    for name in COLUMNS:
        if name in excluded:
            continue
        if name not in frame:
            raise ValueError(f"Missing BPE input column: {name}")
        cents = _cents(frame[name])
        integers, inverse = np.unique(cents // 100, return_inverse=True)
        tokens = [_encoding().encode_ordinary(f" {int(value)}") for value in integers]
        first = np.asarray([row[0] for row in tokens], dtype=np.int32)[inverse]
        second = np.asarray(
            [row[1] if len(row) > 1 else -1 for row in tokens], dtype=np.int32
        )[inverse]
        result[name + SUFFIXES[0]] = categorical_strings(first)
        result[name + SUFFIXES[1]] = categorical_strings(second)
        result[name + SUFFIXES[2]] = categorical_strings(
            np.char.add(np.char.add(second.astype(str), ":"), (cents % 100).astype(str))
        )
    return result


def manifest() -> dict:
    """Pin the tokenizer and official vocabulary assets in experiment evidence."""
    encoding = _encoding()
    return {
        "tiktoken": TOKENIZER_VERSION,
        "encoding": encoding.name,
        "vocabulary_size": encoding.max_token_value + 1,
        "encoding_files": list(ENCODING_FILES),
        "input_columns": list(COLUMNS),
        "output_suffixes": list(SUFFIXES),
        "representation": "cents=np.rint(float64_USD*100); ' ' + str(cents // 100)",
        "rounding": "NumPy rint; nearest representable cent, halfway ties to even",
        "second_token_missing": "-1",
        "third_key": "second_token_id + ':' + str(cents % 100)",
        "more_than_two_tokens": "retain first two; discard later integer tokens",
        "label_usage": "none in extraction; fold-local cross-fitted TE downstream",
    }


In [ ]:
%%writefile r23-source/src/fraudlab/data.py
"""Input contracts and aggregate inspection. Raw data stays local."""

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd


def fingerprint(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def validate_ids(frame: pd.DataFrame, id_column: str) -> None:
    if (
        frame.empty
        or id_column not in frame
        or frame[id_column].isna().any()
        or frame[id_column].duplicated().any()
    ):
        raise ValueError(
            f"{id_column}: rows and IDs must exist; IDs must be non-null and unique."
        )


def validate_submission(
    frame: pd.DataFrame,
    expected: pd.DataFrame,
    id_column: str = "id",
    target: str = "is_fraud",
) -> None:
    if list(frame.columns) != [id_column, target]:
        raise ValueError(
            f"Submission columns must be exactly {[id_column, target]} in that order."
        )
    validate_ids(frame, id_column)
    validate_ids(expected, id_column)
    if frame[id_column].tolist() != expected[id_column].tolist():
        raise ValueError(
            "Submission IDs must match the expected IDs and their order exactly."
        )
    try:
        probabilities = frame[target].to_numpy(dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError("Submission probabilities must be numeric.") from error
    if (
        not np.isfinite(probabilities).all()
        or ((probabilities < 0) | (probabilities > 1)).any()
    ):
        raise ValueError(
            "Submission probabilities must be finite values between 0 and 1."
        )


def load_data(
    directory: Path, id_column: str = "id", target: str = "is_fraud"
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(directory / "train.csv")
    test = pd.read_csv(directory / "test.csv")
    sample = pd.read_csv(directory / "sample_submission.csv")
    if train.empty or test.empty:
        raise ValueError("Training and test data must not be empty.")
    validate_ids(train, id_column)
    validate_ids(test, id_column)
    if set(train[id_column]) & set(test[id_column]):
        raise ValueError("Training and test row IDs must be disjoint.")
    if (
        target not in train
        or train[target].isna().any()
        or set(train[target].unique()) != {0, 1}
    ):
        raise ValueError(
            f"{target}: expected a non-null binary target containing both 0 and 1."
        )
    features = [column for column in train if column not in (id_column, target)]
    if not features or set(test.columns) != {id_column, *features}:
        raise ValueError(
            "Training and test feature schemas must match, excluding target and ID."
        )
    if list(sample.columns) != [id_column, target]:
        raise ValueError("sample_submission.csv has an unexpected schema.")
    validate_ids(sample, id_column)
    if set(sample[id_column]) != set(test[id_column]):
        raise ValueError("Sample submission IDs must cover the test IDs exactly.")
    return train, test[[id_column, *features]], sample


def submission(
    test_ids: pd.Series, predictions: np.ndarray, sample: pd.DataFrame, target: str
) -> pd.DataFrame:
    values = pd.Series(np.asarray(predictions), index=test_ids)
    result = sample.copy()
    result[target] = values.reindex(sample[test_ids.name]).to_numpy()
    validate_submission(result, sample, test_ids.name, target)
    return result


def inspect_data(
    directory: Path, id_column: str = "id", target: str = "is_fraud"
) -> dict:
    train, test, _ = load_data(directory, id_column, target)
    features = [column for column in test if column != id_column]
    columns = {}
    for name in features:
        columns[name] = {
            "dtype": str(train[name].dtype),
            "train_missing": int(train[name].isna().sum()),
            "test_missing": int(test[name].isna().sum()),
            "train_unique": int(train[name].nunique(dropna=False)),
            "test_unique": int(test[name].nunique(dropna=False)),
        }
    duplicate_mask = train.duplicated(features, keep=False)
    return {
        "rows": {"train": len(train), "test": len(test)},
        "target": {
            "name": target,
            "positive_rate": float(train[target].mean()),
            "counts": {str(k): int(v) for k, v in train[target].value_counts().items()},
        },
        "features": columns,
        "duplicate_feature_rows": int(duplicate_mask.sum()),
        "train_test_feature_overlap": int(
            pd.util.hash_pandas_object(train[features], index=False)
            .isin(pd.util.hash_pandas_object(test[features], index=False))
            .sum()
        ),
        "potential_split_columns": [
            name
            for name in features
            if any(
                word in name.lower()
                for word in (
                    "date",
                    "time",
                    "customer",
                    "account",
                    "user",
                    "card_id",
                    "merchant_id",
                )
            )
        ],
        "validation_warning": "Inspect time, entities and duplicate rows before choosing a split. Candidate names alone do not establish the required split.",
        "data_hashes": {
            name: fingerprint(directory / name)
            for name in ("train.csv", "test.csv", "sample_submission.csv")
        },
    }


In [ ]:
%%writefile r23-source/src/fraudlab/experiment.py
"""Cross-validation, immutable run artifacts and explicit-weight blending."""

from __future__ import annotations

import importlib.metadata
import json
import platform
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from .data import (
    fingerprint,
    load_data,
    submission,
    validate_ids,
    validate_submission,
    write_json,
)
from .features import transform_features
from .models import feature_types, fit_predict
from .splits import make_splits


def versions() -> dict:
    result = {"python": platform.python_version()}
    for package in (
        "fraudlab",
        "numpy",
        "pandas",
        "scikit-learn",
        "catboost",
        "lightgbm",
        "torch",
        "tabm",
        "rtdl_num_embeddings",
        "pytabkit",
        "pytorch-lightning",
        "torchmetrics",
        "lightning-utilities",
        "cloudpickle",
        "tiktoken",
        "regex",
    ):
        try:
            result[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            continue
    return result


def git_revision() -> str | None:
    result = subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=False
    )
    return result.stdout.strip() if result.returncode == 0 else None


def source_provenance(feature_set: str | None = None) -> dict:
    status = subprocess.run(
        ["git", "status", "--porcelain"], capture_output=True, text=True, check=False
    )
    result = {
        "source_hashes": {
            file.name: fingerprint(file)
            for file in sorted(Path(__file__).parent.glob("*.py"))
        },
        "git_revision": git_revision(),
        "git_dirty": bool(status.stdout.strip()) if status.returncode == 0 else None,
        "source_hashes_timing": "captured before reading data or fitting models",
    }
    if feature_set == "bpe_digits":
        from .bpe import manifest

        result["feature_encoding"] = manifest()
    elif feature_set == "token_digits":
        from .tokens import manifest

        result["feature_encoding"] = manifest()
    elif feature_set == "joint_bpe":
        from .joints import manifest

        result["feature_encoding"] = manifest()
    return result


def create_run(output: Path, name: str) -> Path:
    if not name or Path(name).name != name or name in (".", ".."):
        raise ValueError("Run name must be a single directory name.")
    directory = output / name
    try:
        directory.mkdir(parents=True, exist_ok=False)
    except FileExistsError as error:
        raise FileExistsError(
            f"Run already exists: {directory}. Choose a new run name."
        ) from error
    return directory


def train(config: dict) -> dict:
    started = time.perf_counter()
    if "_frequency_reference" in config:
        raise ValueError(
            "Frequency reference maps are runtime-only, not persisted configuration."
        )
    if config.get("glm_transductive_frequency"):
        from .transductive import (
            build_reference_counts,
            validate_config as validate_transductive,
        )

        validate_transductive(config)
    original_features = original_labels = original_metadata = None
    if (
        config.get("original_data")
        or config.get("original_sha256")
        or config.get("original_id_column", "transaction_id") != "transaction_id"
    ):
        from .original import load_original, validate_config, verify_unchanged

        validate_config(config)
    provenance = source_provenance(config.get("feature_set"))
    data_dir = Path(config["data"])
    frame, test, sample = load_data(data_dir, config["id_column"], config["target"])
    target, id_column = config["target"], config["id_column"]
    total_rows = len(frame)
    if config.get("original_data"):
        original_path = Path(config["original_data"])
        original_features, original_labels, original_metadata = load_original(
            original_path,
            config["original_sha256"],
            frame,
            test,
            original_id_column=config.get("original_id_column", "transaction_id"),
            id_column=id_column,
            target=target,
        )
        original_features = transform_features(
            original_features, config.get("feature_set", "raw"), (id_column, target)
        )
    holdout_fraction = config.get("holdout_fraction", 0.0)
    if holdout_fraction:
        development, _ = train_test_split(
            np.arange(len(frame)),
            test_size=holdout_fraction,
            stratify=frame[target],
            random_state=config["holdout_seed"],
        )
        frame = frame.iloc[np.sort(development)].reset_index(drop=True)
    frame = transform_features(
        frame, config.get("feature_set", "raw"), (id_column, target)
    )
    test = transform_features(
        test, config.get("feature_set", "raw"), (id_column, target)
    )
    excluded = [id_column, target]
    if config["split"] == "group":
        excluded.append(config["group_column"])
    if config["split"] == "time":
        excluded.append(config["time_column"])
    features = [name for name in frame if name not in excluded]
    if not features:
        raise ValueError(
            "No model features remain after excluding ID, target and split columns."
        )
    if original_features is not None and original_features.columns.tolist() != features:
        raise ValueError(
            "Engineered original feature columns must match competition order."
        )
    frequency_metadata = None
    fit_config = config
    if config.get("glm_transductive_frequency"):
        corpus_hashes = {
            name: fingerprint(data_dir / name)
            for name in ("train.csv", "test.csv", "sample_submission.csv")
        }
        references, frequency_metadata = build_reference_counts(
            frame, test, features, corpus_hashes, excluded=(id_column, target)
        )
        config = {**config, "transductive_frequency": frequency_metadata}
        fit_config = {**config, "_frequency_reference": references}
    splits = make_splits(
        frame,
        target,
        config["split"],
        config["folds"],
        config["seed"],
        config["group_column"],
        config["time_column"],
    )
    run = create_run(Path(config["output"]), config["name"])
    write_json(run / "config.json", config)
    config_hash = fingerprint(run / "config.json")
    write_json(
        run / "status.json",
        {"status": "running", "started_at": datetime.now(timezone.utc).isoformat()},
    )
    hashes = {
        name: fingerprint(data_dir / name)
        for name in ("train.csv", "test.csv", "sample_submission.csv")
    }
    oof = np.full(len(frame), np.nan)
    fold_ids = np.full(len(frame), -1, dtype=int)
    test_predictions = np.zeros(len(test))
    scores, details = [], []
    try:
        if (
            frequency_metadata is not None
            and hashes != frequency_metadata["corpus_hashes"]
        ):
            raise ValueError(
                "Official corpus bytes changed while constructing reference counts."
            )
        for fold, (training_rows, validation_rows) in enumerate(splits):
            training = frame.iloc[training_rows][features]
            labels = frame.iloc[training_rows][target].to_numpy()
            if original_features is not None:
                verify_unchanged(original_path, config["original_sha256"])
                training = pd.concat([training, original_features], ignore_index=True)
                labels = np.concatenate([labels, original_labels])
            valid_prediction, test_prediction, model_details = fit_predict(
                config["model"],
                training,
                labels,
                frame.iloc[validation_rows][features],
                frame.iloc[validation_rows][target].to_numpy(),
                test[features],
                fit_config,
                config["seed"] + fold,
                run / f"fold-{fold}",
            )
            if original_features is not None:
                verify_unchanged(original_path, config["original_sha256"])
                model_details.update(
                    training_rows=len(training),
                    competition_training_rows=len(training_rows),
                    additional_training_rows=len(original_labels),
                )
            if (
                not np.isfinite(valid_prediction).all()
                or not np.isfinite(test_prediction).all()
            ):
                raise ValueError("Model produced non-finite predictions.")
            oof[validation_rows] = valid_prediction
            fold_ids[validation_rows] = fold
            test_predictions += test_prediction / len(splits)
            score = float(
                roc_auc_score(frame.iloc[validation_rows][target], valid_prediction)
            )
            scores.append(score)
            details.append(model_details)
            print(
                f"fold={fold} roc_auc={score:.8f} train={len(training)} validation={len(validation_rows)}",
                flush=True,
            )
        covered = np.isfinite(oof)
        oof_frame = pd.DataFrame(
            {
                id_column: frame.loc[covered, id_column],
                "fold": fold_ids[covered],
                "target": frame.loc[covered, target],
                "prediction": oof[covered],
            }
        )
        result = submission(test[id_column], test_predictions, sample, target)
        if original_features is not None:
            verify_unchanged(original_path, config["original_sha256"])
        oof_frame.to_csv(run / "oof.csv", index=False)
        result.to_csv(run / "predictions.csv", index=False)
        result.to_csv(run / "submission.csv", index=False)
        numeric, categorical = feature_types(frame[features])
        metrics = {
            "status": "complete",
            "phase": config.get("phase", "development"),
            "config_sha256": config_hash,
            "model": config["model"],
            "roc_auc": float(roc_auc_score(frame.loc[covered, target], oof[covered])),
            "fold_scores": scores,
            "fold_std": float(np.std(scores)),
            "rows": {
                "train": total_rows,
                "test": len(test),
                "oof": int(covered.sum()),
                **(
                    {"development": len(frame), "holdout": total_rows - len(frame)}
                    if holdout_fraction
                    else {}
                ),
            },
            "features": {
                "count": len(features),
                "numeric": len(numeric),
                "categorical": len(categorical),
                "names": features,
                "excluded": list(dict.fromkeys(excluded)),
            },
            "target": {"name": target, "positive_rate": float(frame[target].mean())},
            "seed": config["seed"],
            "validation": {
                "method": config["split"],
                "n_splits": len(splits),
                "note": config["validation_note"],
                "group_column": config["group_column"],
                "time_column": config["time_column"],
                "holdout_fraction": holdout_fraction,
                "holdout_seed": (
                    config.get("holdout_seed") if holdout_fraction else None
                ),
                "independent_holdout": bool(holdout_fraction),
                "early_stopping_uses_validation": config["model"]
                in ("catboost", "lightgbm", "mlp")
                and not config.get("fixed_training"),
                **(
                    {
                        "inner_epoch_selection": {
                            "fraction": 0.15,
                            "seed": 42,
                            "metric": "roc_auc",
                            "fresh_outer_refit": True,
                        }
                    }
                    if config["model"] in ("tabm", "neural_offset", "realmlp")
                    else {}
                ),
            },
            "elapsed_seconds": round(time.perf_counter() - started, 3),
            "versions": versions(),
            "data_hashes": hashes,
            **provenance,
            "model_details": {
                **details[0],
                "feature_set": config.get("feature_set", "raw"),
                "folds": details,
            },
            "created_at": datetime.now(timezone.utc).isoformat(),
            "submission_sha256": fingerprint(run / "submission.csv"),
            "limitations": [
                "OOF scores used for model selection are not an independent final assessment.",
                "Public leaderboard and final private leaderboard scores must be obtained from Kaggle.",
                "Temporal OOF, when selected, excludes the initial training-only window.",
                *(
                    [
                        "Exploratory full-data CV uses all labels, including the historical holdout; no independent holdout remains for this stage."
                    ]
                    if config.get("phase") == "exploratory_full_data_cv"
                    else []
                ),
            ],
        }
        if original_metadata is not None:
            metrics["additional_training_data"] = {
                **original_metadata,
                "component_run": str(run),
                "component_model": config["model"],
                "component_scope": "this fitted component only; other blend sources are not retrained",
                "fold_training_rows": [fold["training_rows"] for fold in details],
            }
        if frequency_metadata is not None:
            metrics["transductive_frequency"] = frequency_metadata
            metrics["validation"]["transductive_feature_counts"] = {
                "rows": frequency_metadata["rows"]["total"],
                "label_usage": "none",
                "counts_sha256": frequency_metadata["counts_sha256"],
                "scalers_and_targets": "outer-training only",
            }
        write_json(run / "metrics.json", metrics)
        write_json(run / "status.json", {"status": "complete"})
        return metrics
    except Exception as error:
        write_json(run / "status.json", {"status": "failed", "error": str(error)})
        raise


def blend(
    runs: list[Path],
    weights: list[float] | None,
    output: Path,
    name: str,
    id_column: str = "id",
    target: str = "is_fraud",
    space: str = "probability",
) -> dict:
    from .scores import LOGIT_CLIP, SPACES, blend_scores

    started = time.perf_counter()
    provenance = source_provenance()
    if space not in SPACES:
        raise ValueError(f"Unknown score space: {space}")
    if len(runs) < 2:
        raise ValueError("Blending requires at least two runs.")
    coefficients = np.asarray(
        weights if weights is not None else [1.0] * len(runs), dtype=float
    ).copy()
    if (
        coefficients.shape != (len(runs),)
        or not np.isfinite(coefficients).all()
        or (coefficients < 0).any()
        or not np.isfinite(coefficients.sum())
        or coefficients.sum() <= 0
    ):
        raise ValueError(
            "Provide one finite nonnegative weight per run, with a positive total."
        )
    coefficients /= coefficients.sum()
    metrics = [
        json.loads((run / "metrics.json").read_text(encoding="utf-8")) for run in runs
    ]
    phases = [metric.get("phase") == "full_data_refit" for metric in metrics]
    if any(phases):
        if not all(phases):
            raise ValueError("Cannot mix development OOF runs with full-data refits.")
        if space != "probability":
            raise ValueError("Full-data refits support probability blending only.")
        from .refit import blend_refits

        return blend_refits(runs, weights, output, name, id_column, target)
    for coefficient, metric in zip(coefficients, metrics):
        components = metric.get("model_details", {}).get("external_components", [])
        if not components or coefficient <= 0:
            continue
        source_space = metric.get("model_details", {}).get(
            "mixture_space", "probability"
        )
        if (
            source_space != "probability"
            or any(
                component.get("mixture_space", "probability") != "probability"
                for component in components
            )
            or (space != "probability" and metric.get("model") == "blend")
        ):
            raise ValueError(
                "Cannot flatten nested external mixtures across nonlinear score spaces."
            )
        if space != "probability" and (
            len(components) != 1 or components[0].get("weight", 1.0) != 1.0
        ):
            raise ValueError(
                "Nonlinear external blending requires an atomic unit-weight source."
            )
    oofs = [pd.read_csv(run / "oof.csv") for run in runs]
    predictions = [pd.read_csv(run / "predictions.csv") for run in runs]
    reference, expected = oofs[0], predictions[0]
    validate_ids(reference, id_column)
    for index, (score, oof, prediction) in enumerate(zip(metrics, oofs, predictions)):
        if (
            score.get("status") != "complete"
            or score["data_hashes"] != metrics[0]["data_hashes"]
        ):
            raise ValueError(
                "Blend sources must be complete runs with identical data fingerprints."
            )
        if not reference[[id_column, "fold", "target"]].equals(
            oof[[id_column, "fold", "target"]]
        ):
            raise ValueError(
                "Blend sources must use exactly the same OOF rows, order, folds and labels."
            )
        if score["validation"] != metrics[0]["validation"]:
            # Models can differ in early stopping; the actual split contract cannot.
            fields = (
                "method",
                "n_splits",
                "group_column",
                "time_column",
                "holdout_fraction",
            )
            if any(
                score["validation"].get(field) != metrics[0]["validation"].get(field)
                for field in fields
            ):
                raise ValueError("Blend sources must use the same validation strategy.")
            if metrics[0]["validation"].get("holdout_fraction", 0) and score[
                "validation"
            ].get("holdout_seed") != metrics[0]["validation"].get("holdout_seed"):
                raise ValueError("Blend sources must reserve exactly the same holdout.")
        validate_submission(prediction, expected, id_column, target)
        if (
            not np.isfinite(oof["prediction"]).all()
            or not oof["prediction"].between(0, 1).all()
        ):
            raise ValueError(f"Source {index} has invalid OOF probabilities.")
    oof_result, prediction_result = reference.copy(), expected.copy()
    if space == "probability":
        # Keep the original arithmetic and its summation order exactly.
        oof_result["prediction"] = sum(
            weight * frame["prediction"].to_numpy()
            for weight, frame in zip(coefficients, oofs)
        )
        prediction_result[target] = sum(
            weight * frame[target].to_numpy()
            for weight, frame in zip(coefficients, predictions)
        )
    else:
        oof_result["prediction"] = blend_scores(
            [frame["prediction"].to_numpy() for frame in oofs], weights, space=space
        )
        prediction_result[target] = blend_scores(
            [frame[target].to_numpy() for frame in predictions], weights, space=space
        )
    fold_scores = [
        float(roc_auc_score(group["target"], group["prediction"]))
        for _, group in oof_result.groupby("fold")
    ]
    run = create_run(output, name)
    oof_result.to_csv(run / "oof.csv", index=False)
    prediction_result.to_csv(run / "predictions.csv", index=False)
    prediction_result.to_csv(run / "submission.csv", index=False)
    external_components = {}
    for weight, metric, source in zip(coefficients, metrics, runs):
        if weight <= 0:
            continue
        for component in metric.get("model_details", {}).get("external_components", []):
            entry = {**component, "mixture_space": space}
            if space == "probability":
                entry["weight"] = float(weight) * component.get("weight", 1.0)
                key = entry["predictions_source_url"]
            else:
                entry.pop("weight", None)
                entry.update(
                    coefficient=float(weight),
                    coefficient_space=space,
                    coefficient_interpretation="direct source coefficient, not a probability share",
                    source_run=str(source),
                    source_prediction_space="probability",
                )
                key = (entry["predictions_source_url"], str(source))
            if key in external_components:
                field = "weight" if space == "probability" else "coefficient"
                external_components[key][field] += entry[field]
            else:
                external_components[key] = entry
    result = {
        **metrics[0],
        "model": "blend",
        "external_components": list(external_components.values()),
        "roc_auc": float(roc_auc_score(oof_result["target"], oof_result["prediction"])),
        "fold_scores": fold_scores,
        "fold_std": float(np.std(fold_scores)),
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "created_at": datetime.now(timezone.utc).isoformat(),
        **provenance,
        "model_details": {
            "method": {
                "probability": "weighted_probability_average",
                "logit": "sigmoid_weighted_logit_average",
                "rank": "weighted_normalized_rank_average",
                "rank_logit": "sigmoid_weighted_rank_logit_average",
            }[space],
            "sources": [str(source) for source in runs],
            "weights": coefficients.tolist(),
            "external_components": list(external_components.values()),
            "mixture_space": space,
        },
        "submission_sha256": fingerprint(run / "submission.csv"),
    }
    blend_config = {
        "sources": [str(source) for source in runs],
        "weights": coefficients.tolist(),
    }
    if space != "probability":
        blend_config["space"] = space
        result["model_details"]["score_transform"] = {
            "input_space": "probability",
            "coefficient_space": space,
            "logit_clip": LOGIT_CLIP if space in ("logit", "rank_logit") else None,
            "rank_method": "average" if space in ("rank", "rank_logit") else None,
            "rank_normalization": (
                "rank/(n+1)" if space in ("rank", "rank_logit") else None
            ),
            "rank_scope": (
                "full OOF vector and full test vector separately; no labels"
                if space in ("rank", "rank_logit")
                else None
            ),
        }
    write_json(
        run / "config.json",
        blend_config,
    )
    result["config_sha256"] = fingerprint(run / "config.json")
    write_json(run / "metrics.json", result)
    write_json(run / "status.json", {"status": "complete"})
    return result


In [ ]:
%%writefile r23-source/src/fraudlab/external.py
"""Import one explicitly approved public prediction release; never execute its code."""

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from .data import fingerprint, load_data, submission, write_json
from .experiment import create_run, source_provenance, versions
from .models import feature_types
from .splits import make_splits

MANIFEST_SHA256 = "0083d23b46f71311ceeed66874b093ec927d25d98b37f6d2e78b1de17f0f22d0"
DATASET = "https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof"
CODE = "https://github.com/Matapanino/s6e11-gpt2-target-first"
CODE_REVISION = "8446fb73c11583df4f7006aa08fa34472249b318"


def import_public_gpt2(source: Path, data: Path, output: Path, name: str) -> dict:
    provenance = source_provenance()
    if fingerprint(source / "manifest.json") != MANIFEST_SHA256:
        raise ValueError(
            "Public manifest does not match the reviewed, approved release."
        )
    manifest = json.loads((source / "manifest.json").read_text(encoding="utf-8"))
    hashes = {
        filename: fingerprint(data / filename)
        for filename in ("train.csv", "test.csv", "sample_submission.csv")
    }
    if (
        hashes["train.csv"] != manifest["inputs"]["train_csv"]["sha256"]
        or hashes["test.csv"] != manifest["inputs"]["test_csv"]["sha256"]
    ):
        raise ValueError(
            "Public arrays are not aligned with these exact train/test CSV bytes."
        )
    expected = {
        **manifest["sha256"],
        "fold_ids.npy": manifest["inputs"]["fold_ids"]["sha256"],
    }
    arrays = {}
    for filename in ("oof.npy", "test.npy", "fold_ids.npy"):
        if fingerprint(source / filename) != expected[filename]:
            raise ValueError(f"Public array hash mismatch: {filename}")
        arrays[filename] = np.load(source / filename, allow_pickle=False)
    train, test, sample = load_data(data)
    for filename, rows in (("oof.npy", len(train)), ("test.npy", len(test))):
        array = arrays[filename]
        if (
            array.shape != (rows,)
            or array.dtype != np.float64
            or not np.isfinite(array).all()
        ):
            raise ValueError(
                f"Expected finite float64 logits in original CSV row order: {filename}"
            )
    fold_ids = arrays["fold_ids.npy"]
    if fold_ids.shape != (len(train),) or fold_ids.dtype != np.int8:
        raise ValueError("Expected int8 fold IDs, one per original training row.")
    expected_folds = np.full(len(train), -1, dtype=np.int8)
    for fold, (_, validation) in enumerate(
        make_splits(train, "is_fraud", "stratified", 5, 42)
    ):
        expected_folds[validation] = fold
    if not np.array_equal(fold_ids, expected_folds):
        raise ValueError(
            "Public folds differ from exact StratifiedKFold(5, shuffle=True, seed42)."
        )
    # Stable sigmoid: the approved arrays contain log-odds, not probabilities.
    oof_probability = np.exp(-np.logaddexp(0.0, -arrays["oof.npy"]))
    test_probability = np.exp(-np.logaddexp(0.0, -arrays["test.npy"]))
    score = float(roc_auc_score(train["is_fraud"], oof_probability))
    if abs(score - manifest["auc"]["pooled"]) > 1e-10:
        raise ValueError(
            "Reproduced public OOF AUC differs from the approved manifest."
        )
    fold_scores = [
        float(
            roc_auc_score(
                train.loc[fold_ids == fold, "is_fraud"],
                oof_probability[fold_ids == fold],
            )
        )
        for fold in range(5)
    ]
    result = submission(test["id"], test_probability, sample, "is_fraud")
    run = create_run(output, name)
    pd.DataFrame(
        {
            "id": train["id"],
            "fold": fold_ids,
            "target": train["is_fraud"],
            "prediction": oof_probability,
        }
    ).to_csv(run / "oof.csv", index=False)
    result.to_csv(run / "predictions.csv", index=False)
    result.to_csv(run / "submission.csv", index=False)
    raw_features = train.drop(columns=["id", "is_fraud"])
    numeric, categorical = feature_types(raw_features)
    component = {
        "name": "GPT-2 target-first",
        "kind": "gpt2_target_first",
        "source_type": "external_predictions",
        "trained_by": "Masaya Kawamata",
        "model_architecture": "GPT-2 target-first classifier",
        "training_code_url": CODE,
        "training_code_commit": CODE_REVISION,
        "code_license": "MIT",
        "predictions_source_url": DATASET,
        "predictions_source_version": 2,
        "predictions_download_ref": "masayakawamata/s6e11-gpt2-target-first-oof/2",
        "predictions_license": "CC BY 4.0",
        "data_license": "CC BY 4.0 (prediction dataset)",
        "method_credits": [
            "Chris Deotte: target-first GPT-2 method",
            "Borisov et al.: GReaT / Language Models are Realistic Tabular Data Generators",
            "uditjain13: original Credit Card Fraud Detection 2026 dataset",
        ],
        "training_executed_locally": False,
        "trained_by_us": False,
        "oof_auc": score,
        "validation": {
            "method": "StratifiedKFold",
            "n_splits": 5,
            "shuffle": True,
            "seed": 42,
        },
        "inference": {
            "method": "target-first label likelihood ratio",
            "target_first": True,
            "likelihood_ratio": True,
            "prior": manifest["settings"]["prior"],
            "orders": manifest["settings"]["orders"],
            "order_aggregation": manifest["settings"]["combine"],
            "test_aggregation": "mean of fold log-odds",
            "test_ensemble_folds": 5,
            "source_test_aggregation": manifest["settings"]["test"],
            "import_transform": "sigmoid(log_odds), float64",
        },
    }
    metrics = {
        "status": "complete",
        "model": "external_gpt2",
        "external_components": [component],
        "phase": "exploratory_full_data_cv",
        "roc_auc": score,
        "fold_scores": fold_scores,
        "fold_std": float(np.std(fold_scores)),
        "rows": {"train": len(train), "test": len(test), "oof": len(train)},
        "features": {
            "count": raw_features.shape[1],
            "numeric": len(numeric),
            "categorical": len(categorical),
            "names": raw_features.columns.tolist(),
            "excluded": ["id", "is_fraud"],
        },
        "target": {
            "name": "is_fraud",
            "positive_rate": float(train["is_fraud"].mean()),
        },
        "seed": 42,
        "validation": {
            "method": "stratified",
            "n_splits": 5,
            "note": "Exploratory full-data CV with externally trained predictions; all labels, including the historical holdout, were used in fold training. The old holdout is not independent for this stage.",
            "group_column": None,
            "time_column": None,
            "holdout_fraction": 0.0,
            "holdout_seed": None,
            "independent_holdout": False,
            "early_stopping_uses_validation": False,
        },
        "model_details": {
            "external_components": [component],
            "training_executed_locally": False,
        },
        "external_provenance": {
            "manifest_sha256": MANIFEST_SHA256,
            "array_sha256": expected,
            "fold_ids_reproduced": True,
            "row_alignment": "Exact raw train/test SHA256 and original row order; no sorting by ID",
        },
        "data_hashes": hashes,
        "versions": versions(),
        **provenance,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "submission_sha256": fingerprint(run / "submission.csv"),
        "limitations": [
            "This neural component was trained by an external author, not by FraudLab. Preserve CC BY 4.0 attribution.",
            "No independent holdout is available for this exploratory full-data stage; the earlier holdout report applies only to the original protocol.",
            "Public test predictions average fold log-odds before sigmoid; OOF rows use only their excluded-fold model.",
        ],
    }
    write_json(
        run / "config.json",
        {
            "model": "external_gpt2",
            "phase": "exploratory_full_data_cv",
            "source": DATASET,
            "manifest_sha256": MANIFEST_SHA256,
            "logits_to_probabilities": "sigmoid",
        },
    )
    write_json(run / "metrics.json", metrics)
    write_json(run / "status.json", {"status": "complete"})
    return metrics


In [ ]:
%%writefile r23-source/src/fraudlab/features.py
"""Small deterministic hypotheses; every feature set is compared by ablation."""

from __future__ import annotations

import numpy as np
import pandas as pd

from .strings import categorical_strings

CONTINUOUS = (
    "amount_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "account_balance_usd",
    "velocity_score",
    "merchant_risk_score",
)


def transform_features(
    frame: pd.DataFrame,
    feature_set: str = "raw",
    excluded: tuple[str, ...] = ("id", "is_fraud"),
) -> pd.DataFrame:
    if feature_set == "joint_bpe":
        from .joints import joint_keys

        keys = joint_keys(frame, excluded)
        base = transform_features(frame, "bpe_digits", excluded)
        return pd.concat([base, keys], axis=1)
    if feature_set == "token_digits":
        from .tokens import token_keys

        keys = token_keys(frame, excluded)
        base = transform_features(frame, "value_digits", excluded)
        return pd.concat([base, keys], axis=1)
    if feature_set == "bpe_digits":
        from .bpe import bpe_keys

        base = transform_features(frame, "value_digits", excluded)
        return pd.concat([base, bpe_keys(frame, excluded)], axis=1)
    if feature_set == "raw":
        return frame.copy()
    if feature_set not in ("behavior", "digits", "value_digits", "paired_digits"):
        raise ValueError(f"Unknown feature set: {feature_set}")
    result = frame.copy()
    for name in CONTINUOUS:
        if name in result and name not in excluded:
            values = result[name].to_numpy(dtype=float)
            result[name + "__log1p"] = np.sign(values) * np.log1p(np.abs(values))
    for name, period in (("time_of_day_hour", 24), ("day_of_week", 7)):
        if name in result and name not in excluded:
            angle = 2 * np.pi * result[name] / period
            result[name + "__sin"] = np.sin(angle)
            result[name + "__cos"] = np.cos(angle)
    if {"amount_usd", "account_balance_usd"}.issubset(result) and not {
        "amount_usd",
        "account_balance_usd",
    }.intersection(excluded):
        result["amount_balance_ratio"] = result["amount_usd"] / (
            np.abs(result["account_balance_usd"]) + 1
        )
    if feature_set in ("digits", "value_digits", "paired_digits"):
        # These views test synthetic rounding patterns; they are not banking rules.
        for name in CONTINUOUS:
            if name not in result or name in excluded:
                continue
            values = np.abs(result[name].to_numpy(dtype=float))
            if not np.isfinite(values).all():
                raise ValueError("Digit views require finite numeric inputs.")
            cents = np.rint(values * 100).astype(np.int64)
            magnitude = np.floor(np.log10(np.maximum(values, 1e-12))).astype(int)
            leading = np.floor(values / np.power(10.0, magnitude)).astype(int)
            result[name + "__value_key"] = categorical_strings(cents)
            result[name + "__cents_key"] = categorical_strings(cents % 100)
            result[name + "__leading_key"] = categorical_strings(leading)
            if feature_set in ("value_digits", "paired_digits"):
                leading_two = np.floor(values / np.power(10.0, magnitude - 1)).astype(
                    int
                )
                result[name + "__integer_key"] = categorical_strings(cents // 100)
                result[name + "__tenths_key"] = categorical_strings(cents // 10)
                result[name + "__leading_two_key"] = categorical_strings(leading_two)
        if feature_set in ("value_digits", "paired_digits"):
            for name in frame.select_dtypes(include="number"):
                if name not in CONTINUOUS and name not in excluded:
                    result[name + "__value_key"] = frame[name].astype(str)
    if feature_set == "paired_digits":
        # Four domain/audit hypotheses fixed before CV; no label-based pair search.
        pairs = (
            ("cvv_retry_count", None, "auth_method", None),
            ("velocity_score", 5, "distance_from_home_km", 5),
            ("velocity_score", 5, "txn_count_last_24h", 3),
            ("velocity_score", 5, "hours_since_last_txn", 2),
        )
        for left, left_width, right, right_width in pairs:
            if not {left, right}.issubset(frame) or {left, right}.intersection(
                excluded
            ):
                continue
            left_key = frame[left] // left_width if left_width else frame[left]
            right_key = frame[right] // right_width if right_width else frame[right]
            # Tuple text escapes embedded separators and keeps the pair unambiguous.
            result[f"{left}__{right}__pair"] = pd.Series(
                list(zip(left_key, right_key)), index=frame.index
            ).astype(str)
    return result


In [ ]:
%%writefile r23-source/src/fraudlab/frequency.py
"""Label-free category counts: fold-local by default, optionally fixed."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class FrequencyEncoder(TransformerMixin, BaseEstimator):
    """Encode log1p support from training or explicit reference; unknowns are zero."""

    # Older persisted encoders have no instance attribute for this new option.
    reference_counts = None

    def __init__(self, reference_counts=None):
        self.reference_counts = reference_counts

    def fit(self, frame: pd.DataFrame, y=None):
        if (
            not isinstance(frame, pd.DataFrame)
            or not frame.columns.is_unique
            or not all(isinstance(name, str) for name in frame.columns)
        ):
            raise ValueError(
                "FrequencyEncoder requires unique string DataFrame columns."
            )
        self.feature_names_in_ = np.asarray(frame.columns, dtype=object)
        self.n_features_in_ = len(self.feature_names_in_)
        if self.reference_counts is None:
            self.counts_ = {
                name: frame[name].value_counts(dropna=False, sort=False)
                for name in self.feature_names_in_
            }
        else:
            if not isinstance(self.reference_counts, dict) or set(
                self.reference_counts
            ) != set(frame):
                raise ValueError(
                    "Reference counts must match frequency columns exactly."
                )
            self.counts_ = {}
            for name in self.feature_names_in_:
                counts = self.reference_counts[name]
                if (
                    not isinstance(counts, pd.Series)
                    or not len(counts)
                    or not all(isinstance(key, str) for key in counts.index)
                    or not counts.index.is_unique
                    or counts.index.hasnans
                    or not pd.api.types.is_numeric_dtype(counts)
                    or pd.api.types.is_bool_dtype(counts)
                    or pd.api.types.is_complex_dtype(counts)
                ):
                    raise ValueError(
                        "Reference counts need unique string keys and numeric counts."
                    )
                values = counts.to_numpy(dtype=np.float64)
                if (
                    not np.isfinite(values).all()
                    or (values <= 0).any()
                    or (values != np.floor(values)).any()
                    or (values >= 2**63).any()
                ):
                    raise ValueError(
                        "Reference counts must be finite positive integers."
                    )
                observed = frame[name].value_counts(dropna=False, sort=False)
                available = counts.reindex(observed.index)
                if (
                    available.isna().any()
                    or (available.to_numpy() < observed.to_numpy()).any()
                ):
                    raise ValueError("Reference counts must cover the fitting rows.")
                self.counts_[name] = counts.astype("int64").copy()
        return self

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        check_is_fitted(self, "counts_")
        if (
            not isinstance(frame, pd.DataFrame)
            or frame.columns.tolist() != self.feature_names_in_.tolist()
        ):
            raise ValueError("Frequency columns must match training columns and order.")
        encoded = np.empty((len(frame), self.n_features_in_), dtype=np.float64)
        for index, name in enumerate(self.feature_names_in_):
            counts = frame[name].map(self.counts_[name]).astype(np.float64).fillna(0)
            encoded[:, index] = np.log1p(counts.to_numpy(dtype=np.float64))
        return encoded

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        check_is_fitted(self, "counts_")
        if input_features is not None and not np.array_equal(
            input_features, self.feature_names_in_
        ):
            raise ValueError("Frequency feature names must match training columns.")
        return np.asarray(
            [f"{name}__log_count" for name in self.feature_names_in_], dtype=object
        )


In [ ]:
%%writefile r23-source/src/fraudlab/holdout.py
"""One final evaluation after development choices have been frozen."""

from __future__ import annotations

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from .data import fingerprint, load_data, write_json
from .features import transform_features
from .models import predict_saved


def evaluate(
    runs: list[Path],
    weights: list[float] | None,
    output: Path,
    device: str = "auto",
    development_run: Path | None = None,
) -> dict:
    if output.exists():
        raise FileExistsError(
            "Holdout evaluation already exists; do not reuse it for model selection."
        )
    if not runs:
        raise ValueError("At least one frozen source run is required.")
    coefficients = np.asarray(
        weights if weights is not None else [1.0] * len(runs), dtype=float
    )
    if (
        len(coefficients) != len(runs)
        or not np.isfinite(coefficients).all()
        or not np.isfinite(coefficients.sum())
        or (coefficients < 0).any()
        or coefficients.sum() <= 0
    ):
        raise ValueError(
            "Provide finite, nonnegative weights with a finite positive total."
        )
    coefficients /= coefficients.sum()
    configs = [
        json.loads((run / "config.json").read_text(encoding="utf-8")) for run in runs
    ]
    metrics = [
        json.loads((run / "metrics.json").read_text(encoding="utf-8")) for run in runs
    ]
    reference = configs[0]
    if not reference.get("holdout_fraction"):
        raise ValueError("Sources must have reserved a holdout before training.")
    fields = ("data", "target", "id_column", "holdout_fraction", "holdout_seed")
    for run, config, metric in zip(runs, configs, metrics):
        if (
            metric.get("config_sha256")
            and fingerprint(run / "config.json") != metric["config_sha256"]
        ):
            raise ValueError("Source configuration changed after training.")
        if config["model"] != metric["model"] or config.get(
            "feature_set", "raw"
        ) != metric["model_details"].get("feature_set", "raw"):
            raise ValueError(
                "Source model or feature set no longer matches its training metadata."
            )
        if any(
            config.get(field) != metric["validation"].get(field)
            for field in ("holdout_fraction", "holdout_seed")
        ):
            raise ValueError(
                "Source config no longer matches the holdout recorded at training time."
            )
        if (
            any(config.get(field) != reference.get(field) for field in fields)
            or metric.get("status") != "complete"
            or metric["data_hashes"] != metrics[0]["data_hashes"]
        ):
            raise ValueError(
                "Sources must be complete runs sharing the same untouched holdout and input data."
            )
    data_dir = Path(reference["data"])
    for filename, expected in metrics[0]["data_hashes"].items():
        if fingerprint(data_dir / filename) != expected:
            raise ValueError("Input files changed after training.")
    frame, _, _ = load_data(data_dir, reference["id_column"], reference["target"])
    _, held_out = train_test_split(
        np.arange(len(frame)),
        test_size=reference["holdout_fraction"],
        stratify=frame[reference["target"]],
        random_state=reference["holdout_seed"],
    )
    frame = frame.iloc[np.sort(held_out)].reset_index(drop=True)
    held_out_ids = set(frame[reference["id_column"]])
    for run in runs:
        if held_out_ids.intersection(
            pd.read_csv(run / "oof.csv", usecols=[reference["id_column"]])[
                reference["id_column"]
            ]
        ):
            raise ValueError("Holdout rows were found in development OOF predictions.")
    output.parent.mkdir(parents=True, exist_ok=True)
    # Persist the chosen configuration before the first holdout prediction is evaluated.
    selection = {
        "sources": [str(run) for run in runs],
        "weights": coefficients.tolist(),
        "metrics_hashes": [fingerprint(run / "metrics.json") for run in runs],
        "config_hashes": [fingerprint(run / "config.json") for run in runs],
        "frozen_at": datetime.now(timezone.utc).isoformat(),
    }
    if development_run is not None:
        dev_config = json.loads(
            (development_run / "config.json").read_text(encoding="utf-8")
        )
        dev_metric = json.loads(
            (development_run / "metrics.json").read_text(encoding="utf-8")
        )
        if (
            [Path(source).resolve() for source in dev_config["sources"]]
            != [source.resolve() for source in runs]
            or not np.allclose(dev_config["weights"], coefficients, rtol=0, atol=1e-12)
            or dev_metric.get("status") != "complete"
            or dev_metric["data_hashes"] != metrics[0]["data_hashes"]
        ):
            raise ValueError(
                "Development blend does not match the frozen sources, weights and input data."
            )
        selection.update(
            source_dev_run=str(development_run),
            development_roc_auc=dev_metric["roc_auc"],
            development_metrics_sha256=fingerprint(development_run / "metrics.json"),
        )
    partition_key = hashlib.sha256(
        json.dumps(
            {
                "train_sha256": metrics[0]["data_hashes"]["train.csv"],
                "target": reference["target"],
                "seed": reference["holdout_seed"],
                "fraction": reference["holdout_fraction"],
            },
            sort_keys=True,
        ).encode()
    ).hexdigest()
    ledger_dir = data_dir / ".holdout-evaluations"
    ledger_dir.mkdir(exist_ok=True)
    ledger = ledger_dir / f"{partition_key}.json"
    selection_key = hashlib.sha256(
        json.dumps(
            {key: value for key, value in selection.items() if key != "frozen_at"},
            sort_keys=True,
        ).encode()
    ).hexdigest()
    if ledger.exists():
        prior = json.loads(ledger.read_text(encoding="utf-8"))
        if prior["status"] == "complete" or prior["selection_key"] != selection_key:
            raise ValueError(
                "This independent holdout partition is already reserved or evaluated; do not retune from it."
            )
    else:
        # The dataset-local ledger follows this partition even if --output changes.
        # A failed evaluation may resume only with the identical frozen sources.
        with ledger.open("x", encoding="utf-8") as stream:
            json.dump(
                {
                    "status": "started",
                    "selection_key": selection_key,
                    "selection": selection,
                },
                stream,
                indent=2,
            )
    write_json(output.with_name(output.stem + "-frozen-selection.json"), selection)
    predictions = np.zeros(len(frame))
    individual = []
    for weight, run, config, metric in zip(coefficients, runs, configs, metrics):
        model_predictions = np.zeros(len(frame))
        inputs = transform_features(
            frame,
            config.get("feature_set", "raw"),
            (config["id_column"], config["target"]),
        )[metric["features"]["names"]]
        for fold in range(metric["validation"]["n_splits"]):
            model_predictions += (
                predict_saved(config["model"], run / f"fold-{fold}", inputs, device)
                / metric["validation"]["n_splits"]
            )
        predictions += weight * model_predictions
        individual.append(
            float(roc_auc_score(frame[reference["target"]], model_predictions))
        )
    if (
        not np.isfinite(predictions).all()
        or ((predictions < 0) | (predictions > 1)).any()
    ):
        raise ValueError("Invalid holdout predictions.")
    result = {
        "status": "complete",
        "kind": "independent_holdout",
        "evaluation_source_hashes": {
            file.name: fingerprint(file)
            for file in sorted(Path(__file__).parent.glob("*.py"))
        },
        "roc_auc": float(roc_auc_score(frame[reference["target"]], predictions)),
        "individual_roc_auc": individual,
        "rows": len(frame),
        "positive_rows": int(frame[reference["target"]].sum()),
        "holdout_seed": reference["holdout_seed"],
        "holdout_fraction": reference["holdout_fraction"],
        "selection": selection,
        "data_hashes": metrics[0]["data_hashes"],
        "evaluated_at": datetime.now(timezone.utc).isoformat(),
        "limitation": "The selection was frozen before this evaluation. Using this result to retune would make the holdout part of model selection.",
    }
    pd.DataFrame(
        {
            reference["id_column"]: frame[reference["id_column"]],
            "target": frame[reference["target"]],
            "prediction": predictions,
        }
    ).to_csv(output.with_suffix(".csv"), index=False)
    write_json(output, result)
    write_json(
        ledger,
        {
            "status": "complete",
            "selection_key": selection_key,
            "report_sha256": fingerprint(output),
            "evaluated_at": result["evaluated_at"],
        },
    )
    return result


In [ ]:
%%writefile r23-source/src/fraudlab/joints.py
"""Eight fixed interaction hypotheses, with no learned selection or statistics.

Public-source inspiration: megayak's joint-key ideas as credited in goodpjw2008's
Part 12, version 16. This extraction is independently written. No G-statistic
screen, quantile fitting, third-party code or prediction arrays are used.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

JOINTS = (
    ("foreign_ip", "is_foreign_transaction", "exact", "ip_country_mismatch", "exact"),
    ("category_risk10", "merchant_category", "exact", "merchant_risk_score", "floor10"),
    ("category_amount_log2", "merchant_category", "exact", "amount_usd", "log2"),
    (
        "scam_velocity5",
        "is_ai_generated_scam_attempt",
        "exact",
        "velocity_score",
        "floor5",
    ),
    ("newmerchant_weekday", "is_new_merchant", "exact", "day_of_week", "exact"),
    ("hour_weekday", "time_of_day_hour", "exact", "day_of_week", "exact"),
    ("age_cardyears", "customer_age", "floor10", "card_age_months", "floor12"),
    ("auth_channel", "auth_method", "exact", "channel", "exact"),
)
OUTPUT_NAMES = tuple(name + "__joint_key" for name, *_ in JOINTS)
MAX_VALUE = 2**53 - 1
# Exact float64 boundaries avoid log2(1+x) rounding up immediately below a tie.
LOG2_BOUNDARIES = np.asarray([2**power - 1 for power in range(54)], dtype=np.float64)
FORMULAS = {
    "exact": "unchanged string; numeric value normalized to float64",
    "floor5": "floor(float64(x)/5)",
    "floor10": "floor(float64(x)/10)",
    "floor12": "floor(float64(x)/12)",
    "log2": "floor(log2(1+x)); interval k starts at exact boundary 2**k-1",
}


def _numeric(series: pd.Series, *, allow_bool: bool = False) -> np.ndarray:
    if (
        not pd.api.types.is_numeric_dtype(series)
        or pd.api.types.is_complex_dtype(series)
        or (pd.api.types.is_bool_dtype(series) and not allow_bool)
    ):
        raise ValueError(f"{series.name} requires real numeric joint inputs.")
    values = series.to_numpy(dtype=np.float64, na_value=np.nan)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f"{series.name} requires finite nonnegative joint inputs.")
    if (values > MAX_VALUE).any():
        raise ValueError(f"{series.name} exceeds the supported joint integer range.")
    return np.where(values == 0, 0.0, values)


def _view(series: pd.Series, mode: str) -> list:
    if mode == "exact":
        if pd.api.types.is_numeric_dtype(series):
            return _numeric(series, allow_bool=True).tolist()
        values = series.tolist()
        if not all(isinstance(value, str) for value in values):
            raise ValueError(f"{series.name} requires nonmissing string categories.")
        return values
    values = _numeric(series)
    if mode == "log2":
        bins = np.searchsorted(LOG2_BOUNDARIES, values, side="right") - 1
    else:
        bins = np.floor(values / int(mode.removeprefix("floor")))
    return bins.astype(np.int64).tolist()


def joint_keys(frame: pd.DataFrame, excluded: tuple[str, ...] = ()) -> pd.DataFrame:
    """Return fixed categorical tuple keys; skip a pair if either field is excluded.

    Only the declared fields are read. IDs/targets/other columns have no effect.
    There is no fitted state, category vocabulary or dependence on other rows.
    """
    if not isinstance(frame, pd.DataFrame) or not frame.columns.is_unique:
        raise ValueError("Joint keys require unique DataFrame columns.")
    result = pd.DataFrame(index=frame.index)
    views = {}
    for name, left, left_mode, right, right_mode in JOINTS:
        if {left, right}.intersection(excluded):
            continue
        for column, mode in ((left, left_mode), (right, right_mode)):
            if column not in frame:
                raise ValueError(f"Missing joint input column: {column}")
            if (column, mode) not in views:
                views[column, mode] = _view(frame[column], mode)
        pairs = zip(views[left, left_mode], views[right, right_mode])
        result[name + "__joint_key"] = pd.Series(
            [repr(pair) for pair in pairs], index=frame.index, dtype=object
        )
    return result


def manifest() -> dict:
    from .bpe import manifest as bpe_manifest

    return {
        "algorithm": "bpe_digits_plus_eight_fixed_joint_categories",
        "base_feature_set": "bpe_digits",
        "bpe": bpe_manifest(),
        "joint_encoding": {
            "output_count": 8,
            "pairs": [
                {
                    "output": name + "__joint_key",
                    "left": left,
                    "left_view": FORMULAS[left_mode],
                    "right": right,
                    "right_view": FORMULAS[right_mode],
                }
                for name, left, left_mode, right, right_mode in JOINTS
            ],
            "category_representation": "repr((left, right)); strings escaped, numbers canonicalized as above; categorical labels, not ordinal magnitudes",
            "numeric_domain": "finite nonnegative float64 at most 2**53-1; negative zero normalized; bins use no cent rounding",
            "log2_boundaries": "2**k-1 for integer k=0..53, left-inclusive via searchsorted(side='right')-1",
            "excluded_fields": "Exclude the entire pair if either input is excluded",
            "selection": "Eight hypotheses fixed before fitting; public-source-informed exploratory selection; no G scan, label ranking or quantile fitting",
            "label_usage": "none in extraction; downstream TE/frequency/scaling remain fitted inside outer training only",
            "concept_source": {
                "url": "https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298",
                "version": 16,
                "source_cell_zero_based": 74,
                "source_body_sha256": "835772526d7efcc70f771aa446690cb1956676a457f40b08d48760bbf3336baf",
                "credit": "megayak's joint-key ideas; independent extraction, no source code copied or executed",
            },
        },
    }


In [ ]:
%%writefile r23-source/src/fraudlab/models.py
"""Small tabular models, each fitted inside its training fold."""

from __future__ import annotations

import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    QuantileTransformer,
    SplineTransformer,
    StandardScaler,
    TargetEncoder,
)
from threadpoolctl import threadpool_limits

from .features import CONTINUOUS


def feature_types(frame: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric = frame.select_dtypes(include="number").columns.tolist()
    return numeric, [name for name in frame if name not in numeric]


def clean_features(
    frame: pd.DataFrame,
    numeric: list[str] | None = None,
    categorical: list[str] | None = None,
) -> pd.DataFrame:
    result = frame.copy()
    if numeric is None or categorical is None:
        numeric, categorical = feature_types(result)
    for name in numeric:
        result[name] = pd.to_numeric(result[name], errors="raise")
    result[numeric] = result[numeric].replace([np.inf, -np.inf], np.nan)
    for name in categorical:
        result[name] = result[name].astype("string").fillna("__MISSING__").astype(str)
    return result


def log_odds(values: np.ndarray) -> np.ndarray:
    probabilities = np.clip(values, 1e-6, 1 - 1e-6)
    return np.log(probabilities / (1 - probabilities))


def fit_logistic(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    numeric, categorical = feature_types(training)
    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                Pipeline(
                    [
                        (
                            "impute",
                            SimpleImputer(strategy="median", keep_empty_features=True),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                numeric,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore", min_frequency=5, dtype=np.float32
                ),
                categorical,
            ),
        ],
        sparse_threshold=1.0,
    )
    model = Pipeline(
        [
            ("preprocessing", preprocessing),
            (
                "classifier",
                LogisticRegression(
                    C=config["logistic_c"], max_iter=1000, random_state=seed
                ),
            ),
        ]
    )
    with threadpool_limits(limits=config["threads"]):
        model.fit(training, y)
    joblib.dump(model, path.with_suffix(".joblib"))
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        {
            "algorithm": "logistic_regression",
            "C": config["logistic_c"],
            "solver": "lbfgs",
            "max_iter": 1000,
            "iterations_used": int(model.named_steps["classifier"].n_iter_[0]),
            "preprocessing": "Fold-local median imputation, standard scaling, one-hot encoding; unknown categories ignored; category min_frequency=5.",
        },
    )


def fit_catboost(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    from catboost import CatBoostClassifier

    _, categorical = feature_types(training)
    device = "GPU" if config["device"] == "cuda" else "CPU"
    parameters = {
        "iterations": config["iterations"],
        "depth": config["depth"],
        "learning_rate": config["learning_rate"],
        "l2_leaf_reg": config["l2_leaf_reg"],
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": seed,
        "thread_count": config["threads"],
        "task_type": device,
        "allow_writing_files": False,
        "verbose": False,
    }
    model = CatBoostClassifier(**parameters)
    if config.get("refit") or config.get("fixed_training"):
        model.fit(training, y, cat_features=categorical)
    else:
        model.fit(
            training,
            y,
            cat_features=categorical,
            eval_set=(validation, validation_y),
            early_stopping_rounds=config["patience"],
            use_best_model=True,
        )
    model.save_model(str(path.with_suffix(".cbm")))
    best_iteration = model.get_best_iteration()
    details = {
        **parameters,
        "best_iteration": int(best_iteration) if best_iteration is not None else None,
        "trees_used": int(model.tree_count_),
        "resolved_parameters": model.get_all_params(),
        "early_stopping_patience": (
            None
            if config.get("refit") or config.get("fixed_training")
            else config["patience"]
        ),
        "preprocessing": "Numeric missing values handled by CatBoost; categorical missing values mapped to __MISSING__; native ordered categorical statistics.",
    }
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


class NeuralPreprocessor:
    """Training-fold-only numeric statistics and categorical vocabularies."""

    def __init__(
        self,
        numeric_transform: str = "standard",
        seed: int = 42,
        categorical_transform: str = "embedding",
        smoothing: float = 20.0,
    ):
        self.numeric_transform = numeric_transform
        self.seed = seed
        self.categorical_transform = categorical_transform
        self.smoothing = smoothing

    def fit(self, frame: pd.DataFrame) -> "NeuralPreprocessor":
        self.numeric, self.categorical = feature_types(frame)
        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)
        self.scaler = (
            QuantileTransformer(
                n_quantiles=min(1000, len(frame)),
                output_distribution="normal",
                subsample=min(100000, len(frame)),
                random_state=self.seed,
            )
            if self.numeric_transform == "quantile"
            else StandardScaler()
        )
        if self.numeric:
            self.scaler.fit(self.imputer.fit_transform(frame[self.numeric]))
        self.vocabularies = {
            name: {
                value: index + 1
                for index, value in enumerate(sorted(frame[name].unique()))
            }
            for name in self.categorical
        }
        self.cardinalities = [
            len(self.vocabularies[name]) + 1 for name in self.categorical
        ]
        return self

    def fit_transform(
        self, frame: pd.DataFrame, y: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray]:
        self.fit(frame)
        if self.categorical_transform == "embedding":
            return self.transform(frame)
        numbers = (
            self.scaler.transform(self.imputer.transform(frame[self.numeric])).astype(
                np.float32
            )
            if self.numeric
            else np.empty((len(frame), 0), dtype=np.float32)
        )
        self.cardinalities = []
        self.target_encoder = TargetEncoder(
            target_type="binary",
            smooth=self.smoothing,
            cv=5,
            shuffle=True,
            random_state=self.seed,
        )
        self.target_scaler = StandardScaler()
        if self.categorical:
            encoded = self.target_encoder.fit_transform(frame[self.categorical], y)
            encoded = self.target_scaler.fit_transform(log_odds(encoded)).astype(
                np.float32
            )
            numbers = np.column_stack([numbers, encoded])
        return numbers, np.empty((len(frame), 0), dtype=np.int64)

    def transform(self, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        numbers = (
            self.scaler.transform(self.imputer.transform(frame[self.numeric])).astype(
                np.float32
            )
            if self.numeric
            else np.empty((len(frame), 0), dtype=np.float32)
        )
        if getattr(self, "categorical_transform", "embedding") == "target":
            if self.categorical:
                encoded = self.target_encoder.transform(frame[self.categorical])
                encoded = self.target_scaler.transform(log_odds(encoded)).astype(
                    np.float32
                )
                numbers = np.column_stack([numbers, encoded])
            return numbers, np.empty((len(frame), 0), dtype=np.int64)
        categories = (
            np.column_stack(
                [
                    frame[name]
                    .map(self.vocabularies[name])
                    .fillna(0)
                    .to_numpy(dtype=np.int64)
                    for name in self.categorical
                ]
            )
            if self.categorical
            else np.empty((len(frame), 0), dtype=np.int64)
        )
        return numbers, categories


def build_network(
    torch,
    numeric_count: int,
    cardinalities: list[int],
    hidden_dims: list[int],
    dropout: float,
):
    nn = torch.nn
    embedding_dims = [
        min(32, max(2, int(np.sqrt(count)) + 1)) for count in cardinalities
    ]

    class TabularMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.embeddings = nn.ModuleList(
                [
                    nn.Embedding(count, width, padding_idx=0)
                    for count, width in zip(cardinalities, embedding_dims)
                ]
            )
            width = numeric_count + sum(embedding_dims)
            layers = []
            for next_width in hidden_dims:
                layers.extend(
                    [nn.Linear(width, next_width), nn.ReLU(), nn.Dropout(dropout)]
                )
                width = next_width
            layers.append(nn.Linear(width, 1))
            self.layers = nn.Sequential(*layers)

        def forward(self, numbers, categories):
            pieces = [
                numbers,
                *[
                    embedding(categories[:, i])
                    for i, embedding in enumerate(self.embeddings)
                ],
            ]
            return self.layers(torch.cat(pieces, dim=1)).squeeze(1)

    return TabularMLP(), embedding_dims


def fit_neural(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    try:
        import torch
    except ImportError as error:
        raise RuntimeError(
            "Neural training requires PyTorch: install fraudlab[neural]."
        ) from error

    torch.set_num_threads(config["threads"])
    torch.manual_seed(seed)
    if config["device"] == "cuda" and not torch.cuda.is_available():
        raise ValueError(
            "CUDA was requested but is not available in this PyTorch installation."
        )
    device = torch.device(
        "cuda" if config["device"] != "cpu" and torch.cuda.is_available() else "cpu"
    )
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    preprocessing = NeuralPreprocessor(
        config.get("neural_numeric_transform", "standard"),
        seed,
        config.get("neural_categorical_transform", "embedding"),
        config.get("target_smoothing", 20.0),
    )
    arrays = [
        preprocessing.fit_transform(training, y),
        preprocessing.transform(validation),
        preprocessing.transform(test),
    ]
    tensors = [
        tuple(torch.as_tensor(value, device=device) for value in pair)
        for pair in arrays
    ]
    labels = torch.as_tensor(y, dtype=torch.float32, device=device)
    model, embedding_dims = build_network(
        torch,
        arrays[0][0].shape[1],
        preprocessing.cardinalities,
        config["hidden_dims"],
        config["dropout"],
    )
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["neural_learning_rate"],
        weight_decay=config["weight_decay"],
    )
    criterion = torch.nn.BCEWithLogitsLoss()

    def predict(pair) -> np.ndarray:
        model.eval()
        with torch.no_grad():
            return np.concatenate(
                [
                    torch.sigmoid(
                        model(
                            pair[0][start : start + config["batch_size"]],
                            pair[1][start : start + config["batch_size"]],
                        )
                    )
                    .cpu()
                    .numpy()
                    for start in range(0, len(pair[0]), config["batch_size"])
                ]
            )

    best_auc, stale, best_epoch, best_state = -np.inf, 0, 0, None
    epochs_completed = 0
    for epoch in range(config["epochs"]):
        model.train()
        order = torch.randperm(len(y), device=device)
        for start in range(0, len(y), config["batch_size"]):
            selected = order[start : start + config["batch_size"]]
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(
                model(tensors[0][0][selected], tensors[0][1][selected]),
                labels[selected],
            )
            if not torch.isfinite(loss):
                raise ValueError(
                    "Neural loss became non-finite; check features or lower the learning rate."
                )
            loss.backward()
            optimizer.step()
        epochs_completed = epoch + 1
        if config.get("refit") or config.get("fixed_training"):
            best_epoch = epochs_completed
            if epochs_completed == config["epochs"]:
                best_state = {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in model.state_dict().items()
                }
            continue
        score = roc_auc_score(validation_y, predict(tensors[1]))
        if score > best_auc + 1e-6:
            best_auc, best_epoch, stale = score, epoch + 1, 0
            best_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
        else:
            stale += 1
        if stale >= config["patience"]:
            break
    if best_state is None:
        raise ValueError("Neural training did not produce a usable checkpoint.")
    model.load_state_dict(best_state)
    details = {
        "architecture": {
            "numeric_features": int(arrays[0][0].shape[1]),
            "raw_numeric_features": len(preprocessing.numeric),
            "categorical_transform": config.get(
                "neural_categorical_transform", "embedding"
            ),
            "categorical_cardinalities": preprocessing.cardinalities,
            "embedding_dims": embedding_dims,
            "hidden_dims": config["hidden_dims"],
            "dropout": config["dropout"],
            "hidden_activation": "ReLU",
            "normalization": "none",
            "output_activation": "sigmoid",
        },
        "optimizer": "AdamW",
        "loss": "BCEWithLogitsLoss",
        "learning_rate": config["neural_learning_rate"],
        "weight_decay": config["weight_decay"],
        "batch_size": config["batch_size"],
        "epochs_completed": epochs_completed,
        "best_epoch": best_epoch,
        "early_stopping_patience": (
            None
            if config.get("refit") or config.get("fixed_training")
            else config["patience"]
        ),
        "device": str(device),
        "numeric_transform": config.get("neural_numeric_transform", "standard"),
        "target_encoder": (
            {
                "internal_folds": 5,
                "smooth": config.get("target_smoothing", 20.0),
                "unknown": "training prior",
                "log_odds": True,
            }
            if config.get("neural_categorical_transform") == "target"
            else None
        ),
        "preprocessing": "Fold-local numeric median imputation and "
        + (
            "quantile-to-normal transformation"
            if config.get("neural_numeric_transform") == "quantile"
            else "standard scaling"
        )
        + (
            "; categorical target-encoding logits cross-fitted into five inner partitions, then scaled using training-only statistics."
            if config.get("neural_categorical_transform") == "target"
            else "; categorical training vocabularies with index 0 reserved for unknown categories."
        ),
    }
    torch.save(
        {"state_dict": best_state, "architecture": details["architecture"]},
        path.with_suffix(".pt"),
    )
    joblib.dump(preprocessing, path.with_suffix(".preprocessing.joblib"))
    return predict(tensors[1]), predict(tensors[2]), details


def fit_predict(
    name: str,
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    numeric, categorical = feature_types(training)
    training, validation, test = [
        clean_features(frame, numeric, categorical)
        for frame in (training, validation, test)
    ]
    if name == "logistic":
        return fit_logistic(training, y, validation, test, config, seed, path)
    if name == "glm":
        return fit_glm(training, y, validation, test, config, seed, path)
    if name == "offset":
        from .offset import fit_offset

        result = fit_offset(training, y, validation, test, config, seed, path)
        if config.get("glm_pooled_tokens"):
            from .pooled import manifest

            result[2]["pooled_tokens"] = manifest(random_state=seed)
        if config.get("glm_transductive_frequency"):
            result[2]["transductive_frequency"] = config.get("transductive_frequency")
            result[2]["preprocessing"] = (
                "Numeric scaling and target statistics use outer training only; category counts "
                "use official train+test features without labels. The same cross-fitted matrix "
                "fits GLM and residual trees; inference sums their margins."
            )
        return result
    if name == "catboost":
        return fit_catboost(
            training, y, validation, validation_y, test, config, seed, path
        )
    if name == "mlp":
        return fit_neural(
            training, y, validation, validation_y, test, config, seed, path
        )
    if name == "tabm":
        try:
            from .tabm import fit_tabm
        except ImportError as error:
            raise RuntimeError(
                "TabM requires the optional fraudlab[tabm] dependencies."
            ) from error
        # Deliberately do not forward outer validation labels to TabM.
        return fit_tabm(training, y, validation, test, config, seed, path)
    if name == "neural_offset":
        try:
            from .neural_offset import fit_neural_offset
        except ImportError as error:
            raise RuntimeError(
                "Neural offset requires the optional fraudlab[neural] dependencies."
            ) from error
        # Outer validation labels cannot enter epoch selection or either GLM fit.
        result = fit_neural_offset(training, y, validation, test, config, seed, path)
        if config.get("glm_pooled_tokens"):
            from .pooled import manifest

            result[2]["pooled_tokens"] = manifest(random_state=seed)
        return result
    if name == "realmlp":
        try:
            from .realmlp import fit_realmlp
        except ImportError as error:
            raise RuntimeError(
                "RealMLP requires the optional fraudlab[realmlp] dependencies."
            ) from error
        return fit_realmlp(training, y, validation, test, config, seed, path)
    if name == "lightgbm":
        return fit_lightgbm(
            training, y, validation, validation_y, test, config, seed, path
        )
    raise ValueError(f"Unknown model: {name}")


def predict_saved(
    name: str, path: Path, frame: pd.DataFrame, device: str = "auto"
) -> np.ndarray:
    """Load only locally trained checkpoints; never download executable artifacts."""
    if name in ("logistic", "glm", "lightgbm", "offset"):
        return joblib.load(path.with_suffix(".joblib")).predict_proba(
            clean_features(frame)
        )[:, 1]
    if name == "catboost":
        from catboost import CatBoostClassifier

        model = CatBoostClassifier()
        model.load_model(str(path.with_suffix(".cbm")))
        return model.predict_proba(clean_features(frame))[:, 1]
    if name == "tabm":
        from .tabm import predict_saved_tabm

        return predict_saved_tabm(path, frame, device)
    if name == "neural_offset":
        from .neural_offset import predict_saved_neural_offset

        return predict_saved_neural_offset(path, frame, device)
    if name == "realmlp":
        from .realmlp import predict_saved_realmlp

        return predict_saved_realmlp(path, frame, device)
    if name != "mlp":
        raise ValueError(f"Unknown saved model: {name}")
    import torch

    if device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is not available.")
    selected_device = torch.device(
        "cuda" if device != "cpu" and torch.cuda.is_available() else "cpu"
    )
    preprocessing = joblib.load(path.with_suffix(".preprocessing.joblib"))
    checkpoint = torch.load(
        path.with_suffix(".pt"), map_location="cpu", weights_only=True
    )
    architecture = checkpoint["architecture"]
    model, _ = build_network(
        torch,
        architecture["numeric_features"],
        architecture["categorical_cardinalities"],
        architecture["hidden_dims"],
        architecture["dropout"],
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.to(selected_device).eval()
    prepared = preprocessing.transform(
        clean_features(frame, preprocessing.numeric, preprocessing.categorical)
    )
    predictions = []
    with torch.no_grad():
        for start in range(0, len(frame), 8192):
            tensors = [
                torch.as_tensor(values[start : start + 8192], device=selected_device)
                for values in prepared
            ]
            predictions.append(torch.sigmoid(model(*tensors)).cpu().numpy())
    return np.concatenate(predictions)


def fit_lightgbm(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    validation_y: np.ndarray,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    from lightgbm import LGBMClassifier, early_stopping

    numeric, categorical = feature_types(training)
    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                SimpleImputer(strategy="median", keep_empty_features=True),
                numeric,
            ),
            (
                "categorical",
                TargetEncoder(
                    target_type="binary",
                    smooth=config.get("target_smoothing", "auto"),
                    cv=5,
                    shuffle=True,
                    random_state=seed,
                ),
                categorical,
            ),
        ]
    )
    # fit_transform cross-fits encodings inside this outer training fold. Validation
    # and test use mappings learned only from that training fold.
    encoded_training = preprocessing.fit_transform(training, y)
    encoded_validation = preprocessing.transform(validation)
    parameters = {
        "n_estimators": config["iterations"],
        "num_leaves": config["num_leaves"],
        "learning_rate": config["learning_rate"],
        "min_child_samples": config["min_child_samples"],
        "reg_lambda": config["l2_leaf_reg"],
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "random_state": seed,
        "n_jobs": config["threads"],
        "deterministic": True,
        "force_col_wise": True,
    }
    classifier = LGBMClassifier(**parameters)
    if config.get("refit") or config.get("fixed_training"):
        classifier.fit(encoded_training, y)
    else:
        classifier.fit(
            encoded_training,
            y,
            eval_X=encoded_validation,
            eval_y=validation_y,
            eval_metric="auc",
            callbacks=[
                early_stopping(
                    config["patience"], first_metric_only=True, verbose=False
                )
            ],
        )
    model = Pipeline([("preprocessing", preprocessing), ("classifier", classifier)])
    joblib.dump(model, path.with_suffix(".joblib"))
    details = {
        **parameters,
        "best_iteration": int(classifier.best_iteration_ or classifier.n_estimators_),
        "target_encoder": {
            "internal_folds": 5,
            "smooth": config.get("target_smoothing", "auto"),
            "unknown": "training prior",
        },
        "preprocessing": "Fold-local median imputation and supervised categorical target encoding, internally cross-fitted into five partitions.",
    }
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


def make_glm_pipeline(
    training: pd.DataFrame,
    config: dict,
    seed: int,
) -> Pipeline:
    """Construct fold-local preprocessing and a regularized logistic model."""
    numeric, categorical = feature_types(training)
    nominal = [name for name in categorical if "__" not in name]
    value_one_hot = config.get("glm_value_one_hot", False)
    one_hot_columns = categorical if value_one_hot else nominal
    encoding = Pipeline(
        [
            (
                "target",
                TargetEncoder(
                    target_type="binary",
                    smooth=config["target_smoothing"],
                    cv=config.get("glm_inner_folds", 5),
                    shuffle=True,
                    random_state=seed,
                ),
            ),
            ("log_odds", FunctionTransformer(log_odds)),
            ("scale", StandardScaler()),
        ]
    )
    preprocessing = ColumnTransformer(
        [
            (
                "numeric",
                Pipeline(
                    [
                        (
                            "impute",
                            SimpleImputer(strategy="median", keep_empty_features=True),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                numeric,
            ),
            ("target_logits", encoding, categorical),
            (
                "one_hot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=value_one_hot,
                    min_frequency=5 if value_one_hot else None,
                    dtype=np.float32,
                ),
                one_hot_columns,
            ),
        ],
        sparse_threshold=1.0 if value_one_hot else 0.3,
    )
    knots = config.get("glm_spline_knots", 0)
    if config.get("glm_frequency"):
        from .frequency import FrequencyEncoder

        if (
            config.get("glm_transductive_frequency")
            and config.get("_frequency_reference") is None
        ):
            raise ValueError("Transductive frequency needs runtime reference counts.")
        if config.get("_frequency_reference") is not None and not config.get(
            "glm_transductive_frequency"
        ):
            raise ValueError(
                "Reference counts require the transductive frequency flag."
            )
        preprocessing.transformers.append(
            (
                "category_frequency",
                Pipeline(
                    [
                        (
                            "frequency",
                            FrequencyEncoder(
                                reference_counts=config.get("_frequency_reference")
                            ),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                categorical,
            )
        )
    if config.get("glm_pooled_tokens"):
        from .pooled import COLUMNS, PooledTokenEncoder

        if config.get("glm_inner_folds", 5) != 5:
            raise ValueError("Pooled tokens require five inner folds.")
        preprocessing.transformers.append(
            (
                "pooled_token_logits",
                Pipeline(
                    [
                        ("pooled", PooledTokenEncoder(random_state=seed)),
                        ("log_odds", FunctionTransformer(log_odds)),
                        ("scale", StandardScaler()),
                    ]
                ),
                list(COLUMNS),
            )
        )
    continuous = [name for name in CONTINUOUS if name in numeric]
    if knots:
        if not continuous:
            raise ValueError("Numeric splines require a supported continuous feature.")
        preprocessing.transformers.append(
            (
                "numeric_splines",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="median")),
                        (
                            "spline",
                            SplineTransformer(
                                n_knots=knots,
                                degree=3,
                                knots="quantile",
                                include_bias=False,
                                extrapolation="constant",
                            ),
                        ),
                        ("scale", StandardScaler()),
                    ]
                ),
                continuous,
            )
        )
    classifier = LogisticRegression(
        C=config["logistic_c"],
        solver="lbfgs" if value_one_hot else "newton-cholesky",
        max_iter=2000 if value_one_hot else 100,
        tol=1e-7,
        random_state=seed,
    )
    return Pipeline([("preprocessing", preprocessing), ("classifier", classifier)])


def fit_glm(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    model = make_glm_pipeline(training, config, seed)
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", ConvergenceWarning)
        with threadpool_limits(limits=config["threads"]):
            model.fit(training, y)
    for warning in captured:
        warnings.warn(warning.message, warning.category, stacklevel=2)
    classifier = model.named_steps["classifier"]
    iterations_used = int(classifier.n_iter_[0])
    converged = (
        not any(
            issubclass(warning.category, ConvergenceWarning) for warning in captured
        )
        and iterations_used < classifier.max_iter
    )
    if config.get("glm_value_one_hot") and not converged:
        raise RuntimeError(
            "GLM value one-hot did not converge; "
            f"iterations_used={iterations_used}, max_iter={classifier.max_iter}. "
            "The candidate is invalid under the fixed protocol."
        )
    joblib.dump(model, path.with_suffix(".joblib"))
    details = {
        "algorithm": "logistic_regression_with_cross_fitted_target_logits",
        "C": config["logistic_c"],
        "solver": classifier.solver,
        "max_iter": classifier.max_iter,
        "tol": classifier.tol,
        "iterations_used": iterations_used,
        "converged": converged,
        "value_one_hot": (
            {
                "features": feature_types(training)[1],
                "min_frequency": 5,
                "handle_unknown": "ignore",
                "sparse_output": True,
            }
            if config.get("glm_value_one_hot")
            else None
        ),
        "category_frequency": (
            (
                "log1p official train+test feature counts, no labels; outer-training scale only"
                if config.get("glm_transductive_frequency")
                else "log1p counts fitted on outer training rows only"
            )
            if config.get("glm_frequency")
            else None
        ),
        "numeric_splines": (
            {
                "n_knots": config.get("glm_spline_knots", 0),
                "degree": 3,
                "knots": "quantile fitted only on outer training rows",
                "features": [name for name in CONTINUOUS if name in training],
                "extrapolation": "constant",
            }
            if config.get("glm_spline_knots", 0)
            else None
        ),
        "target_encoder": {
            "internal_folds": config.get("glm_inner_folds", 5),
            "smooth": config["target_smoothing"],
            "unknown": "training prior",
            "log_odds": True,
        },
        "preprocessing": "Fold-local numeric imputation and scaling, cross-fitted target-encoding logits with scaling, plus "
        + (
            "sparse one-hot encoding of all categorical features; counts below 5 pooled per training fold; unknown categories ignored."
            if config.get("glm_value_one_hot")
            else "one-hot encoding of original categorical features."
        ),
    }
    if config.get("glm_pooled_tokens"):
        from .pooled import manifest

        details["pooled_tokens"] = manifest(random_state=seed)
    if config.get("glm_transductive_frequency"):
        details["transductive_frequency"] = config.get("transductive_frequency")
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


In [ ]:
%%writefile r23-source/src/fraudlab/neural_offset.py
"""A fresh neural correction to a frozen GLM logit, selected inside outer training.

The outer validation labels are deliberately absent from this module's API.
Both models learn from the identical, internally cross-fitted training matrix.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits

from .models import clean_features, feature_types


@dataclass(frozen=True)
class NeuralOffsetConfig:
    hidden_width: int = 256
    dropout: float = 0.1
    learning_rate: float = 0.001
    weight_decay: float = 0.0001
    batch_size: int = 4096
    epochs: int = 40
    patience: int = 6
    gradient_clip: float = 1.0
    inner_fraction: float = 0.15
    inner_seed: int = 42

    @classmethod
    def from_training_config(cls, config: dict):
        if config.get("fixed_training") or config.get("refit"):
            raise ValueError("Neural offset requires inner epoch selection.")
        hidden_dims = config.get("hidden_dims")
        if hidden_dims is not None and hidden_dims != [256, 256]:
            raise ValueError("Neural offset requires --hidden-dims 256 256.")
        if config.get("glm_value_one_hot"):
            raise ValueError("Neural offset requires the dense GLM representation.")
        if config.get("neural_numeric_transform", "standard") != "standard":
            raise ValueError("Neural offset uses the GLM's standard numeric scaling.")
        if config.get("neural_categorical_transform") not in (None, "target"):
            raise ValueError("Neural offset uses the GLM's cross-fitted target logits.")
        settings = cls(
            dropout=config.get("dropout", 0.1),
            learning_rate=config.get("neural_learning_rate", 0.001),
            weight_decay=config.get("weight_decay", 0.0001),
            batch_size=config.get("batch_size", 4096),
            epochs=config.get("epochs", 40),
            patience=6 if config.get("patience") is None else config["patience"],
        )
        if not 1 <= settings.epochs <= 40 or settings.patience < 1:
            raise ValueError("Neural offset needs 1..40 epochs and positive patience.")
        if settings.batch_size < 1 or settings.learning_rate <= 0:
            raise ValueError("Batch size and learning rate must be positive.")
        if not 0 <= settings.dropout < 1 or settings.weight_decay < 0:
            raise ValueError("Invalid dropout or weight decay.")
        return settings


def build_residual(input_features: int, settings: NeuralOffsetConfig):
    width = settings.hidden_width
    model = torch.nn.Sequential(
        torch.nn.Linear(input_features, width),
        torch.nn.SiLU(),
        torch.nn.Dropout(settings.dropout),
        torch.nn.Linear(width, width),
        torch.nn.SiLU(),
        torch.nn.Dropout(settings.dropout),
        torch.nn.Linear(width, 1),
    )
    # The initial combined predictor is the GLM with its margin cast to float32.
    torch.nn.init.zeros_(model[-1].weight)
    torch.nn.init.zeros_(model[-1].bias)
    return model


def prepare_training(frame, y, config, seed):
    # Local import permits later lazy dispatch without a circular import.
    from .models import make_glm_pipeline

    glm = make_glm_pipeline(frame, config, seed)
    preprocessing = glm.named_steps["preprocessing"]
    classifier = glm.named_steps["classifier"]
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", ConvergenceWarning)
        with threadpool_limits(limits=config["threads"]):
            encoded = preprocessing.fit_transform(frame, y)
            classifier.fit(encoded, y)
            margin = classifier.decision_function(encoded)
    for warning in captured:
        warnings.warn(warning.message, warning.category, stacklevel=2)
    if (
        any(issubclass(w.category, ConvergenceWarning) for w in captured)
        or int(classifier.n_iter_[0]) >= classifier.max_iter
    ):
        raise RuntimeError(
            "Neural offset GLM did not converge; candidate invalid under fixed protocol."
        )
    # NEVER transform training again with the full target maps.
    return glm, _float_matrix(encoded), np.asarray(margin, dtype=np.float32)


def _float_matrix(encoded):
    if hasattr(encoded, "toarray"):
        raise ValueError("Neural offset requires a dense GLM training matrix.")
    matrix = np.ascontiguousarray(encoded, dtype=np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError("Neural offset received nonfinite encoded features.")
    return matrix


def transform_frame(glm, frame):
    encoded = glm.named_steps["preprocessing"].transform(frame)
    margin = glm.named_steps["classifier"].decision_function(encoded)
    return _float_matrix(encoded), np.asarray(margin, dtype=np.float32)


@torch.no_grad()
def predict_array(model, numbers, margin, batch_size):
    model.eval()
    return np.concatenate(
        [
            (offset + model(batch).squeeze(-1)).sigmoid().cpu().numpy()
            for batch, offset in zip(
                numbers.split(batch_size), margin.split(batch_size)
            )
        ]
    )


def _train(numbers, y, margin, settings, seed, device, *, validation=None, epochs=None):
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    model = build_residual(numbers.shape[1], settings).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=settings.learning_rate,
        weight_decay=settings.weight_decay,
    )
    x = torch.as_tensor(numbers, device=device)
    labels = torch.as_tensor(y, dtype=torch.float32, device=device)
    offsets = torch.as_tensor(margin, dtype=torch.float32, device=device).detach()
    best_auc, best_epoch, stale, history = -np.inf, 0, 0, []
    for epoch in range(1, (epochs or settings.epochs) + 1):
        model.train()
        for selected in torch.randperm(len(y), device=device).split(
            settings.batch_size
        ):
            optimizer.zero_grad(set_to_none=True)
            logits = offsets[selected] + model(x[selected]).squeeze(-1)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                logits, labels[selected]
            )
            if not torch.isfinite(loss):
                raise ValueError("Neural offset loss became nonfinite.")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), settings.gradient_clip)
            optimizer.step()
        if validation is None:
            best_epoch = epoch
            continue
        vx, vy, vm = validation
        score = roc_auc_score(vy, predict_array(model, vx, vm, settings.batch_size))
        history.append({"epoch": epoch, "inner_auc": float(score)})
        print(
            f"neural_offset seed={seed} inner_epoch={epoch} roc_auc={score:.8f}",
            flush=True,
        )
        if score > best_auc + 1e-6:
            best_auc, best_epoch, stale = score, epoch, 0
        else:
            stale += 1
        if stale >= settings.patience:
            break
    return model, best_epoch, history, epoch


def fit_neural_offset(training, y, validation, test, config, seed, path: Path):
    """Choose E internally, discard that stage, and refit both models from scratch."""
    settings = NeuralOffsetConfig.from_training_config(config)
    config = {
        "feature_set": "bpe_digits",
        "glm_frequency": True,
        "glm_pooled_tokens": True,
        "glm_inner_folds": 5,
        "glm_spline_knots": 0,
        "target_smoothing": 50.0,
        "logistic_c": 0.3,
        "threads": 4,
        **config,
    }
    torch.set_num_threads(config["threads"])
    requested_device = config.get("device", "cpu")
    if requested_device not in ("cpu", "cuda", "auto"):
        raise ValueError("Unknown neural offset device.")
    if requested_device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is not available.")
    device = torch.device(
        "cuda" if requested_device != "cpu" and torch.cuda.is_available() else "cpu"
    )
    numeric, categorical = feature_types(training)
    training = clean_features(training, numeric, categorical)
    y = np.asarray(y)
    inner_train, inner_valid = train_test_split(
        np.arange(len(y)),
        test_size=settings.inner_fraction,
        random_state=settings.inner_seed,
        stratify=y,
    )
    inner_glm, inner_x, inner_margin = prepare_training(
        training.iloc[inner_train], y[inner_train], config, seed
    )
    inner_classifier = inner_glm.named_steps["classifier"]
    inner_glm_iterations = int(inner_classifier.n_iter_[0])
    inner_glm_max_iter = inner_classifier.max_iter
    inner_vx, inner_vm = transform_frame(inner_glm, training.iloc[inner_valid])
    selection_model, selected_epoch, history, selection_epochs = _train(
        inner_x,
        y[inner_train],
        inner_margin,
        settings,
        seed,
        device,
        validation=(
            torch.as_tensor(inner_vx, device=device),
            y[inner_valid],
            torch.as_tensor(inner_vm, device=device),
        ),
    )
    # No preprocessing, GLM, network weights or optimizer crosses this boundary.
    del (
        selection_model,
        inner_glm,
        inner_classifier,
        inner_x,
        inner_margin,
        inner_vx,
        inner_vm,
    )
    print(
        f"neural_offset seed={seed} fresh_outer_refit_epochs={selected_epoch}",
        flush=True,
    )
    glm, numbers, margin = prepare_training(training, y, config, seed)
    model, _, _, refit_epochs = _train(
        numbers, y, margin, settings, seed, device, epochs=selected_epoch
    )
    predictions = []
    for frame in (validation, test):
        vx, vm = transform_frame(glm, clean_features(frame, numeric, categorical))
        predictions.append(
            predict_array(
                model,
                torch.as_tensor(vx, device=device),
                torch.as_tensor(vm, device=device),
                settings.batch_size,
            )
        )
    epoch_metadata = {
        "selected_epoch": selected_epoch,
        "selection_epochs_completed": selection_epochs,
        "refit_epochs_completed": refit_epochs,
        "inner_history": history,
        "inner_training_rows": len(inner_train),
        "inner_validation_rows": len(inner_valid),
        "refit_rows": len(training),
        "inner_glm_iterations_used": inner_glm_iterations,
        "inner_glm_converged": inner_glm_iterations < inner_glm_max_iter,
    }
    architecture = {
        "name": "GLM_logit_plus_neural_residual",
        "input_features": numbers.shape[1],
        "hidden_dims": [256, 256],
        "activation": "SiLU",
        "dropout": settings.dropout,
        "zero_initialized_head": True,
        "glm_margin_dtype": "float32",
        "parameters": sum(p.numel() for p in model.parameters()),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {"glm": glm, "numeric": numeric, "categorical": categorical},
        path.with_suffix(".glm.joblib"),
    )
    torch.save(
        {
            "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "config": asdict(settings),
            "architecture": architecture,
            "epoch_metadata": epoch_metadata,
        },
        path.with_suffix(".pt"),
    )
    return *predictions, {
        "architecture": architecture,
        "config": asdict(settings),
        **epoch_metadata,
        "glm": {
            key: config[key]
            for key in (
                "feature_set",
                "logistic_c",
                "target_smoothing",
                "glm_inner_folds",
                "glm_frequency",
                "glm_pooled_tokens",
                "glm_spline_knots",
            )
        },
        "glm_iterations_used": int(glm.named_steps["classifier"].n_iter_[0]),
        "glm_converged": int(glm.named_steps["classifier"].n_iter_[0])
        < glm.named_steps["classifier"].max_iter,
        "optimizer": "AdamW",
        "loss": "unweighted mean BCEWithLogits",
        "prediction": "sigmoid(frozen GLM decision_function + neural residual)",
        "training_encoding": "same cross-fitted matrix for GLM and residual; no full-map training retransform",
        "epoch_selection": "inner 15% AUC; fresh GLM, preprocessing, network and AdamW for full outer refit",
        "outer_validation_usage": "inference only; no labels accepted",
        "seed": seed,
        "threads": torch.get_num_threads(),
        "device": str(device),
    }


def predict_saved_neural_offset(path: Path, frame: pd.DataFrame, device="cpu"):
    if device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is not available.")
    device = "cuda" if device != "cpu" and torch.cuda.is_available() else "cpu"
    payload = torch.load(path.with_suffix(".pt"), map_location="cpu", weights_only=True)
    settings = NeuralOffsetConfig(**payload["config"])
    model = build_residual(payload["architecture"]["input_features"], settings).to(
        device
    )
    model.load_state_dict(payload["state_dict"])
    saved = joblib.load(path.with_suffix(".glm.joblib"))
    numbers, margin = transform_frame(
        saved["glm"], clean_features(frame, saved["numeric"], saved["categorical"])
    )
    return predict_array(
        model,
        torch.as_tensor(numbers, device=device),
        torch.as_tensor(margin, device=device),
        settings.batch_size,
    )


In [ ]:
%%writefile r23-source/src/fraudlab/offset.py
"""Fixed small trees correct a GLM margin using one cross-fitted training matrix."""

from __future__ import annotations

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits


class OffsetClassifier:
    """Persist both terms: LightGBM does not add its training init_score at inference."""

    def __init__(self, glm, residual):
        self.glm = glm
        self.residual = residual

    def predict_proba(self, frame: pd.DataFrame) -> np.ndarray:
        encoded = self.glm.named_steps["preprocessing"].transform(frame)
        margin = self.glm.named_steps["classifier"].decision_function(encoded)
        margin = margin + self.residual.predict(encoded, raw_score=True)
        probability = np.exp(-np.logaddexp(0.0, -margin))
        return np.column_stack((1 - probability, probability))


def fit_offset(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    from lightgbm import LGBMClassifier

    # Local import keeps model dispatch free to import this module lazily.
    from .models import make_glm_pipeline

    glm = make_glm_pipeline(training, config, seed)
    preprocessing = glm.named_steps["preprocessing"]
    classifier = glm.named_steps["classifier"]
    with threadpool_limits(limits=config["threads"]):
        encoded_training = preprocessing.fit_transform(training, y)
        classifier.fit(encoded_training, y)
        initial_score = classifier.decision_function(encoded_training)
    parameters = {
        "n_estimators": config["iterations"],
        "num_leaves": config["num_leaves"],
        "learning_rate": config["learning_rate"],
        "reg_lambda": config["l2_leaf_reg"],
        "min_child_samples": config["min_child_samples"],
        "objective": "binary",
        "verbosity": -1,
        "random_state": seed,
        "n_jobs": config["threads"],
        "deterministic": True,
        "force_col_wise": True,
    }
    residual = LGBMClassifier(**parameters)
    residual.fit(encoded_training, y, init_score=initial_score)
    model = OffsetClassifier(glm, residual)
    joblib.dump(model, path.with_suffix(".joblib"))
    details = {
        "algorithm": "glm_logit_offset_with_lightgbm_residual",
        "prediction": "sigmoid(GLM decision_function + LightGBM raw_score)",
        "fixed_training": True,
        "early_stopping_patience": None,
        "best_iteration": int(residual.n_estimators_),
        "encoded_features": int(encoded_training.shape[1]),
        "glm": {
            "C": config["logistic_c"],
            "solver": classifier.solver,
            "iterations_used": int(classifier.n_iter_[0]),
            "spline_knots": config.get("glm_spline_knots", 0),
        },
        "target_encoder": {
            "internal_folds": config.get("glm_inner_folds", 5),
            "smooth": config["target_smoothing"],
            "unknown": "training prior",
            "log_odds": True,
        },
        "residual_parameters": parameters,
        "preprocessing": (
            "Fold-local GLM preprocessing, including internally cross-fitted target "
            "logits, is fitted once. The identical encoded training matrix fits the "
            "GLM, supplies its init_score and fits residual trees. Training rows are "
            "never re-encoded with the full target map. Validation and test use "
            "training-only mappings. Both fitted models are saved together."
        ),
    }
    return (
        model.predict_proba(validation)[:, 1],
        model.predict_proba(test)[:, 1],
        details,
    )


In [ ]:
%%writefile r23-source/src/fraudlab/original.py
"""Validate explicitly pinned original data for outer-training-only augmentation."""

from __future__ import annotations

import hashlib
from io import BytesIO
from pathlib import Path
import re

import numpy as np
import pandas as pd

from .data import fingerprint, validate_ids

NUMERIC = (
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
)
NOMINAL = ("merchant_category", "card_type", "auth_method", "channel", "device_type")
FLAGS = (
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
)
COLUMNS = NUMERIC + NOMINAL + FLAGS
AUDITED_SHA256 = "4e32b1ad4880cae452d1f2e5aa53224d38eb913c185455c00a3759df5eb3da61"
SOURCE_URL = "https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026/versions/5"


def validate_config(config: dict) -> None:
    path, digest = config.get("original_data"), config.get("original_sha256")
    if not path:
        if (
            digest
            or config.get("original_id_column", "transaction_id") != "transaction_id"
        ):
            raise ValueError("Original-data options require --original-data.")
        return
    if not isinstance(digest, str) or not re.fullmatch(r"[a-fA-F0-9]{64}", digest):
        raise ValueError("--original-data requires a valid --original-sha256.")
    if (
        config.get("phase") != "exploratory_full_data_cv"
        or config.get("split") != "stratified"
        or config.get("holdout_fraction", 0)
        or config.get("folds") != 5
        or config.get("seed") != 42
        or config.get("refit")
    ):
        raise ValueError(
            "Original data requires exploratory stratified CV5/42, no holdout/refit."
        )
    if config.get("model") in ("catboost", "lightgbm", "mlp") and not config.get(
        "fixed_training"
    ):
        raise ValueError(
            "Original augmentation cannot use outer-validation early stopping."
        )


def verify_unchanged(path: Path, expected_sha256: str) -> None:
    if fingerprint(path) != expected_sha256.lower():
        raise ValueError(
            "Original CSV bytes changed; SHA-256 no longer matches the pin."
        )


def load_original(
    path: Path,
    expected_sha256: str,
    competition: pd.DataFrame,
    test: pd.DataFrame,
    *,
    original_id_column="transaction_id",
    id_column="id",
    target="is_fraud",
) -> tuple[pd.DataFrame, np.ndarray, dict]:
    """Return only the 24 ordered features and labels; never carry source IDs."""
    if not isinstance(expected_sha256, str) or not re.fullmatch(
        r"[a-fA-F0-9]{64}", expected_sha256
    ):
        raise ValueError("A valid original SHA-256 pin is required.")
    if original_id_column == target or {original_id_column, id_column, target} & set(
        COLUMNS
    ):
        raise ValueError("Original IDs and target must not be feature columns.")
    raw = path.read_bytes()
    digest = hashlib.sha256(raw).hexdigest()
    if digest != expected_sha256.lower():
        raise ValueError("Original CSV SHA-256 does not match the expected pin.")
    original = pd.read_csv(BytesIO(raw))
    if set(original) != {*COLUMNS, original_id_column, target}:
        raise ValueError(
            "Original schema must contain exactly 24 common features, ID and target."
        )
    order = [name for name in competition if name not in (id_column, target)]
    if (
        len(order) != 24
        or set(order) != set(COLUMNS)
        or set(test) != {*COLUMNS, id_column}
    ):
        raise ValueError(
            "Competition schema must contain the same 24 original features."
        )
    validate_ids(original, original_id_column)
    if original.isna().any().any():
        raise ValueError("Original CSV must not contain missing values.")
    labels = original[target]
    if not pd.api.types.is_numeric_dtype(labels) or set(labels.unique()) != {0, 1}:
        raise ValueError("Original target must contain binary numeric labels 0 and 1.")
    for name in order:
        values = original[name]
        if values.dtype != competition[name].dtype or values.dtype != test[name].dtype:
            raise ValueError(f"Original dtype differs from competition: {name}")
        if name in NUMERIC:
            if (
                not pd.api.types.is_numeric_dtype(values)
                or pd.api.types.is_bool_dtype(values)
                or pd.api.types.is_complex_dtype(values)
                or not np.isfinite(values.to_numpy()).all()
            ):
                raise ValueError(f"Original numeric feature is invalid: {name}")
        elif name in FLAGS:
            if not pd.api.types.is_bool_dtype(values):
                raise ValueError(f"Original flag must be boolean: {name}")
        elif not values.map(lambda value: isinstance(value, str)).all():
            raise ValueError(f"Original nominal feature must contain strings: {name}")
    features = original[order].copy()
    if features.duplicated().any():
        raise ValueError("Original CSV contains duplicate feature rows.")
    hashes = pd.util.hash_pandas_object(features, index=False)
    for name, reference in (("train", competition), ("test", test)):
        reference_features = reference[order]
        candidates = reference_features.loc[
            pd.util.hash_pandas_object(reference_features, index=False).isin(hashes)
        ]
        if (
            not candidates.empty
            and not features.merge(candidates.drop_duplicates(), on=order).empty
        ):
            raise ValueError(f"Original features overlap competition {name} rows.")
    audited = digest == AUDITED_SHA256
    metadata = {
        "source": str(path),
        "sha256": digest,
        "source_url": SOURCE_URL if audited else None,
        "version": 5 if audited else None,
        "license": "CC0-1.0" if audited else None,
        "provenance": (
            "audited public original"
            if audited
            else "explicit local CSV; public provenance not asserted"
        ),
        "rows": len(original),
        "positives": int(labels.sum()),
        "usage": "outer-training only",
        "excluded_columns": [original_id_column],
        "competition_validation_rows_added": 0,
        "competition_oof_rows_added": 0,
        "feature_overlap": "zero exact feature rows against competition train and test",
    }
    verify_unchanged(path, digest)
    return features, labels.to_numpy(dtype=np.int64), metadata


In [ ]:
%%writefile r23-source/src/fraudlab/pooled.py
"""Hierarchical decimal/token target rates with honest inner cross-fitting.

Concept: megayak's token pooling, credited in goodpjw2008's Part 12 notebook
(version 16, gboost.py cell). This implementation is independently written;
no third-party code is copied or executed. The hypothesis concerns observed
numeric patterns, not a verified data-generating mechanism.
"""

from __future__ import annotations

from decimal import Decimal

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.validation import check_is_fitted

POOL2 = (
    "amount_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "account_balance_usd",
)
POOL1 = ("velocity_score", "merchant_risk_score")
COLUMNS = POOL2 + POOL1
GROUPS = ((0, 1, 2, 3), (4, 5, 6, 7), (8, 9), (10, 11))
OUTPUT_NAMES = (
    tuple(f"{name}__pooled_decimal_rate" for name in POOL2)
    + tuple(f"{name}__pooled_first_decimal_rate" for name in POOL2)
    + tuple(f"{name}__pooled_decimal_rate" for name in POOL1)
    + tuple(f"{name}__pooled_last_integer_token_rate" for name in POOL1)
    + tuple(f"{name}__pooled_markov_rate" for name in POOL2)
)


def canonical_decimal(value: float) -> str:
    """Represent a finite nonnegative float64 without exponent or new rounding."""
    if np.iscomplexobj(value):
        raise ValueError("Decimal token inputs must be real numbers.")
    number = float(value)
    if not np.isfinite(number) or number < 0:
        raise ValueError("Decimal token inputs must be finite and nonnegative.")
    text = format(Decimal(str(number if number else 0.0)), "f")
    return text if "." in text else text + ".0"


def _keys(frame: pd.DataFrame) -> list[np.ndarray]:
    from .bpe import _encoding

    decimals, first, last, pairs = {}, {}, {}, {}
    tokenizer = _encoding()
    integer_tokens = {}
    for name in COLUMNS:
        series = frame[name]
        if (
            not pd.api.types.is_numeric_dtype(series)
            or pd.api.types.is_bool_dtype(series)
            or pd.api.types.is_complex_dtype(series)
        ):
            raise ValueError(f"{name} requires numeric decimal token inputs.")
        values = series.to_numpy(dtype=np.float64)
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError(f"{name} requires finite nonnegative values.")
        unique, inverse = np.unique(values, return_inverse=True)
        parts = [canonical_decimal(value).split(".", 1) for value in unique]
        integer = [part[0] for part in parts]
        for text in set(integer) - integer_tokens.keys():
            integer_tokens[text] = tokenizer.encode_ordinary(" " + text)[-1]
        dec = [part[1] for part in parts]
        token = [integer_tokens[text] for text in integer]
        decimals[name] = np.asarray(dec, dtype=object)[inverse]
        first[name] = np.asarray([text[0] for text in dec], dtype=object)[inverse]
        last[name] = np.asarray(token, dtype=np.int32)[inverse]
        if name in POOL2:
            pairs[name] = np.asarray(
                [f"{key}:{text}" for key, text in zip(token, dec)], dtype=object
            )[inverse]
    return (
        [decimals[name] for name in POOL2]
        + [first[name] for name in POOL2]
        + [decimals[name] for name in POOL1]
        + [last[name] for name in POOL1]
        + [pairs[name] for name in POOL2]
    )


def manifest(random_state: int = 42) -> dict:
    from .bpe import ENCODING, ENCODING_FILES, TOKENIZER_VERSION

    return {
        "algorithm": "hierarchical_pooled_decimal_token_rates",
        "input_columns": list(COLUMNS),
        "output_features": list(OUTPUT_NAMES),
        "output_count": 16,
        "groups": [
            {"fields": list(POOL2), "key": "complete decimal text"},
            {"fields": list(POOL2), "key": "first decimal digit"},
            {"fields": list(POOL1), "key": "complete decimal text"},
            {"fields": list(POOL1), "key": "last GPT-2 integer token ID"},
        ],
        "markov_fields": list(POOL2),
        "markov_key": "last GPT-2 integer token ID + complete decimal text",
        "smoothing": {
            "pooled_to_prior": 50.0,
            "field_to_pool": 2000.0,
            "markov_to_field_decimal": 200.0,
        },
        "crossfit": {
            "n_splits": 5,
            "shuffle": True,
            "random_state": random_state,
            "seed_scope": "model seed; CV runner passes run seed + outer fold index",
            "prior": "inner-training mean only",
            "inference": "outer-training mappings only",
        },
        "canonical_decimal": "Decimal(str(float64)), fixed notation, at least .0; normalize negative zero; no cent rounding or decimal padding",
        "integer_token": "last token of one ASCII space + canonical integer part",
        "tokenizer": {
            "tiktoken": TOKENIZER_VERSION,
            "encoding": ENCODING,
            "encoding_files": list(ENCODING_FILES),
        },
        "pipeline": "rates -> clipped log_odds -> outer-training StandardScaler",
        "concept_sources": [
            {
                "author": "megayak",
                "url": "https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack",
                "scope": "token pooling concept credited as v7 in the Part 12 source",
            },
            {
                "author": "goodpjw2008",
                "url": "https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298",
                "version": 16,
                "cell": 74,
                "source_body_sha256": "835772526d7efcc70f771aa446690cb1956676a457f40b08d48760bbf3336baf",
            },
        ],
        "implementation": "independently written; concept credit only, no source code copied or executed",
        "interpretation": "numeric-pattern hypothesis; generator not verified",
    }


class PooledTokenEncoder(TransformerMixin, BaseEstimator):
    """Pool target rates across fields, then shrink field and conditional rates.

    fit_transform returns inner-held-out rates and retains full-training maps
    only for later transform calls. IDs and any extra columns are ignored.
    """

    def __init__(
        self,
        *,
        n_splits=5,
        random_state=42,
        pool_smoothing=50.0,
        field_smoothing=2000.0,
        markov_smoothing=200.0,
    ):
        self.n_splits = n_splits
        self.random_state = random_state
        self.pool_smoothing = pool_smoothing
        self.field_smoothing = field_smoothing
        self.markov_smoothing = markov_smoothing

    def _validate(self, frame, y=None):
        if not isinstance(frame, pd.DataFrame) or not frame.columns.is_unique:
            raise ValueError("PooledTokenEncoder requires unique DataFrame columns.")
        missing = set(COLUMNS) - set(frame)
        if missing:
            raise ValueError(f"Missing pooled-token columns: {sorted(missing)}")
        if y is None:
            return None
        labels = np.asarray(y, dtype=np.float64)
        if (
            labels.shape != (len(frame),)
            or not len(labels)
            or not np.isin(labels, [0, 1]).all()
        ):
            raise ValueError("PooledTokenEncoder requires one binary label per row.")
        if (
            not isinstance(self.n_splits, int)
            or isinstance(self.n_splits, bool)
            or self.n_splits < 2
        ):
            raise ValueError("n_splits must be an integer of at least two.")
        for value in (self.pool_smoothing, self.field_smoothing, self.markov_smoothing):
            if not np.isfinite(value) or value <= 0:
                raise ValueError("Pooled-token smoothing values must be positive.")
        return labels

    def _learn(self, keys, labels, rows):
        prior = float(labels[rows].mean())
        fields, pools, markov = [], [], []
        for group in GROUPS:
            tables = [
                pd.DataFrame({"key": keys[column][rows], "y": labels[rows]})
                .groupby("key", sort=False)["y"]
                .agg(["sum", "count"])
                for column in group
            ]
            pooled = pd.concat(tables).groupby(level=0, sort=False).sum()
            rate = (pooled["sum"] + self.pool_smoothing * prior) / (
                pooled["count"] + self.pool_smoothing
            )
            pools.append(rate)
            for table in tables:
                fields.append(
                    (table["sum"] + self.field_smoothing * rate.reindex(table.index))
                    / (table["count"] + self.field_smoothing)
                )
        for column in range(12, 16):
            markov.append(
                pd.DataFrame({"key": keys[column][rows], "y": labels[rows]})
                .groupby("key", sort=False)["y"]
                .agg(["sum", "count"])
            )
        return {"prior": prior, "pools": pools, "fields": fields, "markov": markov}

    def _encode(self, keys, maps, rows):
        result = np.empty((len(rows), 16), dtype=np.float64)
        for index, group in enumerate(GROUPS):
            for column in group:
                query = pd.Series(keys[column][rows], copy=False)
                fallback = query.map(maps["pools"][index]).fillna(maps["prior"])
                result[:, column] = query.map(maps["fields"][column]).fillna(fallback)
        for index in range(4):
            counts = maps["markov"][index].reindex(keys[index + 12][rows], fill_value=0)
            rate = (
                counts["sum"].to_numpy() + self.markov_smoothing * result[:, index]
            ) / (counts["count"].to_numpy() + self.markov_smoothing)
            result[:, index + 12] = np.where(
                counts["count"].to_numpy() == 0, result[:, index], rate
            )
        return result

    def _retain(self, maps):
        self.maps_ = maps
        self.feature_names_in_ = np.asarray(COLUMNS, dtype=object)
        self.n_features_in_ = len(COLUMNS)

    def fit(self, frame, y):
        labels = self._validate(frame, y)
        keys = _keys(frame)
        self._retain(self._learn(keys, labels, np.arange(len(frame))))
        return self

    def fit_transform(self, frame, y, **fit_params):
        labels = self._validate(frame, y)
        if (
            min(np.count_nonzero(labels == 0), np.count_nonzero(labels == 1))
            < self.n_splits
        ):
            raise ValueError("Cross-fitting requires at least n_splits rows per class.")
        keys = _keys(frame)
        result = np.empty((len(frame), 16), dtype=np.float64)
        splitter = StratifiedKFold(
            n_splits=self.n_splits, shuffle=True, random_state=self.random_state
        )
        for training, validation in splitter.split(np.zeros(len(frame)), labels):
            maps = self._learn(keys, labels, training)
            result[validation] = self._encode(keys, maps, validation)
        self._retain(self._learn(keys, labels, np.arange(len(frame))))
        return result

    def transform(self, frame):
        check_is_fitted(self, "maps_")
        self._validate(frame)
        return self._encode(_keys(frame), self.maps_, np.arange(len(frame)))

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "maps_")
        if input_features is not None and list(input_features) != list(COLUMNS):
            raise ValueError(
                "Pooled-token feature names must match the six input columns."
            )
        return np.asarray(OUTPUT_NAMES, dtype=object)


In [ ]:
%%writefile r23-source/src/fraudlab/realmlp.py
"""Official RealMLP with train-only statistics and fresh outer refits.

RealMLP: https://github.com/dholzmueller/pytabkit (Apache-2.0).
Fixed representation/recipe adapted from yekenot's public PyTabKit notebook v15:
https://www.kaggle.com/code/yekenot/ps-s6-e11-realmlp-pytabkit
Outer validation labels are deliberately absent from this module's API.
"""

from __future__ import annotations

import gc
import importlib.metadata
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import cloudpickle
import numpy as np
import pandas as pd
import torch
from pytabkit.models.alg_interfaces.base import InterfaceResources, SplitIdxs
from pytabkit.models.alg_interfaces.nn_interfaces import NNAlgInterface
from pytabkit.models.data.conversion import ToDictDatasetConverter
from pytabkit.models.data.data import DictDataset, TensorInfo
from pytabkit.models.sklearn.default_params import DefaultParams
from pytabkit.models.training.logging import StdoutLogger
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

from .models import clean_features
from .strings import categorical_strings

NUMERIC = (
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
)
CATEGORICAL = (
    "merchant_category",
    "card_type",
    "auth_method",
    "channel",
    "device_type",
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
)
DECIMALS = (
    "amount_usd",
    "hours_since_last_txn",
    "velocity_score",
    "distance_from_home_km",
)
BINS = (
    ("merchant_risk_score", 7),
    ("hours_since_last_txn", 500),
    ("hours_since_last_txn", 1000),
)
COMBOS = (
    ("merchant_category", "card_type", "txn_count_last_24h"),
    ("merchant_category", "customer_age"),
)
TE_COLUMNS = ("combo_0", "combo_1", "hours_since_last_txn")
METRIC = "1-auc_ovr"


@dataclass(frozen=True)
class RealMLPConfig:
    epochs: int = 3
    batch_size: int = 288
    learning_rate: float = 0.073
    weight_decay: float = 0.038
    dropout: float = 0.05
    inner_fraction: float = 0.15
    inner_seed: int = 42

    @classmethod
    def from_training_config(cls, config: dict):
        if config.get("fixed_training") or config.get("refit"):
            raise ValueError(
                "RealMLP requires inner selection; full-data refit is undefined."
            )
        required = {
            "feature_set": "raw",
            "hidden_dims": [512, 256, 128],
            "neural_categorical_transform": "embedding",
            "neural_numeric_transform": "realmlp",
            "target_smoothing": "auto",
        }
        for key, expected in required.items():
            if config.get(key) != expected:
                raise ValueError(f"RealMLP requires {key}={expected!r}.")
        epochs = config.get("epochs", 3)
        if not 1 <= epochs <= 3 or config.get("patience", 3) != 3:
            raise ValueError(
                "RealMLP requires epochs in 1..3 and patience=3 (no early termination)."
            )
        return cls(
            epochs=epochs,
            batch_size=config["batch_size"],
            learning_rate=config["neural_learning_rate"],
            weight_decay=config["weight_decay"],
            dropout=config["dropout"],
        )


def official_config(settings: RealMLPConfig, selection: bool) -> dict:
    """Resolve the actual 1.8.0 defaults, including PBLD's linear second layer."""
    if importlib.metadata.version("pytabkit") != "1.8.0":
        raise ValueError("RealMLP requires exactly pytabkit==1.8.0.")
    return {
        **DefaultParams.RealMLP_TD_CLASS,
        "n_ens": 16,
        "n_epochs": settings.epochs,
        "batch_size": settings.batch_size,
        "predict_batch_size": 1024,
        "use_early_stopping": False,
        "ens_av_before_softmax": False,
        "use_best_epoch": selection,
        "val_metric_name": METRIC,
        "lr": settings.learning_rate,
        "wd": settings.weight_decay,
        "sq_mom": 0.988,
        "lr_sched": "flat_anneal",
        "wd_sched": "cos_log_15",
        "first_layer_lr_factor": 0.25,
        "embedding_size": 5,
        "max_one_hot_cat_size": 16,
        "hidden_sizes": [512, 256, 128],
        "act": "silu",
        "p_drop": settings.dropout,
        "p_drop_sched": "pow5",
        "num_emb_type": "pbld",
        "use_plr_embeddings": True,
        "plr_hidden_1": 16,
        "plr_hidden_2": 8,
        "plr_act_name": "linear",
        "plr_use_densenet": True,
        "plr_use_cos_bias": True,
        "plr_lr_factor": 0.1151,
        "plr_sigma": 0.75,
        "ls_eps": 0.01,
        "ls_eps_sched": "sqrt_cos",
        "use_ls_prior": False,
        "smooth_clip_max_abs_value": 3.0,
        "robust_scale_eps": 1e-30,
        "l2_normalize_eps": 1e-8,
        "add_front_scale": False,
        "bias_init_mode": "neg-uniform-dynamic-2",
        "tfms": [
            "one_hot",
            "median_center",
            "robust_scale",
            "smooth_clip",
            "embedding",
            "l2_normalize",
        ],
    }


class RealMLPPreprocessor:
    """53 deterministic views; every learned object is fitted on this stage's train."""

    def __init__(self, seed: int):
        self.seed = seed

    def _base(self, frame: pd.DataFrame) -> pd.DataFrame:
        if set(frame) != set(NUMERIC + CATEGORICAL):
            raise ValueError(
                "RealMLP requires exactly the 24 raw features, without ID/target."
            )
        return clean_features(
            frame.loc[:, list(NUMERIC + CATEGORICAL)], list(NUMERIC), list(CATEGORICAL)
        )

    def _views(self, frame: pd.DataFrame, fit: bool) -> pd.DataFrame:
        result = self._base(frame)
        if fit:
            self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)
            values = self.imputer.fit_transform(result[list(NUMERIC)])
            self.bins = []
        else:
            values = self.imputer.transform(result[list(NUMERIC)])
        result[list(NUMERIC)] = values
        for name in DECIMALS:
            result[f"decimal_{name}"] = (result[name] % 1).round(2).astype(np.float32)
        for name in NUMERIC:
            floors = np.floor(result[name].to_numpy())
            if np.any(np.abs(floors) >= 2**63):
                raise ValueError("A numerical floor exceeds int64 support.")
            result[f"category_{name}"] = categorical_strings(floors.astype(np.int64))
        for name in DECIMALS:
            cents = np.rint(
                result[f"decimal_{name}"].to_numpy(dtype=np.float64) * 100
            ).astype(np.int64)
            result[f"category_decimal_{name}"] = categorical_strings(cents)
        for index, (name, count) in enumerate(BINS):
            if fit:
                encoder = KBinsDiscretizer(
                    n_bins=count,
                    encode="ordinal",
                    strategy="quantile",
                    subsample=None,
                    quantile_method="linear",
                )
                encoded = encoder.fit_transform(result[[name]])
                self.bins.append(encoder)
            else:
                encoded = self.bins[index].transform(result[[name]])
            result[f"bin_{index}"] = categorical_strings(
                encoded.ravel().astype(np.int64)
            )
        for index, columns in enumerate(COMBOS):
            # Tuple representation avoids ambiguous delimiter concatenations.
            codes, uniques = pd.factorize(
                pd.MultiIndex.from_frame(result[list(columns)]), sort=False
            )
            keys = np.array([repr(tuple(row)) for row in uniques], dtype=object)
            result[f"combo_{index}"] = keys[codes]
        return result

    def fit_transform(self, frame: pd.DataFrame, y: np.ndarray) -> DictDataset:
        self.fit_rows = len(frame)
        result = self._views(frame, fit=True)
        self.target_encoder = TargetEncoder(
            cv=5, smooth="auto", shuffle=True, random_state=self.seed
        )
        result[["te_0", "te_1", "te_2"]] = self.target_encoder.fit_transform(
            result[list(TE_COLUMNS)], y
        )
        self.columns = result.columns.tolist()
        self.categorical = [
            name
            for name in self.columns
            if name not in NUMERIC and not name.startswith(("decimal_", "te_"))
        ]
        self.converter = ToDictDatasetConverter(
            cat_features=[name in self.categorical for name in self.columns]
        )
        return self.converter.fit_transform(result)

    def transform(self, frame: pd.DataFrame) -> DictDataset:
        result = self._views(frame, fit=False)
        result[["te_0", "te_1", "te_2"]] = self.target_encoder.transform(
            result[list(TE_COLUMNS)]
        )
        return self.converter.transform(result[self.columns])

    def metadata(self) -> dict:
        return {
            "columns": self.columns,
            "total_features": 53,
            "numeric_features": 20,
            "categorical_features": 33,
            "training_rows": self.fit_rows,
            "categorical_cardinalities": self.converter.tensor_infos["x_cat"]
            .get_cat_sizes()
            .tolist(),
            "quantile_bin_edges": [
                encoder.bin_edges_[0].tolist() for encoder in self.bins
            ],
            "target_encoding": {
                "columns": list(TE_COLUMNS),
                "smooth": "auto",
                "cv": 5,
                "shuffle": True,
                "seed": self.seed,
                "training": "crossfit fit_transform once",
            },
        }


def _train(
    numbers: DictDataset,
    y: np.ndarray,
    validation,
    settings: RealMLPConfig,
    seed: int,
    device: str,
    threads: int,
    stop_epoch: int | None = None,
):
    """The official implementation owns architecture, optimization and epoch callbacks."""
    started = time.perf_counter()
    if device.startswith("cuda"):
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    tensors = numbers.tensors.copy()
    labels = np.asarray(y)
    if validation is not None:
        valid_numbers, valid_y = validation
        tensors = {
            key: torch.cat((value, valid_numbers.tensors[key]))
            for key, value in tensors.items()
        }
        labels = np.concatenate((labels, valid_y))
    tensors["y"] = torch.as_tensor(labels[:, None], dtype=torch.long)
    ds = DictDataset(tensors, {**numbers.tensor_infos, "y": TensorInfo(cat_sizes=[2])})
    split = SplitIdxs(
        train_idxs=torch.arange(len(y))[None, :],
        val_idxs=(
            torch.arange(len(y), len(labels))[None, :]
            if validation is not None
            else None
        ),
        test_idxs=None,
        split_seed=seed,
        sub_split_seeds=[seed],
        split_id=0,
    )
    parameters = official_config(settings, selection=validation is not None)
    fit_params = None if stop_epoch is None else [{"stop_epoch": {METRIC: stop_epoch}}]
    model = NNAlgInterface(fit_params=fit_params, **parameters)
    model.fit(
        ds,
        [split],
        InterfaceResources(threads, [device] if device.startswith("cuda") else []),
        StdoutLogger(verbosity_level=2),
        [None],
        "FraudLab RealMLP",
    )
    if device.startswith("cuda"):
        torch.cuda.synchronize()
    selected = (
        int(model.fit_params[0]["stop_epoch"][METRIC])
        if validation is not None
        else stop_epoch
    )
    details = {
        "elapsed_seconds": time.perf_counter() - started,
        "parameters": sum(p.numel() for p in model.model.model.parameters()),
        "completed_epochs": int(round(model.model.progress.epoch_float)),
        "selected_epoch": selected,
        "resolved_parameters": parameters,
        "cuda_memory_scope": "reset before official model construction; synchronized after fit; includes train, inner validation and optimizer; excludes outer/test prediction",
        "cuda_max_memory_allocated_bytes": (
            torch.cuda.max_memory_allocated() if device.startswith("cuda") else None
        ),
        "cuda_max_memory_reserved_bytes": (
            torch.cuda.max_memory_reserved() if device.startswith("cuda") else None
        ),
    }
    return model, details


def _predict(model: NNAlgInterface, numbers: DictDataset) -> np.ndarray:
    # NNAlgInterface already returns log(mean(softmax(member logits))).
    return torch.softmax(model.predict(numbers), dim=-1).mean(dim=0)[:, 1].numpy()


def fit_realmlp(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
):
    settings = RealMLPConfig.from_training_config(config)
    if config["device"] == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is not available.")
    device = (
        "cuda:0" if config["device"] != "cpu" and torch.cuda.is_available() else "cpu"
    )
    torch.set_num_threads(config["threads"])
    inner_train, inner_valid = train_test_split(
        np.arange(len(training)),
        test_size=settings.inner_fraction,
        stratify=y,
        random_state=settings.inner_seed,
    )
    inner_started = time.perf_counter()
    preprocessing = RealMLPPreprocessor(seed)
    numbers = preprocessing.fit_transform(training.iloc[inner_train], y[inner_train])
    inner_metadata = preprocessing.metadata()
    model, selection = _train(
        numbers,
        y[inner_train],
        (preprocessing.transform(training.iloc[inner_valid]), y[inner_valid]),
        settings,
        seed,
        device,
        config["threads"],
    )
    epoch = selection["selected_epoch"]
    if not 1 <= epoch <= settings.epochs:
        raise ValueError("RealMLP did not select a finite inner validation epoch.")
    selection["stage_elapsed_seconds"] = time.perf_counter() - inner_started
    del preprocessing, numbers, model
    gc.collect()
    if device.startswith("cuda"):
        torch.cuda.empty_cache()
    print(f"RealMLP selected_epoch={epoch}; fresh outer refit", flush=True)
    outer_started = time.perf_counter()
    preprocessing = RealMLPPreprocessor(seed)
    numbers = preprocessing.fit_transform(training, y)
    model, refit = _train(
        numbers,
        y,
        None,
        settings,
        seed + 10000,
        device,
        config["threads"],
        stop_epoch=epoch,
    )
    refit["stage_elapsed_seconds"] = time.perf_counter() - outer_started
    del numbers
    predictions = [
        _predict(model, preprocessing.transform(frame)) for frame in (validation, test)
    ]
    details = {
        "architecture": {
            "library": "pytabkit==1.8.0",
            "name": "RealMLP_TD",
            "n_ens": 16,
            "hidden_dims": [512, 256, 128],
            "activation": "parametric SiLU",
            "numeric_embedding": "PBLD: periodic16, linear7 + original-value skip =8",
            "categorical_embedding_dim": 5,
            "parameters": sum(p.numel() for p in model.model.model.parameters()),
        },
        "config": asdict(settings),
        "preprocessing": preprocessing.metadata(),
        "inner_preprocessing": inner_metadata,
        "selection": selection,
        "refit": refit,
        "selected_epoch": epoch,
        "inner_training_rows": len(inner_train),
        "inner_validation_rows": len(inner_valid),
        "refit_rows": len(training),
        "epoch_selection": f"inner 15% AUC, cap{settings.epochs}; discard all state, fresh outer refit for selected E; LR schedules retain cap{settings.epochs} horizon",
        "outer_validation_usage": "inference only; labels absent from API",
        "prediction": "mean of member softmax probabilities",
        "loss": "sum across16 members of batch-mean cross entropy; label smoothing .01 sqrt_cos",
        "seed": seed,
        "refit_seed": seed + 10000,
        "device": device,
        "threads": config["threads"],
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    _move(model, "cpu")
    # The official scheduler contains lambdas; ordinary pickle/joblib cannot save it.
    # Only locally trained, hash-audited bundles are loaded, as for other model files.
    with path.with_suffix(".pkl").open("wb") as stream:
        cloudpickle.dump(
            {"model": model, "preprocessing": preprocessing, "details": details}, stream
        )
    return *predictions, details


def predict_saved_realmlp(
    path: Path, frame: pd.DataFrame, device: str = "cpu"
) -> np.ndarray:
    if importlib.metadata.version("pytabkit") != "1.8.0":
        raise ValueError("RealMLP requires exactly pytabkit==1.8.0.")
    if device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA was requested but is not available.")
    with path.with_suffix(".pkl").open("rb") as stream:
        saved = cloudpickle.load(stream)
    _move(
        saved["model"],
        "cuda:0" if device != "cpu" and torch.cuda.is_available() else "cpu",
    )
    return _predict(saved["model"], saved["preprocessing"].transform(frame))


def _move(model: NNAlgInterface, device: str) -> None:
    model.to(device)
    # This fitted preprocessing layer belongs to the non-Module creator object.
    model.model.creator.static_model.to(device)
    model.model.creator.device_info = device


In [ ]:
%%writefile r23-source/src/fraudlab/refit.py
"""Refit a frozen choice once on all labeled rows, without another selection loop."""

from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from .data import fingerprint, load_data, submission, validate_submission, write_json
from .experiment import create_run, source_provenance, versions
from .features import transform_features
from .models import feature_types, fit_predict


def refit(source: Path, holdout_report: Path, output: Path, name: str) -> dict:
    started = time.perf_counter()
    config = json.loads((source / "config.json").read_text(encoding="utf-8"))
    if config.get("original_data"):
        raise ValueError("Original-data augmentation has no full-data refit contract.")
    if config.get("glm_transductive_frequency"):
        raise ValueError("Transductive frequency has no full-data refit contract.")
    if config["model"] == "realmlp":
        raise ValueError("RealMLP has no full-data refit contract.")
    if config["model"] == "neural_offset":
        raise ValueError(
            "Neural offset uses the CV checkpoint ensemble; a full-data refit protocol is not defined."
        )
    if config["model"] == "tabm":
        raise ValueError(
            "TabM uses the CV checkpoint ensemble; a full-data refit protocol is not defined."
        )
    provenance = source_provenance(config.get("feature_set"))
    selected = json.loads((source / "metrics.json").read_text(encoding="utf-8"))
    holdout = json.loads(holdout_report.read_text(encoding="utf-8"))
    if (
        selected.get("status") != "complete"
        or holdout.get("kind") != "independent_holdout"
        or holdout.get("status") != "complete"
    ):
        raise ValueError(
            "A completed development run and frozen independent holdout evaluation are required."
        )
    sources = [Path(path).resolve() for path in holdout["selection"]["sources"]]
    if source.resolve() not in sources:
        raise ValueError("This model was not part of the frozen holdout selection.")
    index = sources.index(source.resolve())
    if holdout["selection"]["weights"][index] <= 0:
        raise ValueError("A model with zero frozen weight does not need a final refit.")
    if (
        fingerprint(source / "metrics.json")
        != holdout["selection"]["metrics_hashes"][index]
        or fingerprint(source / "config.json")
        != holdout["selection"]["config_hashes"][index]
    ):
        raise ValueError(
            "Development metrics or configuration changed after the selection was frozen."
        )
    data_dir = Path(config["data"])
    if any(
        fingerprint(data_dir / filename) != expected
        for filename, expected in selected["data_hashes"].items()
    ):
        raise ValueError("Input data changed after development training.")
    folds = selected["model_details"]["folds"]
    if config["model"] == "mlp":
        config["epochs"] = int(
            round(float(np.median([fold["best_epoch"] for fold in folds])))
        )
    elif config["model"] in ("catboost", "lightgbm"):
        config["iterations"] = int(
            round(
                float(
                    np.median(
                        [
                            (
                                fold["trees_used"]
                                if config["model"] == "catboost"
                                else fold["best_iteration"]
                            )
                            for fold in folds
                        ]
                    )
                )
            )
        )
    config.update(
        refit=True,
        holdout_fraction=0.0,
        name=name,
        output=str(output),
        selection_source=str(source),
        holdout_report=str(holdout_report),
    )
    frame, test, sample = load_data(data_dir, config["id_column"], config["target"])
    target, id_column = config["target"], config["id_column"]
    frame = transform_features(
        frame, config.get("feature_set", "raw"), (id_column, target)
    )
    test = transform_features(
        test, config.get("feature_set", "raw"), (id_column, target)
    )
    features = selected["features"]["names"]
    run = create_run(output, name)
    write_json(run / "config.json", config)
    write_json(run / "status.json", {"status": "running"})
    try:
        _, predictions, details = fit_predict(
            config["model"],
            frame[features],
            frame[target].to_numpy(),
            test[features],
            None,
            test[features],
            config,
            config["seed"],
            run / "model",
        )
        result = submission(test[id_column], predictions, sample, target)
        result.to_csv(run / "predictions.csv", index=False)
        result.to_csv(run / "submission.csv", index=False)
        numeric, categorical = feature_types(frame[features])
        metrics = {
            "status": "complete",
            "model": config["model"],
            "phase": "full_data_refit",
            "roc_auc": None,
            "fold_scores": [],
            "fold_std": None,
            "rows": {"train": len(frame), "test": len(test), "oof": 0},
            "features": {
                "count": len(features),
                "numeric": len(numeric),
                "categorical": len(categorical),
                "names": features,
                "excluded": selected["features"]["excluded"],
            },
            "target": {"name": target, "positive_rate": float(frame[target].mean())},
            "seed": config["seed"],
            "validation": {
                "method": "no_new_selection_full_data_refit",
                "n_splits": 0,
                "early_stopping_uses_validation": False,
            },
            "selection": {
                "source_dev_run": str(source),
                "development_roc_auc": selected["roc_auc"],
                "holdout_report_sha256": fingerprint(holdout_report),
                "frozen_iterations_rule": "median best epoch/tree count across development folds; model hyperparameters unchanged",
            },
            "elapsed_seconds": round(time.perf_counter() - started, 3),
            "versions": versions(),
            "data_hashes": selected["data_hashes"],
            **provenance,
            "model_details": {
                **details,
                "feature_set": config.get("feature_set", "raw"),
            },
            "created_at": datetime.now(timezone.utc).isoformat(),
            "submission_sha256": fingerprint(run / "submission.csv"),
            "limitations": [
                "All labels were used to refit a previously frozen choice. No new independent score was measured for this final model.",
                "Wall-clock training times from concurrent runs are not controlled performance benchmarks.",
            ],
        }
        write_json(run / "metrics.json", metrics)
        write_json(run / "status.json", {"status": "complete"})
        return metrics
    except Exception as error:
        write_json(run / "status.json", {"status": "failed", "error": str(error)})
        raise


def blend_refits(
    runs: list[Path],
    weights: list[float] | None,
    output: Path,
    name: str,
    id_column: str,
    target: str,
) -> dict:
    """Combine final predictions with exactly the pre-holdout frozen weights."""
    started = time.perf_counter()
    provenance = source_provenance()
    configs = [
        json.loads((run / "config.json").read_text(encoding="utf-8")) for run in runs
    ]
    metrics = [
        json.loads((run / "metrics.json").read_text(encoding="utf-8")) for run in runs
    ]
    report_path = Path(configs[0]["holdout_report"])
    report = json.loads(report_path.read_text(encoding="utf-8"))
    report_hash = fingerprint(report_path)
    frozen = {
        Path(source).resolve(): weight
        for source, weight in zip(
            report["selection"]["sources"], report["selection"]["weights"]
        )
        if weight > 0
    }
    source_paths = [Path(config["selection_source"]).resolve() for config in configs]
    if set(source_paths) != set(frozen) or len(set(source_paths)) != len(source_paths):
        raise ValueError(
            "Final blend must contain every nonzero frozen source exactly once."
        )
    coefficients = np.array([frozen[source] for source in source_paths], dtype=float)
    if weights is not None:
        requested = np.array(weights, dtype=float)
        if len(requested) != len(coefficients) or not np.allclose(
            requested / requested.sum(), coefficients, rtol=0, atol=1e-12
        ):
            raise ValueError(
                "Final blend weights cannot change after independent holdout evaluation."
            )
    for config, metric in zip(configs, metrics):
        if (
            metric.get("status") != "complete"
            or metric.get("phase") != "full_data_refit"
            or metric["data_hashes"] != metrics[0]["data_hashes"]
            or metric["selection"]["holdout_report_sha256"] != report_hash
            or fingerprint(Path(config["holdout_report"])) != report_hash
        ):
            raise ValueError(
                "Final sources must be completed refits of the same frozen selection and input data."
            )
    predictions = [pd.read_csv(run / "predictions.csv") for run in runs]
    result = predictions[0].copy()
    for frame in predictions:
        validate_submission(frame, result, id_column, target)
    result[target] = sum(
        weight * frame[target].to_numpy()
        for weight, frame in zip(coefficients, predictions)
    )
    validate_submission(result, predictions[0], id_column, target)
    run = create_run(output, name)
    result.to_csv(run / "predictions.csv", index=False)
    result.to_csv(run / "submission.csv", index=False)
    details = {
        "method": "weighted_probability_average",
        "sources": [str(source) for source in runs],
        "weights": coefficients.tolist(),
    }
    metadata = {
        **metrics[0],
        **provenance,
        "model": "blend",
        "phase": "full_data_refit",
        "roc_auc": None,
        "fold_scores": [],
        "fold_std": None,
        "model_details": details,
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "submission_sha256": fingerprint(run / "submission.csv"),
        "selection": {
            "source_dev_run": report["selection"].get("source_dev_run"),
            "development_roc_auc": report["selection"].get("development_roc_auc"),
            "development_metrics_sha256": report["selection"].get(
                "development_metrics_sha256"
            ),
            "holdout_report_sha256": report_hash,
            "weights_frozen_before_holdout": True,
        },
    }
    write_json(run / "config.json", details)
    write_json(run / "metrics.json", metadata)
    write_json(run / "status.json", {"status": "complete"})
    return metadata


In [ ]:
%%writefile r23-source/src/fraudlab/scores.py
"""Pure weighted score combinations; ranking never reads target labels."""

from __future__ import annotations

import numpy as np
from scipy.stats import rankdata

SPACES = ("probability", "logit", "rank", "rank_logit")
LOGIT_CLIP = 1e-6


def blend_scores(columns, weights=None, *, space="probability") -> np.ndarray:
    """Blend whole aligned vectors; rank OOF and test in separate calls."""
    if space not in SPACES:
        raise ValueError(f"Unknown score space: {space}")
    values = [np.asarray(column, dtype=np.float64) for column in columns]
    if not values or any(
        column.ndim != 1
        or not len(column)
        or column.shape != values[0].shape
        or not np.isfinite(column).all()
        or ((column < 0) | (column > 1)).any()
        for column in values
    ):
        raise ValueError("Scores must be aligned nonempty finite vectors in [0, 1].")
    coefficients = np.asarray(
        weights if weights is not None else [1.0] * len(values), dtype=np.float64
    ).copy()
    if (
        coefficients.shape != (len(values),)
        or not np.isfinite(coefficients).all()
        or (coefficients < 0).any()
        or not np.isfinite(coefficients.sum())
        or coefficients.sum() <= 0
    ):
        raise ValueError("Provide finite nonnegative weights with a positive total.")
    coefficients /= coefficients.sum()
    if space in ("rank", "rank_logit"):
        values = [
            rankdata(column, method="average") / (len(column) + 1) for column in values
        ]
    if space in ("logit", "rank_logit"):
        values = [np.clip(column, LOGIT_CLIP, 1 - LOGIT_CLIP) for column in values]
        values = [np.log(column / (1 - column)) for column in values]
    result = sum(weight * column for weight, column in zip(coefficients, values))
    return (
        np.exp(-np.logaddexp(0.0, -result))
        if space in ("logit", "rank_logit")
        else result
    )


In [ ]:
%%writefile r23-source/src/fraudlab/splits.py
"""Explicit split strategies shared by every model."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold,
    TimeSeriesSplit,
)


def make_splits(
    frame: pd.DataFrame,
    target: str,
    method: str,
    folds: int,
    seed: int,
    group_column: str | None = None,
    time_column: str | None = None,
) -> list[tuple[np.ndarray, np.ndarray]]:
    if folds < 2:
        raise ValueError("At least two folds are required.")
    y = frame[target]
    if method == "stratified":
        if y.value_counts().min() < folds:
            raise ValueError("Every class needs at least one row per fold.")
        splits = list(
            StratifiedKFold(folds, shuffle=True, random_state=seed).split(frame, y)
        )
    elif method == "group":
        if not group_column or group_column not in frame:
            raise ValueError("Group validation requires an existing --group-column.")
        groups = frame[group_column]
        if groups.isna().any() or groups.nunique() < folds:
            raise ValueError(
                "Group column must be non-null and contain at least one group per fold."
            )
        splits = list(
            StratifiedGroupKFold(folds, shuffle=True, random_state=seed).split(
                frame, y, groups
            )
        )
    elif method == "time":
        if not time_column or time_column not in frame:
            raise ValueError("Time validation requires an existing --time-column.")
        values = frame[time_column]
        if not pd.api.types.is_numeric_dtype(values):
            values = pd.to_datetime(values, errors="raise", utc=True)
        if values.isna().any():
            raise ValueError("Time column must not contain missing values.")
        # Split unique timestamps, so simultaneous events never cross a boundary.
        times = np.sort(values.unique())
        if len(times) <= folds:
            raise ValueError("Time validation needs more unique timestamps than folds.")
        splits = [
            (
                np.flatnonzero(values.isin(times[a])),
                np.flatnonzero(values.isin(times[b])),
            )
            for a, b in TimeSeriesSplit(folds).split(times)
        ]
    else:
        raise ValueError(f"Unknown validation method: {method}")
    seen: set[int] = set()
    for training, validation in splits:
        if set(training) & set(validation) or seen & set(validation):
            raise ValueError(
                "Training and validation rows must be disjoint; each OOF row is predicted once."
            )
        seen.update(validation)
        if y.iloc[training].nunique() != 2 or y.iloc[validation].nunique() != 2:
            raise ValueError(
                "Every training and validation fold needs both classes for ROC-AUC."
            )
    return splits


In [ ]:
%%writefile r23-source/src/fraudlab/strings.py
"""Object string arrays whose repeated values reference the same Python string."""

import numpy as np


def categorical_strings(values: np.ndarray) -> np.ndarray:
    """Preserve integer/Unicode text while converting only distinct values.

    This is storage sharing, not a learned vocabulary or pandas categorical dtype.
    Float/object inputs are excluded because their formatting is not interchangeable
    with pandas' existing raw-value conversion.
    """
    values = np.asarray(values)
    if values.ndim != 1 or values.dtype.kind not in "iuU":
        raise ValueError(
            "Shared categorical strings require integer or Unicode arrays."
        )
    unique, inverse = np.unique(values, return_inverse=True)
    return unique.astype(str).astype(object)[inverse]


In [ ]:
%%writefile r23-source/src/fraudlab/tabm.py
"""TabM with inner epoch selection and fresh outer-training refits.

The outer validation labels are deliberately absent from this module's API.
Architecture: https://github.com/yandex-research/tabm (Apache-2.0).
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from .models import NeuralPreprocessor, clean_features
from rtdl_num_embeddings import PiecewiseLinearEmbeddings, compute_bins
from tabm import TabM


@dataclass(frozen=True)
class TabMConfig:
    n_blocks: int = 2
    d_block: int = 256
    k: int = 16
    dropout: float = 0.1
    d_embedding: int = 12
    n_bins: int = 48
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 1024
    epochs: int = 40
    patience: int = 6
    gradient_clip: float = 1.0
    inner_fraction: float = 0.15
    inner_seed: int = 42
    numeric_transform: str = "standard"
    target_smoothing: float = 20.0

    @classmethod
    def from_training_config(cls, config: dict):
        if config.get("fixed_training") or config.get("refit"):
            raise ValueError(
                "TabM requires inner epoch selection; fixed-training/full-data refit is not defined."
            )
        if config.get("neural_categorical_transform", "target") != "target":
            raise ValueError("TabM requires --neural-categorical-transform target.")
        if config.get("hidden_dims", [256, 256]) != [256, 256]:
            raise ValueError("TabM architecture is fixed: --hidden-dims 256 256.")
        return cls(
            dropout=config["dropout"],
            learning_rate=config["neural_learning_rate"],
            weight_decay=config["weight_decay"],
            batch_size=config["batch_size"],
            epochs=config["epochs"],
            patience=config["patience"],
            numeric_transform=config.get("neural_numeric_transform", "standard"),
            target_smoothing=config["target_smoothing"],
        )


def build_tabm(bins: list[torch.Tensor], config: TabMConfig) -> TabM:
    return TabM.make(
        n_num_features=len(bins),
        d_out=1,
        num_embeddings=PiecewiseLinearEmbeddings(
            bins, config.d_embedding, activation=False, version="B"
        ),
        arch_type="tabm",
        n_blocks=config.n_blocks,
        d_block=config.d_block,
        k=config.k,
        dropout=config.dropout,
    )


def member_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """One BCE per row AND member. Never BCE on the average prediction."""
    if logits.ndim != 3 or logits.shape[-1] != 1:
        raise ValueError("Expected [batch, k, 1] logits.")
    return torch.nn.functional.binary_cross_entropy_with_logits(
        logits[..., 0], y[:, None].expand_as(logits[..., 0])
    )


def mean_probability(logits: torch.Tensor) -> torch.Tensor:
    return logits[..., 0].sigmoid().mean(dim=1)


def prepare_training(frame: pd.DataFrame, y: np.ndarray, seed: int, config: TabMConfig):
    preprocessing = NeuralPreprocessor(
        numeric_transform=config.numeric_transform,
        seed=seed,
        categorical_transform="target",
        smoothing=config.target_smoothing,
    )
    numbers, categories = preprocessing.fit_transform(frame, y)
    assert categories.shape == (len(frame), 0)
    # The official binning API rejects constants. Select using training only;
    # the same fixed mask is applied to every later partition and checkpoint.
    active = np.any(numbers != numbers[0], axis=0)
    if not active.any():
        raise ValueError("TabM needs at least one nonconstant training feature.")
    numbers = np.ascontiguousarray(numbers[:, active])
    bins = compute_bins(torch.from_numpy(numbers), n_bins=config.n_bins)
    return preprocessing, active, numbers, bins


@torch.no_grad()
def predict_array(model, numbers: torch.Tensor, batch_size: int) -> np.ndarray:
    model.eval()
    return np.concatenate(
        [
            mean_probability(model(batch)).cpu().numpy()
            for batch in numbers.split(batch_size)
        ]
    )


def _train(numbers, y, bins, config, seed, device, *, validation=None, epochs=None):
    """Select E on inner validation, or train exactly E from a new seed state."""
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    model = build_tabm(bins, config).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    x = torch.as_tensor(numbers, device=device)
    labels = torch.as_tensor(y, dtype=torch.float32, device=device)
    best_auc, best_epoch, stale, history = -np.inf, 0, 0, []
    for epoch in range(1, (epochs or config.epochs) + 1):
        model.train()
        for selected in torch.randperm(len(y), device=device).split(config.batch_size):
            optimizer.zero_grad(set_to_none=True)
            loss = member_loss(model(x[selected]), labels[selected])
            if not torch.isfinite(loss):
                raise ValueError("TabM loss became nonfinite.")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
            optimizer.step()
        if validation is None:
            best_epoch = epoch
            continue
        vx, vy = validation
        score = roc_auc_score(vy, predict_array(model, vx, config.batch_size))
        history.append({"epoch": epoch, "inner_auc": float(score)})
        print(f"tabm seed={seed} inner_epoch={epoch} roc_auc={score:.8f}", flush=True)
        if score > best_auc + 1e-6:
            best_auc, best_epoch, stale = score, epoch, 0
        else:
            stale += 1
        if stale >= config.patience:
            break
    return model, best_epoch, history, epoch


def fit_tabm(
    training: pd.DataFrame,
    y: np.ndarray,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    config: dict,
    seed: int,
    path: Path,
):
    """Outer validation is inference-only: its labels cannot enter this API."""
    settings = TabMConfig.from_training_config(config)
    torch.set_num_threads(config["threads"])
    if config["device"] == "cuda" and not torch.cuda.is_available():
        raise ValueError(
            "CUDA was requested but is not available in this PyTorch installation."
        )
    device = torch.device(
        "cuda" if config["device"] != "cpu" and torch.cuda.is_available() else "cpu"
    )
    inner_train, inner_valid = train_test_split(
        np.arange(len(y)),
        test_size=settings.inner_fraction,
        random_state=settings.inner_seed,
        stratify=y,
    )
    inner_pre, inner_active, inner_x, inner_bins = prepare_training(
        training.iloc[inner_train], y[inner_train], seed, settings
    )
    inner_vx, _ = inner_pre.transform(training.iloc[inner_valid])
    selection_model, selected_epoch, history, selection_epochs_completed = _train(
        inner_x,
        y[inner_train],
        inner_bins,
        settings,
        seed,
        device,
        validation=(
            torch.as_tensor(inner_vx[:, inner_active], device=device),
            y[inner_valid],
        ),
    )
    # Discard ALL inner preprocessing, bins, weights and optimizer state.
    del selection_model, inner_pre, inner_active, inner_x, inner_bins, inner_vx
    print(f"tabm seed={seed} fresh_outer_refit_epochs={selected_epoch}", flush=True)
    preprocessing, active, outer_x, bins = prepare_training(training, y, seed, settings)
    model, _, _, refit_epochs_completed = _train(
        outer_x, y, bins, settings, seed, device, epochs=selected_epoch
    )
    predictions = []
    for frame in (validation, test):
        numbers, _ = preprocessing.transform(frame)
        predictions.append(
            predict_array(
                model,
                torch.as_tensor(numbers[:, active], device=device),
                settings.batch_size,
            )
        )
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {"preprocessing": preprocessing, "active": active},
        path.with_suffix(".preprocessing.joblib"),
    )
    torch.save(
        {
            "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "bins": [b.cpu() for b in bins],
            "config": asdict(settings),
            "selected_epoch": selected_epoch,
        },
        path.with_suffix(".pt"),
    )
    return *predictions, {
        "architecture": {
            "name": "TabM",
            "n_blocks": settings.n_blocks,
            "d_block": settings.d_block,
            "k": settings.k,
            "dropout": settings.dropout,
            "d_out": 1,
            "num_embeddings": {
                "name": "PiecewiseLinearEmbeddings",
                "version": "B",
                "activation": False,
                "d_embedding": settings.d_embedding,
                "n_bins": settings.n_bins,
            },
            "parameters": sum(parameter.numel() for parameter in model.parameters()),
        },
        "config": asdict(settings),
        "selected_epoch": selected_epoch,
        "selection_epochs_completed": selection_epochs_completed,
        "refit_epochs_completed": refit_epochs_completed,
        "inner_history": history,
        "inner_training_rows": len(inner_train),
        "inner_validation_rows": len(inner_valid),
        "refit_rows": len(training),
        "input_features": len(active),
        "active_features": int(active.sum()),
        "active_columns": np.flatnonzero(active).tolist(),
        "numeric_features": preprocessing.numeric,
        "categorical_features": preprocessing.categorical,
        "bin_edges_per_active_feature": [len(b) for b in bins],
        "optimizer": "AdamW",
        "loss": "mean BCEWithLogits over rows and members",
        "inference": "mean sigmoid probability over members",
        "learning_rate": settings.learning_rate,
        "weight_decay": settings.weight_decay,
        "batch_size": settings.batch_size,
        "device": str(device),
        "seed": seed,
        "threads": torch.get_num_threads(),
        "target_encoder": {
            "internal_folds": 5,
            "smooth": settings.target_smoothing,
            "log_odds": True,
            "training_values": "crossfit",
        },
        "numeric_transform": settings.numeric_transform,
        "bins_fit": "training only per stage; quantiles, no labels",
        "epoch_selection": "inner 15% AUC; fresh full outer-training refit",
        "outer_validation_usage": "inference only; no labels accepted",
    }


def predict_saved_tabm(path: Path, frame: pd.DataFrame, device: str = "cpu"):
    if device == "cuda" and not torch.cuda.is_available():
        raise ValueError(
            "CUDA was requested but is not available in this PyTorch installation."
        )
    device = "cuda" if device != "cpu" and torch.cuda.is_available() else "cpu"
    payload = torch.load(path.with_suffix(".pt"), map_location="cpu", weights_only=True)
    config = TabMConfig(**payload["config"])
    model = build_tabm(payload["bins"], config).to(device)
    model.load_state_dict(payload["state_dict"])
    saved = joblib.load(path.with_suffix(".preprocessing.joblib"))
    preprocessing = saved["preprocessing"]
    numbers, _ = preprocessing.transform(
        clean_features(frame, preprocessing.numeric, preprocessing.categorical)
    )
    return predict_array(
        model,
        torch.as_tensor(numbers[:, saved["active"]], device=device),
        config.batch_size,
    )


In [ ]:
%%writefile r23-source/src/fraudlab/tokens.py
"""Complete-number GPT-2 keys and integer-prefix views, without labels.

Inspired by megayak's numeric-token views and goodpjw2008's Part 12 notebook
(version 16). Independently implemented; no notebook code or predictions are
copied. Canonical float64 text intentionally differs from original CSV lexemes.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from .bpe import ENCODING, ENCODING_FILES, TOKENIZER_VERSION, _encoding
from .pooled import COLUMNS, canonical_decimal

SUFFIXES = (
    "__fulltoken_first_key",
    "__fulltoken_second_key",
    "__fulltoken_last_key",
    "__fulltoken_length_key",
    "__integer_tens_key",
    "__integer_length_first_key",
    "__integer_length_two_key",
)


def token_keys(frame: pd.DataFrame, excluded: tuple[str, ...] = ()) -> pd.DataFrame:
    """Return seven categorical keys per allowed continuous column.

    Inputs must also fit the exact-cent range inherited by value_digits. The
    complete decimal text retains fractional cents; there is no cent rounding.
    A missing second token is -1. Prefixes come from floor(x), with zero length1.
    """
    result = pd.DataFrame(index=frame.index)
    encoding = _encoding()
    for name in COLUMNS:
        if name in excluded:
            continue
        if name not in frame:
            raise ValueError(f"Missing full-token input column: {name}")
        series = frame[name]
        if (
            not pd.api.types.is_numeric_dtype(series)
            or pd.api.types.is_bool_dtype(series)
            or pd.api.types.is_complex_dtype(series)
        ):
            raise ValueError(f"{name} requires numeric full-token inputs.")
        values = series.to_numpy(dtype=np.float64)
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError(f"{name} requires finite nonnegative values.")
        if (values > (2**53 - 1) / 100).any():
            raise ValueError(
                f"{name} exceeds the exact-cent range inherited from value_digits."
            )
        unique, inverse = np.unique(values, return_inverse=True)
        keys = []
        for value in unique:
            text = canonical_decimal(value)
            pieces = encoding.encode_ordinary(" " + text)
            integer = text.split(".", 1)[0]
            keys.append(
                (
                    str(pieces[0]),
                    str(pieces[1] if len(pieces) > 1 else -1),
                    str(pieces[-1]),
                    str(len(pieces)),
                    str(int(integer) // 10),
                    f"{len(integer)}:{integer[:1]}",
                    f"{len(integer)}:{integer[:2]}",
                )
            )
        columns = np.asarray(keys, dtype=object).reshape(-1, len(SUFFIXES))[inverse]
        for index, suffix in enumerate(SUFFIXES):
            result[name + suffix] = columns[:, index]
    return result


def manifest() -> dict:
    encoding = _encoding()
    return {
        "algorithm": "complete_numeric_token_and_integer_prefix_keys",
        "tiktoken": TOKENIZER_VERSION,
        "encoding": ENCODING,
        "vocabulary_size": encoding.max_token_value + 1,
        "encoding_files": list(ENCODING_FILES),
        "input_columns": list(COLUMNS),
        "output_suffixes": list(SUFFIXES),
        "output_count": len(COLUMNS) * len(SUFFIXES),
        "base_feature_set": "value_digits",
        "canonical_text": "float64 -> Decimal(str(float(value))) -> fixed decimal notation, at least .0; one leading ASCII space",
        "zero": "-0.0 normalized to 0.0",
        "fractional_cents": "retained; no cent rounding or decimal padding",
        "csv_lexemes": "not preserved; trailing zeros and exponent spelling may differ",
        "second_token_missing": "-1",
        "integer_keys": "floor(x/10), integer-digit-length:first-digit, integer-digit-length:first-two-digits; floor(0) has length1",
        "numeric_range": "finite, nonnegative, at most (2**53-1)/100 to support inherited exact-cent views",
        "category_semantics": "all seven keys are categorical labels, not ordered token magnitudes",
        "label_usage": "none; only the six declared input columns are read",
        "inspiration": [
            "https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack",
            "https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298",
        ],
    }


In [ ]:
%%writefile r23-source/src/fraudlab/transductive.py
"""Label-free categorical counts over official training and test features."""

from __future__ import annotations

import hashlib
import json

import pandas as pd

from .models import clean_features, feature_types


def validate_config(config: dict) -> None:
    if not config.get("glm_transductive_frequency"):
        return
    if (
        not config.get("glm_frequency")
        or config.get("model") not in ("glm", "offset")
        or config.get("phase") != "exploratory_full_data_cv"
        or config.get("split") != "stratified"
        or config.get("folds") != 5
        or config.get("seed") != 42
        or config.get("holdout_fraction", 0)
        or config.get("original_data")
        or config.get("refit")
    ):
        raise ValueError(
            "Transductive frequency requires GLM/offset, glm_frequency and exploratory CV5/42 without original data, holdout or refit."
        )


def counts_sha256(counts: dict[str, pd.Series]) -> str:
    """Hash sorted UTF-8 category/count records with unambiguous JSON framing."""
    digest = hashlib.sha256()
    for name in sorted(counts):
        column = counts[name]
        digest.update((json.dumps([name], ensure_ascii=False) + "\n").encode("utf-8"))
        for key, value in column.sort_index().items():
            digest.update(
                (json.dumps([key, int(value)], ensure_ascii=False) + "\n").encode(
                    "utf-8"
                )
            )
    return digest.hexdigest()


def build_reference_counts(
    training: pd.DataFrame,
    test: pd.DataFrame,
    feature_columns: list[str],
    corpus_hashes: dict,
    excluded: tuple[str, ...] = ("id", "is_fraud"),
) -> tuple[dict[str, pd.Series], dict]:
    """Count one column at a time; never concatenate a million-row wide frame."""
    columns = [name for name in feature_columns if name not in excluded]
    if (
        training.empty
        or test.empty
        or not training.columns.is_unique
        or not test.columns.is_unique
        or len(columns) != len(set(columns))
        or not all(isinstance(name, str) for name in columns)
        or not set(columns).issubset(training)
        or not set(columns).issubset(test)
    ):
        raise ValueError(
            "Transductive counts require nonempty frames and matching feature columns."
        )
    # Zero-row selection preserves exactly feature_types' dtype semantics.
    _, categorical = feature_types(training.iloc[:0][columns])
    if not categorical:
        raise ValueError("Transductive frequency requires categorical features.")
    reference = {}
    for name in categorical:
        left = clean_features(training[[name]], [], [name])[name].value_counts(
            sort=False
        )
        right = clean_features(test[[name]], [], [name])[name].value_counts(sort=False)
        reference[name] = left.add(right, fill_value=0).astype("int64")
    metadata = {
        "scope": "transductive official train + test feature counts",
        "label_usage": "none",
        "rows": {
            "train": len(training),
            "test": len(test),
            "total": len(training) + len(test),
        },
        "corpus_hashes": dict(corpus_hashes),
        "columns": categorical,
        "counts_sha256": counts_sha256(reference),
        "normalization": "clean_features categorical string conversion and __MISSING__",
        "supervised_statistics": "target encoding, pooled target rates and all scalers fit on outer training only",
    }
    return reference, metadata


In [ ]:
%%writefile r23-source/scripts/refit_selected_cv.py
"""Refit one audited exploratory-CV recipe on all labels; no new AUC or submission.

This explicit contract is separate from the independent-holdout refit command.
Only the pinned core imports after verification; fit parameters come unchanged
from the audited CV configurations, and the external predictions remain fixed.
"""

from __future__ import annotations

import argparse
import gc
import hashlib
import importlib
import json
import os
from pathlib import Path
import subprocess
import sys
import time
from datetime import datetime, timezone

ENTRYPOINT = Path(__file__).resolve()
ROLES = ("offset", "lightgbm", "external")
WEIGHTS = [0.5625, 0.1875, 0.25]
EXECUTION = {"data", "output", "name", "phase", "refit", "validation_note"}


def require(condition, message):
    if not condition:
        raise ValueError(message)


def digest(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(block)
    return result.hexdigest()


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def write(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )


def parameters_sha256(config):
    values = {key: value for key, value in config.items() if key not in EXECUTION}
    return hashlib.sha256(
        json.dumps(values, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()


def checked_inputs(root, plan):
    require(plan["weights"] == WEIGHTS, "Wrong probability weights.")
    require(set(plan["inputs"]) == {"selection", *ROLES}, "Wrong source roles.")
    loaded, frozen = {}, {}
    for role, spec in plan["inputs"].items():
        directory = (root / spec["path"]).resolve()
        require(directory.is_relative_to(root), "Source outside the workspace.")
        needed = {"config.json", "metrics.json", "status.json"}
        if role in ROLES[:2]:
            needed |= {f"fold-{i}.joblib" for i in range(5)}
        require(needed <= spec["sha256"].keys(), "Missing source pins.")
        for name, expected in spec["sha256"].items():
            require(Path(name).name == name, "Pinned file must be a basename.")
            path = directory / name
            require(digest(path) == expected, "Input changed: " + str(path))
            frozen[str(path)] = expected
        metric, cfg = read(directory / "metrics.json"), read(directory / "config.json")
        require(
            metric["status"] == read(directory / "status.json")["status"] == "complete",
            "Incomplete source.",
        )
        require(metric["phase"] == "exploratory_full_data_cv", "Not exploratory CV.")
        require(
            metric["rows"] == {**plan["rows"], "oof": plan["rows"]["train"]},
            "Wrong source rows.",
        )
        require(metric["data_hashes"] == plan["data_hashes"], "Wrong source data.")
        if role in ROLES[:2]:
            require(
                cfg["model"] == role
                and cfg["phase"] == "exploratory_full_data_cv"
                and cfg["folds"] == 5
                and cfg["split"] == "stratified"
                and cfg["holdout_fraction"] == 0
                and cfg["seed"] == 42
                and cfg["fixed_training"],
                "Wrong refit protocol.",
            )
            require(
                not cfg.get("original_data")
                and not cfg.get("glm_transductive_frequency"),
                "Extra/transductive data excluded.",
            )
            require(
                metric["config_sha256"] == spec["sha256"]["config.json"],
                "Config receipt differs.",
            )
            require(
                parameters_sha256(cfg) == spec["parameters_sha256"],
                "Parameters changed.",
            )
            require(
                all(
                    cfg[key] == value
                    for key, value in spec.get("parameters_summary", {}).items()
                ),
                "Fixed recipe summary differs.",
            )
        loaded[role] = metric, cfg
    receipts = {}
    for kind in ("audit", "public_proof"):
        spec, path = plan[kind], root / plan[kind]["path"]
        require(path.resolve().is_relative_to(root), "Receipt outside the workspace.")
        require(digest(path) == spec["sha256"], "Receipt changed: " + kind)
        receipts[kind], frozen[str(path)] = read(path), spec["sha256"]
    metric = loaded["selection"][0]
    name = Path(plan["inputs"]["selection"]["path"]).name
    audit, proof = receipts["audit"], receipts["public_proof"]
    require(
        audit["status"] == "passed" and audit["source_clean_at_start_and_finish"],
        "Audit did not pass.",
    )
    require(
        audit["source_revision"] == plan["source"]["revision"]
        and audit["competition_data_hashes"] == plan["data_hashes"],
        "Audit source/data differ.",
    )
    require(
        audit["selected_within_round"]["run"] == name
        and audit["selected_within_round"]["eligible_to_submit"] is True
        and audit["selected_within_round"]["oof_auc"]
        == metric["roc_auc"]
        == plan["reference_cv_auc"],
        "Audited selection differs.",
    )
    audited = {
        k.replace("\\", "/").removeprefix("./"): v
        for k, v in audit["inputs_sha256"].items()
    }
    for role, spec in plan["inputs"].items():
        for filename in ("config.json", "metrics.json"):
            require(
                audited.get(spec["path"] + "/" + filename) == spec["sha256"][filename],
                "Unaudited source: " + role,
            )
    require(
        proof["run"] == name
        and proof["submission_file_verified"]
        and proof["ref"] == plan["public_proof"]["ref"],
        "Wrong public proof.",
    )
    require(
        proof["submission_sha256"]
        == proof["downloaded_submission_sha256"]
        == metric["submission_sha256"],
        "Public CSV differs.",
    )
    recipe = metric["model_details"]
    actual = {
        Path(p.replace("\\", "/")).name: w
        for p, w in zip(recipe["sources"], recipe["weights"])
        if w
    }
    expected = {Path(plan["inputs"][r]["path"]).name: w for r, w in zip(ROLES, WEIGHTS)}
    require(actual == expected, "Selected probability recipe differs.")
    component = metric["external_components"]
    require(
        len(component) == 1
        and component[0]["predictions_source_version"] == 2
        and component[0]["weight"] == 0.25,
        "External release differs.",
    )
    require(
        {"predictions.csv", "submission.csv"}
        <= plan["inputs"]["external"]["sha256"].keys(),
        "External CSVs unpinned.",
    )
    return loaded, frozen


def final_config(original, name, data, output):
    config = {
        **original,
        "name": name,
        "data": str(data),
        "output": str(output),
        "phase": "full_data_refit",
        "refit": True,
        "validation_note": "All-label refit of a fixed exploratory-CV selection; no new local score or independent holdout.",
    }
    require(
        parameters_sha256(config) == parameters_sha256(original),
        "Refit changed training parameters.",
    )
    return config


def pin_core(worktree, spec):
    git = lambda *args: subprocess.check_output(
        ["git", "-C", str(worktree), *args], text=True
    ).strip()
    require(
        git("rev-parse", "HEAD") == spec["revision"]
        and not git("status", "--porcelain"),
        "Core must be clean at its pinned revision.",
    )
    require(
        {p.name: digest(p) for p in sorted((worktree / "src/fraudlab").glob("*.py"))}
        == spec["sha256"],
        "Core source hashes differ.",
    )
    require("fraudlab" not in sys.modules, "Core imported before source pinning.")
    sys.path.insert(0, str(worktree / "src"))
    core = importlib.import_module("fraudlab")
    require(
        Path(core.__file__).resolve().is_relative_to(worktree / "src"),
        "Wrong core import.",
    )
    os.chdir(worktree)


def fit_and_reload(config, training, y, test, path, fit_predict, predict_saved):
    import numpy as np

    _, prediction, details = fit_predict(
        config["model"], training, y, test, None, test, config, config["seed"], path
    )
    require(
        len(prediction) == len(test)
        and np.isfinite(prediction).all()
        and ((prediction >= 0) & (prediction <= 1)).all(),
        "Invalid full-data predictions.",
    )
    count = min(1024, len(test))
    reloaded = predict_saved(config["model"], path, test.iloc[:count], "cpu")
    error = float(np.max(np.abs(reloaded - prediction[:count])))
    require(error <= 1e-12, "Full-data checkpoint reload differs.")
    return (
        prediction,
        details,
        {
            "test_rows": count,
            "max_absolute_error": error,
            "atol": 1e-12,
            "rtol": 0,
            "checkpoint_sha256": digest(path.with_suffix(".joblib")),
        },
    )


def execute(plan_path, root, worktree):
    started = time.perf_counter()
    plan_path, root, worktree = plan_path.resolve(), root.resolve(), worktree.resolve()
    plan = read(plan_path)
    require(
        digest(ENTRYPOINT) == plan["entrypoint_sha256"],
        "Entrypoint differs from the published plan.",
    )
    loaded, frozen = checked_inputs(root, plan)
    frozen.update(
        {str(plan_path): digest(plan_path), str(ENTRYPOINT): digest(ENTRYPOINT)}
    )
    pin_core(worktree, plan["source"])
    import numpy as np
    import pandas as pd
    from fraudlab.data import load_data, submission, validate_submission
    from fraudlab.experiment import create_run, source_provenance, versions
    from fraudlab.features import transform_features
    from fraudlab.models import fit_predict, predict_saved

    data, output = root / plan["data"], root / plan["output"]
    require(
        data.resolve().is_relative_to(root) and output.resolve().is_relative_to(root),
        "Data/output outside the workspace.",
    )
    for filename, expected in plan["data_hashes"].items():
        require(digest(data / filename) == expected, "Official data changed.")
        frozen[str(data / filename)] = expected
    raw, raw_test, sample = load_data(data, plan["id_column"], plan["target"])
    require(
        len(raw) == plan["rows"]["train"] and len(raw_test) == plan["rows"]["test"],
        "Row counts differ.",
    )
    reference = root / plan["inputs"]["selection"]["path"]
    selection = {
        "source_dev_run": str(reference),
        "reference_cv_run": reference.name,
        "development_metrics_sha256": plan["inputs"]["selection"]["sha256"][
            "metrics.json"
        ],
        "development_roc_auc": plan["reference_cv_auc"],
        "development_roc_auc_scope": "Historical exploratory full-data CV of the selected hybrid; not an AUC of this refit.",
        "independent_holdout": False,
    }
    provenance = source_provenance()
    provenance["source_hashes_timing"] = (
        "Pinned core checked before reading CSVs; hashes captured again before fitting."
    )
    common = {
        **provenance,
        "status": "complete",
        "phase": "full_data_refit",
        "roc_auc": None,
        "fold_scores": [],
        "fold_std": None,
        "rows": {**plan["rows"], "oof": 0},
        "data_hashes": plan["data_hashes"],
        "selection": selection,
        "versions": versions(),
        "plan_sha256": digest(plan_path),
        "refit_script_sha256": digest(ENTRYPOINT),
        "validation": {
            "method": "all_labels_refit_of_audited_exploratory_cv",
            "n_splits": 0,
            "independent_holdout": False,
            "early_stopping_uses_validation": False,
        },
        "limitations": [
            "No OOF or independent local score exists for these full-data fits.",
            "Replacing five folds by one all-label model changes inference; improved performance is not established.",
        ],
    }
    predictions, bundles = {}, {}
    for role in ROLES[:2]:
        metric, original = loaded[role]
        spec = plan["inputs"][role]
        config = final_config(original, plan["refit_names"][role], data, output)
        run = create_run(output, config["name"])
        write(run / "config.json", config)
        write(run / "status.json", {"status": "running"})
        try:
            train = transform_features(
                raw, config["feature_set"], (config["id_column"], config["target"])
            )
            test = transform_features(
                raw_test, config["feature_set"], (config["id_column"], config["target"])
            )
            names = metric["features"]["names"]
            require(
                [
                    name
                    for name in train
                    if name not in (config["id_column"], config["target"])
                ]
                == names,
                "Engineered schema differs from the selected CV.",
            )
            prediction, details, reload = fit_and_reload(
                config,
                train[names],
                train[config["target"]].to_numpy(),
                test[names],
                run / "model",
                fit_predict,
                predict_saved,
            )
            if role == "offset":
                require(
                    details["encoded_features"] == spec["encoded_features"],
                    "Full-data encoded feature count differs.",
                )
            frame = submission(
                test[config["id_column"]], prediction, sample, config["target"]
            )
            for filename in ("predictions.csv", "submission.csv"):
                frame.to_csv(run / filename, index=False)
            result = {
                **common,
                "model": role,
                "config_sha256": digest(run / "config.json"),
                "features": metric["features"],
                "model_details": details,
                "checkpoint_reload": reload,
                "source_cv_run": spec["path"],
                "source_parameters_sha256": spec["parameters_sha256"],
                "submission_sha256": digest(run / "submission.csv"),
            }
            if metric.get("feature_encoding") is not None:
                result["feature_encoding"] = metric["feature_encoding"]
            write(run / "metrics.json", result)
            write(run / "status.json", {"status": "complete"})
            predictions[role] = frame[config["target"]].to_numpy()
            bundles[role] = {
                "path": str(run),
                "sha256": {
                    name: digest(run / name)
                    for name in (
                        "config.json",
                        "metrics.json",
                        "model.joblib",
                        "submission.csv",
                    )
                },
            }
            del train, test, prediction
            gc.collect()
        except Exception as error:
            write(run / "status.json", {"status": "failed", "error": str(error)})
            raise
    external = root / plan["inputs"]["external"]["path"]
    external_frame = pd.read_csv(external / "predictions.csv")
    validate_submission(external_frame, sample, plan["id_column"], plan["target"])
    predictions["external"] = external_frame[plan["target"]].to_numpy()
    credited = [
        {k: v for k, v in item.items() if k != "oof_auc"}
        for item in loaded["selection"][0]["external_components"]
    ]
    values = sum(weight * predictions[role] for role, weight in zip(ROLES, WEIGHTS))
    final = submission(sample[plan["id_column"]], values, sample, plan["target"])
    require(
        np.array_equal(final[plan["target"]].to_numpy(), values),
        "Final probability formula differs.",
    )
    for path, expected in frozen.items():
        require(digest(path) == expected, "Input changed during refit: " + path)
    require(
        source_provenance()["source_hashes"] == plan["source"]["sha256"]
        and source_provenance()["git_revision"] == plan["source"]["revision"]
        and source_provenance()["git_dirty"] is False,
        "Pinned core changed.",
    )
    run = create_run(output, plan["name"])
    config = {
        "operation": "refit_selected_exploratory_cv",
        "plan_sha256": digest(plan_path),
        "effective_weights": dict(zip(ROLES, WEIGHTS)),
        "inputs": plan["inputs"],
        "refits": bundles,
    }
    write(run / "config.json", config)
    for filename in ("predictions.csv", "submission.csv"):
        final.to_csv(run / filename, index=False)
    written = pd.read_csv(run / "submission.csv")
    validate_submission(written, sample, plan["id_column"], plan["target"])
    part_paths = [Path(bundles[r]["path"]) for r in ROLES[:2]] + [external]
    parts = [pd.read_csv(p / "submission.csv")[plan["target"]] for p in part_paths]
    expected = sum(w * values.to_numpy() for w, values in zip(WEIGHTS, parts))
    formula_error = float(np.max(np.abs(written[plan["target"]] - expected)))
    require(formula_error <= 1e-15, "Serialized full-test formula differs.")
    metric = {
        **common,
        "model": "blend",
        "feature_encoding": loaded["offset"][0].get("feature_encoding"),
        "features": loaded["offset"][0]["features"],
        "config_sha256": digest(run / "config.json"),
        "effective_weights": dict(zip(ROLES, WEIGHTS)),
        "model_details": {
            "method": "weighted_probability_average",
            "sources": [bundles[role]["path"] for role in ROLES[:2]] + [str(external)],
            "weights": WEIGHTS,
        },
        "refits": bundles,
        "external_components": credited,
        "formula_rows_verified": len(final),
        "formula_max_absolute_error": formula_error,
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "submission_sha256": digest(run / "submission.csv"),
    }
    write(run / "metrics.json", metric)
    write(run / "status.json", {"status": "complete"})
    return metric


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--plan", type=Path, required=True)
    parser.add_argument("--root", type=Path, required=True)
    parser.add_argument("--source-worktree", type=Path, required=True)
    args = parser.parse_args()
    result = execute(args.plan, args.root, args.source_worktree)
    print(
        json.dumps(
            {
                key: result[key]
                for key in ("phase", "roc_auc", "rows", "submission_sha256")
            }
        )
    )


In [ ]:
%%writefile r23-source/scripts/refit_neural_cv.py
"""Refit one audited RealMLP CV recipe; reuse the audited R20 full models.

An explicit all-label contract, separate from the independent-holdout refit.
No new AUC, OOF, epoch search, tree fit or submission is performed.
"""

from __future__ import annotations

import argparse
import hashlib
import importlib.util
import importlib.metadata
import json
from pathlib import Path
import statistics
import sys
import time

ENTRYPOINT = Path(__file__).resolve()
WEIGHTS = [0.421875, 0.140625, 0.1875, 0.25]
ROLES = ("offset", "realmlp", "lightgbm", "external_gpt2")


def require(condition, message):
    if not condition:
        raise ValueError(message)


def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def write(path, value):
    Path(path).write_bytes(
        (
            json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
        ).encode()
    )


def chosen_epoch(plan, own_metric, selected_metric, audit22):
    require(
        own_metric["status"] == selected_metric["status"] == "complete"
        and audit22["status"] == "passed",
        "Incomplete CV or independent audit.",
    )
    require(
        own_metric["phase"] == selected_metric["phase"] == "exploratory_full_data_cv",
        "Selection must be exploratory CV.",
    )
    epochs = [item["selected_epoch"] for item in own_metric["model_details"]["folds"]]
    require(epochs == plan["cv_epochs"] == [3, 2, 3, 3, 3], "CV epochs changed.")
    selected = audit22.get("selected_within_round") or {}
    require(
        selected.get("run") == Path(plan["inputs"]["selection"]["path"]).name
        and selected.get("eligible") is True
        and selected.get("weights") == WEIGHTS
        and selected.get("delta_vs_r18", 0) > 0
        and selected.get("delta_vs_r10", 0) > 0.0001
        and selected.get("improved_folds_vs_r18", 0) >= 4
        and selected.get("roc_auc")
        == selected_metric["roc_auc"]
        == plan["reference_cv_auc"],
        "Audited CV selection/gate/weights changed.",
    )
    require(
        selected_metric["model_details"]["weights"] == WEIGHTS,
        "Selected mixture differs.",
    )
    epoch = statistics.median(epochs)
    require(epoch == 3, "Expected median epoch3.")
    return int(epoch)


def fit_full_neural(api, config, frame, y, test, epoch, path):
    """Use the frozen official implementation once, without any validation set."""
    import cloudpickle
    import numpy as np
    import torch

    settings = api.RealMLPConfig.from_training_config(config)
    require(
        settings.epochs == epoch == 3,
        "The fixed horizon and median epoch must both be3.",
    )
    require(config["device"] in ("cpu", "cuda"), "Choose an explicit device.")
    require(
        config["device"] != "cuda" or torch.cuda.is_available(),
        "CUDA unavailable; no fallback permitted.",
    )
    torch.set_num_threads(config["threads"])
    started = time.perf_counter()
    prep = api.RealMLPPreprocessor(42)
    numbers = prep.fit_transform(frame, y)
    model, detail = api._train(
        numbers,
        y,
        None,
        settings,
        10042,
        "cuda:0" if config["device"] == "cuda" else "cpu",
        config["threads"],
        stop_epoch=epoch,
    )
    del numbers
    require(
        detail["completed_epochs"] == epoch and model.model.progress.max_epochs == 3,
        "Full fit did not complete the fixed horizon.",
    )
    prediction = api._predict(model, prep.transform(test)).astype(np.float64)
    details = {
        "architecture": {
            "library": "pytabkit==1.8.0",
            "name": "RealMLP_TD",
            "n_ens": 16,
            "hidden_dims": [512, 256, 128],
            "activation": "parametric SiLU",
            "numeric_embedding": "PBLD: periodic16, linear7 + original-value skip =8",
            "categorical_embedding_dim": 5,
            "parameters": detail["parameters"],
        },
        "preprocessing": prep.metadata(),
        "refit": detail,
        "refit_rows": len(frame),
        "epochs": epoch,
        "preprocessing_seed": 42,
        "network_seed": 10042,
        "validation_rows": 0,
        "epoch_selection": "No selection in this fit; fixed median of the five audited CV-selected epochs.",
        "elapsed_seconds": time.perf_counter() - started,
    }
    api._move(model, "cpu")
    with path.with_suffix(".pkl").open("wb") as stream:
        cloudpickle.dump(
            {"model": model, "preprocessing": prep, "details": details}, stream
        )
    count = min(1024, len(test))
    reloaded = api.predict_saved_realmlp(path, test.iloc[:count], "cpu")
    error = float(np.max(np.abs(reloaded - prediction[:count])))
    require(
        np.isfinite(prediction).all()
        and ((prediction >= 0) & (prediction <= 1)).all()
        and error <= 1e-6,
        "Full neural probability/reload check failed.",
    )
    return (
        prediction,
        details,
        {
            "rows": count,
            "max_abs_error": error,
            "atol": 1e-6,
            "rtol": 0,
            "checkpoint_sha256": digest(path.with_suffix(".pkl")),
        },
    )


def execute(plan_path, root, worktree):
    started = time.perf_counter()
    root, worktree, plan_path = root.resolve(), worktree.resolve(), plan_path.resolve()
    plan, frozen, loaded = read(plan_path), {}, {}
    require(
        plan["weights"] == WEIGHTS and digest(ENTRYPOINT) == plan["entrypoint_sha256"],
        "Entrypoint/weights changed.",
    )
    for role, spec in plan["inputs"].items():
        directory = (root / spec["path"]).resolve()
        require(directory.is_relative_to(root), "Input escaped workspace.")
        for name, expected in spec["sha256"].items():
            require(Path(name).name == name, "Pinned input must be a basename.")
            path = directory / name
            require(digest(path) == expected, "Input changed: " + str(path))
            frozen[str(path)] = expected
        loaded[role] = read(directory / "metrics.json"), read(directory / "config.json")
        require(
            loaded[role][0]["status"]
            == read(directory / "status.json")["status"]
            == "complete"
            and loaded[role][0]["data_hashes"] == plan["data_hashes"],
            "Incomplete or different-data input.",
        )
    receipts = {}
    for role, spec in plan["proofs"].items():
        path = (root / spec["path"]).resolve()
        require(
            path.is_relative_to(root) and digest(path) == spec["sha256"],
            "Proof changed: " + role,
        )
        frozen[str(path)] = spec["sha256"]
        receipts[role] = read(path)
    epoch = chosen_epoch(
        plan, loaded["neural_cv"][0], loaded["selection"][0], receipts["r22_audit"]
    )
    require(
        receipts["r22_audit"]["core_revision"] == plan["source"]["revision"]
        and receipts["r22_audit"]["plan_sha256"]
        == plan["proofs"]["r22_plan"]["sha256"],
        "CV audit source/plan differs.",
    )
    audited = {
        key.replace("\\", "/"): value
        for key, value in receipts["r22_audit"]["input_sha256"].items()
    }
    for role in ("neural_cv", "selection"):
        spec = plan["inputs"][role]
        for name in ("config.json", "metrics.json"):
            require(
                audited.get(spec["path"] + "/" + name) == spec["sha256"][name],
                "CV source differs from independent audit.",
            )
    require(
        receipts["r20_audit"]["status"] == "passed", "R20 full-data audit did not pass."
    )
    for role in ("offset", "lightgbm"):
        metric, _ = loaded[role]
        require(
            metric["phase"] == "full_data_refit"
            and metric["roc_auc"] is None
            and metric["rows"] == {**plan["rows"], "oof": 0},
            "Reuse requires complete all-label R20 models.",
        )
        proof = next(
            item
            for item in receipts["r20_audit"]["checkpoints"]
            if item["role"] == role
        )
        require(
            proof["model_sha256"] == plan["inputs"][role]["sha256"]["model.joblib"]
            and proof["parameters_unchanged"],
            "Reused model was not audited.",
        )
    helper = ENTRYPOINT.with_name("refit_selected_cv.py")
    require(
        digest(helper) == plan["support_script_sha256"],
        "Source pinning helper changed.",
    )
    spec = importlib.util.spec_from_file_location("_selected_refit_contracts", helper)
    contracts = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(contracts)
    for role in ("offset", "lightgbm"):
        require(
            contracts.parameters_sha256(loaded[role][1])
            == loaded[role][0]["source_parameters_sha256"],
            "Reused R20 parameters changed.",
        )
    contracts.pin_core(worktree, plan["source"])
    import numpy as np
    import pandas as pd
    from fraudlab import realmlp
    from fraudlab.data import load_data, submission, validate_submission
    from fraudlab.experiment import create_run, source_provenance, versions
    from fraudlab.features import transform_features
    from fraudlab.models import predict_saved

    require(
        all(
            importlib.metadata.version(name) == version
            for name, version in plan["dependency_versions"].items()
        ),
        "Pinned dependency versions changed.",
    )

    frozen.update(
        {
            str(plan_path): digest(plan_path),
            str(ENTRYPOINT): digest(ENTRYPOINT),
            str(helper): digest(helper),
        }
    )
    data, output = root / plan["data"], root / plan["output"]
    require(
        data.resolve().is_relative_to(root) and output.resolve().is_relative_to(root),
        "Data/output escaped workspace.",
    )
    for name, expected in plan["data_hashes"].items():
        require(digest(data / name) == expected, "Raw data changed.")
        frozen[str(data / name)] = expected
    raw, test_raw, sample = load_data(data, "id", "is_fraud")
    require(
        len(raw) == plan["rows"]["train"] and len(test_raw) == plan["rows"]["test"],
        "Wrong data sizes.",
    )
    config = loaded["neural_cv"][1]
    require(
        loaded["neural_cv"][0]["config_sha256"]
        == plan["inputs"]["neural_cv"]["sha256"]["config.json"],
        "Neural source config differs from its receipt.",
    )
    require(
        config["seed"] == 42 and config["device"] == "cuda" and config["threads"] == 2,
        "Frozen neural execution settings differ.",
    )
    require(
        not (output / plan["neural_name"]).exists()
        and not (output / plan["name"]).exists(),
        "Output exists; no retry/overwrite permitted.",
    )
    nn_run = create_run(output, plan["neural_name"])
    hybrid_run = create_run(output, plan["name"])
    selection = {
        "source_dev_run": str(root / plan["inputs"]["selection"]["path"]),
        "reference_cv_run": Path(plan["inputs"]["selection"]["path"]).name,
        "development_metrics_sha256": plan["inputs"]["selection"]["sha256"][
            "metrics.json"
        ],
        "development_roc_auc": plan["reference_cv_auc"],
        "development_roc_auc_scope": "Historical exploratory CV of the selected R22 hybrid; no score of these full-data predictions.",
        "independent_holdout": False,
    }
    common = {
        **source_provenance(),
        "phase": "full_data_refit",
        "roc_auc": None,
        "fold_scores": [],
        "fold_std": None,
        "rows": {**plan["rows"], "oof": 0},
        "data_hashes": plan["data_hashes"],
        "selection": selection,
        "versions": versions(),
        "plan_sha256": digest(plan_path),
        "refit_script_sha256": digest(ENTRYPOINT),
        "validation": {
            "method": "fixed_median_epoch_refit_of_audited_exploratory_cv",
            "n_splits": 0,
            "independent_holdout": False,
            "early_stopping_uses_validation": False,
        },
    }
    for run in (nn_run, hybrid_run):
        write(run / "status.json", {"status": "running"})
    try:
        nn_config = {
            **config,
            "name": plan["neural_name"],
            "phase": "full_data_refit",
            "refit": True,
            "epochs": epoch,
            "network_seed": 10042,
            "preprocessing_seed": 42,
            "selection": selection,
            "validation_note": "Fixed median E3 from audited R22 CV; fresh all-label refit, no validation and no new local score.",
        }
        write(nn_run / "config.json", nn_config)
        values, details, reload = fit_full_neural(
            realmlp,
            config,
            raw.drop(columns=["id", "is_fraud"]),
            raw.is_fraud.to_numpy(),
            test_raw.drop(columns="id"),
            epoch,
            nn_run / "model",
        )
        nn_frame = submission(test_raw.id, values, sample, "is_fraud")
        for filename in ("predictions.csv", "submission.csv"):
            nn_frame.to_csv(nn_run / filename, index=False)
        nn_metric = {
            **common,
            "status": "complete",
            "model": "realmlp",
            "features": loaded["neural_cv"][0]["features"],
            "model_details": details,
            "checkpoint_reload": reload,
            "config_sha256": digest(nn_run / "config.json"),
            "submission_sha256": digest(nn_run / "submission.csv"),
        }
        write(nn_run / "metrics.json", nn_metric)
        predictions, reuse = {
            "realmlp": pd.read_csv(nn_run / "predictions.csv").is_fraud.to_numpy()
        }, {}
        for role in ("offset", "lightgbm", "external_gpt2"):
            path = root / plan["inputs"][role]["path"]
            frame = pd.read_csv(path / "predictions.csv")
            validate_submission(frame, sample, "id", "is_fraud")
            predictions[role] = frame.is_fraud.to_numpy()
            if role != "external_gpt2":
                metric, cfg = loaded[role]
                transformed = transform_features(
                    test_raw.iloc[:1024], cfg["feature_set"], ("id", "is_fraud")
                )
                result = predict_saved(
                    role,
                    path / "model",
                    transformed[metric["features"]["names"]],
                    "cpu",
                )
                error = float(np.max(np.abs(result - predictions[role][: len(result)])))
                require(error <= 1e-12, "Reused R20 checkpoint reload differs.")
                reuse[role] = {
                    "path": str(path),
                    "checkpoint_sha256": digest(path / "model.joblib"),
                    "reload_rows": len(result),
                    "max_abs_error": error,
                    "new_fit": False,
                }
        formula = sum(
            weight * predictions[role] for role, weight in zip(ROLES, WEIGHTS)
        )
        final = submission(test_raw.id, formula, sample, "is_fraud")
        for filename in ("predictions.csv", "submission.csv"):
            final.to_csv(hybrid_run / filename, index=False)
        written = pd.read_csv(hybrid_run / "submission.csv")
        validate_submission(written, sample, "id", "is_fraud")
        error = float(np.max(np.abs(written.is_fraud.to_numpy() - formula)))
        require(error <= 1e-15, "Full-test serialized formula differs.")
        for path, expected in frozen.items():
            require(digest(path) == expected, "Input changed during fit: " + path)
        provenance = source_provenance()
        require(
            provenance["git_revision"] == plan["source"]["revision"]
            and provenance["git_dirty"] is False
            and provenance["source_hashes"] == plan["source"]["sha256"],
            "Frozen core changed.",
        )
        cfg = {
            "operation": "full_data_neural_refit_reuse_audited_trees",
            "effective_weights": dict(zip(ROLES, WEIGHTS)),
            "plan_sha256": digest(plan_path),
            "inputs": plan["inputs"],
            "neural_run": str(nn_run),
        }
        write(hybrid_run / "config.json", cfg)
        source_credit = (
            loaded["selection"][0].get("external_components")
            or loaded["selection"][0]["model_details"]["external_components"]
        )
        credited = [
            {key: value for key, value in item.items() if key != "oof_auc"}
            for item in source_credit
        ]
        metric = {
            **common,
            "status": "complete",
            "model": "blend",
            "features": loaded["selection"][0]["features"],
            "effective_weights": dict(zip(ROLES, WEIGHTS)),
            "external_components": credited,
            "model_details": {
                "method": "weighted_probability_average",
                "sources": [
                    (
                        str(nn_run)
                        if role == "realmlp"
                        else str(root / plan["inputs"][role]["path"])
                    )
                    for role in ROLES
                ],
                "weights": WEIGHTS,
            },
            "config_sha256": digest(hybrid_run / "config.json"),
            "submission_sha256": digest(hybrid_run / "submission.csv"),
            "reused_models": reuse,
            "neural_checkpoint_reload": reload,
            "formula_rows_verified": len(final),
            "formula_max_absolute_error": error,
            "elapsed_seconds": time.perf_counter() - started,
            "new_fits": 1,
            "new_submissions": 0,
        }
        write(hybrid_run / "metrics.json", metric)
        for run in (nn_run, hybrid_run):
            write(run / "status.json", {"status": "complete"})
        return metric
    except Exception as error:
        for run in (nn_run, hybrid_run):
            if (run / "metrics.json").exists():
                failed = read(run / "metrics.json")
                failed.update(status="failed", error=str(error))
                write(run / "metrics.json", failed)
            write(run / "status.json", {"status": "failed", "error": str(error)})
        raise


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--plan", type=Path, required=True)
    parser.add_argument("--workspace", type=Path, required=True)
    parser.add_argument("--core-worktree", type=Path, required=True)
    args = parser.parse_args()
    print(json.dumps(execute(args.plan, args.workspace, args.core_worktree), indent=2))


In [ ]:
%%writefile r23-source/evidence/round22-plan.json
{
  "date": "2026-11-17",
  "status": "prospective; no competitive R22 fit or AUC exists",
  "hypothesis": "An independently trained neural family using learned categorical embeddings and periodic numerical bases may complement the joint-GLM residual trees. No improvement is promised.",
  "own_run": "round22-realmlp53",
  "cli_argv": [
    "train",
    "--name",
    "round22-realmlp53",
    "--model",
    "realmlp",
    "--feature-set",
    "raw",
    "--phase",
    "exploratory_full_data_cv",
    "--folds",
    "5",
    "--seed",
    "42",
    "--holdout-fraction",
    "0",
    "--epochs",
    "3",
    "--patience",
    "3",
    "--batch-size",
    "288",
    "--hidden-dims",
    "512",
    "256",
    "128",
    "--dropout",
    ".05",
    "--neural-learning-rate",
    ".073",
    "--weight-decay",
    ".038",
    "--neural-numeric-transform",
    "realmlp",
    "--neural-categorical-transform",
    "embedding",
    "--target-smoothing",
    "auto",
    "--threads",
    "2",
    "--device",
    "cuda",
    "--validation-note",
    "Exploratory 5-fold/seed42 all-label CV. Inner15% selects E<=3; discard all state and refit fresh on full outer train. Outer validation is inference only. No independent holdout."
  ],
  "config": {
    "data": "data/raw",
    "original_data": null,
    "original_sha256": null,
    "original_id_column": "transaction_id",
    "output": "artifacts/runs",
    "name": "round22-realmlp53",
    "phase": "exploratory_full_data_cv",
    "model": "realmlp",
    "feature_set": "raw",
    "id_column": "id",
    "target": "is_fraud",
    "split": "stratified",
    "validation_note": "Exploratory 5-fold/seed42 all-label CV. Inner15% selects E<=3; discard all state and refit fresh on full outer train. Outer validation is inference only. No independent holdout.",
    "group_column": null,
    "time_column": null,
    "folds": 5,
    "seed": 42,
    "holdout_fraction": 0.0,
    "holdout_seed": 2026,
    "threads": 2,
    "fixed_training": false,
    "device": "cuda",
    "logistic_c": 1.0,
    "glm_inner_folds": 5,
    "glm_value_one_hot": false,
    "glm_frequency": false,
    "glm_transductive_frequency": false,
    "glm_pooled_tokens": false,
    "glm_spline_knots": 0,
    "iterations": 2000,
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3.0,
    "num_leaves": 4,
    "min_child_samples": 100,
    "target_smoothing": "auto",
    "epochs": 3,
    "hidden_dims": [
      512,
      256,
      128
    ],
    "dropout": 0.05,
    "neural_learning_rate": 0.073,
    "weight_decay": 0.038,
    "neural_numeric_transform": "realmlp",
    "batch_size": 288,
    "neural_categorical_transform": "embedding",
    "patience": 3
  },
  "execution": {
    "requires_published_commit_and_clean_worktree": true,
    "requires_exact_core_hashes": true,
    "data_and_output_overrides": "Only replace --data/--output with absolute workspace data/raw and artifacts/runs; record actual invocation before fit.",
    "pythonpath": "Frozen worktree/src plus private vendor with official pinned optional packages; never import mutable main/src.",
    "no_competitive_fit_before_public_freeze": true
  },
  "core_source_hashes": {
    "__init__.py": "14479eff95a0233e8bf2d9f359843c36f53f36b92cc95c3e452f75bfb2982dcc",
    "__main__.py": "ad2a700e81de61cd32f5885092d03ff1109a227dc50289df19b8b23f4bdbfd97",
    "bpe.py": "fb71799ca075c657a8344f85d6a7e91653d1f319e24b5e287c17565cea456d4e",
    "data.py": "f22355cc51fa6e6162b24e639923f40a11a23af00afcb65553090176271c8326",
    "experiment.py": "51fab0578a28f05078148ace515191b993682c34f8afeb0d36f5d2c45d49951b",
    "external.py": "05ec4ec64e4a2c5cf1137c949646bed14fe83fb04506b8f2f973c2c7633c5e08",
    "features.py": "184fe327468194f34563b25aee2268348ca854d59e4916f8e3eacdd1b6420424",
    "frequency.py": "de06b4f581f6864066d94b140d67e4ed16f7cef054a30ef20bc408a972279ce6",
    "holdout.py": "e3702891766326710c3b56dd7a7fe69f2f778acea3316665f130aa6955e2f4e3",
    "joints.py": "620851854c56aceceba02d5855fe22525e748973e80ffe0f83040f5e75a271e8",
    "models.py": "761f72365c5d510bc77b217ccd51c3f5b46538263a5c7bf7965fbdccd02d1a77",
    "neural_offset.py": "042eec5ea088477bb37b045ace2f537c6e6e39ff49b3e810dc5e501b4ea8e487",
    "offset.py": "a4ea551e4ddf89aedc87f313eb07425aebc60f7d967986524a873a73565fdc4f",
    "original.py": "4db9d8d54929ba9d9a36fd55b8f8655611a800d22fade59af079e0497ce7b934",
    "pooled.py": "6f15acb31c8a878efc262410a0ddf3f6b733b1a636a7bfa917758d568164601a",
    "realmlp.py": "5bf0ec39dd482c71f82f30f6670082e2a2c70707cd5469ee9a64c486bdf4f278",
    "refit.py": "0768665f10806858d9eeca83160bcb5b819d5417f26b775d3bc8262b988c01e4",
    "scores.py": "8bae07482b56eb488a388c54ecd7ec0cf87a8644d9e98e53d7c1fcc75d2e58a1",
    "splits.py": "41215280332984010f5b616bbf6a675c623b7af3e3f00604eb3238e79c709af9",
    "strings.py": "f3248a9c0e7bbda655e29ecac02886dfa4cd4f34f35007ad90cf99a17efccbb0",
    "tabm.py": "df22ff1a120c8a6fde59052ea7a1b509ff1b7e7acc83d774f670a1daf4a4e667",
    "tokens.py": "8e3abfcbe2e437eee859d120f2a89b0a45bd84dfe389a06c11b622ee5ac9cbb4",
    "transductive.py": "b2848df445f94d1135df27b99e7586048c279860b244dcf9544c30687263ca3b"
  },
  "data_hashes": {
    "train.csv": "87eaa10b8d17a7e4f657b8da1c66242431338bdd8b6ad45861521ac0c0e9fbd5",
    "test.csv": "10f516959020357b565abaaa4fd25baae97fc2f536d0805695bb7cc15c1660f0",
    "sample_submission.csv": "13fc6ed8b73de5329884ec490f2a0b750cf29a7cda772b78cfa0cf486f6f35fa"
  },
  "expected_rows": {
    "train": 699960,
    "test": 299984
  },
  "representation": {
    "total_views": 53,
    "numeric_views": 20,
    "categorical_views": 33,
    "raw_numeric": [
      "amount_usd",
      "hours_since_last_txn",
      "txn_count_last_24h",
      "distance_from_home_km",
      "card_age_months",
      "customer_age",
      "account_balance_usd",
      "cvv_retry_count",
      "velocity_score",
      "time_of_day_hour",
      "day_of_week",
      "merchant_risk_score",
      "prior_disputes"
    ],
    "raw_categorical": [
      "merchant_category",
      "card_type",
      "auth_method",
      "channel",
      "device_type",
      "is_foreign_transaction",
      "is_new_merchant",
      "used_vpn",
      "ip_country_mismatch",
      "billing_shipping_mismatch",
      "is_ai_generated_scam_attempt"
    ],
    "missing": "Train-only median imputation with keep_empty_features=True; categorical missing string __MISSING__. Inf is treated as missing.",
    "extra_numeric": "Four (x%1).round(2).float32 decimals plus three cross-fitted TE views.",
    "decimal_columns": [
      "amount_usd",
      "hours_since_last_txn",
      "velocity_score",
      "distance_from_home_km"
    ],
    "extra_categorical": "13 floor(x) integer strings; 4 rint(decimal*100) strings; 3 train-only quantile-bin codes; 2 fixed tuple-repr combinations. Integer strings share repeated objects without changing values.",
    "quantile_bins": [
      [
        "merchant_risk_score",
        7
      ],
      [
        "hours_since_last_txn",
        500
      ],
      [
        "hours_since_last_txn",
        1000
      ]
    ],
    "quantile_method": "linear",
    "subsample": null,
    "combinations": [
      [
        "merchant_category",
        "card_type",
        "txn_count_last_24h"
      ],
      [
        "merchant_category",
        "customer_age"
      ]
    ],
    "target_encoding": {
      "columns": [
        "combo_0",
        "combo_1",
        "hours_since_last_txn"
      ],
      "implementation": "sklearn TargetEncoder",
      "smooth": "auto",
      "cv": 5,
      "shuffle": true,
      "random_state": "42+outer_fold",
      "training": "fit_transform once, cross-fitted; validation/test only transform"
    },
    "unknown_category": "Train-fitted official OrdinalEncoder unknown=-1, then +1 => reserved0; outer/inner validation never contributes vocabulary or statistics."
  },
  "resolved_parameters": {
    "hidden_sizes": [
      512,
      256,
      128
    ],
    "max_one_hot_cat_size": 16,
    "embedding_size": 5,
    "weight_param": "ntk",
    "bias_lr_factor": 0.1,
    "act": "silu",
    "use_parametric_act": true,
    "act_lr_factor": 0.1,
    "block_str": "w-b-a-d",
    "p_drop": 0.05,
    "p_drop_sched": "pow5",
    "add_front_scale": false,
    "scale_lr_factor": 6.0,
    "bias_init_mode": "neg-uniform-dynamic-2",
    "weight_init_mode": "std",
    "wd": 0.038,
    "wd_sched": "cos_log_15",
    "bias_wd_factor": 0.0,
    "use_ls": true,
    "ls_eps": 0.01,
    "num_emb_type": "pbld",
    "plr_sigma": 0.75,
    "plr_hidden_1": 16,
    "plr_hidden_2": 8,
    "plr_lr_factor": 0.1151,
    "lr": 0.073,
    "tfms": [
      "one_hot",
      "median_center",
      "robust_scale",
      "smooth_clip",
      "embedding",
      "l2_normalize"
    ],
    "n_epochs": 3,
    "lr_sched": "flat_anneal",
    "opt": "adam",
    "sq_mom": 0.988,
    "n_ens": 16,
    "batch_size": 288,
    "predict_batch_size": 1024,
    "use_early_stopping": false,
    "ens_av_before_softmax": false,
    "use_best_epoch": true,
    "val_metric_name": "1-auc_ovr",
    "first_layer_lr_factor": 0.25,
    "use_plr_embeddings": true,
    "plr_act_name": "linear",
    "plr_use_densenet": true,
    "plr_use_cos_bias": true,
    "ls_eps_sched": "sqrt_cos",
    "use_ls_prior": false,
    "smooth_clip_max_abs_value": 3.0,
    "robust_scale_eps": 1e-30,
    "l2_normalize_eps": 1e-08
  },
  "architecture": {
    "networks": 16,
    "hidden_dims": [
      512,
      256,
      128
    ],
    "head_logits": 2,
    "activation": "u + alpha*(SiLU(u)-u), learned alpha",
    "numerical_embedding": "PBLD:16 cosine coordinates, learned linear7 plus original-value skip =>8 outputs per numeric input",
    "categorical_embedding_dim": 5,
    "small_categories": "one-hot for support<=16; support includes unknown0",
    "loss": "sum over16 members of batch-mean smoothed cross entropy; uniform label smoothing .01 sqrt_cos",
    "prediction": "mean of16 member softmax probabilities; then mean of5 outer-fold probabilities",
    "parameter_count": "Measure sum(numel) separately for each actual inner/refit model. Cardinalities and internal feature widths may vary by stage/fold; the smoke count is not a competitive model count."
  },
  "protocol": {
    "outer": "StratifiedKFold(5,shuffle=True,random_state=42), all699960 labels; no independent holdout",
    "inner_fraction": 0.15,
    "inner_split_seed": 42,
    "inner_max_epochs": 3,
    "inner_metric": "ROC AUC",
    "inner_early_termination": false,
    "choose_epoch": "Official best mean ensemble validation epoch among1..3; ties choose latest. Inner validation only.",
    "fresh_refit": true,
    "discard": "Discard inner preprocessor, encoder, bins, model and optimizer; refit all train statistics, vocabulary, network and Adam from scratch.",
    "fit_seed": "42+outer_fold",
    "refit_seed": "10042+outer_fold",
    "refit_stop": "StopAtEpochsCallback at chosen E; no validation set. LR/WD/dropout/label-smoothing schedules retain the original3-epoch horizon.",
    "outer_validation_labels": "Never passed to the model, preprocessing, callbacks or epoch selection. Only calculate final OOF AUC after the frozen fits.",
    "telemetry": "Record actual per-stage elapsed time/numel; synchronize and reset CUDA peaks immediately before official model construction, record allocated/reserved after fit. These are not process RSS.",
    "persistence": "Cloudpickle3.1.2 local trusted CPU bundle.pkl; move registered modules and creator.static_model toCPU. Save fitted preprocessing, resolved parameters, chosen E and telemetry."
  },
  "sources": [
    {
      "run": "round18-joint-bpe-pooled-offset",
      "config_sha256": "0780db71a21e97d5fbd8cbc338403ae7cf31506f962e525f807df6a4a79565e6",
      "metrics_sha256": "4523e4220096af21367ca76f1de6157d7d1a5a6d3052f1319f7b13b0f49c9b55",
      "oof_sha256": "4c268f8a229c644904e765660532669f9e3729ffb1d030783b3c8fefe3a5b767",
      "predictions_sha256": "51b34e8ff57a9e41633de7f5ba93ef841666409048ea98b8b5fe3da49f3e0bde"
    },
    {
      "run": "explore5-lgbm",
      "config_sha256": "50968cd9ae1c1734ae439657839a24658480341e5dbd7aeb9a78a8e4dbb7230a",
      "metrics_sha256": "cbbec3ffbcc1461e2f34835955808e3de7ef9306336ea93bf21f3453a0950e4d",
      "oof_sha256": "2a188175419c819bf5d5098125423afa8e95cd79cd965d2108b834cc4016a234",
      "predictions_sha256": "54c9fc1229d1b8aa253520b79c4493b0ae97902ee1f76b73baa9b09d48ccd142"
    },
    {
      "run": "explore5-public-gpt2-v2",
      "config_sha256": "843e20a6839bc22dc5522a4f12c7d46f64472f20782bad6a606c295dfe741d31",
      "metrics_sha256": "14f9e508e5f78c926cd5a4341259623f490872da3e73d5b96f1ae8903fe29bc7",
      "oof_sha256": "4c37722b34d520a6de7d94565140cd0681baad6fb6c012d0b7a86a25b82a4dcb",
      "predictions_sha256": "78ada898faa132accf86daa3311ba114e44a472d31f0648357d10000ab243d58"
    }
  ],
  "reference": {
    "R18": {
      "run": "round18-pooled-share100-public25",
      "config_sha256": "929e7de4e6b6c76fe397672cea0d6187df4db6e9a88b3770af93a3f3bd59a90e",
      "metrics_sha256": "a027c62b262accdf9bf23f0d2bf29baf1465ddade0f1202472f1163f25799563",
      "oof_sha256": "0eda1f90dcfa9f829d4d042da64d1fbe8d3e1c95fd583bc942e0b166133e11b8",
      "predictions_sha256": "18c92eab3d28dcefed12c5a1eebdd9380f45e13972875bf5b31cd4a0e6151e53",
      "roc_auc": 0.9516524825255244,
      "fold_scores": [
        0.9506790019581457,
        0.9520317507242809,
        0.9566745172306466,
        0.9518625895354618,
        0.9470964306274507
      ]
    },
    "R10": {
      "run": "round10-offset-bpe-frequency-public25",
      "config_sha256": "3777c2c58519005942badff767be6d5878bbd3f730a38141bd7d7173c54a3875",
      "metrics_sha256": "66150258799c3ac2e5066ce29383afc02962c2dd35766126e7a995d9fcf57d57",
      "oof_sha256": "038bc4bd9c03f777e97fa73738f261bd9881d0267e33502525697526fd577fea",
      "predictions_sha256": "274cbe4ea90d9dc4893140cad55682f3627ced02163fff48655df4b095e428e7",
      "roc_auc": 0.9514921382152453
    }
  },
  "fixed_mixtures": [
    {
      "name": "round22-realmlp-offset010-public25",
      "replaced_fraction_of_offset": 0.1,
      "sources": [
        "round18-joint-bpe-pooled-offset",
        "round22-realmlp53",
        "explore5-lgbm",
        "explore5-public-gpt2-v2"
      ],
      "weights": [
        0.50625,
        0.05625,
        0.1875,
        0.25
      ],
      "space": "probability"
    },
    {
      "name": "round22-realmlp-offset025-public25",
      "replaced_fraction_of_offset": 0.25,
      "sources": [
        "round18-joint-bpe-pooled-offset",
        "round22-realmlp53",
        "explore5-lgbm",
        "explore5-public-gpt2-v2"
      ],
      "weights": [
        0.421875,
        0.140625,
        0.1875,
        0.25
      ],
      "space": "probability"
    },
    {
      "name": "round22-realmlp-offset050-public25",
      "replaced_fraction_of_offset": 0.5,
      "sources": [
        "round18-joint-bpe-pooled-offset",
        "round22-realmlp53",
        "explore5-lgbm",
        "explore5-public-gpt2-v2"
      ],
      "weights": [
        0.28125,
        0.28125,
        0.1875,
        0.25
      ],
      "space": "probability"
    },
    {
      "name": "round22-realmlp-offset100-public25",
      "replaced_fraction_of_offset": 1.0,
      "sources": [
        "round18-joint-bpe-pooled-offset",
        "round22-realmlp53",
        "explore5-lgbm",
        "explore5-public-gpt2-v2"
      ],
      "weights": [
        0.0,
        0.5625,
        0.1875,
        0.25
      ],
      "space": "probability"
    }
  ],
  "selection": {
    "all_required": [
      "All provenance/data/checkpoint/CSV verifications pass",
      "Pooled mixture OOF strictly greater than R18",
      "Pooled mixture OOF minus R10 strictly greater than0.0001",
      "At least4/5 fold AUCs strictly better than R18"
    ],
    "choose": "Highest pooled OOF among eligible predeclared mixtures; exact ties use the smaller replacement fraction.",
    "public_evaluation": "At most one submission, only by root after independent audit and gate. No calibration/weight/architecture changes from public feedback. No retry or extra search in this round."
  },
  "verification": {
    "raw_files": "Rehash all3 raw CSVs before/after; exact match to every source",
    "alignment": "699960 unique OOF IDs; exact id/fold/label equality against every source and recreated5/42 split;299984 test IDs in sample order; all probabilities finite[0,1]",
    "scores": "Recompute pooled/per-fold AUC from saved OOF CSV; compare metadata",
    "reload": "Each of5 own bundles predicts1024 of its own outer-validation IDs without refitting; compare to OOF with atol1e-6,rtol0 (CUDAfloat32 vsCPUfloat32). Hash every bundle.",
    "mixtures": "Recompute all four declared probability formulas for all OOF/test rows, atol1e-15,rtol0; validate_submission and submission CSV SHA",
    "full_data_refit": "Undefined and refused; no new full-data neural refit in this round."
  },
  "synthetic_smoke": {
    "kind": "synthetic capacity/throughput/reload smoke; not competitive validation",
    "rows_resident": 559968,
    "numeric_features": 20,
    "categorical_features": 33,
    "optimizer_batches": 100,
    "batch_size": 288,
    "n_ens": 16,
    "parameters": 6123552,
    "preprocessing_seconds": 4.420939399977215,
    "fit_seconds": 7.649439199944027,
    "cuda_max_memory_allocated_bytes": 1578438656,
    "cuda_max_memory_reserved_bytes": 2485125120,
    "gpu": "NVIDIA GeForce RTX 4070 Laptop GPU",
    "reload_rows": 1024,
    "cuda_to_cpu_max_abs_error": 1.6763806343078613e-08,
    "fresh_process_cuda_disabled_reload": {
      "cuda_available": false,
      "rows": 1024,
      "max_abs_error": 0.0,
      "kind": "GPU-trained CPU-saved bundle loaded in a fresh CUDA-disabled process"
    },
    "report_sha256": "ede2101b81aac3db99a9dbbbe2fb88acfec67c0a65804962d83bfc45cf3e6809",
    "scope": "100 batches only, not a complete epoch; no competitive rows or labels used for training. Only X cardinalities sized the independent synthetic table. This is not a wall-clock forecast for CV."
  },
  "primary_attribution": [
    {
      "title": "RealMLP/PyTabKit official implementation",
      "url": "https://github.com/dholzmueller/pytabkit",
      "license": "Apache-2.0",
      "version": "1.8.0",
      "wheel_sha256": "11637959137946de1447e18192c81aae903262b25c3ed49524eaf61d1ef98e9e"
    },
    {
      "title": "Better by Default (NeurIPS2024)",
      "url": "https://arxiv.org/abs/2407.04491"
    },
    {
      "title": "yekenot RealMLP PyTabKit",
      "url": "https://www.kaggle.com/code/yekenot/ps-s6-e11-realmlp-pytabkit",
      "version": 15,
      "source_sha256": "b19284e15e63e9d3af3bcbcfb4692cc11e0bf5df42a18f199a4839bd792981a4",
      "adaptation": "Same fixed family/feature definitions, with train-only imputation/bins/vocabulary and inner selection/fresh outer refit instead of outer validation selection. PBLD effective linear projection is explicit, not the overridden GELU field."
    },
    {
      "title": "DavutKursun ablation (author-reported, not reproduced)",
      "url": "https://www.kaggle.com/code/davutkursun/s6e11-what-worked-what-didn-t-lb-0-9526",
      "version": 18,
      "reported": "RealMLP~0.95000 alone; adding it to author-owned trees0.95066→0.95092 nested CV and0.95138→0.95178 public. This supports complementarity as a hypothesis, not a promised gain in our stricter protocol."
    }
  ],
  "core_revision": "11b139f77206101d3f5beeb5648fb459f4cd304a"
}


In [ ]:
%%writefile r23-source/evidence/round23-plan.json
{
  "date": "2026-11-18",
  "status": "prospective; no competitive R23 fit or inference exists",
  "operation": "full_data_neural_refit_reuse_audited_trees",
  "name": "round23-refit-realmlp-offset025-public25",
  "neural_name": "round23-refit-realmlp",
  "entrypoint": "scripts/refit_neural_cv.py",
  "entrypoint_sha256": "a25e9d6971f07a459e7de0532a64584f8e394d81e678c5b9ed641b166ac631e1",
  "support_script_sha256": "b58c0b9919a6aa17874170c3276cc528c26a86639c50012cb0a35aa26e3bc754",
  "source": {
    "revision": "11b139f77206101d3f5beeb5648fb459f4cd304a",
    "sha256": {
      "__init__.py": "14479eff95a0233e8bf2d9f359843c36f53f36b92cc95c3e452f75bfb2982dcc",
      "__main__.py": "ad2a700e81de61cd32f5885092d03ff1109a227dc50289df19b8b23f4bdbfd97",
      "bpe.py": "fb71799ca075c657a8344f85d6a7e91653d1f319e24b5e287c17565cea456d4e",
      "data.py": "f22355cc51fa6e6162b24e639923f40a11a23af00afcb65553090176271c8326",
      "experiment.py": "51fab0578a28f05078148ace515191b993682c34f8afeb0d36f5d2c45d49951b",
      "external.py": "05ec4ec64e4a2c5cf1137c949646bed14fe83fb04506b8f2f973c2c7633c5e08",
      "features.py": "184fe327468194f34563b25aee2268348ca854d59e4916f8e3eacdd1b6420424",
      "frequency.py": "de06b4f581f6864066d94b140d67e4ed16f7cef054a30ef20bc408a972279ce6",
      "holdout.py": "e3702891766326710c3b56dd7a7fe69f2f778acea3316665f130aa6955e2f4e3",
      "joints.py": "620851854c56aceceba02d5855fe22525e748973e80ffe0f83040f5e75a271e8",
      "models.py": "761f72365c5d510bc77b217ccd51c3f5b46538263a5c7bf7965fbdccd02d1a77",
      "neural_offset.py": "042eec5ea088477bb37b045ace2f537c6e6e39ff49b3e810dc5e501b4ea8e487",
      "offset.py": "a4ea551e4ddf89aedc87f313eb07425aebc60f7d967986524a873a73565fdc4f",
      "original.py": "4db9d8d54929ba9d9a36fd55b8f8655611a800d22fade59af079e0497ce7b934",
      "pooled.py": "6f15acb31c8a878efc262410a0ddf3f6b733b1a636a7bfa917758d568164601a",
      "realmlp.py": "5bf0ec39dd482c71f82f30f6670082e2a2c70707cd5469ee9a64c486bdf4f278",
      "refit.py": "0768665f10806858d9eeca83160bcb5b819d5417f26b775d3bc8262b988c01e4",
      "scores.py": "8bae07482b56eb488a388c54ecd7ec0cf87a8644d9e98e53d7c1fcc75d2e58a1",
      "splits.py": "41215280332984010f5b616bbf6a675c623b7af3e3f00604eb3238e79c709af9",
      "strings.py": "f3248a9c0e7bbda655e29ecac02886dfa4cd4f34f35007ad90cf99a17efccbb0",
      "tabm.py": "df22ff1a120c8a6fde59052ea7a1b509ff1b7e7acc83d774f670a1daf4a4e667",
      "tokens.py": "8e3abfcbe2e437eee859d120f2a89b0a45bd84dfe389a06c11b622ee5ac9cbb4",
      "transductive.py": "b2848df445f94d1135df27b99e7586048c279860b244dcf9544c30687263ca3b"
    }
  },
  "data": "data/raw",
  "output": "artifacts/runs",
  "rows": {
    "train": 699960,
    "test": 299984
  },
  "data_hashes": {
    "train.csv": "87eaa10b8d17a7e4f657b8da1c66242431338bdd8b6ad45861521ac0c0e9fbd5",
    "test.csv": "10f516959020357b565abaaa4fd25baae97fc2f536d0805695bb7cc15c1660f0",
    "sample_submission.csv": "13fc6ed8b73de5329884ec490f2a0b750cf29a7cda772b78cfa0cf486f6f35fa"
  },
  "reference_cv_auc": 0.9517074903582017,
  "reference_cv_scope": "Historical exploratory R22 hybrid CV5/42, all699960labels/noindependentholdout. Not an AUC of the new full-data NN or hybrid.",
  "cv_epochs": [
    3,
    2,
    3,
    3,
    3
  ],
  "epochs": 3,
  "scheduling_horizon": 3,
  "preprocessing_seed": 42,
  "network_seed": 10042,
  "weights": [
    0.421875,
    0.140625,
    0.1875,
    0.25
  ],
  "roles": [
    "offset",
    "realmlp",
    "lightgbm",
    "external_gpt2"
  ],
  "inputs": {
    "neural_cv": {
      "path": "artifacts/runs/round22-realmlp53",
      "sha256": {
        "config.json": "3b681f2dd2fd46244e5f639994f0f4b0f9de5878a1653f2ced7fbefd463e8872",
        "metrics.json": "62e330bf87793a903e1c7b10d6ccd172263f48833509166702030002309b0152",
        "status.json": "970587076af8c4ebf6e94dc3999b017a4c3792faf7660c408eb91572fd86128b",
        "predictions.csv": "d501cf906fc2b44ce5b6f005af616d1b1aa1f1fff7f7a1c4936ce23d75c99fac",
        "submission.csv": "d501cf906fc2b44ce5b6f005af616d1b1aa1f1fff7f7a1c4936ce23d75c99fac",
        "oof.csv": "44038fe9d9f9833380de3f0dabeb8ab0d813abbce3ec8cb04dd249c08c99e226",
        "fold-0.pkl": "44e4be3d981b0661c8ec6c09fceb75d6ff64dd0ecb4e98dc0819dd36c82782b9",
        "fold-1.pkl": "6bc3ca09ff62f6deef092562d3f444d14a457d6d13fd797a2f307d6887d38475",
        "fold-2.pkl": "19fbf5251a45e6d38ad6cdeead95c4e5cae49c85fffe32ee281431071cb2a2a0",
        "fold-3.pkl": "eb1a8e704f9dfc20a1be90ff7786ef69d937243983d9382c3f3d6a66dcd44c58",
        "fold-4.pkl": "6aa9ca7c6f19f6ac7d302dd471b6916bed6ea5cae43431c3449a6f79af7fa2be"
      }
    },
    "selection": {
      "path": "artifacts/runs/round22-realmlp-offset025-public25",
      "sha256": {
        "config.json": "ccb022a90158df1b545829437c4997ca4c544d0cebd432b4fef1950cca6d69b7",
        "metrics.json": "a14d3e1fa6b3b8924de1271c0de8297888339293e4967dd3d3c48c1b52ca4b21",
        "status.json": "970587076af8c4ebf6e94dc3999b017a4c3792faf7660c408eb91572fd86128b",
        "predictions.csv": "f15150a2ef40abdfa316d77f6cdd1f701ef2a75529ea44c5fbe3aec321ef8dfa",
        "submission.csv": "f15150a2ef40abdfa316d77f6cdd1f701ef2a75529ea44c5fbe3aec321ef8dfa",
        "oof.csv": "7949cb49208efd3b4879e3ff8e22425f588bd96c6d22e7c1a12ea705739b135d"
      }
    },
    "offset": {
      "path": "artifacts/runs/round20-refit-joint-offset",
      "sha256": {
        "config.json": "030bbde9757a78953d976987d9183a7daa5c830272941027626ef83066432538",
        "metrics.json": "5bdf77bfb811a184c294ff7e7e4ae010082d6772fd37b07e9e1288543f62990d",
        "status.json": "970587076af8c4ebf6e94dc3999b017a4c3792faf7660c408eb91572fd86128b",
        "predictions.csv": "67a497a92fc9a3ba985e2f0b36c0a7cb0b782925b8e605c9e2090bfb1f525150",
        "submission.csv": "67a497a92fc9a3ba985e2f0b36c0a7cb0b782925b8e605c9e2090bfb1f525150",
        "model.joblib": "43da6386631e966647d2600a77e1c811dc3dc984925525ab1f7e7e8a7dbef17a"
      }
    },
    "lightgbm": {
      "path": "artifacts/runs/round20-refit-lgbm",
      "sha256": {
        "config.json": "98f7065463f113024e8127bb0357bf91598aea495ba2c01ca78c84fb9c3edb34",
        "metrics.json": "6242e56c04d51242d90b499502dab744bd32762f38fc2896061544ee4e064649",
        "status.json": "970587076af8c4ebf6e94dc3999b017a4c3792faf7660c408eb91572fd86128b",
        "predictions.csv": "5b31b6061c48d5e875acb48dccef7d6c07e2cc0545c9373202c49be676e2ca35",
        "submission.csv": "5b31b6061c48d5e875acb48dccef7d6c07e2cc0545c9373202c49be676e2ca35",
        "model.joblib": "93566d24f2e38273f02cdf5f29b55820f3f9137500601339e81392436138f315"
      }
    },
    "external_gpt2": {
      "path": "artifacts/runs/explore5-public-gpt2-v2",
      "sha256": {
        "config.json": "843e20a6839bc22dc5522a4f12c7d46f64472f20782bad6a606c295dfe741d31",
        "metrics.json": "14f9e508e5f78c926cd5a4341259623f490872da3e73d5b96f1ae8903fe29bc7",
        "status.json": "970587076af8c4ebf6e94dc3999b017a4c3792faf7660c408eb91572fd86128b",
        "predictions.csv": "78ada898faa132accf86daa3311ba114e44a472d31f0648357d10000ab243d58",
        "submission.csv": "78ada898faa132accf86daa3311ba114e44a472d31f0648357d10000ab243d58",
        "oof.csv": "4c37722b34d520a6de7d94565140cd0681baad6fb6c012d0b7a86a25b82a4dcb"
      }
    }
  },
  "proofs": {
    "r22_audit": {
      "path": ".local/research/realmlp/independent-audit.json",
      "sha256": "25124a10b21661a6ba8e815765f7903783210cf437ecec92b75157a1d563b34a"
    },
    "r22_plan": {
      "path": "evidence/round22-plan.json",
      "sha256": "8d8d8f9c7f0d5a814c4b23b0d725ef63d8a9dde863d0e5e6294146af89623f6b"
    },
    "r22_results": {
      "path": ".local/research/realmlp/results.json",
      "sha256": "8a71faf6d50df135f5c176d79e1fc6f731c08b742cf672de1f017d347034b047"
    },
    "r20_audit": {
      "path": ".local/research/round20/independent-audit.json",
      "sha256": "c91c29b6851f40d98fb4d2f878faaa69a5cb2ad28b24b579eadcb8f3ad7c7b7d"
    },
    "r20_public": {
      "path": "evidence/round20-public.json",
      "sha256": "ba59f4bad3a6707d115caf02787ae344c111a32bcf2666dc27c459c623c2b919"
    },
    "r20_results": {
      "path": "evidence/round20-results.json",
      "sha256": "71bda8d422bfe92fe18f1b9f9c0b17a863c4c88be608e0f82eada182dbc9afae"
    }
  },
  "dependency_versions": {
    "numpy": "2.2.6",
    "pandas": "2.3.3",
    "scikit-learn": "1.7.2",
    "torch": "2.11.0+cu128",
    "pytabkit": "1.8.0",
    "pytorch-lightning": "2.6.6",
    "torchmetrics": "1.9.0",
    "lightning-utilities": "0.15.3",
    "cloudpickle": "3.1.2",
    "tiktoken": "0.14.0",
    "regex": "2026.9.29",
    "lightgbm": "4.7.0"
  },
  "neural_source_parameters": {
    "hidden_sizes": [
      512,
      256,
      128
    ],
    "max_one_hot_cat_size": 16,
    "embedding_size": 5,
    "weight_param": "ntk",
    "bias_lr_factor": 0.1,
    "act": "silu",
    "use_parametric_act": true,
    "act_lr_factor": 0.1,
    "block_str": "w-b-a-d",
    "p_drop": 0.05,
    "p_drop_sched": "pow5",
    "add_front_scale": false,
    "scale_lr_factor": 6.0,
    "bias_init_mode": "neg-uniform-dynamic-2",
    "weight_init_mode": "std",
    "wd": 0.038,
    "wd_sched": "cos_log_15",
    "bias_wd_factor": 0.0,
    "use_ls": true,
    "ls_eps": 0.01,
    "num_emb_type": "pbld",
    "plr_sigma": 0.75,
    "plr_hidden_1": 16,
    "plr_hidden_2": 8,
    "plr_lr_factor": 0.1151,
    "lr": 0.073,
    "tfms": [
      "one_hot",
      "median_center",
      "robust_scale",
      "smooth_clip",
      "embedding",
      "l2_normalize"
    ],
    "n_epochs": 3,
    "lr_sched": "flat_anneal",
    "opt": "adam",
    "sq_mom": 0.988,
    "n_ens": 16,
    "batch_size": 288,
    "predict_batch_size": 1024,
    "use_early_stopping": false,
    "ens_av_before_softmax": false,
    "use_best_epoch": true,
    "val_metric_name": "1-auc_ovr",
    "first_layer_lr_factor": 0.25,
    "use_plr_embeddings": true,
    "plr_act_name": "linear",
    "plr_use_densenet": true,
    "plr_use_cos_bias": true,
    "ls_eps_sched": "sqrt_cos",
    "use_ls_prior": false,
    "smooth_clip_max_abs_value": 3.0,
    "robust_scale_eps": 1e-30,
    "l2_normalize_eps": 1e-08
  },
  "fixed_protocol": {
    "new_competitive_fits": 1,
    "all_labels": 699960,
    "new_validation_rows": 0,
    "new_oof_rows": 0,
    "new_roc_auc": null,
    "epoch_rule": "Median[3,2,3,3,3]=3 from the five auditedR22inner selections; no new epoch selection.",
    "fresh_objects": "New RealMLPPreprocessor(seed42), train-only imputation/bins/vocabulary/5-foldcrossfitTE on all699960labels; new official NNAlgInterface/Adam with networkseed10042; validation=None,stop_epoch3; schedulehorizon3.",
    "representation": "Exactly the same53views20num33cat and architecture/optimizer as11b R22. Full-data category support/parameter count is measured rather than copied from CV or smoke.",
    "reused_models": "Use the actual existing R20FULL offset and LightGBM checkpoint/CSV bytes; no tree refit and no averaging of CVfold predictions.",
    "teacher": "Same approved MasayaGPT2predictiondatasetv2, unchanged25%probabilityshare; no new external arrays.",
    "full_data_mixture": "0.421875*R20offsetFULL +0.140625*newRealMLPFULL +0.1875*R20LGBFULL +0.25*MasayaGPT2v2",
    "seed_scope": "Preprocessing seed42 is the R22fold0fit seed; fresh neural seed10042 follows its outer-refit rule. No automatic extra10000offset.",
    "selection_justification": "Recipe and four weights selected by the prospective R22 CV gate; only the amount of labeled training data changes. R22public feedback is not used to retune parameters/weights."
  },
  "verification": {
    "requires_public_source_and_plan_before_fit": true,
    "source": "Frozen11bworktree clean, exact all-moduleSHA manifest; entrypoint/helper/rawdata/config/metrics/checkpoints/proofs SHA checked before/after. All dependency versions fixed.",
    "reused_models": "AuditedR20 checkpoint hashes and original parameterSHA must agree; reload each on1024actualtestrows and compare to saved full-dataCSVatol1e-12.",
    "neural_reload": "SaveCPUcloudpickle bundle likeR22, reload1024actualtestrows onCPU vs original CUDA inferenceatol1e-6,rtol0.",
    "composition": "Validate all299984IDs/order and finite[0,1] predictions; compare all serialized probabilities to the fixed direct formulaatol1e-15; hashbundle/config/metrics/CSV.",
    "score_contract": "phasefull_data_refit,roc_aucnull,fold_scores[],fold_stdnull,rows.oof0,selection.source_dev_run/development_roc_auc explicitly historicalR22. Do not invent an OOF or holdout score.",
    "immutability": "Refuse existing output directories; failure statuses prevent promotion; no retry/budget/calibration/architecturesearch in this round.",
    "public_evaluation": "At mostone root submission after independent audit; compare public inference to the existing R20 full-data baseline. Script never submits."
  },
  "observed_reference": {
    "R22_own_auc": 0.9501312330938791,
    "R22_selected_hybrid_auc": 0.9517074903582017,
    "R22_improved_folds_vs_R18": 4,
    "R20_public_submission": 58483010,
    "R20_public_auc": 0.95162,
    "scope": "Historical observations and prospective rationale, not a result of R23."
  }
}


In [ ]:
import hashlib
import json
from pathlib import Path
CURRENT_PROTOCOL = {'scope': 'source_export_only; no current-model training or inference in this appendix',
 'core_revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
 'script_revision': 'c95ff468619d299c94764b45e5e15af5ed9f4674',
 'effective_weights': {'offset': 0.421875,
                       'realmlp': 0.140625,
                       'lightgbm': 0.1875,
                       'external_gpt2': 0.25},
 'dependencies': {'numpy': '2.2.6',
                  'pandas': '2.3.3',
                  'scikit-learn': '1.7.2',
                  'torch': '2.11.0+cu128',
                  'pytabkit': '1.8.0',
                  'pytorch-lightning': '2.6.6',
                  'torchmetrics': '1.9.0',
                  'lightning-utilities': '0.15.3',
                  'cloudpickle': '3.1.2',
                  'tiktoken': '0.14.0',
                  'regex': '2026.9.29',
                  'lightgbm': '4.7.0'},
 'source_files': {'r23-source/src/fraudlab/__init__.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                          'path': 'src/fraudlab/__init__.py',
                                                          'sha256': '14479eff95a0233e8bf2d9f359843c36f53f36b92cc95c3e452f75bfb2982dcc'},
                  'r23-source/src/fraudlab/__main__.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                          'path': 'src/fraudlab/__main__.py',
                                                          'sha256': 'ad2a700e81de61cd32f5885092d03ff1109a227dc50289df19b8b23f4bdbfd97'},
                  'r23-source/src/fraudlab/bpe.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                     'path': 'src/fraudlab/bpe.py',
                                                     'sha256': 'fb71799ca075c657a8344f85d6a7e91653d1f319e24b5e287c17565cea456d4e'},
                  'r23-source/src/fraudlab/data.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                      'path': 'src/fraudlab/data.py',
                                                      'sha256': 'f22355cc51fa6e6162b24e639923f40a11a23af00afcb65553090176271c8326'},
                  'r23-source/src/fraudlab/experiment.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                            'path': 'src/fraudlab/experiment.py',
                                                            'sha256': '51fab0578a28f05078148ace515191b993682c34f8afeb0d36f5d2c45d49951b'},
                  'r23-source/src/fraudlab/external.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                          'path': 'src/fraudlab/external.py',
                                                          'sha256': '05ec4ec64e4a2c5cf1137c949646bed14fe83fb04506b8f2f973c2c7633c5e08'},
                  'r23-source/src/fraudlab/features.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                          'path': 'src/fraudlab/features.py',
                                                          'sha256': '184fe327468194f34563b25aee2268348ca854d59e4916f8e3eacdd1b6420424'},
                  'r23-source/src/fraudlab/frequency.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                           'path': 'src/fraudlab/frequency.py',
                                                           'sha256': 'de06b4f581f6864066d94b140d67e4ed16f7cef054a30ef20bc408a972279ce6'},
                  'r23-source/src/fraudlab/holdout.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                         'path': 'src/fraudlab/holdout.py',
                                                         'sha256': 'e3702891766326710c3b56dd7a7fe69f2f778acea3316665f130aa6955e2f4e3'},
                  'r23-source/src/fraudlab/joints.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                        'path': 'src/fraudlab/joints.py',
                                                        'sha256': '620851854c56aceceba02d5855fe22525e748973e80ffe0f83040f5e75a271e8'},
                  'r23-source/src/fraudlab/models.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                        'path': 'src/fraudlab/models.py',
                                                        'sha256': '761f72365c5d510bc77b217ccd51c3f5b46538263a5c7bf7965fbdccd02d1a77'},
                  'r23-source/src/fraudlab/neural_offset.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                               'path': 'src/fraudlab/neural_offset.py',
                                                               'sha256': '042eec5ea088477bb37b045ace2f537c6e6e39ff49b3e810dc5e501b4ea8e487'},
                  'r23-source/src/fraudlab/offset.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                        'path': 'src/fraudlab/offset.py',
                                                        'sha256': 'a4ea551e4ddf89aedc87f313eb07425aebc60f7d967986524a873a73565fdc4f'},
                  'r23-source/src/fraudlab/original.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                          'path': 'src/fraudlab/original.py',
                                                          'sha256': '4db9d8d54929ba9d9a36fd55b8f8655611a800d22fade59af079e0497ce7b934'},
                  'r23-source/src/fraudlab/pooled.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                        'path': 'src/fraudlab/pooled.py',
                                                        'sha256': '6f15acb31c8a878efc262410a0ddf3f6b733b1a636a7bfa917758d568164601a'},
                  'r23-source/src/fraudlab/realmlp.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                         'path': 'src/fraudlab/realmlp.py',
                                                         'sha256': '5bf0ec39dd482c71f82f30f6670082e2a2c70707cd5469ee9a64c486bdf4f278'},
                  'r23-source/src/fraudlab/refit.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                       'path': 'src/fraudlab/refit.py',
                                                       'sha256': '0768665f10806858d9eeca83160bcb5b819d5417f26b775d3bc8262b988c01e4'},
                  'r23-source/src/fraudlab/scores.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                        'path': 'src/fraudlab/scores.py',
                                                        'sha256': '8bae07482b56eb488a388c54ecd7ec0cf87a8644d9e98e53d7c1fcc75d2e58a1'},
                  'r23-source/src/fraudlab/splits.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                        'path': 'src/fraudlab/splits.py',
                                                        'sha256': '41215280332984010f5b616bbf6a675c623b7af3e3f00604eb3238e79c709af9'},
                  'r23-source/src/fraudlab/strings.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                         'path': 'src/fraudlab/strings.py',
                                                         'sha256': 'f3248a9c0e7bbda655e29ecac02886dfa4cd4f34f35007ad90cf99a17efccbb0'},
                  'r23-source/src/fraudlab/tabm.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                      'path': 'src/fraudlab/tabm.py',
                                                      'sha256': 'df22ff1a120c8a6fde59052ea7a1b509ff1b7e7acc83d774f670a1daf4a4e667'},
                  'r23-source/src/fraudlab/tokens.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                        'path': 'src/fraudlab/tokens.py',
                                                        'sha256': '8e3abfcbe2e437eee859d120f2a89b0a45bd84dfe389a06c11b622ee5ac9cbb4'},
                  'r23-source/src/fraudlab/transductive.py': {'revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                                                              'path': 'src/fraudlab/transductive.py',
                                                              'sha256': 'b2848df445f94d1135df27b99e7586048c279860b244dcf9544c30687263ca3b'},
                  'r23-source/scripts/refit_selected_cv.py': {'revision': 'c95ff468619d299c94764b45e5e15af5ed9f4674',
                                                              'path': 'scripts/refit_selected_cv.py',
                                                              'sha256': 'b58c0b9919a6aa17874170c3276cc528c26a86639c50012cb0a35aa26e3bc754'},
                  'r23-source/scripts/refit_neural_cv.py': {'revision': 'c95ff468619d299c94764b45e5e15af5ed9f4674',
                                                            'path': 'scripts/refit_neural_cv.py',
                                                            'sha256': 'a25e9d6971f07a459e7de0532a64584f8e394d81e678c5b9ed641b166ac631e1'},
                  'r23-source/evidence/round22-plan.json': {'revision': 'c95ff468619d299c94764b45e5e15af5ed9f4674',
                                                            'path': 'evidence/round22-plan.json',
                                                            'sha256': '8d8d8f9c7f0d5a814c4b23b0d725ef63d8a9dde863d0e5e6294146af89623f6b'},
                  'r23-source/evidence/round23-plan.json': {'revision': 'c95ff468619d299c94764b45e5e15af5ed9f4674',
                                                            'path': 'evidence/round23-plan.json',
                                                            'sha256': '46ba3e33786e111fff9492904c462a7ec279288889d64bf9dd2d45f4981f987c'}},
 'observed_local_result': {'round': 23,
                           'status': 'complete_selected',
                           'phase': 'full_data_refit',
                           'source_revision': '11b139f77206101d3f5beeb5648fb459f4cd304a',
                           'new_neural_fits': 1,
                           'reused_own_models': 2,
                           'training_rows': 699960,
                           'test_rows': 299984,
                           'neural_ensemble_members': 16,
                           'neural_parameters': 6128752,
                           'selected_epoch': 3,
                           'epoch_selection': 'Median[3,2,3,3,3] from audited R22, frozen before '
                                              'full refit',
                           'local_auc': None,
                           'reference_cv_auc': 0.9517074903582017,
                           'reference_cv_run': 'round22-realmlp-offset025-public25',
                           'reference_cv_scope': 'Historical exploratory source only; no new OOF '
                                                 'or independent holdout',
                           'weights': {'offset': 0.421875,
                                       'realmlp': 0.140625,
                                       'lightgbm': 0.1875,
                                       'external_gpt2': 0.25},
                           'elapsed_seconds': 204.90454979997594,
                           'elapsed_scope': 'Observed full-data neural fit, inference and checks; '
                                            'not a controlled benchmark',
                           'public_result': 'evidence/round23-public.json',
                           'verification': 'evidence/round23-verification.json',
                           'full_metrics_sha256': '000cc843935e6c0fbca0cd16fd6535decc2f8565317bbcbb7854053423ace7a0',
                           'preflight_correction': 'An import-scope error was corrected and '
                                                   'republished before any competitive R23 fit or '
                                                   'output was created.'},
 'processed_public_result': {'ref': 58490653,
                             'run': 'round23-refit-realmlp-offset025-public25',
                             'submission_sha256': '937dc4dd355229bc49fda67c3099ea97867a23fb9c9bbe97a2bf57fae6cf24de',
                             'downloaded_submission_sha256': '937dc4dd355229bc49fda67c3099ea97867a23fb9c9bbe97a2bf57fae6cf24de',
                             'submission_file_verified': True,
                             'public_auc': 0.95183,
                             'ranking': 626,
                             'total_teams': 2135,
                             'observed_at': '2026-11-18T02:12:21.191510+00:00',
                             'source': 'https://www.kaggle.com/competitions/playground-series-s6e11/leaderboard',
                             'plan': 'evidence/round23-plan.json',
                             'integrity_audit': 'evidence/round23-verification.json',
                             'phase': 'full_data_refit',
                             'local_oof_auc': None,
                             'reference_cv_run': 'round22-realmlp-offset025-public25',
                             'reference_cv_auc': 0.9517074903582017,
                             'previous_public_auc': 0.95162,
                             'public_delta': 0.00021,
                             'protocol_deviation': False,
                             'winning_position_reached': False}}
CURRENT_SOURCE_SHA256 = {'r23-source/evidence/round22-plan.json': '8d8d8f9c7f0d5a814c4b23b0d725ef63d8a9dde863d0e5e6294146af89623f6b',
 'r23-source/evidence/round23-plan.json': '46ba3e33786e111fff9492904c462a7ec279288889d64bf9dd2d45f4981f987c',
 'r23-source/scripts/refit_neural_cv.py': 'a25e9d6971f07a459e7de0532a64584f8e394d81e678c5b9ed641b166ac631e1',
 'r23-source/scripts/refit_selected_cv.py': 'b58c0b9919a6aa17874170c3276cc528c26a86639c50012cb0a35aa26e3bc754',
 'r23-source/src/fraudlab/__init__.py': '14479eff95a0233e8bf2d9f359843c36f53f36b92cc95c3e452f75bfb2982dcc',
 'r23-source/src/fraudlab/__main__.py': 'ad2a700e81de61cd32f5885092d03ff1109a227dc50289df19b8b23f4bdbfd97',
 'r23-source/src/fraudlab/bpe.py': 'fb71799ca075c657a8344f85d6a7e91653d1f319e24b5e287c17565cea456d4e',
 'r23-source/src/fraudlab/data.py': 'f22355cc51fa6e6162b24e639923f40a11a23af00afcb65553090176271c8326',
 'r23-source/src/fraudlab/experiment.py': '51fab0578a28f05078148ace515191b993682c34f8afeb0d36f5d2c45d49951b',
 'r23-source/src/fraudlab/external.py': '05ec4ec64e4a2c5cf1137c949646bed14fe83fb04506b8f2f973c2c7633c5e08',
 'r23-source/src/fraudlab/features.py': '184fe327468194f34563b25aee2268348ca854d59e4916f8e3eacdd1b6420424',
 'r23-source/src/fraudlab/frequency.py': 'de06b4f581f6864066d94b140d67e4ed16f7cef054a30ef20bc408a972279ce6',
 'r23-source/src/fraudlab/holdout.py': 'e3702891766326710c3b56dd7a7fe69f2f778acea3316665f130aa6955e2f4e3',
 'r23-source/src/fraudlab/joints.py': '620851854c56aceceba02d5855fe22525e748973e80ffe0f83040f5e75a271e8',
 'r23-source/src/fraudlab/models.py': '761f72365c5d510bc77b217ccd51c3f5b46538263a5c7bf7965fbdccd02d1a77',
 'r23-source/src/fraudlab/neural_offset.py': '042eec5ea088477bb37b045ace2f537c6e6e39ff49b3e810dc5e501b4ea8e487',
 'r23-source/src/fraudlab/offset.py': 'a4ea551e4ddf89aedc87f313eb07425aebc60f7d967986524a873a73565fdc4f',
 'r23-source/src/fraudlab/original.py': '4db9d8d54929ba9d9a36fd55b8f8655611a800d22fade59af079e0497ce7b934',
 'r23-source/src/fraudlab/pooled.py': '6f15acb31c8a878efc262410a0ddf3f6b733b1a636a7bfa917758d568164601a',
 'r23-source/src/fraudlab/realmlp.py': '5bf0ec39dd482c71f82f30f6670082e2a2c70707cd5469ee9a64c486bdf4f278',
 'r23-source/src/fraudlab/refit.py': '0768665f10806858d9eeca83160bcb5b819d5417f26b775d3bc8262b988c01e4',
 'r23-source/src/fraudlab/scores.py': '8bae07482b56eb488a388c54ecd7ec0cf87a8644d9e98e53d7c1fcc75d2e58a1',
 'r23-source/src/fraudlab/splits.py': '41215280332984010f5b616bbf6a675c623b7af3e3f00604eb3238e79c709af9',
 'r23-source/src/fraudlab/strings.py': 'f3248a9c0e7bbda655e29ecac02886dfa4cd4f34f35007ad90cf99a17efccbb0',
 'r23-source/src/fraudlab/tabm.py': 'df22ff1a120c8a6fde59052ea7a1b509ff1b7e7acc83d774f670a1daf4a4e667',
 'r23-source/src/fraudlab/tokens.py': '8e3abfcbe2e437eee859d120f2a89b0a45bd84dfe389a06c11b622ee5ac9cbb4',
 'r23-source/src/fraudlab/transductive.py': 'b2848df445f94d1135df27b99e7586048c279860b244dcf9544c30687263ca3b'}
for filename, expected in CURRENT_SOURCE_SHA256.items():
    assert hashlib.sha256(Path(filename).read_bytes()).hexdigest() == expected, filename
print(json.dumps({'scope': CURRENT_PROTOCOL['scope'], 'files_verified': len(CURRENT_SOURCE_SHA256),
                  'observed_local_result': CURRENT_PROTOCOL['observed_local_result'],
                  'processed_public_result': CURRENT_PROTOCOL['processed_public_result']}, indent=2))
